# Quenched galaxies as starbursts and as aging discs

**The selection is a region of a map, not a threshold on a number.** Part 1b draws one quantity —
$\log M_{\rm dust}/M_\star$ by default — against the clock, one column per epoch, and finds the **two bright
regions** that appear after the clock point. It peels them off in the order the eye reads them: the upper
region becomes group 1, the map is redrawn *with those tracks removed*, the brightest region of what is left
becomes group 2, and the galaxies that never entered either are group 3, the intermediates. The regions are
calibrated on the two big boxes pooled and **the same cells** are peeled off every box, so the 25 Mpc box,
with 82 galaxies, is never asked to find structure it cannot resolve — only whether its galaxies fall in it.

**Everything downstream is those three groups.** Part 2 adopts them (`P2_ADOPT = "peel"`), so the clocks of
Part 3, the observer's question of Part 4, the dust lifetimes of Part 5 and the CGM of Part 6 all describe
the same selection. Group 1 becomes the `burst` key and group 2 the `aging` one; the intermediates are left
out of both, the way a tercile middle is.

**What the selection turns out to be.** Carried onto the other quantities it is a **kinematic** split, not a
dust one: the galaxies in the upper dust region are the ones whose gas is still rotating (AUC 0.87–1.00 in
$\kappa_{\rm rot}$(gas)), in **extended** rather than contracted gas (AUC 0.1–0.3 in
$-\log R_{50,\rm gas}/R_{50,\star}$), and the difference is already there at the truncation itself, before the
dust has separated at all. Part 1c then asks whether it also divides the normalised quenching time — the one
axis in this notebook that is genuinely bimodal — or whether the two are independent.

**Why this notebook exists.** `dusty_qgs_kappa_selection.ipynb` grew to 51 cells and a dependency chain in
which almost nothing runs on its own. This one is deliberately small: **one pass, one division you choose, and the checks on it.**
It reads that notebook's inputs and one of its caches; it writes nothing into its output directory and
changes none of its results.

| Part | The question |
|---|---|
| 0 | configuration: the sample, the clocks, the shared helpers |
| 1 | one pass over the branch chains — the event times, the tracks, the raw series |
| 1b | **THE SELECTION — the peel**: the map of one quantity against the clock, its two bright regions taken off one at a time (calibrated on the big boxes, applied to all of them), and the intermediates left behind as a third group |
| 1c | does the selection divide the **normalised quenching time**, or is it an independent axis? |
| 2 | **the division**: it adopts the selection by default (`P2_ADOPT = "peel"`); a spec axis puts the browser back |
| 3 | the grid of clocks: the two groups through the truncation |
| 4 | **the observer's question**: a quenched galaxy caught at a random snapshot with $f_{\rm dust} > 10^{-4}$ — how likely is it an aging one and not a starburst? And the same for a high $\kappa_{\rm gas}$ |
| 5 | how long the dust lasts: the time to fall *permanently* below $f_{\rm dust} = 10^{-4}$, and whether it is set by the rate or by the head start |
| 6 | the ISM–CGM relation (m25): $T_{\rm CGM}$ on the clock, the inflow/outflow history, the CGM dust share — and $P(\text{cold-rich CGM} \mid \text{dusty QG})$ |
| 10 | **paper figure 1**: the map the selection is cut from, and the same dust as a distribution at the five events of a galaxy's own history and at five redshifts — *where the two modes are, and where they are not* |
| 10b | **paper figure 1b**: the selection alone — the three jet-aligned dust maps with the dusty and dust-poor regions on them, nothing else |
| 10c | **paper figure 1c**: the peel stage by stage in m100n1024 — the whole map and the dusty region, the map with those tracks removed and the dust-poor region, the residual; each column dimmed by the share still in play |
| 11 | **paper figure 2**: the two groups on the truncation clock in six quantities, with the number of galaxies behind every track drawn in the back of its panel |
| 12 | **paper figure 3**: the gas at the onset — $\kappa_{\rm gas}$, $\Sigma_{\rm H_2}$ and the compactness as full tracks on the jet clock with their bootstrap bands, and the critical points (jet, quenching onset, observation) read at the events on them |
| 13 | **paper figure 4**: the ledger of the surviving dust — kept, regrown or brought in; the ex-situ growth after quenching as the merger axis |
| 14 | **paper figure 5**: the timescale ladder — growth, destruction, supply and the observed lifetime on one axis |
| 15 | **paper figure 6**: the timeline — every event of each pathway on the truncation clock |
| 16 | **paper figure 7**: the observer's guidance — one panel per redshift, P(dusty | disky QG) against how dusty "dusty" has to be, for three degrees of "disky" (stellar $\kappa_\star$, as B/T) |
| 17 | **paper figure 8**: the radial profiles — $\Sigma_{\rm dust}$, $\Sigma_{\rm H_2}$, $M_{\rm dust}/M_\star$, D/G, $A_V$ and age per annulus out to 100 kpc, the two groups against the star-forming galaxies |
| 18 | **paper figure 9**: where the dust goes and whether it comes back — the inside/outside shares on the clock, how much is left, the return fraction, the return against the in-situ growth |
| 19 | **paper figure 10**: the Kennicutt–Schmidt plane — the two groups' tracks $t_{q,\rm start} \to t_{q,\rm end} \to$ end against the VERTICO relation, on one IMF |

**The counterfactual is a column, not a footnote.** `cis50nox` (jets on, SIMBA's X-ray heating compiled out)
is drawn beside the three X-ray-on boxes in every figure of Parts 2–5, so anything the X-ray channel is
credited with has to be absent there.

In [ ]:
# ── Part 0 — configuration: the products read, the sample, the clocks, and the helpers every part shares ──
# This notebook is the SIMPLE one: one pass over the branch chains (Part 1), ONE division (Part 2), and four
# checks on it (Parts 3-6).  It reads the products of paper_ism_prediction_boxes and ONE cache of
# dusty_qgs_kappa_selection (Part 13's CGM node table); it writes only into its own output directory.
import os
import warnings
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu, spearmanr, gaussian_kde

IN   = os.path.join(os.getcwd(), "output", "box_resolution", "ism_prediction")   # paper_ism_prediction_boxes' products
KIN  = os.path.join(os.getcwd(), "output", "box_resolution", "agn_states")       # the kinematic sidecar (READ ONLY)
KSEL = os.path.join(os.getcwd(), "output", "box_resolution", "kappa_selection")  # dusty_qgs_kappa_selection's products
OUT  = os.path.join(os.getcwd(), "output", "box_resolution", "burst_aging")      # everything this notebook writes
CACHE_FITS = os.path.join(IN, "ism_prediction_catalogue.fits")                   # its Part 1: every galaxy log M* > 9.5 per (box, anchor)
CLASS_CSV  = os.path.join(IN, "ism_prediction_agn_classes.csv")                  # its Part 6a: t_SFT / t_QT / tau_q per galaxy
HIST_FILE  = os.path.join(IN, "ism_prediction_histories_{box}.hdf5")             # its Part 6a: the main-branch chains
KIN_FILE   = os.path.join(KIN, "agn_states_kin_histories_{box}.hdf5")            # kappa_rot(gas), R50(gas), R50(*), N_gas along the branch
os.makedirs(OUT, exist_ok=True)

BOXES   = ["cis100", "cis50", "cis25"]      # the three X-ray-on resolutions, in this order everywhere
NOX_BOX = "cis50nox"                        # the counterfactual: the same 50 Mpc box with SIMBA's X-ray channel compiled out
BOXES_ALL = BOXES + [NOX_BOX]               # the drawing order of every figure: the control is the LAST column
BOX_LABEL = {"cis100": "m100n1024", "cis50": "m50n512", "cis25": "m25n512",
             "cis50nox": "m50n512 s50nox (X-ray off)"}     # the runs' REAL names, on every figure

# ── THE SAMPLE: the m25 quenched rule of paper_ism_prediction_boxes Part 0, verbatim, plus the mass / redshift / age cuts ──
MASS_FLOOR, PASSIVE_FACTOR, NGAS_MIN, NSTAR_MIN, DUST_TO_H2_MIN = 10.0, 0.2, 50, 500, 1e-15
MASS_MIN   = 10.25       # log M* strictly above this
Z_MAX      = None        # anchors at or below this redshift (None = every anchor of the ladder)
AGE_MIN    = 1.5         # Gyr, the mass-weighted stellar age at the anchor: at or above this
AGE_MAX    = None        # Gyr, an upper bound if wanted (None = none)
DUSTY_CUT  = -4.0        # log M_dust/M*: THE observable threshold of this notebook — the user's f_dust > 1e-4
FLOOR      = -6.5        # zero dust is kept at this log (the end point of the removal, never dropped)
SSFR_FLOOR = 1e-19
HE  = 1.36               # helium correction on the simulated (hydrogen-only) M_H2
KEY = ["box", "snap", "gal_id"]

POPS = {"dusty": dict(color="#d7191c", lw=2.4, label=rf"dusty QG  ($\log M_{{\rm dust}}/M_\star \geq {DUSTY_CUT:g}$)"),
        "pure":  dict(color="#2c7bb6", lw=2.2, label=rf"dust-poor QG  ($< {DUSTY_CUT:g}$)")}

# ── THE TWO GROUPS.  Named here so every part uses the same colours; Part 2 is what assigns them. ──
# "starburst"  = COMPACT H2 when the jet turns on (high Sigma_H2)   -> the post-starburst branch
# "aging"      = DIFFUSE H2 when the jet turns on (low Sigma_H2)    -> the secularly declining branch
GK   = ["burst", "aging"]
GCOL = {"burst": "#d95f02", "aging": "#7570b3"}
GLAB = {"burst": r"starburst  (compact H$_2$ at $t_{\rm jet}$)", "aging": r"aging  (diffuse H$_2$ at $t_{\rm jet}$)"}
GSH  = {"burst": "starburst", "aging": "aging"}
GLAB0, GSH0 = dict(GLAB), dict(GSH)         # the defaults Part 2 restores whenever P2_LABELS is None

# ── kappa_rot(gas) against the isotropic null.  IN THIS NOTEBOOK IT IS AN OBSERVABLE, NEVER A SELECTION. ──
# For an isotropic velocity field E[kappa] = 1/3 and Var(v_phi^2/v^2) = 4/45; the v^2 weighting of the
# denominator gives N_eff = 0.6 N for a Maxwellian, so s_iso(N) = sqrt(4/45 / 0.6 N) = 0.385 / sqrt(N).
# Part 4 asks whether a galaxy caught with a SIGNIFICANTLY rotating gas disc is an aging one — the significance,
# not kappa itself, because a gas-poor galaxy can read "rotating" on a handful of aligned particles.
KAPPA_ISO, KAPPA_VARF, KAPPA_WEFF = 1.0 / 3.0, 4.0 / 45.0, 0.6
KAPPA_C  = float(np.sqrt(KAPPA_VARF / KAPPA_WEFF))      # 0.385
ROT_SIG  = 3.0                                          # "high kappa" in Part 4 = kappa > 1/3 + ROT_SIG s_iso(N)
sig_iso  = lambda n: KAPPA_C / np.sqrt(np.where(np.asarray(n, float) > 0, np.asarray(n, float), np.nan))
kappa_z  = lambda k, n: (np.asarray(k, float) - KAPPA_ISO) / sig_iso(n)

# ── the AGN states: SIMBA's own feedback branches evaluated at every snapshot of the main branch ──
JET_LOGMBH, JET_FEDD = 7.5, 0.2       # SIMBA's jet criterion: M_BH above 10^7.5 AND f_Edd below 0.2
W_FULL        = 0.999                 # "full-velocity jet": w_jet = clip(log10(0.2/f_Edd), 0, 1) at/above this
XRAY_FGAS_MAX = float(os.environ.get("XRAY_FGAS_MAX", "0.2"))   # SIMBA's X-ray condition, M_gas/M_* below this (Dave+19)
ETA_RAD       = 0.1
MDOT_PER_MBH  = 1.26e38 / (ETA_RAD * 8.98755e20) / 1.98892e33 * 3.15576e7   # Msun/yr per Msun of BH at f_Edd = 1

# ── the clock and the estimator settings ──
TG    = np.arange(-3.0, 4.001, 0.25)   # THE CLOCK: t - t_SFT (also t - t_jet, t - t_x1) [Gyr]
NMIN  = 5                              # an epoch is drawn only where this many galaxies contribute
NBOOT = int(os.environ.get("BA_NBOOT", "400"))    # the bootstrap band; drop it for a smoke test
SEED  = 11
PAPER_DPI = 300
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9,
                     "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "axes.titleweight": "bold", "pdf.fonttype": 42})
# THE PAPER FIGURES (Parts 3b, 10, 12, 14) draw inside plt.rc_context(PAPER_RC): larger, no titles, the real
# run names, one histogram per panel and no diagnostic text — every statistic they drop stays in their printout.
PAPER_RC = {"font.size": 18, "axes.labelsize": 21, "xtick.labelsize": 18, "ytick.labelsize": 18,
            "legend.fontsize": 17, "axes.titlesize": 18, "axes.grid": False}     # large, and no grids


def need(path, made_by):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: {made_by}")
    return path


def paper_save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{stem}.{ext}"), dpi=PAPER_DPI, bbox_inches="tight")
    print("figure ->", os.path.join(OUT, f"{stem}.{{png,pdf}}"))


def mw_auc(x, m1):
    """x over two populations (m1 = population 1): AUC = P(x_1 > x_0), the two-sided Mann-Whitney p, the two counts."""
    x = np.asarray(x, float); m1 = np.asarray(m1, bool)
    a, b = x[m1 & np.isfinite(x)], x[(~m1) & np.isfinite(x)]
    if len(a) < 3 or len(b) < 3 or np.nanstd(np.concatenate([a, b])) == 0:
        return np.nan, np.nan, len(a), len(b)
    u = mannwhitneyu(a, b, alternative="two-sided")
    return float(u.statistic) / (len(a) * len(b)), float(u.pvalue), len(a), len(b)


BETA_PRIOR = "flat"       # THE BAND ON EVERY FRACTION IN THIS NOTEBOOK (2026-09-18, replaces the Wilson interval): the
                          # Beta posterior of a binomial fraction — Beta(k + a, n - k + a) with a = 1 ("flat") or 1/2
                          # ("jeffreys") — read at the quantiles Phi(-z), Phi(+z), so z = 1 is the 68 % credible band.
                          # Exact at k = 0 and k = n and inside [0, 1] by construction; the point estimate stays k / n.
                          # It treats the galaxies as independent draws: it is the SAMPLING band of one realisation, not
                          # the cosmic variance of a 100 Mpc box (which needs positions and an octant jackknife).
from scipy.stats import beta as _beta_dist, norm as _norm_dist


def frac_band(k, n, z=1.0, prior=None):
    """(k/n, lo, hi): the observed fraction and its Beta-posterior credible band at +-z (68 % for z = 1).
    Vectorised over k; n a scalar.  NaN for n <= 0."""
    a = {"flat": 1.0, "jeffreys": 0.5}[prior or BETA_PRIOR]
    k = np.asarray(k, float); n = float(n)
    if n <= 0:
        return (np.full(k.shape, np.nan),) * 3 if k.ndim else (np.nan,) * 3
    q = _norm_dist.cdf([-abs(z), abs(z)])
    lo = _beta_dist.ppf(q[0], k + a, n - k + a); hi = _beta_dist.ppf(q[1], k + a, n - k + a)
    p = k / n
    return (p, lo, hi) if k.ndim else (float(p), float(lo), float(hi))


wilson = frac_band        # every earlier cell calls it by this name; the band it returns is the Beta one


# ── the KDE and the multimodality test, shared by Parts 2, 2b and 2c ──────────────────────────────────
# Pure statistics, no knob of this notebook in them: the number of modes of a Gaussian KDE at a given
# bandwidth, Silverman's critical bandwidth (the smallest one that leaves at most k modes), his
# smoothed-bootstrap p for "at most k modes", and the interior minimum between the two highest peaks —
# the candidate place to cut.  KDE_NGRID is the grid every one of them is evaluated on.
KDE_NGRID = 512


def _nmode(x, h, ngrid=None):
    """The number of modes of the Gaussian KDE of x at bandwidth h."""
    gx = np.linspace(x.min() - 3 * h, x.max() + 3 * h, KDE_NGRID if ngrid is None else ngrid)
    d = np.exp(-0.5 * ((gx[:, None] - x[None, :]) / h) ** 2).sum(axis=1)
    return int(((d[1:-1] > d[:-2]) & (d[1:-1] >= d[2:])).sum()), gx, d


def _hcrit(x, k=1, lo=None, hi=None, it=40):
    """Silverman's critical bandwidth: the SMALLEST h whose KDE has at most k modes.  Monotone in h, so bisect."""
    s = np.std(x, ddof=1)
    lo = s * 1e-3 if lo is None else lo
    hi = s * 3.0 if hi is None else hi
    while _nmode(x, hi)[0] > k and hi < s * 100:
        hi *= 2.0
    for _ in range(it):
        m = np.sqrt(lo * hi)
        if _nmode(x, m)[0] > k:
            lo = m
        else:
            hi = m
    return hi


def _silverman(x, k=1, nboot=None, seed=SEED, nmin=20):
    """Silverman's bootstrap p for H0 = 'at most k modes'.  Resamples from the density smoothed at h_crit
    (rescaled to preserve the variance) and counts how often it needs a LARGER critical bandwidth than the data.
    A small p means one mode is not enough."""
    nb = 200 if nboot is None else nboot
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < nmin or nb <= 0:
        return np.nan, np.nan
    h = _hcrit(x, k)
    s, xb, rng = np.std(x, ddof=1), x.mean(), np.random.default_rng(seed)
    cnt = 0
    for _ in range(nb):
        i = rng.integers(0, len(x), len(x))
        y = xb + (x[i] - xb + h * rng.standard_normal(len(x))) / np.sqrt(1.0 + h * h / (s * s))
        cnt += int(_hcrit(y, k) > h)
    return float(cnt) / nb, float(h)


def _dip(x, bw=None):
    """The KDE and the interior minimum between its two highest maxima; (nan, ...) when there is only one mode."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    try:
        kde = gaussian_kde(x, bw_method=bw)
    except Exception:                       # a degenerate sample (every value identical) has no density
        return np.nan, np.array([]), np.array([]), 0
    gx = np.linspace(x.min() - 0.5 * x.std(), x.max() + 0.5 * x.std(), KDE_NGRID)
    d = kde(gx)
    pk = np.flatnonzero((d[1:-1] > d[:-2]) & (d[1:-1] >= d[2:])) + 1
    if len(pk) < 2:
        return np.nan, gx, d, len(pk)
    a, b = sorted(pk[np.argsort(d[pk])[-2:]])
    return float(gx[a + int(np.argmin(d[a:b + 1]))]), gx, d, len(pk)


# ── the confounds every controlled score is computed inside ───────────────────────────────────────────
# The groups of this notebook differ in cosmic time and in stellar mass whatever they are divided on, so
# every AUC is quoted twice: raw, and on the value RANKED INSIDE a cell of these two.  The second one is
# the one to believe.
CTRL_COLS = ["t_anchor", "log_mstar"]
CTRL_Q = 3                       # quantiles per control axis


def _cellrank(D, col, ctrl=None, q=None):
    """The value of `col` replaced by its RANK INSIDE a cell of the control quantiles — the standard way to ask
    a question at fixed epoch and mass.  Cells with fewer than 4 rows return NaN.  Returns values in [0, 1]."""
    ctrl = CTRL_COLS if ctrl is None else ctrl
    q = CTRL_Q if q is None else q
    y = D[col].to_numpy(float)
    lab = np.zeros(len(D), int)
    for c in ctrl:
        if c not in D.columns:
            continue
        v = D[c].to_numpy(float)
        e = np.nanpercentile(v[np.isfinite(v)], np.linspace(0, 100, q + 1)[1:-1]) if np.isfinite(v).sum() > q else []
        lab = lab * (q + 1) + np.digitize(v, e)
    out = np.full(len(D), np.nan)
    for u in np.unique(lab):
        m = (lab == u) & np.isfinite(y)
        if m.sum() >= 4:
            out[m] = (np.argsort(np.argsort(y[m])) + 0.5) / m.sum()
    return out


def _auc_ctrl(D, col, m1, ctrl=None):
    """AUC of `col` toward m1, computed on the within-cell rank: the same statistic at fixed epoch and mass."""
    return mw_auc(_cellrank(D, col, ctrl), np.asarray(m1, bool))


def med_band(A, nmin=None, seed=SEED, nboot=None, grid=None):
    """A (n_gal, n_t) stack -> the median track, its bootstrap 16-84 band and the per-epoch finite count."""
    A = np.asarray(A, float)
    g = TG if grid is None else grid
    nmin = NMIN if nmin is None else nmin
    nb = NBOOT if nboot is None else nboot
    if A.ndim != 2 or not len(A):
        return (np.full(len(g), np.nan),) * 3 + (np.zeros(len(g), int),)
    n_t = np.isfinite(A).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        cent = np.nanmedian(A, axis=0)
        rng = np.random.default_rng(seed)
        bs = np.array([np.nanmedian(A[rng.integers(0, len(A), len(A))], axis=0) for _ in range(nb)])
        lo, hi = np.nanpercentile(bs, 16, axis=0), np.nanpercentile(bs, 84, axis=0)
    ok = n_t >= nmin
    return np.where(ok, cent, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan), n_t


def to_grid(trel, y, grid=None):
    """One galaxy's track interpolated onto the clock; nothing is extrapolated past the sampled range."""
    g = TG if grid is None else grid
    ok = np.isfinite(trel) & np.isfinite(y)
    return np.interp(g, np.asarray(trel)[ok], np.asarray(y)[ok], left=np.nan, right=np.nan) if ok.sum() >= 3 else np.full(len(g), np.nan)


def agn_states(mstar, mgas, mbh, fedd):
    """The four exclusive AGN states of an (n_t, n_gal) history block, plus w_jet, f_gas, Mdot_BH and the sub-masks.
    A cell where the galaxy does not exist is NaN everywhere; a cell whose BH columns are absent counts as dormant.
    NOTE f_gas = M_gas/M_* is SIMBA's OWN X-ray gate (Dave+19) and stays on the TOTAL gas — a feedback criterion,
    not an ISM diagnostic."""
    with np.errstate(all="ignore"):
        exist = np.isfinite(mstar) & (mstar > 0)
        fgas  = np.where(exist, mgas / np.where(exist, mstar, np.nan), np.nan)
        bh    = np.isfinite(mbh) & (mbh > 0)
        acc   = exist & bh & np.isfinite(fedd) & (fedd > 0)                       # the BH is accreting
        big   = bh & (mbh > 10 ** JET_LOGMBH)
        wjet  = np.where(acc & big, np.clip(np.log10(JET_FEDD / np.clip(fedd, 1e-12, None)), 0.0, 1.0), 0.0)
        mdot  = np.where(acc, MDOT_PER_MBH * np.nan_to_num(fedd) * np.nan_to_num(mbh), 0.0)
        jet   = acc & big & (fedd < JET_FEDD)                                     # SIMBA's jet criterion
        xray  = jet & (wjet >= W_FULL) & (fgas < XRAY_FGAS_MAX)                   # full velocity AND gas-poor
        S = {"xray": xray, "choked": jet & ~xray, "radiative": acc & ~jet, "dormant": exist & ~acc}
        D = {"nobh": exist & ~bh, "big": big, "gate": exist & (fgas < XRAY_FGAS_MAX), "jet": jet}
    f = lambda m: np.where(exist, m.astype(float), np.nan)
    return ({k: f(v) for k, v in S.items()}, {k: f(v) for k, v in D.items()},
            dict(exist=exist, fgas=fgas, wjet=np.where(exist, wjet, np.nan), mdot=np.where(exist, mdot, np.nan)))

# ── the catalogue cache -> the quenched sample -> the dust LABEL ──
CAT = Table.read(need(CACHE_FITS, "paper_ism_prediction_boxes Part 1")).to_pandas()
CAT["box"] = [(v.decode() if isinstance(v, bytes) else str(v)).strip() for v in CAT["box"]]
_ms, _sfr = CAT["mstar"].to_numpy(float), CAT["sfr"].to_numpy(float)
with np.errstate(divide="ignore", invalid="ignore"):
    _ssfr = np.where(_ms > 0, _sfr / _ms, np.nan)
    _lm   = np.log10(np.where(_ms > 0, _ms, np.nan))
_quenched = ((_lm > MASS_FLOOR) & (_ssfr < PASSIVE_FACTOR / (CAT["t_H_gyr"].to_numpy(float) * 1e9))
             & (CAT["ngas"].to_numpy(float) >= NGAS_MIN) & (CAT["nstar"].to_numpy(float) >= NSTAR_MIN)
             & (CAT["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * CAT["mh2"].to_numpy(float)))
QM = CAT[_quenched].copy()
with np.errstate(divide="ignore", invalid="ignore"):
    ms = np.where(QM["mstar"].to_numpy(float) > 0, QM["mstar"].to_numpy(float), np.nan)
    QM["log_mstar"] = np.log10(ms)
    fd = QM["mdust"].to_numpy(float) / ms
    QM["lfd"]   = np.where(fd > 0, np.log10(np.where(fd > 0, fd, 1.0)), FLOOR)     # zero dust at the floor, never dropped
    QM["lfgas"] = np.log10(np.where(QM["mgas"].to_numpy(float) > 0, QM["mgas"].to_numpy(float) / ms, np.nan))
    QM["lfh2"]  = np.log10(np.where(QM["mh2"].to_numpy(float) > 0, HE * QM["mh2"].to_numpy(float) / ms, np.nan))
    QM["lssfr"] = np.log10(np.clip(np.maximum(QM["sfr"].to_numpy(float), 0.0) / ms, SSFR_FLOOR, None))
    mbh = QM["mbh"].to_numpy(float)
    QM["log_mbh"] = np.where(mbh > 0, np.log10(np.where(mbh > 0, mbh, 1.0)), np.nan)
    QM["kz_cat"]  = kappa_z(QM["kappa_gas"].to_numpy(float), QM["ngas"].to_numpy(float))

_n_before = len(QM)
_cuts = [(f"log M* > {MASS_MIN:g}", QM["log_mstar"].to_numpy(float) > MASS_MIN)]
if Z_MAX is not None:
    _cuts.append((f"z <= {Z_MAX:g}", QM["z"].to_numpy(float) <= float(Z_MAX) + 1e-9))
if AGE_MIN is not None:
    _cuts.append((f"age >= {AGE_MIN:g} Gyr", QM["age"].to_numpy(float) >= float(AGE_MIN)))
if AGE_MAX is not None:
    _cuts.append((f"age <= {AGE_MAX:g} Gyr", QM["age"].to_numpy(float) <= float(AGE_MAX)))
_keep, _drop = np.ones(len(QM), bool), []
for _lab, _m in _cuts:                                    # each cut's own toll, in the order it is applied
    _drop.append((_lab, int((_keep & ~_m).sum())))
    _keep &= _m
QM = QM[_keep].reset_index(drop=True)
QM["pop"] = np.where(QM["lfd"].to_numpy(float) >= DUSTY_CUT, "dusty", "pure")

CLS = pd.read_csv(need(CLASS_CSV, "paper_ism_prediction_boxes Part 6a"))
CLS_COLS = [c for c in ["t_sft", "t_qt", "tau_q", "w_pre", "agn_class", "agn_onset"] if c in CLS.columns]
QM = QM.merge(CLS[KEY + CLS_COLS], on=KEY, how="left")    # left join: a box / anchor without histories keeps NaN
with np.errstate(divide="ignore", invalid="ignore"):
    QM["ltauq"] = np.log10(np.where((QM["tau_q"].to_numpy(float) > 0) & (QM["t_qt"].to_numpy(float) > 0),
                                    QM["tau_q"].to_numpy(float) / QM["t_qt"].to_numpy(float), np.nan))
FAST_CUT = -1.25          # log10(tau_q / t_QT) below this = a FAST quencher (paper_xray_classes_m25's own cut)

print("THE SAMPLE: the m25 quenched rule on the catalogue cache, then " + ", ".join(l for l, _ in _cuts)
      + f"  ->  {_n_before} -> {len(QM)} rows"
      + ("  (" + ", ".join(f"{l}: -{n}" for l, n in _drop if n) + ")" if any(n for _, n in _drop) else ""))
print("  per box: " + ", ".join(f"{b} {int((QM['box'] == b).sum())}" for b in BOXES_ALL if (QM['box'] == b).any())
      + f";  anchors z = {QM['z'].min():.2f}-{QM['z'].max():.2f}, age = {QM['age'].min():.2f}-{QM['age'].max():.2f} Gyr")
print(f"  the dust threshold of this notebook, log M_dust/M* = {DUSTY_CUT:g} (f_dust = 1e{DUSTY_CUT:g}): "
      f"dusty {int((QM['pop'] == 'dusty').sum())}, dust-poor {int((QM['pop'] == 'pure').sum())} — "
      "an OUTCOME scored in Parts 4-6, never the division (Part 2 divides on the H2 at the jet).")
print(f"  the X-ray condition in force: f_gas < {XRAY_FGAS_MAX:g} (env XRAY_FGAS_MAX); full jet at w_jet >= {W_FULL:g}, "
      f"i.e. f_Edd <= {JET_FEDD * 10 ** -W_FULL:.3f}")

# ── what is on disk, and what each part will therefore be able to do ──
HAVE, BOXES_OK = {}, []
for _b in BOXES_ALL:
    _h, _k = HIST_FILE.format(box=_b), KIN_FILE.format(box=_b)
    _oh, _ok = os.path.exists(_h), os.path.exists(_k)
    HAVE[_b] = dict(hist=_oh, kin=_ok)
    if _oh:
        BOXES_OK.append(_b)
    print(f"  {_b:9s} histories {'ok' if _oh else 'MISSING [paper_ism_prediction_boxes Part 6a]'};  "
          f"kinematic sidecar {'ok' if _ok else 'MISSING [dusty_qgs_kappa_selection Part 3 builds it] -> Sigma_H2 falls back to f_H2'}")
_NODES = os.path.join(KSEL, "kappa_m25_cgm_nodes.csv")
print(f"  CGM node table (Part 6): {'ok' if os.path.exists(_NODES) else 'MISSING [dusty_qgs_kappa_selection Part 13] -> Part 6 skips'}")
print(f"  boxes this run will use: {BOXES_OK}    bootstrap NBOOT = {NBOOT} (env BA_NBOOT)")

## Part 1 — one pass over the branch chains

Everything below reads what this cell builds, and nothing re-opens an HDF5 file. One pass per box over
`ism_prediction_histories_<box>.hdf5` and its kinematic sidecar `agn_states_kin_histories_<box>.hdf5`
produces three objects:

- **`F`** — one row per branch: the event times ($t_{\rm jet}$, $t_{\rm SFT}$, $t_{\rm x1}$, $t_{\rm gate}$,
  $t_{\rm anchor}$), every tracked quantity **read at each of those events** *and* its **maximum over the
  interval between the jet and the truncation** (the `_pk` columns, which is where Part 2's axis comes from),
  the burst diagnostics ($f_{\rm burst}$, the sSFR peak) and the anchor properties;
- **`T[q][clock]`** — the same quantities as $(n_{\rm gal}, n_t)$ stacks interpolated onto the clock `TG`,
  on **three** clocks: $t-t_{\rm SFT}$, $t-t_{\rm jet}$, $t-t_{\rm x1}$;
- **`R1`** — the **raw** snapshot series per branch, *every* tracked quantity of it. Parts 4 and 5 need the
  actual sampling (a random observation lands on a snapshot, and a threshold crossing happens between two of
  them), not a track resampled onto a regular grid — and **Part 2 evaluates its axis specs on these**, which
  is what lets `"comp @ sft > qt+0.5"` be answered in a second without this cell being run again. One entry
  per row of `F`, in the same order; `F["ridx"]` is where a row's series lives.

**The de-duplication.** The same physical galaxy is a quenched anchor at several redshifts — in these boxes
about twice in the median. Part 4's question ("without resampling the same source") and Part 5's lifetimes
are meaningless without collapsing those. The branches are fingerprinted by $\log M_\star$ interpolated at
three fixed cosmic times; anchors of one galaxy share their whole early branch and so agree to numerical
precision, and the **latest** anchor is kept. This is the box-generic route — the exact progenitor index
exists only for m25.

**The `_pk` window, and how to move it.** A point read at $t_{\rm jet}$ catches the reservoir on its way up:
SIMBA's jet criterion is a black-hole condition, not an ISM one, and it is met while the cold gas is still
being compressed. `<q>_pk` is therefore the **maximum** of $q$ over a window — with `<q>_mn` its minimum, `<q>_w0` and
`<q>_w1` its values at the window's two ends and `<q>_dw` $=$ `w1` $-$ `w0` the **change across it** — and
`t_peak` / `dt_peak` record when the axis quantity got there. The `_w` columns are read at snapshots that
exist, so they never go NaN for want of one near an end time; Part 2's `"comp @ sft > qt+0.5"` is a pair of
*point* reads instead, each needing a snapshot within `P1_TOL` of its epoch, and goes NaN when there is none.
Both are legitimate, the second is stricter, and the coverage of either is printed. An interval shorter than one snapshot is bracketed by the two snapshots around it rather than left
empty, and the cell reports how often that happened.

**The window is a knob, not a hard-wired jet leg.** `P1_WIN` names its two ends and `P1_WOFF` shifts either of
them in Gyr, so the same machinery answers a different question in one line:

| `P1_WIN`, `P1_WOFF` | the window |
|---|---|
| `("jet", "sft"), (0, 0)` | **the default** — the jet leg, $t_{\rm jet} \to t_{\rm SFT}$ (~9 snapshots, 0.9 Gyr) |
| `("ign", "sft"), (0, 0)` | the whole AGN-on leg, from the first *radiative* state (~30 snapshots, 2.6 Gyr) |
| `("sft", "sft"), (-0.4, 0)` | the last 0.4 Gyr before the truncation, no event needed (~4 snapshots) |
| `("jet", "x1"), (0, 0)` | from the jet to the first X-ray firing |

The events available are `jet`, `x1`, `gate`, `big` (the BH crossing the jet mass), **`ign`** (the first
radiative state — the AGN lights up long before it meets the jet criterion), `acc` (the first accretion of any
mode), `sft`, `qt`, `anc` and `first`. The ends may be written in either order; they are sorted, and the one
written **first** is the "opening" end that `<q>_w0` is read at and that $C$ is measured from. Everything
downstream follows: Part 2's labels, $C$, and $t_{\rm feed}$, which *is* the window's width.

**The window read the other way round: $C$, the compactification.** The same interval also gives the
**minimum** of every quantity (`<q>_mn`) and its value at the **jet end** (`<q>_w0`), and the gas half-mass
radius turns those into $C = \log_{10} R_{50,\rm gas}(t_{\rm jet}) / R^{\min}_{50,\rm gas}$ — how far the cold
gas shrank between the jet firing and the truncation, in dex. Two properties of that number are measured here
rather than argued about later:

- $C \ge 0$ **by construction**, because the numerator is itself inside the window. A branch whose gas only
  ever grows after the jet sits at exactly 0, and that pile-up against a hard boundary is counted and printed.
- **the minimum of $n$ noisy points drifts down as $n$ grows**, and the window holds ~5 snapshots in m25
  against ~16 in the nox box. `comp_c0` is the *same statistic* over an equally long stretch of the branch
  ending at the jet — the noise floor of the measurement plus whatever contraction was already under way —
  and `comp_cx` $= C - C_0$ is the excess the jet leg actually added.

**$\Sigma_{\rm H_2}$ is a density, not a resolved size.** caesar carries no H$_2$ radius, so the H$_2$ mass
is spread over the *cold-gas* half-mass radius: $\Sigma_{\rm H_2} = 0.5\,\cdot 1.36\,M_{\rm H_2}/(\pi R_{50,\rm gas}^2)$.
Without the sidecar the axis degrades to the mass fraction $f_{\rm H_2}$ and the cell says so loudly.

**Broken chains** (2026-09-18). A main-progenitor line whose $\log M_\star$ falls by more than `P1_JUMP` (0.5 dex) between
consecutive snapshots is not one galaxy's history: the link jumped to a bigger object, and its $t_{\rm jet}$ / $t_{\rm SFT}$ belong
to that object. The largest one-step fall and its time are kept (`lms_jump`, `t_jump`), the branch is flagged `broken`, and
`P1_DROP_BROKEN` (default on) removes it before the peel, the tracks and the gallery. Part 8's formation redshift exposed the
first case: (116, 105) in m25, whose anchor has $\log M_\star = 10.25$ and a 2.2 Gyr age while its "chain" holds a 10.8 galaxy
aged 4.6 Gyr one snapshot earlier.


In [ ]:
# ── Part 1 — ONE PASS over the branch chains: the event times, the tracks, and the raw series ──
# One open of the histories file and one of the kinematic sidecar per box.  Everything downstream reads F / T / R1
# and never touches an HDF5 file again.
P1_BOXES  = None          # None = every box with a histories cache (BOXES_OK)
P1_NMIN   = 3             # a branch is kept only with this many existing snapshots
P1_JUMP   = 0.5           # dex: A BROKEN CHAIN (2026-09-18).  A main-progenitor line whose log M* FALLS by more than this
                          # between consecutive snapshots is not one galaxy's history: the progenitor link jumped to a
                          # bigger object (the anchor of (116, 105) in m25 sits at log M* 10.25 with a 2.2 Gyr age; the
                          # snapshot before it on its "chain" is a 10.83 galaxy aged 4.6 Gyr — the host's history, whose
                          # t_jet / t_SFT then get pinned on a galaxy they do not belong to; the formation redshift of Part 8
                          # exposed it).  The largest one-step fall is kept in F["lms_jump"] (with its time, "t_jump") and
                          # the branch is flagged F["broken"]; m100 / m50: 0.2 / 0 % of the branches, m25: 6 % of the galaxies.
P1_DROP_BROKEN = True     # drop the broken branches before anything downstream (the peel, the tracks, the gallery); False
                          # keeps them, flagged, and the printout says how many
P1_TOL    = 0.4           # Gyr: an event scalar is read at the nearest snapshot within this of the event time
P1_BURST  = 1.0           # Gyr before the SFT over which f_burst (the mass formed in the last burst) is integrated
P1_PKW    = 3.0           # Gyr before the SFT in which the sSFR peak is looked for
P1_FP     = (0.8, 1.5, 2.5)   # the de-duplication fingerprint: log M* interpolated at these cosmic times [Gyr]
P1_FPDEC  = 2             # dex decimals the fingerprint is rounded to (branches of one galaxy agree exactly)
P1_FALLBACK = True        # without the sidecar, let Sigma_H2 degrade to the mass fraction f_H2 (and say so)
P1_PKQ    = "lsig_h2"     # the quantity whose PEAK TIME is recorded (t_peak / dt_peak); the peak VALUE is stored for all
P1_MNQ    = "lr_gas"      # the quantity whose MINIMUM inside the window defines C (below); its time is t_min / dt_min
P1_TRK = ["lsig_h2", "lsig_gas", "lr_gas", "comp", "kappa_gas", "kz", "lfgas", "lfd", "lfh2", "ldh2",
          "ldg", "ldtm", "lssfr", "lms", "age", "lmdot", "xr"]
P1_ZDUST  = False         # THE METAL BUDGET of the dust-to-metal ratio.  False = the histories' zgas is the
                          # GAS-PHASE metallicity and SIMBA's grains are metals taken OUT of it, so the budget
                          # is Z_gas M_gas + M_dust (agn_states_dusty_vs_pure's PF_Z_HAS_DUST convention,
                          # verbatim).  True = zgas already counts the grain metals and dust is not added twice.
P1_CLOCKS = ["sft", "jet", "x1", "qt"]     # the clocks the tracks are gridded on: any key of P1_EV below
P1_LAB = {"lsig_h2": r"$\log \Sigma_{\rm H_2}$  [$M_\odot$ kpc$^{-2}$]", "lsig_gas": r"$\log \Sigma_{\rm gas}$",
          "lr_gas": r"$\log R_{50,\rm gas}$  [kpc]",
          "comp": r"$-\log R_{50,\rm gas}/R_{50,\star}$", "kappa_gas": r"$\kappa_{\rm rot}$(gas)",
          "kz": r"$(\kappa_{\rm gas}-1/3)/s_{\rm iso}$", "lfgas": r"$\log M_{\rm gas}/M_\star$",
          "lfd": r"$\log M_{\rm dust}/M_\star$", "lfh2": r"$\log M_{\rm H_2}/M_\star$",
          "lssfr": r"$\log$ sSFR  [yr$^{-1}$]", "lms": r"$\log M_\star$", "xr": "X-ray channel on",
          "age": r"mass-weighted stellar age  [Gyr]", "lmdot": r"$\log \dot{M}_{\rm BH}$  [$M_\odot$ yr$^{-1}$]",
          "ldh2": r"$\log M_{\rm dust}/M_{\rm H_2}$", "ldg": r"$\log M_{\rm dust}/M_{\rm gas}$   (D/G)",
           "ldtm": r"$\log M_{\rm dust}/M_{\rm metals}$   (D/M)"}
EPOCHS = ["jet", "sft", "x1", "qt", "anc"]    # the five POINTS every tracked quantity is read at
# "qt" is the QUENCHING TIME itself (the end of the transition, not its start): "<q>_qt" is what an observer
# who catches the galaxy exactly as it arrives on the red sequence would measure.
# ── "pk" is a fifth epoch, and it is a WINDOW statistic, not a point read ──────────────────────────────────
# Reading a quantity AT t_jet samples the rise, not the state: the jet criterion is met while the cold gas is
# still being compressed, and the clocks of Part 3 show the H2 density peaking between the jet and the
# truncation.  "<q>_pk" is therefore the MAXIMUM of q over the interval spanned by t_jet and t_SFT (in either
# order — a galaxy whose jet fires after its truncation has the same interval, walked backwards), and
# "t_peak" / "dt_peak" record when the axis quantity got there, absolutely and relative to the truncation.
PK_EPOCH = "pk"
# ── THE WINDOW IS A KNOB: which two events bound it, and how far either end is shifted ────────────────────
# Each end is (an event key from P1_EV, an offset in Gyr).  The default is the jet leg, t_jet -> t_SFT, but
#     P1_WIN, P1_WOFF = ("ign", "sft"), (0.0, 0.0)      the whole AGN-on leg, from the first radiative state
#     P1_WIN, P1_WOFF = ("sft", "sft"), (-0.4, 0.0)     the last 0.4 Gyr before the truncation
#     P1_WIN, P1_WOFF = ("jet", "x1"),  (0.0, 0.0)      from the jet to the first X-ray firing
# are all one line.  The ends may be given in either order; they are sorted, and the FIRST one written is the
# "opening" end that "<q>_w0" is read at and that C is measured from.
P1_WIN  = ("sft", "qt")
P1_WOFF = (0.0, 0.5)
P1_EV = {"jet": "t_jet", "x1": "t_x1", "gate": "t_gate", "big": "t_big", "ign": "t_ign", "acc": "t_acc",
         "sft": "t_sft", "qt": "t_qt", "anc": "t_anchor", "first": "t_first"}
# t_ign = the first RADIATIVE state (the AGN lights up long before it meets the jet criterion);
# t_acc = the first accretion of any mode;  t_big = M_BH crossing the jet mass;  t_gate = the f_gas gate;
# t_first = the branch's earliest existing snapshot.
_EVTEX = {"jet": r"t_{\rm jet}", "x1": r"t_{\rm x1}", "gate": r"t_{\rm gate}", "big": r"t_{\rm bigBH}",
          "ign": r"t_{\rm ign}", "acc": r"t_{\rm acc}", "sft": r"t_{\rm SFT}", "qt": r"t_{\rm QT}",
          "anc": r"t_{\rm anchor}", "first": r"t_{\rm first}"}
_wl = lambda k, o: P1_EV.get(k, k) + (f" {o:+.2f} Gyr" if o else "")
_wt = lambda k, o: _EVTEX.get(k, k) + (f" {o:+.2f}" if o else "")
WIN_LAB = f"{_wl(P1_WIN[0], P1_WOFF[0])} -> {_wl(P1_WIN[1], P1_WOFF[1])}"        # for prints
WIN_TEX = f"$[{_wt(P1_WIN[0], P1_WOFF[0])},\\, {_wt(P1_WIN[1], P1_WOFF[1])}]$"    # for axis labels
# ── the same window read the other way: "<q>_mn" the MINIMUM over it, "<q>_w0" / "<q>_w1" its two ENDS ──────
# "<q>_w0" is q at the first in-window snapshot walking from the opening end (the end written FIRST in P1_WIN),
# "<q>_w1" at the last one, and "<q>_dw" = w1 - w0 is the CHANGE ACROSS THE WINDOW.  With the default window
# "comp_dw" is therefore how much -log R50,gas/R50,* moved over the whole leg.  These are always measured on
# snapshots that exist, so they never go NaN for want of a snapshot near the end time — which is the difference
# between them and Part 2's "comp @ sft > qt+0.5", a pair of POINT reads that each need a snapshot within
# P1_TOL of their epoch.  Both are legitimate; the second is stricter and says so by going NaN.
# Those three are what the COMPACTIFICATION is built from:
#
#     C  =  log10 R50,gas(t_jet) / R50,gas(min over the window)   =   lr_gas_w0 - lr_gas_mn      [dex]
#
# the factor by which the cold gas shrank between the jet firing and the truncation.  Two properties of it have
# to be kept in view, and both are measured and printed rather than argued about:
#   * C >= 0 BY CONSTRUCTION -- the numerator is itself inside the window, so a branch whose gas only ever grows
#     after the jet sits at exactly 0.  A spike at a hard boundary next to a continuum will make any
#     multimodality test reject, so the size of that spike is reported separately in Part 2.
#   * the minimum of n noisy points drifts down as n grows, and the window holds ~5 snapshots in m25 against ~18
#     in the nox box.  "comp_c0" is the SAME statistic over an equally long stretch ending at the jet -- the
#     noise floor of the measurement, and whatever contraction was already under way before the jet fired.
#     "comp_cx" = C - C0 is the excess of the one over the other: the contraction the jet leg ADDED.


def _kinread1(box):
    """The kinematic sidecar as {anchor: dict(snaps, gal_ids, kappa_gas, r_gas, r_star, ngas)}.
    It is joined to the histories BY VALUE (snapshot number, galaxy id) and never by array order: the two
    files build their snapshot and galaxy lists independently."""
    p = KIN_FILE.format(box=box)
    if not os.path.exists(p):
        return None
    K = {}
    with h5py.File(p, "r") as f:
        for gk in f:
            g = f[gk]
            K[int(gk[4:])] = dict(snaps=g["snaps"][:].astype(int), gal_ids=g["gal_ids"][:].astype(int),
                                  **{c: g[c][:].astype(float) for c in ("kappa_gas", "r_gas", "r_star", "ngas") if c in g})
    return K


def _align1(ks, snaps, gal_ids, col, shape):
    """One sidecar column put on the histories' (snapshot x galaxy) grid; anything absent stays NaN."""
    out = np.full(shape, np.nan)
    if ks is None or col not in ks:
        return out
    smap = {int(s): i for i, s in enumerate(ks["snaps"])}
    gmap = {int(g): j for j, g in enumerate(ks["gal_ids"])}
    ri = np.array([smap.get(int(s), -1) for s in snaps])
    ci = np.array([gmap.get(int(g), -1) for g in gal_ids])
    rr, cc = np.flatnonzero(ri >= 0), np.flatnonzero(ci >= 0)
    if len(rr) and len(cc):
        out[np.ix_(rr, cc)] = ks[col][np.ix_(ri[rr], ci[cc])]
    return out


def _pass1(box):
    """One pass over the box's chains -> (F, T, R).  F one row per branch, T the gridded tracks on three
    clocks, R the raw snapshot series."""
    sub = QM[QM["box"] == box]
    if not len(sub):
        return None
    KS = _kinread1(box)
    rows, R = [], []
    T = {q: {c: [] for c in P1_CLOCKS} for q in P1_TRK}
    _trapz = getattr(np, "trapezoid", None) or np.trapz
    with h5py.File(need(HIST_FILE.format(box=box), "paper_ism_prediction_boxes Part 6a"), "r") as f:
        for a, Sa in sub.groupby("snap"):
            gk = f"snap{int(a):03d}"
            if gk not in f:
                continue
            g = f[gk]
            o = np.argsort(g["t_yr"][:])                        # the file is written newest-first
            t = g["t_yr"][:][o] / 1e9                           # cosmic time [Gyr]; t_yr IS t_H(z) at that step
            snaps, ids = g["snaps"][:][o], g["gal_ids"][:].astype(int)
            zs = g["z"][:][o] if "z" in g else np.full(len(t), np.nan)   # the ladder's OWN redshifts (the
            # histories write them beside t_yr), so Part 10's redshift row needs no cosmology of its own
            H = {k: (g[k][:][o] if k in g else np.full((len(t), len(ids)), np.nan))
                 for k in ("mstar", "sfr", "mgas", "mbh", "fedd", "mdust", "mh2", "age", "zgas")}
            ks = None if KS is None else KS.get(int(a))
            KAP = _align1(ks, snaps, ids, "kappa_gas", H["mstar"].shape)
            RG  = _align1(ks, snaps, ids, "r_gas",  H["mstar"].shape)
            RS  = _align1(ks, snaps, ids, "r_star", H["mstar"].shape)
            NG  = _align1(ks, snaps, ids, "ngas",   H["mstar"].shape)
            S, D, X = agn_states(H["mstar"], H["mgas"], H["mbh"], H["fedd"])
            with np.errstate(all="ignore"):
                ex   = X["exist"]
                ms   = np.where(ex, H["mstar"], np.nan)
                ssfr = np.where(ex, np.maximum(H["sfr"], 0.0) / ms, np.nan)
                V = {}
                V["lms"]   = np.log10(ms)
                V["lfd"]   = np.log10(np.clip(H["mdust"] / ms, 10 ** FLOOR, None))     # floored: zero dust is a value
                V["lfh2"]  = np.log10(np.where(H["mh2"] > 0, HE * H["mh2"] / ms, np.nan))
                V["lfgas"] = np.log10(np.where(H["mgas"] > 0, H["mgas"] / ms, np.nan))
                V["lssfr"] = np.log10(np.clip(ssfr, SSFR_FLOOR, None))
                V["lsig_gas"] = np.log10(np.where(RG > 0, 0.5 * H["mgas"] / (np.pi * RG ** 2), np.nan))
                V["lsig_h2"]  = np.log10(np.where((RG > 0) & (H["mh2"] > 0), 0.5 * HE * H["mh2"] / (np.pi * RG ** 2), np.nan))
                if P1_FALLBACK and not np.isfinite(V["lsig_h2"]).any():
                    V["lsig_h2"] = V["lfh2"]                                            # no sidecar: the axis degrades
                V["lr_gas"] = np.log10(np.where(RG > 0, RG, np.nan))
                V["comp"]  = -np.log10(np.where((RG > 0) & (RS > 0), RG / RS, np.nan))
                # THE TWO RATIOS, on the same helium convention as lfh2 (M_H2 is hydrogen-only in SIMBA)
                V["ldh2"]  = np.log10(np.where((H["mdust"] > 0) & (H["mh2"] > 0),
                                               H["mdust"] / (HE * np.where(H["mh2"] > 0, H["mh2"], np.nan)), np.nan))
                V["ldg"]   = np.log10(np.where((H["mdust"] > 0) & (H["mgas"] > 0),
                                               H["mdust"] / np.where(H["mgas"] > 0, H["mgas"], np.nan), np.nan))
                # THE DEPLETION: the same ratio with the ENRICHMENT divided out, so a dust-to-gas difference
                # that is only a difference in metallicity does not survive into it.  NaN wherever the box's
                # histories carry no 'zgas' (a cache built before that column existed).
                _mz  = np.where((H["zgas"] > 0) & (H["mgas"] > 0), H["zgas"] * H["mgas"], np.nan)
                _mzt = _mz if P1_ZDUST else _mz + np.where(H["mdust"] > 0, H["mdust"], 0.0)
                V["ldtm"] = np.log10(np.where((H["mdust"] > 0) & (_mzt > 0),
                                              H["mdust"] / np.where(_mzt > 0, _mzt, np.nan), np.nan))
                V["age"]   = np.where(ex, H["age"], np.nan)          # the mass-weighted stellar age [Gyr]
                V["lmdot"] = np.log10(np.where(X["mdot"] > 0, X["mdot"], np.nan))   # log Mdot_BH [Msun/yr]
                V["kappa_gas"] = np.where(ex, KAP, np.nan)
                V["kz"]    = kappa_z(KAP, NG)
                V["xr"]    = S["xray"]
                # THE QUENCHED FLAG AT EVERY SNAPSHOT — Part 4's "a QG observed there".  The catalogue rule's
                # sSFR criterion and its mass floor; the particle-count cuts are anchor-only and are not imposed here.
                qflag = (ssfr < PASSIVE_FACTOR / (t[:, None] * 1e9)) & (V["lms"] > MASS_FLOOR) & ex
            for _, r0 in Sa.iterrows():
                gid = int(r0["gal_id"])
                j = int(np.searchsorted(ids, gid))
                if j >= len(ids) or int(ids[j]) != gid:
                    continue
                e = ex[:, j]
                if e.sum() < P1_NMIN:
                    continue
                ka = int(np.flatnonzero(e)[-1])
                first = lambda m: float(t[m & e][0]) if (m & e).any() else np.nan
                r = dict(box=box, snap=int(a), gal_id=gid, t_anchor=float(t[ka]),
                         t_jet=first(D["jet"][:, j] > 0), t_x1=first(S["xray"][:, j] > 0),
                         t_gate=first(D["gate"][:, j] > 0), t_big=first(D["big"][:, j] > 0),
                         t_ign=first(S["radiative"][:, j] > 0), t_first=float(t[e][0]),
                         t_acc=first((np.nan_to_num(S["radiative"][:, j]) + np.nan_to_num(S["choked"][:, j])
                                      + np.nan_to_num(S["xray"][:, j])) > 0),
                         t_sft=float(r0["t_sft"]), t_qt=float(r0["t_qt"]), tau_q=float(r0["tau_q"]),
                         ltauq=float(r0["ltauq"]), t_span=float(t[e][-1] - t[e][0]), n_step=int(e.sum()))
                r["t_post"] = r["t_anchor"] - r["t_sft"]
                r["lag_x1"] = r["t_x1"] - r["t_jet"]
                r["lag_sft"] = r["t_sft"] - r["t_jet"]
                r["tq_span"] = r["t_qt"] - r["t_sft"]                  # the transition itself, for comparison

                def _at(y, te):
                    """y read at the snapshot nearest te, but only if one lies within P1_TOL of it."""
                    if not np.isfinite(te):
                        return np.nan
                    k = np.flatnonzero(e)
                    i = k[int(np.argmin(np.abs(t[k] - te)))]
                    return float(y[i]) if abs(t[i] - te) <= P1_TOL and np.isfinite(y[i]) else np.nan

                # THE WINDOW the "pk" and "mn" epochs are taken over — P1_WIN / P1_WOFF, the jet leg by default.
                _wm, _ij, _iwj = np.zeros(len(t), bool), -1, np.array([], int)
                r["n_win"], r["w_win"], r["win_brack"] = 0, np.nan, False
                r["n_win0"], r["comp_c0"] = 0, np.nan
                r["t_w0"] = r.get(P1_EV.get(P1_WIN[0], ""), np.nan) + P1_WOFF[0]
                r["t_w1"] = r.get(P1_EV.get(P1_WIN[1], ""), np.nan) + P1_WOFF[1]
                if np.isfinite(r["t_w0"]) and np.isfinite(r["t_w1"]):
                    _lo, _hi = sorted((r["t_w0"], r["t_w1"]))
                    r["w_win"] = float(_hi - _lo)
                    _wm = e & (t >= _lo) & (t <= _hi)
                    if not _wm.any():                       # shorter than one snapshot: bracket it instead
                        _k = np.flatnonzero(e)
                        _b = _k[t[_k] <= _lo]; _a = _k[t[_k] >= _hi]
                        _i0 = int(_b[-1]) if len(_b) else int(_k[0])
                        _i1 = int(_a[0]) if len(_a) else int(_k[-1])
                        _wm = np.zeros(len(t), bool); _wm[min(_i0, _i1):max(_i0, _i1) + 1] = True; _wm &= e
                        r["win_brack"] = True
                    r["n_win"] = int(_wm.sum())
                    # the in-window snapshots walked FROM THE JET END outwards: _iwj[0] is where C is measured from
                    _iw = np.flatnonzero(_wm)
                    if len(_iw):
                        _iwj = _iw if r["t_w0"] <= r["t_w1"] else _iw[::-1]
                        _ij = int(_iwj[0])
                        # THE NULL WINDOW: as many snapshots as the real one, ending where the real one starts.
                        _k = np.flatnonzero(e)
                        _pos = int(np.flatnonzero(_k == _ij)[0])
                        _p0 = max(0, _pos - r["n_win"] + 1)
                        if _pos - _p0 >= 1:
                            _yn = V[P1_MNQ][_k[_p0:_pos + 1], j]
                            _f0 = np.flatnonzero(np.isfinite(_yn))
                            r["n_win0"] = int(_pos - _p0 + 1)
                            if len(_f0):
                                r["comp_c0"] = float(_yn[_f0[0]] - np.nanmin(_yn))
                for q in P1_TRK:
                    y = V[q][:, j]
                    for ep, te in zip(EPOCHS, (r["t_jet"], r["t_sft"], r["t_x1"], r["t_qt"], r["t_anchor"])):
                        r[f"{q}_{ep}"] = float(y[ka]) if ep == "anc" and np.isfinite(y[ka]) else _at(y, te)
                    _yw = np.where(_wm & np.isfinite(y), y, np.nan)
                    _any = bool(np.isfinite(_yw).any())
                    r[f"{q}_pk"] = float(np.nanmax(_yw)) if _any else np.nan
                    r[f"{q}_mn"] = float(np.nanmin(_yw)) if _any else np.nan
                    _fj = np.flatnonzero(np.isfinite(y[_iwj])) if len(_iwj) else np.array([], int)
                    r[f"{q}_w0"] = float(y[_iwj[_fj[0]]]) if len(_fj) else np.nan
                    r[f"{q}_w1"] = float(y[_iwj[_fj[-1]]]) if len(_fj) else np.nan
                    r[f"{q}_dw"] = r[f"{q}_w1"] - r[f"{q}_w0"]     # the CHANGE across the window, end minus start
                    if q == P1_PKQ and _any:
                        _ip = int(np.nanargmax(_yw))
                        r["t_peak"] = float(t[_ip])
                        r["dt_peak"] = float(t[_ip] - r["t_sft"])
                    if q == P1_MNQ and _any:
                        _im = int(np.nanargmin(_yw))
                        r["t_min"] = float(t[_im])
                        r["dt_min"] = float(t[_im] - r["t_sft"])
                    for c in P1_CLOCKS:
                        te = r.get(P1_EV.get(c, c), np.nan)
                        T[q][c].append(to_grid(t - te, y) if np.isfinite(te) else np.full(len(TG), np.nan))
                r.setdefault("t_peak", np.nan); r.setdefault("dt_peak", np.nan)
                r.setdefault("t_min", np.nan); r.setdefault("dt_min", np.nan)
                # C, THE COMPACTIFICATION: how far the gas half-mass radius fell below its jet-time value
                # before the truncation.  NaN when the sidecar has no radius on either side of the pair.
                # t_feed, THE FEEDING TIME: the WIDTH OF THE WINDOW, which with the default P1_WIN is the jet
                # leg t_SFT - t_jet exactly.  It follows P1_WIN, so re-pointing the window re-points Part 2b too.
                r["t_feed"] = r["w_win"]
                r["lfeed_n"] = (np.log10(r["t_feed"] / r["t_qt"])      # t_feed in units of the epoch it happened at
                                if np.isfinite(r["t_feed"]) and r["t_feed"] > 0 and r["t_qt"] > 0 else np.nan)
                # t_feed spans a decade, so it is also carried in log: multimodality is not scale-invariant and
                # a skewed positive quantity has to be tested on both scales before it is called unimodal.
                r["lt_feed"] = np.log10(r["t_feed"]) if np.isfinite(r["t_feed"]) and r["t_feed"] > 0 else np.nan
                r["comp_c"] = r[f"{P1_MNQ}_w0"] - r[f"{P1_MNQ}_mn"]
                r["comp_cx"] = r["comp_c"] - r["comp_c0"]    # the EXCESS over its own pre-jet null
                # the burst diagnostics: the mass formed in the last P1_BURST Gyr before the SFT, over M* there,
                # integrated from the branch's OWN SFR so a merger cannot fake it; and the pre-SFT sSFR peak.
                r["f_burst"] = r["b_ssfr"] = r["t_pk"] = np.nan
                if np.isfinite(r["t_sft"]):
                    w = e & (t >= r["t_sft"] - P1_BURST) & (t <= r["t_sft"])
                    m_sft = _at(ms[:, j], r["t_sft"])
                    if w.sum() >= 2 and np.isfinite(m_sft) and m_sft > 0:
                        r["f_burst"] = float(_trapz(np.nan_to_num(H["sfr"][w, j]), t[w] * 1e9) / m_sft)
                    pw = e & (t >= r["t_sft"] - P1_PKW) & (t <= r["t_sft"]) & np.isfinite(ssfr[:, j])
                    if pw.any():
                        ip = np.flatnonzero(pw)[int(np.argmax(ssfr[pw, j]))]
                        r["b_ssfr"], r["t_pk"] = float(np.log10(max(ssfr[ip, j], SSFR_FLOOR))), float(t[ip])
                # THE CHAIN'S INTEGRITY: the largest one-step fall of log M* along it (P1_JUMP)
                _lme = V["lms"][e, j]; _dlm = np.diff(_lme); _fin = np.isfinite(_dlm)
                r["lms_jump"] = float(np.nanmin(_dlm)) if _fin.any() else np.nan
                r["t_jump"] = float(t[e][1:][_fin][int(np.nanargmin(_dlm[_fin]))]) if _fin.any() else np.nan
                r["broken"] = bool(np.isfinite(r["lms_jump"]) and r["lms_jump"] < -float(P1_JUMP))
                # the fingerprint the de-duplication is done on
                fp = np.interp(P1_FP, t[e], V["lms"][e, j], left=np.nan, right=np.nan)
                r["fp"] = "|".join("nan" if not np.isfinite(v) else f"{v:.{P1_FPDEC}f}" for v in fp)
                for c in ("z", "age", "log_mstar", "log_mbh", "lfd", "lfh2", "lfgas", "lssfr", "pop", "kz_cat", "central"):
                    if c in r0:
                        r[c] = r0[c]
                rows.append(r)
                # THE SERIES THEMSELVES, one entry per row of F and in the same order.  Parts 4 and 5 read
                # "t", "q" and "lfd" out of these; Part 2 evaluates ITS AXIS SPECS on them, which is what lets
                # a new axis ("comp @ sft > qt+0.5") cost nothing but a loop over branches — no second pass
                # over the HDF5 files, and no re-run of this cell when the question changes.
                R.append(dict(t=t[e].copy(), z=zs[e].copy(), snap=snaps[e].astype(int).copy(), q=qflag[e, j].copy(),
                              **{k: V[k][e, j].copy() for k in P1_TRK}))
    if not rows:
        return None
    _nbrk = int(sum(r["broken"] for r in rows))
    if P1_DROP_BROKEN and _nbrk:                              # the broken chains leave HERE, before F / T / R are aligned
        _keep = [i for i, r in enumerate(rows) if not r["broken"]]
        rows = [rows[i] for i in _keep]; R = [R[i] for i in _keep]
        for q in P1_TRK:
            for c in P1_CLOCKS:
                T[q][c] = [T[q][c][i] for i in _keep]
    _fate = "DROPPED" if P1_DROP_BROKEN else "kept, flagged F['broken']"
    print(f"{box:9s} {_nbrk:4d} broken chains (log M* falls by > {P1_JUMP:g} dex in one step: the progenitor line jumped to "
          f"another galaxy) — {_fate}")
    if not rows:
        return None
    F = pd.DataFrame(rows).reset_index(drop=True)
    for q in P1_TRK:
        for c in P1_CLOCKS:
            T[q][c] = np.asarray(T[q][c], float).reshape(len(F), len(TG))
    return F, T, R


B1, FR, R1 = {}, [], []
_BX1 = BOXES_OK if P1_BOXES is None else [b for b in P1_BOXES if b in BOXES_OK]
for _b in _BX1:
    _r = _pass1(_b)
    if _r is None:
        print(f"{_b}: no branch survived the pass"); continue
    _F, _T, _R = _r
    # ── THE DE-DUPLICATION: one row per physical galaxy, the LATEST anchor kept ──
    _F["gid"] = _b + ":" + _F["fp"]
    _bad = _F["fp"].str.count("nan") >= 2                       # a branch too short to fingerprint keeps its own key
    _F.loc[_bad, "gid"] = [f"{_b}:row{i}" for i in _F.index[_bad]]
    _o = _F.sort_values(["gid", "t_anchor"]).groupby("gid").tail(1).index
    _F["uniq"] = False; _F.loc[_o, "uniq"] = True
    _F["ridx"] = np.arange(len(_F))            # B1[box]["R"][ridx] is this row's series — Part 2's axis specs
    B1[_b] = dict(F=_F, T=_T, R=_R)
    FR.append(_F)
    print(f"{_b:9s} {len(_F):5d} branches -> {int(_F['uniq'].sum()):5d} galaxies "
          f"({len(_F) / max(1, int(_F['uniq'].sum())):.2f} anchors each);  "
          f"t_jet {np.isfinite(_F['t_jet']).mean() * 100:3.0f} %, t_x1 {np.isfinite(_F['t_x1']).mean() * 100:3.0f} %, "
          f"t_SFT {np.isfinite(_F['t_sft']).mean() * 100:3.0f} %;  "
          f"{P1_PKQ} peak inside the window {np.isfinite(_F[f'{P1_PKQ}_pk']).mean() * 100:3.0f} % "
          f"(median {np.nanmedian(_F['n_win']):.0f} snapshots over {np.nanmedian(_F['w_win']):.2f} Gyr, "
          f"{_F['win_brack'].mean() * 100:.0f} % bracketed) "
          + ("[SIDECAR MISSING -> f_H2 fallback]" if not os.path.exists(KIN_FILE.format(box=_b)) else ""))

FR = pd.concat(FR, ignore_index=True) if FR else pd.DataFrame()
FR.to_csv(os.path.join(OUT, "ba_galaxies.csv"), index=False)
print("\ntable ->", os.path.join(OUT, "ba_galaxies.csv"), f"({len(FR)} branch rows, {int(FR['uniq'].sum())} unique galaxies)")
print(f"  the dust-to-metal track (ldtm, budget = Z_gas M_gas{'' if P1_ZDUST else ' + M_dust'}) is defined on "
      f"{np.isfinite(FR['ldtm_anc']).mean() * 100:3.0f} % of branch rows and the ladder redshift on "
      f"{float(np.mean([np.isfinite(r['z']).any() for b in B1 for r in B1[b]['R']])) * 100:3.0f} % of the series "
      f"(both need the histories' 'zgas' / 'z': a cache built before them keeps NaN).")
print(f"the five POINT epochs every quantity is read at: {EPOCHS}  (nearest snapshot within {P1_TOL:g} Gyr of the event)")
print(f"every branch's SERIES is kept in memory as well ({len(FR)} of them, {P1_TRK}), so Part 2 can be asked for\n"
      f"  any quantity at any epoch — 'comp @ sft > qt+0.5' — without this cell being run again.")
print(f"plus the WINDOW epochs '{PK_EPOCH}' / 'mn': the maximum and the minimum over  {WIN_LAB}  (P1_WIN / P1_WOFF); "
      f"t_peak / dt_peak record when {P1_PKQ} got there and t_min / dt_min when {P1_MNQ} did (both relative to the "
      f"truncation).")
print(f"  the peak sits {np.nanmedian(FR['dt_peak']):+.2f} Gyr from the truncation in the median "
      f"({float(np.nanmean(FR['dt_peak'] < 0)) * 100:.0f} % of branches peak BEFORE it); "
      f"the window is {np.nanmedian(FR['w_win']):.2f} Gyr / {np.nanmedian(FR['n_win']):.0f} snapshots wide, "
      f"and {float(FR['win_brack'].mean()) * 100:.0f} % were too short to contain a snapshot and were bracketed.")
print(f"and the WINDOW MINIMUM of {P1_MNQ}, from which C = {P1_MNQ}_w0 - {P1_MNQ}_mn = "
      f"log R50,gas(at {P1_EV.get(P1_WIN[0], P1_WIN[0])}{P1_WOFF[0]:+.2f}) / R50,gas(min in the window) is formed:")
_cA = FR["comp_c"].to_numpy(float); _c0 = FR["comp_c0"].to_numpy(float)
print(f"  C measured on {np.isfinite(_cA).mean() * 100:3.0f} % of branches, median {np.nanmedian(_cA):.3f} dex "
      f"(a factor {10 ** np.nanmedian(_cA):.2f} of shrinking), {float(np.nanmean(_cA[np.isfinite(_cA)] <= 1e-9)) * 100:.0f} % sit at exactly 0 "
      f"(the gas never contracts once the window opens); the gas is most compact {np.nanmedian(FR['dt_min']):+.2f} Gyr "
      f"from the truncation in the median.")
print(f"  the NULL over an equally long stretch BEFORE the window: median {np.nanmedian(_c0):.3f} dex over "
      f"{np.nanmedian(FR['n_win0']):.0f} snapshots — C above it is contraction, C at it is the noise floor of a "
      f"minimum taken over {np.nanmedian(FR['n_win']):.0f} points.")

## Part 1b — the selection: the peel

**This cell is the selection.** It draws the map of one quantity against the clock, finds the two bright
regions in it, and turns them into the three groups every later part of the notebook works with. The knobs
come in two families: `P1B_*` is how the **map** and its valley are measured (the clock, the bandwidth, what
counts as a mode), `P1D_*` is how the **regions** are peeled off it (how bright, how long a track must stay,
how strict).

| | |
|---|---|
| **stage 1** | the whole map. The bright region **above the valley** is outlined; every galaxy whose track spends at least `P1D_NHIT` epochs inside it becomes **group 1** |
| **stage 2** | the same map **with group 1's tracks removed**. If the low region empties out as well, those galaxies were passing through it on their way down and the two regions are one story; if it stays put, they are two. The most populated bright blob of what is left becomes **group 2** |
| **stage 3** | the residual — the galaxies that never entered either region: **group 3**, the intermediates |

**Calibrate where the statistics are.** `P1D_CAL = ["cis100", "cis50"]` finds the regions **once**, on those
two boxes pooled, and then peels **the same cells** off every box — so the 25 Mpc box, with 82 galaxies, is
never asked to find regions it cannot resolve; it is only asked whether its galaxies fall in them. `None`
lets every box find its own.

**How it is drawn.** Every epoch column is scaled to its own peak — that is what makes two ridges visible at
all — but re-doing that at each stage would renormalise the map back to full brightness and the whole point
of the peel, *watching a region empty*, would be invisible. So the scaling is fixed at stage 1 and each
column is **dimmed by the fraction of its galaxies still in play**. An empty cell is black, not white. The
cyan line is the median of what is still in play at that stage, with its 16–84 spread dotted, so the three
panels can be read against each other.

**How a region is found.** A cell is *bright* when it stands within `P1D_BRIGHT` of the peak **of its own
side of the valley** — so a minor upper ridge is not erased by a major lower one — and the valley row itself
is left out, so the two sides can never be connected through it. Connectivity is worked out on a mask
dilated by `P1D_BRIDGE` epochs along the clock (a ridge graded at +1.0 and at +1.5 is one ridge, not two a
track can never be inside twice) and the columns jumped are then filled in, so the region is a band and not
a comb. Of the candidate blobs the **most populated** wins — "the bright region" in the eye means the one
most galaxies are in, not the tallest or the largest. `P1D_REG` replaces all of that with two rectangles you
type.

**The number the peel exists for** is printed under stage 2: *what fraction of group 1 also enters group 2's
region*. High means the two bright regions are the same galaxies caught at two times — one story, seen
twice. Low means two populations. At the default settings on the jet clock it is **7 / 3 / 7 %**
(m100 / m50 / m25) — with `P1D_PURE` on, the few that do qualify for both are sent to the intermediates
rather than counted as members of either.

The second figure carries the three groups onto every other quantity — the gas compactness, $\Sigma_{\rm H_2}$,
$\kappa_{\rm rot}$(gas), **sSFR, the mass-weighted stellar age and $\dot{M}_{\rm BH}$** (`P1D_Q`) — with the
field median dotted behind them and the AUC between the two extreme groups printed in each panel. The
membership is written into
`FR["grp_peel"]` and `ba_peel_members.csv`. **Part 2 adopts this selection by default** (`P2_ADOPT = "peel"`), so Parts 3–6 all run on these three
groups: group 1 becomes the `burst` key, group 2 the `aging` one, and the intermediates are left out of
both, the way a tercile middle is.

In [ ]:
# ── Part 1b — THE SELECTION: the peel ─────────────────────────────────────────────────────────────────────
# THIS CELL IS THE SELECTION.  It builds the map of one quantity against the clock, finds the two bright
# regions in it, and turns them into the three groups every later part of this notebook works with.
#
# The knobs come in two families and both are here:
#   P1B_*  how the MAP and its valley are measured — the clock, the bandwidth, what counts as a mode
#   P1D_*  how the REGIONS are peeled off it — how bright, how long a track must stay, how strict
#
#   STAGE 1  the whole map.  The brightest connected region at HIGH f_dust is outlined; every galaxy whose
#            track spends at least P1D_NHIT epochs inside it becomes GROUP 1.
#   STAGE 2  the same map WITH GROUP 1'S TRACKS REMOVED.  If the low region empties out as well, those
#            galaxies were passing through it on their way down and the two regions are one story; if it
#            stays put, they are two.  The brightest connected region at LOW f_dust is outlined -> GROUP 2.
#   STAGE 3  what is left: the galaxies that never entered either region -> GROUP 3, the intermediates.
#
# HOW IT IS DRAWN.  Part 1b scales every epoch column to its own peak, which is what makes the two ridges
# visible at all — but re-doing that at each stage would renormalise the map back to full brightness and the
# whole point of the peel, watching a region empty, would be invisible.  So each column keeps Part 1b's shape
# and is DIMMED BY THE FRACTION OF ITS GALAXIES STILL IN PLAY: same scaling, honest depletion.  The regions
# are found on the shape and weighed by the COUNTS behind it, so "the bright region" means the one most
# galaxies are in and not the one that happens to be tallest.
# ── the map: the clock it is drawn on, and what counts as a mode in a column ──────────────────────────────
P1B_CLOCK = "jet"        # the event the abscissa is zeroed on: any of P1_CLOCKS  (sft | jet | x1 | qt)
P1B_UNIQ  = True         # galaxies, not anchors
P1B_NMIN  = 3            # an epoch column is used at all only with this many galaxies in it
P1B_NPER  = 50           # ... and its VALLEY is trusted only with this many: a mode needs a crowd
P1B_YLIM  = (1, 99)      # the percentile range of the y axis (over every epoch at once)
P1B_BW    = 0.5          # the KDE bandwidth per column, as a multiple of Scott's rule (smaller = more modes)
P1B_PROM  = 0.15         # a peak counts only if it stands this far above the column's lowest point
P1B_DEEP  = 0.75         # ... and two peaks are two modes only if the valley falls below this x the lower one
P1B_FLOORS = {"lfd": FLOOR, "lssfr": np.log10(SSFR_FLOOR)}   # a pile-up AT A KNOWN FLOOR is censoring, not a
                         # population: zero dust is stored at FLOOR and zero SFR at SSFR_FLOOR, so a mode
                         # sitting on one of those is never counted and never used as a valley
P1B_CMAP  = "magma"
_CTX = {"sft": r"t-t_{\rm SFT}", "jet": r"t-t_{\rm jet}", "x1": r"t-t_{\rm x1}", "qt": r"t-t_{\rm QT}"}
if P1B_CLOCK not in P1_CLOCKS:
    raise KeyError(f"P1B_CLOCK = {P1B_CLOCK!r} is not one of the clocks Part 1 built: {P1_CLOCKS}")

# ── the peel ──────────────────────────────────────────────────────────────────────────────────────────────
P1D_ON    = "lfd"        # the quantity whose heatmap is peeled
P1D_T0    = 0.0          # Gyr: only look for regions after this point on the clock
P1D_NY    = 60           # the cells of the map
P1D_BRIGHT = 0.6        # a cell is "bright" within this fraction of the peak OF ITS OWN SIDE of the valley.
                         # THE FIRST KNOB OF RESTRICTIVENESS: 0.45 takes the whole ridge, 0.75 only its core.
P1D_MINCELL = 3          # a bright blob smaller than this many cells is not a region
P1D_MINGAL = 10          # ... and a region fewer than this many tracks ever enter is not one either
P1D_BRIDGE = 1           # epochs of gap a region may jump: a valley is only GRADED in some columns,
                         # and the ridge between two of them is the same ridge.  0 = no bridging,
                         # which leaves one-column regions no track can be inside twice.
P1D_NHIT  = 3            # THE SECOND: epochs a track has to spend inside a region to belong to it.  2 lets a
                         # galaxy that merely crosses the region join it; 4 asks it to live there.
P1D_FHIT  = 0.0          # THE THIRD: and at least this FRACTION of the epochs it exists for inside the
                         # region's own time window — so a long track is not favoured over a short one.
                         # 0 = off.  0.5 asks a galaxy to be inside the region for half of its stay.
P1D_PURE  = True         # a track that qualifies for BOTH regions belongs to NEITHER: it is a crosser, not a
                         # member, and it goes to the intermediates (the overlap is measured and printed
                         # BEFORE this is applied, so the diagnostic survives).  False = the first region
                         # wins, which is what a sequential peel does on its own.
P1D_SCAN  = True         # print what other (P1D_BRIGHT, P1D_NHIT) settings would give, before choosing
P1D_REG   = None         # None = find the two regions automatically (above the valley, then the residual);
                         # or give them by hand as [(t_lo, t_hi, y_lo, y_hi), (t_lo, t_hi, y_lo, y_hi)]
P1D_CAL   = ["cis100", "cis50"]     # THE BOXES THE REGIONS ARE CALIBRATED ON, pooled — and then the SAME
                         # CELLS are peeled off every box, so a box too small to find its own regions
                         # (m25 has 82 galaxies) is not asked to.  None = every box finds its own.
P1D_Q     = ["comp", "lsig_h2", "kappa_gas", "lssfr", "age", "lmdot"]    # what FIG 2 carries the three
                         # groups onto (the peeled quantity itself is always dropped from that list)
P1D_NAMES = ("dusty QGs", "dust-poor QGs", "intermediate")   # the three groups, and what every figure
                         # from here down calls them.  The FIRST is the upper region of the map, the
                         # second the lower one, the third what never entered either.
P1D_COL   = dict(zip(P1D_NAMES, ["#d7191c", "#2166ac", "#9970ab"]))
P1D_KEEP  = True         # write the membership into FR["grp_peel"] (Part 2 can adopt it: P2_ADOPT = "peel")
_G3 = list(P1D_NAMES)


def _col_kde(v, gy):
    """One epoch column: the KDE of v on the grid gy, scaled to its own maximum."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 5 or np.std(v) <= 0:
        return None
    try:
        d = gaussian_kde(v)
        d.set_bandwidth(d.factor * P1B_BW)
        y = d(gy)
    except Exception:                       # a degenerate column has no density
        return None
    m = float(np.max(y))
    return y / m if m > 0 else None


def _at_floor(q, v, tol):
    """Is this mode sitting on the quantity's known censoring floor rather than somewhere physical?"""
    f = P1B_FLOORS.get(q)
    return f is not None and abs(v - f) <= tol


def _col_modes(y, gy, q=None):
    """The modes of one column, the valley between the two highest and HOW DEEP it is as a fraction of the
    lower of the two peaks (smaller = cleaner; 1 = no valley at all).
    A peak has to stand P1B_PROM above the column's floor; two peaks are two modes only if the valley between
    them falls to P1B_DEEP of the lower peak.  Anything else is a shoulder."""
    if y is None:
        return [], np.nan, 1.0
    pk = np.flatnonzero((y[1:-1] > y[:-2]) & (y[1:-1] >= y[2:])) + 1
    pk = pk[y[pk] - y.min() >= P1B_PROM]
    _tol = 3.0 * float(gy[1] - gy[0])
    pk = np.array([i for i in pk if not _at_floor(q, gy[i], _tol)], int)   # a censoring floor is not a mode
    if len(pk) < 2:
        return [float(gy[i]) for i in pk], np.nan, 1.0
    a, b = sorted(pk[np.argsort(y[pk])[-2:]])
    lo = a + int(np.argmin(y[a:b + 1]))
    if y[lo] > P1B_DEEP * min(y[a], y[b]):
        return [float(gy[i]) for i in pk], np.nan, 1.0          # a shoulder, not a valley
    return [float(gy[i]) for i in pk], float(gy[lo]), float(y[lo] / min(y[a], y[b]))


def _label_cc(mask):
    """Connected components of a boolean mask, 4-connectivity.  scipy if it is there, a flood fill if not."""
    try:
        from scipy.ndimage import label as _lab
        return _lab(mask)[0]
    except Exception:
        out = np.zeros(mask.shape, int)
        cur = 0
        for s0 in range(mask.shape[0]):
            for s1 in range(mask.shape[1]):
                if mask[s0, s1] and not out[s0, s1]:
                    cur += 1
                    stack = [(s0, s1)]
                    while stack:
                        a, b = stack.pop()
                        if not (0 <= a < mask.shape[0] and 0 <= b < mask.shape[1]):
                            continue
                        if out[a, b] or not mask[a, b]:
                            continue
                        out[a, b] = cur
                        stack += [(a + 1, b), (a - 1, b), (a, b + 1), (a, b - 1)]
        return out


def _stage_dens(A, rows, gy, q=None):
    """The per-column KDE of the rows still in play — the SHAPE the regions are found on — together with the
    per-column valley (Part 1b's rule, on a column with enough galaxies) and the counts behind each column."""
    D = np.full((len(gy), len(TG)), np.nan)
    spl, n = np.full(len(TG), np.nan), np.zeros(len(TG), int)
    for j in range(len(TG)):
        v = A[rows, j]; v = v[np.isfinite(v)]
        n[j] = len(v)
        if len(v) >= max(5, P1B_NMIN):
            y = _col_kde(v, gy)
            if y is not None:
                D[:, j] = y
                if n[j] >= max(P1B_NMIN, P1B_NPER):
                    spl[j] = _col_modes(y, gy, P1D_ON if q is None else q)[1]
    return D, spl, n


def _bright(D, spl, gy, thr=None):
    """The bright cells, SIDE BY SIDE OF EACH COLUMN'S VALLEY: a cell is bright when it stands within
    P1D_BRIGHT of the peak OF ITS OWN SIDE.  A minor mode is therefore not erased by the major one — the
    upper ridge of a column whose lower ridge is ten times taller is still found — and the valley row itself
    is left out, so the two sides cannot be connected THROUGH a column that has one."""
    thr = P1D_BRIGHT if thr is None else thr
    B = np.zeros(D.shape, bool)
    for j in range(D.shape[1]):
        col = D[:, j]
        if not np.isfinite(col).any() or np.nanmax(col) <= 0:
            continue
        if np.isfinite(spl[j]):
            k = int(np.argmin(np.abs(gy - spl[j])))
            for sl in (slice(0, k), slice(k + 1, None)):
                seg = col[sl]
                if np.isfinite(seg).any() and np.nanmax(seg) > 0:
                    B[sl, j] = np.nan_to_num(seg) >= thr * np.nanmax(seg)
        else:
            B[:, j] = np.nan_to_num(col) >= thr * np.nanmax(col)
    return B


def _stage_cnt(A, rows, ye):
    """The galaxies per cell — the ABSOLUTE picture, so a region that empties looks empty."""
    C = np.zeros((len(ye) - 1, len(TG)))
    for j in range(len(TG)):
        v = A[rows, j]; v = v[np.isfinite(v)]
        C[:, j] = np.histogram(v, bins=ye)[0]
    return C


def _side_mask(spl, gy, side):
    """The cells above ("high") or below ("low") each column's valley; a column without one contributes
    nothing to "high" (there is no upper ridge to speak of there) and everything to "low"."""
    M = np.zeros((len(gy), len(TG)), bool)
    for j in range(len(TG)):
        if np.isfinite(spl[j]):
            M[:, j] = gy > spl[j] if side == "high" else gy < spl[j]
        elif side == "low":
            M[:, j] = True
    return M


def _pick_region(B, cnt, gy, taken=None):
    """The MOST POPULATED connected blob of bright cells — most populated, not highest or largest, because
    "the bright region" in the eye means the one most galaxies are in.  None when nothing is big enough."""
    m = B.copy()
    m[:, TG < P1D_T0] = False
    if taken is not None:
        m &= ~taken
    # the connectivity is worked out on a mask DILATED along the clock, so a ridge graded at +1.0 and at
    # +1.5 is one region and not two a track can never be inside twice; the cells themselves are unchanged.
    mb = m.copy()
    for _s in range(1, int(P1D_BRIDGE) + 1):
        mb[:, _s:] |= m[:, :-_s]; mb[:, :-_s] |= m[:, _s:]
    lab = np.where(m, _label_cc(mb), 0)
    best, bmask = -1.0, None
    for u in range(1, int(lab.max()) + 1):
        mm = lab == u
        if mm.sum() < P1D_MINCELL:
            continue
        mass = float(cnt[mm].sum())
        if mass > best:
            best, bmask = mass, mm
    return bmask


def _rect_mask(rect, gy):
    """A hand-given (t_lo, t_hi, y_lo, y_hi) as a mask on the same grid."""
    t0, t1, y0, y1 = rect
    M = np.zeros((len(gy), len(TG)), bool)
    M[np.ix_((gy >= y0) & (gy <= y1), (TG >= t0) & (TG <= t1))] = True
    return M


def _fill_gaps(M, B=None):
    """A region graded at +1.00 and +1.50 is ONE region, and the epoch between them belongs to it: every
    column the bridging jumped is filled across the y-range its two neighbours span.  Without this the
    region comes out as stripes — a band with holes in it — and a track can be inside it once per stripe."""
    if M is None or not M.any():
        return M
    out = M.copy()
    cols = np.flatnonzero(M.any(axis=0))
    for a, b in zip(cols[:-1], cols[1:]):
        if 1 < b - a <= int(P1D_BRIDGE) + 1:
            lo = min(np.flatnonzero(M[:, a]).min(), np.flatnonzero(M[:, b]).min())
            hi = max(np.flatnonzero(M[:, a]).max(), np.flatnonzero(M[:, b]).max())
            for j in range(a + 1, b):
                out[lo:hi + 1, j] = True
    return out


def _hits(A, mask, ye):
    """How many epochs each track spends inside the mask (a track is in the group at P1D_NHIT or more)."""
    n = np.zeros(len(A), int)
    for j in range(len(TG)):
        v = A[:, j]
        ok = np.isfinite(v)
        if not ok.any():
            continue
        ib = np.clip(np.digitize(v[ok], ye) - 1, 0, len(ye) - 2)
        n[np.flatnonzero(ok)] += mask[ib, j].astype(int)
    return n


# ── the regions: found once, on the calibration boxes, then peeled off every box ──────────────────────────
if P1D_ON not in P1_TRK:
    raise KeyError(f"P1D_ON = {P1D_ON!r} is not tracked.  P1_TRK = {P1_TRK}")
_pool = np.concatenate([B1[b]["T"][P1D_ON][P1B_CLOCK].ravel() for b in _BX1])
_pool = _pool[np.isfinite(_pool)]
_ylo, _yhi = np.nanpercentile(_pool, P1B_YLIM)
_YE = np.linspace(_ylo, _yhi, P1D_NY + 1)
_YC = 0.5 * (_YE[1:] + _YE[:-1])


def _tracks(box):
    """One box's tracks of the peeled quantity on the clock, over galaxies (or anchor rows)."""
    A = B1[box]["T"][P1D_ON][P1B_CLOCK]
    return A[B1[box]["F"]["uniq"].to_numpy(bool)] if P1B_UNIQ else A


def _inside(A, M, nhit=None, fhit=None):
    """Which tracks are IN a region: at least `nhit` epochs inside it, and at least `fhit` of the epochs the
    galaxy exists for inside the region's own time window (0 = that second condition is off)."""
    nhit = P1D_NHIT if nhit is None else nhit
    fhit = P1D_FHIT if fhit is None else fhit
    if M is None:
        return np.zeros(len(A), bool)
    n = _hits(A, M, _YE)
    ok = n >= nhit
    if fhit > 0:
        cols = M.any(axis=0)
        have = np.isfinite(A[:, cols]).sum(axis=1)
        ok &= n >= fhit * np.maximum(have, 1)
    return ok


def _find_regions(A, thr=None, nhit=None):
    """The two regions, peeled in order off whatever sample is handed in.
    STAGE 1 takes the bright cells ABOVE the valley, and only in the columns that HAVE one — that is what
    "the upper of two bright regions" means, and it is the same valley Part 1b grades and draws.
    STAGE 2 takes the most populated bright blob of WHAT IS LEFT, below any valley that survives."""
    masks, note, rows = [], [], np.ones(len(A), bool)
    taken = np.zeros((len(_YC), len(TG)), bool)
    for k, side in enumerate(("high", "low")):
        D, spl, nc = _stage_dens(A, rows, _YC)
        C = _stage_cnt(A, rows, _YE)
        B = _bright(D, spl, _YC, thr)
        if P1D_REG is not None:
            M = _rect_mask(P1D_REG[k], _YC)
        elif k == 0:
            M = _pick_region(B & _side_mask(spl, _YC, "high"), C, _YC) if np.isfinite(spl).any() else None
            if M is None:
                note.append("no bright region above a valley — the most populated bright blob instead")
                M = _pick_region(B, C, _YC)
            M = _fill_gaps(M, B)
        else:
            M = _fill_gaps(_pick_region(B & _side_mask(spl, _YC, "low"), C, _YC, taken=taken), B)
        if M is None:
            masks.append(None); continue
        n_in = int((_inside(A, M, nhit) & rows).sum())
        if n_in < P1D_MINGAL:
            note.append(f"the {side} region holds only {n_in} tracks (< P1D_MINGAL) — dropped")
            masks.append(None); continue
        taken = taken | M
        rows = rows & ~(_inside(A, M, nhit) & rows)
        masks.append(M)
    return masks, note


_CALB = [b for b in (P1D_CAL or []) if b in _BX1]
print(f"THE PEEL on {P1D_ON} over the {P1B_CLOCK} clock  —  regions after t = {P1D_T0:+g} Gyr, bright = "
      f"{P1D_BRIGHT:g} of the peak of its own side of the valley, a track joins at {P1D_NHIT} epochs inside")
if _CALB:
    _MASKS, _NOTE = _find_regions(np.vstack([_tracks(b) for b in _CALB]))
    print(f"  CALIBRATED on {' + '.join(_CALB)} pooled ({sum(len(_tracks(b)) for b in _CALB)} galaxies) and the "
          f"SAME CELLS peeled off every box — a box is never asked to find regions it has no statistics for."
          + ("   [" + "; ".join(_NOTE) + "]" if _NOTE else ""))
else:
    print("  each box finds its own regions (P1D_CAL = None)")

def _assign(A, masks, nhit=None, fhit=None, pure=None):
    """The three groups, and the two crossing fractions measured BEFORE the purity rule is applied."""
    pure = P1D_PURE if pure is None else pure
    has = np.isfinite(A[:, TG >= P1D_T0]).any(axis=1)
    inh = _inside(A, masks[0], nhit, fhit)
    inl = _inside(A, masks[1], nhit, fhit)
    xo = float(np.mean(inl[inh])) if inh.any() and masks[1] is not None else np.nan
    xo2 = float(np.mean(inh[inl])) if inl.any() and masks[0] is not None else np.nan
    both = inh & inl
    grp = np.array([""] * len(A), dtype=object)
    if pure:
        grp[inh & ~both] = _G3[0]
        grp[inl & ~both] = _G3[1]
    else:
        grp[inl] = _G3[1]
        grp[inh] = _G3[0]                       # the sequential peel: the first region wins
    grp[(grp == "") & has] = _G3[2]
    return grp, xo, xo2, int(both.sum())


# ── how restrictive to be: what the other settings would give ─────────────────────────────────────────────
if P1D_SCAN and _CALB:
    _Ac = np.vstack([_tracks(b) for b in _CALB])
    print(f"  HOW RESTRICTIVE?  the calibration pool under other settings — the row in [] is the one in force:")
    _w3 = max(10, max(len(str(g)) for g in _G3) + 2)          # the columns fit the group NAMES
    print(f"      {'bright':>8s} {'nhit':>5s}" + "".join(f"{g:>{_w3}s}" for g in _G3)
          + f"{'crossers':>10s}{'overlap':>9s}")
    for _thr in sorted({0.45, 0.6, 0.75, P1D_BRIGHT}):
        _mk, _ = _find_regions(_Ac, thr=_thr)
        for _nh in sorted({2, 3, 4, P1D_NHIT}):
            _g, _x, _x2, _nb = _assign(_Ac, _mk, nhit=_nh)
            _mark = " <-" if (_thr == P1D_BRIGHT and _nh == P1D_NHIT) else "   "
            print(f"      {_thr:8.2f} {_nh:5d}" + "".join(f"{int((_g == g).sum()):{_w3}d}" for g in _G3)
                  + f"{_nb:10d}" + (f"{_x * 100:8.0f} %" if np.isfinite(_x) else "      n/a") + _mark)
    print("      crossers = tracks that qualify for BOTH regions (P1D_PURE sends them to the intermediates);\n"
          "      overlap = the share of the upper group that also enters the lower region, measured BEFORE that.")

# ── the membership, box by box, with those cells ──────────────────────────────────────────────────────────
PEEL, ROW1D = {}, []
for _b in _BX1:
    _A = _tracks(_b)
    _masks, _note = (_MASKS, list(_NOTE)) if _CALB else _find_regions(_A)
    _grp, _xo, _xo2, _nb = _assign(_A, _masks)
    _rows, _st = np.ones(len(_A), bool), []
    for _k in range(2):
        _D, _spl, _nc = _stage_dens(_A, _rows, _YC)
        _st.append(dict(dens=_D, n=_nc, mask=_masks[_k], rows=_rows.copy()))
        _rows = _rows & (_grp != _G3[_k])
    _D3, _s3, _n3 = _stage_dens(_A, _rows, _YC)
    _st.append(dict(dens=_D3, n=_n3, mask=None, rows=_rows.copy()))
    PEEL[_b] = dict(A=_A, stages=_st, grp=_grp, masks=_masks, overlap=_xo)
    print(f"  {_b:9s} " + "  ".join(f"{g} {int((_grp == g).sum()):4d}" for g in _G3)
          + f"   unlabelled {int((_grp == '').sum()):4d} of {len(_A)}"
          + (f";  {_xo * 100:3.0f} % of the {_G3[0]} group ALSO enters the {_G3[1]} region"
             if np.isfinite(_xo) else
             ";  no " + ("upper" if _masks[0] is None else "lower") + " region")
          + (f", {_xo2 * 100:3.0f} % the other way" if np.isfinite(_xo2) else "")
          + (f";  {_nb} crossers -> the intermediates" if _nb and P1D_PURE else
             f";  {_nb} crossers kept in {_G3[0]}" if _nb else "")
          + ("   [" + "; ".join(_note) + "]" if _note and not _CALB else ""))
    for _g in _G3:
        ROW1D.append(dict(box=_b, group=_g, n=int((_grp == _g).sum()), overlap_hi_into_lo=_xo,
                          calibrated_on="+".join(_CALB) if _CALB else _b, quantity=P1D_ON, clock=P1B_CLOCK))
print("      THE OVERLAP IS THE POINT OF STAGE 2: a high fraction means the two bright regions are the SAME\n"
      "      galaxies caught at two times — one story, seen twice.  A low one means they are two populations.")

# ── FIG 1 — the three stages, and the three groups on the quantity they were peeled from ──────────────────
fig, AX = plt.subplots(len(_BX1), 4, figsize=(16.4, 2.9 * len(_BX1) + 0.7), squeeze=False)
for _r, _b in enumerate(_BX1):
    _P = PEEL[_b]
    _n1 = np.maximum(_P["stages"][0]["n"].astype(float), 1.0)
    for _k in range(3):
        ax = AX[_r, _k]
        _sh = _P["stages"][_k]["dens"] * (_P["stages"][_k]["n"] / _n1)[None, :]
        ax.set_facecolor("k")          # an empty cell is EMPTY, not white — white reads as bright
        ax.pcolormesh(TG, _YC, np.ma.masked_invalid(_sh), cmap=P1B_CMAP, shading="nearest", vmin=0, vmax=1)
        # the median of what is STILL IN PLAY at this stage, with its 16-84 spread dotted — the same line
        # the map carried before the peel, so the three panels can be read against each other
        _rw = _P["stages"][_k]["rows"]
        if int(_rw.sum()) >= 3:
            _sub = _P["A"][_rw]
            _ok3 = np.array([np.isfinite(_sub[:, j]).sum() >= 3 for j in range(len(TG))])
            _md = np.array([np.nanmedian(_sub[:, j]) if _ok3[j] else np.nan for j in range(len(TG))])
            _l1 = np.array([np.nanpercentile(_sub[:, j][np.isfinite(_sub[:, j])], 16) if _ok3[j] else np.nan
                            for j in range(len(TG))])
            _l2 = np.array([np.nanpercentile(_sub[:, j][np.isfinite(_sub[:, j])], 84) if _ok3[j] else np.nan
                            for j in range(len(TG))])
            ax.plot(TG, _md, color="#67e8f9", lw=1.8)
            ax.plot(TG, _l1, color="#67e8f9", lw=0.9, ls=":"); ax.plot(TG, _l2, color="#67e8f9", lw=0.9, ls=":")
        _M = _P["stages"][_k]["mask"]
        if _M is not None:
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=[P1D_COL[_G3[_k]]], linewidths=2.0)
        ax.axvline(P1D_T0, color="w", ls="--", lw=1.0, alpha=0.7)
        _nl = int((_P["grp"] == _G3[_k]).sum()) if _k < 2 else int((_P["grp"] == _G3[2]).sum())
        ax.text(0.02, 0.04, [f"stage 1: the whole map\n{_G3[0]} = {_nl} galaxies",
                             f"stage 2: {_G3[0]} removed\n{_G3[1]} = {_nl}",
                             f"stage 3: the residual\n{_G3[2]} = {_nl}"][_k],
                transform=ax.transAxes, va="bottom", ha="left", fontsize=8.5, color="w",
                bbox=dict(fc="k", ec="none", alpha=0.45, pad=1.8))
        ax.set_ylabel(f"{BOX_LABEL.get(_b, _b).split(' ')[0]}\n" + P1_LAB.get(P1D_ON, P1D_ON).split("  [")[0]
                      if _k == 0 else "")
        if _r == len(_BX1) - 1:
            ax.set_xlabel(f"${_CTX.get(P1B_CLOCK, P1B_CLOCK)}$  [Gyr]")
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(_YE[0], _YE[-1])
    ax = AX[_r, 3]
    for _g in _G3:
        _m = _P["grp"] == _g
        if _m.sum() < 3:
            continue
        _md = np.array([np.nanmedian(_P["A"][_m, j]) if np.isfinite(_P["A"][_m, j]).sum() >= 3 else np.nan
                        for j in range(len(TG))])
        _q1 = np.array([np.nanpercentile(_P["A"][_m, j][np.isfinite(_P["A"][_m, j])], 16)
                        if np.isfinite(_P["A"][_m, j]).sum() >= 3 else np.nan for j in range(len(TG))])
        _q2 = np.array([np.nanpercentile(_P["A"][_m, j][np.isfinite(_P["A"][_m, j])], 84)
                        if np.isfinite(_P["A"][_m, j]).sum() >= 3 else np.nan for j in range(len(TG))])
        ax.fill_between(TG, _q1, _q2, color=P1D_COL[_g], alpha=0.16, lw=0)
        ax.plot(TG, _md, color=P1D_COL[_g], lw=2.2, label=f"{_g}  ({int(_m.sum())})")
    ax.axvline(P1D_T0, color="k", ls="--", lw=1.0, alpha=0.5)
    ax.legend(fontsize=8, frameon=False, loc="lower left")
    ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(_YE[0], _YE[-1])
    if _r == 0:
        ax.set_title("the three groups", fontsize=10)
    if _r == len(_BX1) - 1:
        ax.set_xlabel(f"${_CTX.get(P1B_CLOCK, P1B_CLOCK)}$  [Gyr]")
for _k, _t in enumerate(["stage 1 — the whole map", "stage 2 — the first group removed",
                         "stage 3 — the residual"]):
    AX[0, _k].set_title(_t, fontsize=10)
fig.suptitle(f"Peeling the {P1_LAB.get(P1D_ON, P1D_ON).split('  [')[0]} map: the same colour scale at every "
             f"stage, so a region that empties looks empty", y=0.999, fontsize=11)
fig.tight_layout(rect=[0, 0.035, 1, 0.965], h_pad=2.2)
fig.legend(handles=[Line2D([], [], color="#67e8f9", lw=2, label="the median of what is still in play, "
                           "16-84 dotted"),
                    Line2D([], [], color=P1D_COL[_G3[0]], lw=2, label=f"the {_G3[0]} region (stage 1)"),
                    Line2D([], [], color=P1D_COL[_G3[1]], lw=2, label=f"the {_G3[1]} region (stage 2)")],
           loc="lower center", ncol=3, fontsize=9, frameon=False)
paper_save(fig, f"ba_peel_{P1D_ON}_{P1B_CLOCK}"); plt.show()

# ── FIG 2 — where the three groups end up on everything else ──────────────────────────────────────────────
_QD = [q for q in P1D_Q if q in P1_TRK and q != P1D_ON]
fig, AX = plt.subplots(len(_QD), len(_BX1), figsize=(4.0 * len(_BX1), 2.8 * len(_QD) + 0.8), squeeze=False)
for _r, _q in enumerate(_QD):
    _pl = np.concatenate([B1[b]["T"][_q][P1B_CLOCK].ravel() for b in _BX1])
    _pl = _pl[np.isfinite(_pl)]
    _lo2, _hi2 = np.nanpercentile(_pl, P1B_YLIM)
    for _k, _b in enumerate(_BX1):
        ax = AX[_r, _k]
        _T = B1[_b]["T"][_q][P1B_CLOCK]
        if P1B_UNIQ:
            _T = _T[B1[_b]["F"]["uniq"].to_numpy(bool)]
        _gp = PEEL[_b]["grp"]
        _fm = np.array([np.nanmedian(_T[:, j]) if np.isfinite(_T[:, j]).sum() >= 5 else np.nan
                        for j in range(len(TG))])
        ax.plot(TG, _fm, color="0.55", lw=1.4, ls=":")
        for _g in _G3:
            _m = _gp == _g
            if _m.sum() < 3:
                continue
            _md = np.array([np.nanmedian(_T[_m, j]) if np.isfinite(_T[_m, j]).sum() >= 3 else np.nan
                            for j in range(len(TG))])
            ax.plot(TG, _md, color=P1D_COL[_g], lw=2.2)
        _mh, _ml = (_gp == _G3[0]), (_gp == _G3[1])
        _au = np.array([mw_auc(np.where(_mh | _ml, _T[:, j], np.nan), _mh)[0] for j in range(len(TG))])
        _jb = int(np.nanargmax(np.abs(_au - 0.5))) if np.isfinite(_au).any() else None
        ax.text(0.02, 0.04, (f"AUC({_G3[0]} vs {_G3[1]}) {_au[_jb]:.2f} at {TG[_jb]:+.1f} Gyr"
                             if _jb is not None else "AUC n/a"), transform=ax.transAxes, va="bottom",
                ha="left", fontsize=8.5, bbox=dict(fc="w", ec="none", alpha=0.75, pad=1.6))
        ax.axvline(P1D_T0, color="k", ls="--", lw=1.0, alpha=0.4)
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(_lo2, _hi2)
        if _r == 0:
            ax.set_title(BOX_LABEL.get(_b, _b), fontsize=10)
        if _r == len(_QD) - 1:
            ax.set_xlabel(f"${_CTX.get(P1B_CLOCK, P1B_CLOCK)}$  [Gyr]")
        ax.set_ylabel(P1_LAB.get(_q, _q).split("  [")[0] if _k == 0 else "")
        ROW1D.append(dict(box=_b, group="auc", quantity=_q, clock=P1B_CLOCK,
                          auc_hi_lo=_au[_jb] if _jb is not None else np.nan,
                          t_best=float(TG[_jb]) if _jb is not None else np.nan))
fig.suptitle(f"Where the three {P1D_ON} groups end up — the field median dotted", y=0.999, fontsize=11)
fig.tight_layout(rect=[0, 0.04, 1, 0.965], h_pad=2.4)
fig.legend(handles=[Line2D([], [], color=P1D_COL[g], lw=2.4, label=f"{g}") for g in _G3]
           + [Line2D([], [], color="0.55", lw=1.4, ls=":", label="every galaxy in the box")],
           loc="lower center", ncol=4, fontsize=9, frameon=False)
paper_save(fig, f"ba_peel_{P1D_ON}_where"); plt.show()

# ── the membership, carried back to the branch table ──────────────────────────────────────────────────────
if P1D_KEEP:
    FR["grp_peel"] = ""
    for _b in _BX1:
        _F = B1[_b]["F"]
        _u = _F["uniq"].to_numpy(bool)
        _gid = _F["gid"].to_numpy()[_u] if "gid" in _F.columns else None
        _map = dict(zip(_gid, PEEL[_b]["grp"])) if _gid is not None else {}
        _all = np.array([_map.get(g, "") for g in _F["gid"].to_numpy()], dtype=object) if _map else \
            np.array([""] * len(_F), dtype=object)
        FR.loc[FR["box"] == _b, "grp_peel"] = _all
        B1[_b]["F"]["grp_peel"] = _all
    print(f"\nthe membership is in FR['grp_peel'] (anchors of one galaxy inherit it through the fingerprint): "
          + ", ".join(f"{g} {int((FR['grp_peel'] == g).sum())}" for g in _G3)
          + f", unlabelled {int((FR['grp_peel'] == '').sum())} branch rows")
    print(f"  SET  P2_ADOPT = \"peel\"  in Part 2 to run Parts 3-6 on them: {_G3[0]} becomes the 'burst' key,\n"
          f"  {_G3[1]} the 'aging' one and {_G3[2]} is left out of both, the way a tercile middle is.")
pd.DataFrame(ROW1D).to_csv(os.path.join(OUT, f"ba_peel_{P1D_ON}_{P1B_CLOCK}.csv"), index=False)
if P1D_KEEP:
    FR.loc[FR["uniq"], KEY + ["grp_peel"]].to_csv(os.path.join(OUT, f"ba_peel_members.csv"), index=False)
print("tables ->", os.path.join(OUT, f"ba_peel_{P1D_ON}_{P1B_CLOCK}.csv"),
      "and", os.path.join(OUT, "ba_peel_members.csv"))

## Part 1c — does the selection divide the quenching time?

The selection was made on the **dust** map and knows nothing about how fast the galaxy quenched.
$\log_{10}(\tau_q/t_{\rm QT})$ — the quenching time in units of the epoch it happened at — is the axis
SIMBA's fast and slow modes live on, and it is the one distribution in this notebook that is genuinely
bimodal: its two modes survive Silverman's critical-bandwidth test where nothing else here does. So the
question this cell asks is whether the three groups fall on **different parts of it**, or whether the peel
has cut straight across it.

Three answers, and the figure is only one of them:

- **the distributions** — the axis drawn per group against the whole box in grey. Does each group sit on one
  of the two modes, or does each straddle both?
- **the fast share** — the fraction of each group below `FAST_CUT`, with a Beta-posterior 68 % band, against the
  box's own rate;
- **the score** — AUC(upper vs lower) raw, and again **inside (epoch × mass) cells**, because the groups
  differ in cosmic time and in mass whatever they are divided on.

And the reverse, printed underneath: what the fast and the slow quenchers are each *made of*.

**An AUC of 0.5 with the same fast share in both groups is a result, not a failure**: it would say the dust
map and the quenching mode are independent axes, and that the groups of Part 1b cannot be re-labelled as
fast and slow quenchers. `P1C_X` points the whole cell at another column (`t_feed`, `tau_q`, …) and
`BA_TEST=1` adds Silverman's p on the axis itself.

In [ ]:
# ── Part 1c — DOES THE SELECTION DIVIDE THE QUENCHING TIME? ────────────────────────────────────────────────
# The selection was made on the DUST map and knows nothing about how fast the galaxy quenched.
# log10(tau_q / t_QT) — the quenching time in units of the epoch it happened at — is the axis SIMBA's fast
# and slow modes live on, and it is the one distribution in this notebook that is genuinely bimodal (its two
# modes survive Silverman's test where nothing else does).  So the question is whether the three groups fall
# on different parts of it, or whether the peel has cut straight across it.
#
# Three answers, none of which is the figure alone:
#   THE DISTRIBUTIONS   the axis drawn per group, against the whole box in grey — does each group sit on one
#                       of the two modes, or does each straddle both?
#   THE FAST SHARE      the fraction of each group below FAST_CUT, with its Beta-posterior 68 % band
#   THE SCORE           AUC(upper vs lower) raw, and AGAIN inside (epoch x mass) cells, because the groups
#                       differ in cosmic time and in mass and tau_q/t_QT is normalised by the epoch already
P1C_X     = "ltauq"      # the axis tested; any column of FR ("tau_q", "t_feed", "lt_feed" also work)
P1C_FAST  = FAST_CUT     # the fast / slow cut drawn and scored against
P1C_BW    = None         # the KDE bandwidth of the drawn curves (None = Scott)
P1C_NB    = 26           # histogram bins
P1C_MINN  = 8            # a group is drawn and scored only above this many galaxies
P1C_TEST  = bool(int(os.environ.get("BA_TEST", "0")))   # run Silverman on the whole axis as well (~1 min)
ROW1C = []

_S1C = FR[FR["uniq"]] if P1B_UNIQ else FR
if "grp_peel" not in _S1C.columns:
    raise KeyError("Part 1c needs the selection: run Part 1b (it writes FR['grp_peel']).")
if P1C_X not in _S1C.columns:
    raise KeyError(f"P1C_X = {P1C_X!r} is not a column of FR.")
_XL = {"ltauq": r"$\log_{10}\, \tau_q / t_{\rm QT}$", "tau_q": r"$\tau_q$  [Gyr]",
       "t_feed": r"$t_{\rm feed}$  [Gyr]", "lt_feed": r"$\log_{10}\, t_{\rm feed}$"}.get(P1C_X, P1C_X)

print(f"DOES THE SELECTION DIVIDE {P1C_X}?   the peel was made on {P1D_ON}, this axis is the quenching mode")
_w1c = max(12, max(len(str(g)) for g in _G3) + 2)             # ... and so do these
print(f"    {'box':9s} " + "".join(f"{g:>{_w1c}s}" for g in _G3)
      + f"{'AUC hi-lo':>11s}{'| cells':>9s}{'p':>9s}   fast share per group")
for _b in _BX1:
    _s = _S1C[(_S1C["box"] == _b) & np.isfinite(_S1C[P1C_X].to_numpy(float))]
    _g = {g: _s[_s["grp_peel"] == g] for g in _G3}
    _med = [np.nanmedian(_g[g][P1C_X].to_numpy(float)) if len(_g[g]) >= P1C_MINN else np.nan for g in _G3]
    _hi, _lo = _g[_G3[0]], _g[_G3[1]]
    if len(_hi) >= P1C_MINN and len(_lo) >= P1C_MINN:
        _two = pd.concat([_hi, _lo])
        _m1 = (_two["grp_peel"] == _G3[0]).to_numpy()
        _a, _p, _, _ = mw_auc(_two[P1C_X].to_numpy(float), _m1)
        _ac, _pc, _, _ = _auc_ctrl(_two, P1C_X, _m1)
    else:
        _a = _p = _ac = _pc = np.nan
    _fs = []
    for g in _G3:
        _v = _g[g][P1C_X].to_numpy(float)
        _f, _l, _h = wilson(int((_v < P1C_FAST).sum()), len(_v)) if len(_v) else (np.nan,) * 3
        _fs.append((g, _f, _l, _h, len(_v)))
        ROW1C.append(dict(box=_b, group=g, n=len(_v), median=np.nanmedian(_v) if len(_v) else np.nan,
                          fast=_f, fast_lo=_l, fast_hi=_h, auc=_a, auc_ctrl=_ac, p=_p, axis=P1C_X))
    print(f"    {_b:9s} " + "".join(f"{m:{_w1c}.2f}" for m in _med)
          + (f"{_a:11.2f}{_ac:9.2f}{_p:9.1g}" if np.isfinite(_a) else f"{'n/a':>11s}{'':9s}{'':9s}")
          + "   " + "  ".join(f"{g} {f * 100:3.0f}% [{l * 100:.0f}-{h * 100:.0f}] ({n})"
                              if np.isfinite(f) else f"{g} -" for g, f, l, h, n in _fs))
print(f"      the medians per group, then AUC(the {_G3[0]} group vs the {_G3[1]} one) raw and inside "
      f"({' x '.join(CTRL_COLS)}) cells,\n"
      f"      then the share of each group that is a FAST quencher ({P1C_X} < {P1C_FAST:g}), Beta-posterior 68 % band.")
print("      AUC 0.5 AND THE SAME FAST SHARE IN BOTH GROUPS = the selection does not divide this axis at all:\n"
      "      the dust map and the quenching mode are independent, and that is a result, not a failure.")

# ── the reverse: what are the fast quenchers made of? ─────────────────────────────────────────────────────
print(f"\nAND THE OTHER WAY ROUND — the composition of each side of {P1C_X} = {P1C_FAST:g}:")
for _b in _BX1:
    _s = _S1C[(_S1C["box"] == _b) & np.isfinite(_S1C[P1C_X].to_numpy(float)) & (_S1C["grp_peel"] != "")]
    if len(_s) < 3 * P1C_MINN:
        print(f"  {_b:9s} too few"); continue
    _f = _s[P1C_X].to_numpy(float) < P1C_FAST
    for _lab, _m in (("fast", _f), ("slow", ~_f)):
        if not _m.any():
            print(f"  {_b:9s} {_lab:4s} (N    0): nothing on this side of the cut"); continue
        print(f"  {_b:9s} {_lab:4s} (N {int(_m.sum()):4d}): "
              + "  ".join(f"{g} {float(np.mean(_s['grp_peel'].to_numpy()[_m] == g)) * 100:3.0f} %" for g in _G3))

# ── FIG — the distribution, and the fast share ────────────────────────────────────────────────────────────
fig, AX = plt.subplots(2, len(_BX1), figsize=(4.0 * len(_BX1), 6.4), squeeze=False)
for _k, _b in enumerate(_BX1):
    _s = _S1C[(_S1C["box"] == _b) & np.isfinite(_S1C[P1C_X].to_numpy(float))]
    _x = _s[P1C_X].to_numpy(float)
    ax = AX[0, _k]
    if len(_x) > 5:
        _bins = np.linspace(np.nanpercentile(_x, 1), np.nanpercentile(_x, 99), P1C_NB)
        ax.hist(_x, bins=_bins, density=True, color="0.85", edgecolor="0.65", lw=0.6)   # the whole box
        _gx = np.linspace(_bins[0], _bins[-1], 400)
        for _g in _G3:                                    # and the three groups, as KDEs only
            _v = _s.loc[_s["grp_peel"] == _g, P1C_X].to_numpy(float)
            if len(_v) < P1C_MINN:
                continue
            try:
                ax.plot(_gx, gaussian_kde(_v, bw_method=P1C_BW)(_gx), color=P1D_COL[_g], lw=3.0)
            except Exception:
                pass
        ax.axvline(P1C_FAST, color="#2ca02c", ls="--", lw=1.6)
        ax.text(P1C_FAST, ax.get_ylim()[1] * 0.97, "  fast | slow", color="#2ca02c", fontsize=8.5, va="top")
        if P1C_TEST:
            _pv, _hv = _silverman(_x, 1, nmin=20)
            _dv = _dip(_x, P1C_BW)[0]
            ax.text(0.02, 0.97, f"Silverman p = {_pv:.3f}" + ("  <- TWO MODES" if _pv < 0.10 else "")
                    + (f"\ndip {_dv:+.2f}" if np.isfinite(_dv) else ""), transform=ax.transAxes,
                    va="top", ha="left", fontsize=8.5, color=("#b2182b" if _pv < 0.10 else "k"))
    ax.set_title(BOX_LABEL.get(_b, _b), fontsize=10)
    ax.set_xlabel(_XL); ax.set_ylabel("density" if _k == 0 else "")
    ax = AX[1, _k]
    _r = [r for r in ROW1C if r["box"] == _b]
    _xs, _ys, _el, _eh, _cl = [], [], [], [], []
    for _i, _g in enumerate(_G3):
        _rr = [r for r in _r if r["group"] == _g]
        if not _rr or not np.isfinite(_rr[0]["fast"]):
            continue
        _xs.append(_i); _ys.append(_rr[0]["fast"]); _el.append(_rr[0]["fast"] - _rr[0]["fast_lo"])
        _eh.append(_rr[0]["fast_hi"] - _rr[0]["fast"]); _cl.append(P1D_COL[_g])
    if _xs:
        ax.bar(_xs, _ys, color=_cl, width=0.62, yerr=[_el, _eh], capsize=4, ecolor="0.3")
        _all = float(np.mean(_s[P1C_X].to_numpy(float) < P1C_FAST))
        ax.axhline(_all, color="k", ls=":", lw=1.4)
        ax.text(0.02, _all, f" the box: {_all * 100:.0f} %", transform=ax.get_yaxis_transform(),
                ha="left", va="bottom", fontsize=8.5)
    ax.set_xticks(range(len(_G3))); ax.set_xticklabels(_G3)
    ax.set_ylim(0, 1.05); ax.set_ylabel("fast quenchers" if _k == 0 else "")
fig.suptitle(f"Does the {P1D_ON} selection divide {P1C_X}?  the three groups against the whole box (grey)",
             y=0.999, fontsize=11)
fig.tight_layout(rect=[0, 0.04, 1, 0.965], h_pad=3.0)
fig.legend(handles=[Line2D([], [], color=P1D_COL[g], lw=2.4, label=g) for g in _G3]
           + [Line2D([], [], color="0.85", lw=8, label="the whole box"),
              Line2D([], [], color="#2ca02c", ls="--", lw=1.6, label=f"{P1C_X} = {P1C_FAST:g}")],
           loc="lower center", ncol=5, fontsize=9, frameon=False)
paper_save(fig, f"ba_peel_{P1C_X}"); plt.show()
pd.DataFrame(ROW1C).to_csv(os.path.join(OUT, f"ba_peel_{P1C_X}.csv"), index=False)
print("\ntable ->", os.path.join(OUT, f"ba_peel_{P1C_X}.csv"))

## Part 2 — the division

**By default this cell does one thing: it adopts Part 1b's selection.** `P2_ADOPT = "peel"` turns the three
groups into the two keys the rest of the notebook draws with — group 1 → `burst`, group 2 → `aging`, the
intermediates in neither, the way a tercile middle is — and Parts 3 to 6 follow from there. Nothing else in
the cell runs.

The rest of it is a **browser** you switch to by putting a spec in `P2_ADOPT` instead, when you want to see
what a scalar axis would have done. An axis is one string, evaluated on the branch series Part 1 kept in
memory, so a new one costs a second and never re-opens an HDF5 file:

| written as | means |
|---|---|
| `"comp @ sft"` | $q$ read at one epoch |
| `"comp @ sft > qt+0.5"` | the **change** $q(B) - q(A)$ between two epochs (the default for a pair) |
| `"lsig_h2 @ jet > sft : max"` | a window statistic: `max`, `min`, `mean`, `rng`, `slope` (per Gyr), `d` |
| `"ltauq"`, `"comp_c"`, `"comp_dw"` | any column of `FR`, taken as it stands |

$q$ is any tracked quantity (`P1_TRK`) and an epoch is any event Part 1 timed, with an offset in Gyr if
wanted: `sft`, `qt+0.5`, `jet-0.3`, `x1`, `ign`, `acc`, `anc`, `first`. `P2_SHOW` is the list drawn — one
row of the figure each, with its KDE, its interior minimum and its median — and `P2_CUT` is where the cut
goes: `"dip"`, `"median"`, `"tercile"`, `"q70"`, a number you type, a `(low, high)` **region** that leaves
the middle in neither group, or a dict of those per box. The printed table gives $p_5$, the terciles, the
median and $p_{95}$ of every axis in every box precisely so a number can be read off it and typed back in.

`"burst"` is always the HIGH side of whatever is adopted and `"aging"` the LOW one; `P2_LABELS` renames them
in every figure downstream.

In [ ]:
# ── Part 2 — THE DIVISION: the distributions, their KDEs, and a cut YOU choose ─────────────────────────────
# This cell is a BROWSER first and a division second.  Nothing about the division is wired in: every candidate
# axis is ONE SPEC STRING, evaluated on the branch series Part 1 kept in memory, drawn with its KDE and its
# interior minimum, and the cut is put wherever P2_CUT says — the KDE minimum, the median, the terciles, a
# percentile, a number you type after looking at FIG 1, or a (low, high) pair that leaves the middle out.
# Whether any of it is BIMODAL is a separate question with a separate answer: Part 2c, after Part 2b.
#
# ── THE SPEC LANGUAGE ─────────────────────────────────────────────────────────────────────────────────────
#   "<q> @ <epoch>"                        q read at that epoch
#   "<q> @ <epochA> > <epochB>"            the CHANGE q(B) - q(A) across the pair   (the default for a pair)
#   "<q> @ <epochA> > <epochB> : <op>"     op = d (the change) | max | min | mean | rng (max - min) | slope (/Gyr)
#   "<column>"                             any column of FR as it stands: ltauq, comp_c, comp_cx, t_feed,
#                                          f_burst, dt_peak, lsig_h2_pk, comp_dw, comp_qt, ...
#   <q>      one of P1_TRK: lsig_h2, lsig_gas, lr_gas, comp, kappa_gas, kz, lfgas, lfd, lfh2, lssfr, lms
#   <epoch>  an event of P1_EV, optionally shifted in Gyr:  sft   qt+0.5   jet-0.3   x1   ign   acc   gate
#            big   anc   first     (the same names Part 1's window knob P1_WIN takes)
#
# "comp @ sft > qt+0.5" is therefore exactly HOW MUCH -log R50,gas/R50,* CHANGED between the truncation and half
# a gigayear after the quenching time: positive = the cold gas shrank relative to the stars over that stretch.
# A point read needs a snapshot within P2_TOL of its epoch or it is NaN — the coverage of every axis is printed
# below, and P2_CLAMP rescues epochs that fall off the end of a branch.
P2_SHOW  = ["comp @ sft > qt+0.5",              # <- the division, by default: how much the gas contracted
            "comp @ sft",                       #    where it started
            "comp_c",                           #    C, the same window read as a contraction against its minimum
            "lsig_h2 @ jet > sft : max",        #    the old axis: the peak H2 density of the jet leg
            "ltauq"]                            #    the reference: SIMBA's fast / slow quenching axis
P2_ADOPT = "peel"        # WHAT DIVIDES THE SAMPLE.  "peel" = Part 1b's three groups, which is what the
                         # rest of this notebook is built on; any spec above puts the browser back instead.
# ── WHERE THE CUT GOES.  This is the knob to reach for after looking at FIG 1. ────────────────────────────
#   "dip"      the KDE's interior minimum, whatever the bimodality test says (the median if there is none)
#   "median" | "tercile" | "q70"     the median, the two terciles (middle dropped), any percentile
#   0.05                             a number you type: high group above it, low group at or below
#   (-0.02, 0.08)                    a REGION: low group at or below -0.02, high group at or above +0.08,
#                                    everything between them dropped from both
#   {"cis100": 0.05, "cis25": (0.0, 0.1), "cis50": "median"}   any of the above, per box
P2_CUT    = "dip"
P2_PERBOX = True          # a cut per box (the boxes differ in resolution, so most axes are offset between them)
P2_UNIQ   = True          # the distributions are over GALAXIES, not anchors
P2_TOL    = P1_TOL        # Gyr: a point read takes the nearest snapshot within this of its epoch, else NaN
P2_CLAMP  = False         # True = an epoch that falls off the end of a branch is read AT the end instead of NaN
P2_SHORT  = None          # the two names in their SHORT form, for the printed tables (None = P2_LABELS cut)
P2_LABELS = None          # ("high side", "low side") renames the two groups in every figure from here down.
                          # The Part 0 names are "starburst" / "aging" and they belong to an H2-DENSITY axis —
                          # say what the groups are when the division is on something else, in a few words:
                          # P2_LABELS = ("the gas contracted", "the gas spread out")
P2_ZCTRL  = False         # True = cut the axis RANKED INSIDE (epoch x mass) cells, not its raw value
P2_MINFRAC = 0.15         # "dip" only: a KDE minimum that leaves less than this share on a side is refused and
                          # the median is used instead (and said out loud).  None = take the dip wherever it is.
P2_BW     = None          # the KDE bandwidth (None = Scott)
P2_NGRID  = 512
P2_MINN   = 20            # no KDE and no dip below this many galaxies
P2_TEST   = bool(int(os.environ.get("BA_TEST", "0")))   # ALSO run Silverman on each axis drawn (env BA_TEST=1)
P2_SILV_K, P2_SILV_N, P2_SILV_P = 1, int(os.environ.get("BA_SILV", "200")), 0.10

# ── the axis labels of the columns that are NOT specs (a spec labels itself out of P1_LAB) ───────────────
P2_DLAB = {"comp_c": r"$C=\log R_{50,\rm gas}$(window open)$/R^{\rm min}_{50,\rm gas}$  [dex]",
           "comp_c0": r"$C_0$, the pre-window null  [dex]", "comp_cx": r"$C-C_0$  [dex]",
           "ltauq": r"$\log_{10}\,\tau_q/t_{\rm QT}$", "f_burst": r"$f_{\rm burst}$",
           "dt_peak": r"$t_{\rm peak}-t_{\rm SFT}$  [Gyr]", "dt_min": r"$t_{\rm min}-t_{\rm SFT}$  [Gyr]",
           "t_feed": r"$t_{\rm feed}$, the width of " + WIN_TEX + "  [Gyr]",
           "lt_feed": r"$\log_{10}\, t_{\rm feed}$", "lfeed_n": r"$\log_{10}\, t_{\rm feed}/t_{\rm QT}$",
           "log_mstar": r"$\log M_\star$", "age": "mass-weighted stellar age  [Gyr]"}

# ══ the machinery.  Nothing below needs editing to ask a different question — change the strings above. ════
import re

# what the suffix of a Part 1 column means, so that "comp_dw" and "lsig_h2_pk" can label themselves too
_SUFTX = {"jet": r"at $t_{\rm jet}$", "sft": r"at $t_{\rm SFT}$", "x1": r"at $t_{\rm x1}$",
          "qt": r"at $t_{\rm QT}$", "anc": "at the anchor", "pk": "peak over " + WIN_TEX,
          "mn": "minimum over " + WIN_TEX, "w0": "at the opening of " + WIN_TEX,
          "w1": "at the close of " + WIN_TEX, "dw": "across " + WIN_TEX}


def _ep(s):
    """'qt+0.5' -> ('qt', 0.5).  The event has to be one Part 1 timed."""
    m = re.match(r"^([A-Za-z_]\w*)\s*([+-]\s*[\d.]+)?$", str(s).strip())
    if not m or m.group(1) not in P1_EV:
        raise ValueError(f"{s!r} is not an epoch.  The events are {sorted(P1_EV)}, each with an optional offset "
                         f"in Gyr: 'sft', 'qt+0.5', 'jet-0.3'.")
    return m.group(1), (float(m.group(2).replace(" ", "")) if m.group(2) else 0.0)


def _spec(s):
    """A spec string -> (the column it is stored in, quantity, epoch A, epoch B, op).  A plain column of FR
    passes straight through and is never recomputed."""
    s = str(s).strip()
    if "@" not in s:
        if s in FR.columns:
            return s, None, None, None, None
        raise KeyError(f"{s!r} is neither a column of FR nor a spec.  Write '<q> @ <epoch>' or "
                       f"'<q> @ <epochA> > <epochB> [: op]' with q in {P1_TRK}.")
    q, rest = (p.strip() for p in s.split("@", 1))
    op = None
    if ":" in rest:
        rest, op = (p.strip() for p in rest.split(":", 1))
    if q not in P1_TRK:
        raise KeyError(f"{q!r} is not a tracked quantity.  P1_TRK = {P1_TRK}  (or name a column of FR outright).")
    _p = [p for p in re.split(r"->|\.\.|>", rest) if p.strip()]
    a = _ep(_p[0]); b = _ep(_p[1]) if len(_p) > 1 else None
    op = (op or ("d" if b else "val")).lower()
    if (b is None) != (op == "val"):
        raise ValueError(f"{s!r}: 'val' takes one epoch and {('d', 'max', 'min', 'mean', 'rng', 'slope')} take two.")
    _w = lambda e: e[0] + (f"{e[1]:+g}" if e[1] else "")
    return (f"{q}@{_w(a)}" if b is None else f"{q}@{_w(a)}>{_w(b)}:{op}"), q, a, b, op


_ETEX = lambda e: _EVTEX.get(e[0], e[0]) + (f"{e[1]:+g}" if e[1] else "")
_num = lambda v: ("     --" if not np.isfinite(v) else
                  f"{v:+8.3f}" if v == 0 or 1e-3 <= abs(v) < 1e4 else f"{v:+8.1e}")


def _spec_lab(s):
    """The axis label a spec deserves, built out of Part 1's own labels."""
    col, q, a, b, op = _spec(s)
    if q is None:                                   # a plain column: its own label, or one built from the suffix
        if col in P2_DLAB or col in P1_LAB:
            return P2_DLAB.get(col) or P1_LAB[col]
        _q, _, _sf = col.rpartition("_")
        if _q in P1_TRK and _sf in _SUFTX:
            return (r"$\Delta$" if _sf == "dw" else "") + P1_LAB[_q].split("  [")[0] + "   " + _SUFTX[_sf]
        return col
    base = P1_LAB.get(q, q).split("  [")[0]
    if b is None:
        return base + f"   at ${_ETEX(a)}$"
    if op == "d":
        return r"$\Delta$" + base + f"   from ${_ETEX(a)}$ to ${_ETEX(b)}$"
    if op == "slope":
        return r"$\mathrm{d}$" + base + f"$/\\mathrm{{d}}t$   over $[{_ETEX(a)},\\, {_ETEX(b)}]$  [/Gyr]"
    return {"max": "peak of ", "min": "minimum of ", "mean": "mean of ", "rng": "range of "}[op] + \
        base + f"   over $[{_ETEX(a)},\\, {_ETEX(b)}]$"


def _read1(t, y, te):
    """y at the snapshot nearest te — within P2_TOL of it, or (P2_CLAMP) at the end of the branch it fell off."""
    if not np.isfinite(te):
        return np.nan
    ok = np.isfinite(y)
    if not ok.any():
        return np.nan
    tt, yy = t[ok], y[ok]
    i = int(np.argmin(np.abs(tt - te)))
    if abs(tt[i] - te) <= P2_TOL or (P2_CLAMP and not (tt[0] <= te <= tt[-1])):
        return float(yy[i])
    return np.nan


def _axis(spec, D=None):
    """Evaluate a spec over the branch table and cache it as a column named after the spec; returns the name.
    The series are the ones Part 1 kept, so this never opens a file."""
    D = FR if D is None else D
    col, q, a, b, op = _spec(spec)
    if col in D.columns:
        return col
    _tv = lambda e: (D[P1_EV[e[0]]].to_numpy(float) + e[1]) if P1_EV[e[0]] in D.columns else np.full(len(D), np.nan)
    ta, tb = _tv(a), (_tv(b) if b else None)
    bx, ri = D["box"].to_numpy(), D["ridx"].to_numpy(int)
    out = np.full(len(D), np.nan)
    for i in range(len(D)):
        Rb = B1.get(bx[i], {}).get("R")
        if Rb is None or ri[i] >= len(Rb):
            continue
        S = Rb[ri[i]]
        t, y = S["t"], S.get(q)
        if y is None:
            continue
        if b is None:
            out[i] = _read1(t, y, ta[i])
        elif op in ("d", "slope"):
            v0, v1 = _read1(t, y, ta[i]), _read1(t, y, tb[i])
            out[i] = (v1 - v0) if op == "d" else ((v1 - v0) / (tb[i] - ta[i]) if tb[i] != ta[i] else np.nan)
        else:
            lo, hi = (ta[i], tb[i]) if ta[i] <= tb[i] else (tb[i], ta[i])
            m = np.isfinite(y) & (t >= lo) & (t <= hi)
            if m.any():
                out[i] = {"max": np.max, "min": np.min, "mean": np.mean,
                          "rng": lambda v: np.max(v) - np.min(v)}[op](y[m])
    D[col] = out
    if b is not None:
        D[col + " | dt"] = tb - ta          # the width of THIS axis' own window, galaxy by galaxy
    return col


_BIMO_MEMO = {}


def _bimo(x, key, test=False):
    """The KDE of one sample, its mode count, its interior minimum and its quantiles — and Silverman's p, but
    ONLY when `test` asks for it and P2_TEST is on.  The KDE part is cheap and is what Part 2 draws; the
    bootstrap is the whole cost of this notebook and belongs to Part 2c.  Memoised per (box, column), so an
    axis that is both the division and a row of the battery is paid for once."""
    out = _BIMO_MEMO.get(key)
    if out is None:
        x = np.asarray(x, float); x = x[np.isfinite(x)]
        if len(x) < P2_MINN:
            out = dict(n=len(x), p=np.nan, h=np.nan, dip=np.nan, npk=0)
        else:
            _d, _gx, _dn, _npk = _dip(x, P2_BW)
            _q = np.nanpercentile(x, [5, 100 / 3, 50, 200 / 3, 95])
            out = dict(n=len(x), p=np.nan, h=np.nan, dip=_d, npk=_npk, gx=_gx, dn=_dn, x=x,
                       p05=_q[0], lo=_q[1], med=_q[2], hi=_q[3], p95=_q[4])
        _BIMO_MEMO[key] = out
    if test and P2_TEST and out["n"] >= P2_MINN and not np.isfinite(out["p"]):
        out["p"], out["h"] = _silverman(out["x"], P2_SILV_K, P2_SILV_N, nmin=P2_MINN)
    return out


def _cut_rule(d, rule):
    """One box's KDE summary and a rule -> (low edge, high edge, how it was chosen).  Equal edges = a plain
    split at that value; different edges = a region, with everything between them left out of both groups."""
    if isinstance(rule, (list, tuple)) and len(rule) == 2:
        return float(rule[0]), float(rule[1]), "the region you typed"
    if isinstance(rule, (int, float)) and not isinstance(rule, bool):
        return float(rule), float(rule), "the value you typed"
    s = str(rule).strip().lower()
    if not np.isfinite(d.get("med", np.nan)):
        return np.nan, np.nan, f"nothing to cut (N = {d['n']})"
    if s == "median":
        return d["med"], d["med"], "the median"
    if s == "tercile":
        return d["lo"], d["hi"], "the terciles, the middle third in neither group"
    if re.match(r"^q[\d.]+$", s):
        v = float(np.nanpercentile(d["x"], float(s[1:])))
        return v, v, f"the {float(s[1:]):g}th percentile"
    if s in ("dip", "dipraw"):
        if not np.isfinite(d["dip"]):
            return d["med"], d["med"], "THE MEDIAN — this KDE has no interior minimum to cut at"
        sh = float(np.mean(d["x"] > d["dip"]))
        if P2_MINFRAC is not None and min(sh, 1 - sh) < P2_MINFRAC:
            return d["med"], d["med"], (f"THE MEDIAN — the dip at {d['dip']:+.3g} splits {sh * 100:.0f}/"
                                        f"{(1 - sh) * 100:.0f} %, past P2_MINFRAC")
        return d["dip"], d["dip"], f"the KDE minimum ({sh * 100:.0f}/{(1 - sh) * 100:.0f} % split)"
    raise ValueError(f"P2_CUT = {rule!r} is not a rule.  Use a number, a (low, high) pair, 'dip', 'median', "
                     f"'tercile', 'qNN', or a dict of those keyed by box.")


# ── THE PEEL AS A DIVISION: Part 1d's three groups, when P2_ADOPT asks for them ───────────────────────────
# The peel is a region of the (epoch x value) map, not a scalar, so it cannot be written as a spec.  It is
# carried here as a code — 2 the upper group, 1 the intermediates, 0 the lower one — and cut as a REGION, so
# the intermediates fall out of both groups exactly the way a tercile middle does.
_PEEL2 = str(P2_ADOPT).strip().lower() == "peel"
if _PEEL2:
    if "grp_peel" not in FR.columns:
        raise KeyError("P2_ADOPT = 'peel' needs Part 1d to have run — it is what writes FR['grp_peel'].")
    _pc = {P1D_NAMES[0]: 2.0, P1D_NAMES[2]: 1.0, P1D_NAMES[1]: 0.0}
    FR["peel"] = np.array([_pc.get(g, np.nan) for g in FR["grp_peel"].to_numpy()], float)
    P2_ADOPT, P2_CUT, P2_PERBOX, P2_SHOW = "peel", (0.5, 1.5), False, []
    P2_DLAB["peel"] = f"Part 1d's peel of {P1D_ON}   (2 = {P1D_NAMES[0]}, 1 = {P1D_NAMES[2]}, 0 = {P1D_NAMES[1]})"
    if P2_LABELS is None:
        P2_LABELS = (str(P1D_NAMES[0]), str(P1D_NAMES[1]))     # the names Part 1b gave them, as they are
        P2_SHORT = tuple(P1D_NAMES[:2])
    print(f"THE DIVISION IS PART 1b's PEEL: {P1D_NAMES[0]} -> the 'burst' key, {P1D_NAMES[1]} -> 'aging', "
          f"{P1D_NAMES[2]} in neither;  P2_CUT forced to (0.5, 1.5) and P2_PERBOX to False.\n")

# THE PAPER COLOURS.  With the peel adopted, every figure meant for the paper (Parts 3b, 10-15) draws the two
# keys in the PEEL's own red and blue, so a region on the map and a track are the same colour; with a spec
# division they fall back to Part 0's orange / purple.
PFC = ({"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]} if _PEEL2 else dict(GCOL))

# ── the axes, evaluated once over every branch row ────────────────────────────────────────────────────────
# A RE-RUN MUST NOT REUSE WHAT THE LAST ONE LEFT BEHIND: the specs, P2_TOL and P2_CLAMP may all have changed
# since, so every evaluated axis is dropped from the table first and rebuilt from the series.
FR.drop(columns=[c for c in FR.columns if "@" in c or c.endswith((" | dt", " | cells"))], inplace=True)
ACOL = _axis(P2_ADOPT)
SHOW = []
for _s in P2_SHOW:
    _c = _axis(_s)
    if _c not in [c for c, _ in SHOW]:
        SHOW.append((_c, _spec_lab(_s)))
if ACOL not in [c for c, _ in SHOW]:                  # the division always gets drawn, first
    SHOW.insert(0, (ACOL, _spec_lab(P2_ADOPT)))
ALAB = dict(SHOW)[ACOL]
if P2_ZCTRL:
    FR[ACOL + " | cells"] = _cellrank(FR, ACOL)
    ACOL = ACOL + " | cells"
    ALAB += r"   ranked inside (epoch $\times$ mass) cells"
    SHOW = [(ACOL, ALAB)] + [(c, l) for c, l in SHOW if not c.startswith(ACOL[:-8])]
S2 = FR[FR["uniq"]] if P2_UNIQ else FR
GLAB.update(GLAB0); GSH.update(GSH0)      # a re-run with P2_LABELS = None puts the Part 0 names back
if P2_LABELS:                             # ... and with it set, THESE are the names every figure below uses
    GLAB["burst"], GLAB["aging"] = P2_LABELS
    GSH["burst"], GSH["aging"] = ((str(_v)[:18] for _v in P2_LABELS) if P2_SHORT is None
                                  else (str(_v) for _v in P2_SHORT))

print(f"THE AXES ON THE TABLE  —  {'unique galaxies' if P2_UNIQ else 'anchor rows'}, "
      f"a point read taking the nearest snapshot within {P2_TOL:g} Gyr"
      + (" (epochs off the end of a branch clamped to it)" if P2_CLAMP else "") + ":")
KDE2 = {}
for _c, _lab in SHOW:
    KDE2[_c] = {}
    print(f"  {_c}" + ("      <- THE DIVISION" if _c == ACOL else ""))
    for _b in _BX1:
        _sb = S2[S2["box"] == _b]
        _d = _bimo(_sb[_c].to_numpy(float), (_b, _c))
        KDE2[_c][_b] = _d
        if _d["n"] < P2_MINN:
            print(f"      {_b:9s} N {_d['n']:5d} / {len(_sb):5d} — too few to draw"); continue
        print(f"      {_b:9s} N {_d['n']:5d} / {len(_sb):5d} ({_d['n'] / max(1, len(_sb)) * 100:3.0f} % defined)   "
              f"p05 {_num(_d['p05'])}  1/3 {_num(_d['lo'])}  med {_num(_d['med'])}  2/3 {_num(_d['hi'])}  "
              f"p95 {_num(_d['p95'])}   KDE modes {_d['npk']}  dip {_num(_d['dip'])}")
print("      those six numbers are there to be read off and typed back into P2_CUT.")

# ── the division ──────────────────────────────────────────────────────────────────────────────────────────
CUT2, DIAG2 = {}, {}
_pool = _bimo(S2[ACOL].to_numpy(float), ("pool", ACOL)) if not P2_PERBOX else None
for _b in _BX1:
    _d = KDE2[ACOL][_b] if P2_PERBOX else _pool
    _rule = P2_CUT.get(_b, "median") if isinstance(P2_CUT, dict) else P2_CUT
    _lo, _hi, _how = _cut_rule(_d, _rule)
    CUT2[_b] = _lo if _lo == _hi else (_lo, _hi)
    DIAG2[_b] = dict({k: v for k, v in _d.items() if k not in ("gx", "dn", "x")}, cut_lo=_lo, cut_hi=_hi, how=_how)

FR["grp"] = ""
for _b in _BX1:
    _m = (FR["box"] == _b).to_numpy()
    _x = FR[ACOL].to_numpy(float)
    _lo, _hi = DIAG2[_b]["cut_lo"], DIAG2[_b]["cut_hi"]
    if _lo == _hi:
        FR.loc[_m & (_x > _hi), "grp"] = "burst"
        FR.loc[_m & (_x <= _lo), "grp"] = "aging"
    else:
        FR.loc[_m & (_x >= _hi), "grp"] = "burst"
        FR.loc[_m & (_x <= _lo), "grp"] = "aging"
for _b in _BX1:                                     # carry the label back into the per-box tables
    B1[_b]["F"]["grp"] = FR.loc[FR["box"] == _b, "grp"].to_numpy()
    B1[_b]["F"]["uniq"] = FR.loc[FR["box"] == _b, "uniq"].to_numpy()
G2 = FR["grp"].to_numpy()
S2 = FR[FR["uniq"]] if P2_UNIQ else FR
XCOL, XLAB = ACOL, ALAB

print(f"\nTHE DIVISION ADOPTED: {ACOL}   [{'per box' if P2_PERBOX else 'one cut for every box'}, "
      f"rule = {P2_CUT!r}]")
for _b in _BX1:
    _d, _n = DIAG2[_b], S2[S2["box"] == _b]
    _c = (_num(_d["cut_lo"]) if _d["cut_lo"] == _d["cut_hi"] else _num(_d["cut_lo"]) + " |" + _num(_d["cut_hi"]))
    print(f"  {_b:9s} cut {_c:>18s}  [{_d['how']}]   "
          + "  ".join(f"{GSH[g]} {int((_n['grp'] == g).sum())}" for g in GK)
          + f"   in neither {int((_n['grp'] == '').sum())}"
          + ("  (the axis is not defined there, or it falls in the gap)"
             if _b == _BX1[0] and int((_n["grp"] == "").sum()) else ""))
_qn = _spec(P2_ADOPT)[1]
if P2_LABELS is None and _qn not in (None, "lsig_h2"):
    print(f"  !! the groups are still called {GSH['burst']!r} / {GSH['aging']!r}, which is an H2-DENSITY story,\n"
          f"     and this division is on {_qn!r}.  Set P2_LABELS = ('the high side', 'the low side') and every\n"
          f"     figure from here down will say what the two groups actually are.")
print("  'burst' is always the HIGH side of the axis and 'aging' the LOW one — the keys Parts 3-6 draw with.\n"
      "  WHETHER THIS IS A POPULATION BOUNDARY OR ONLY A CUT is asked in Part 2c, not here.")

# ── FIG 1 — one row per axis, one column per box ──────────────────────────────────────────────────────────
fig, AX = plt.subplots(len(SHOW), len(_BX1), figsize=(4.1 * len(_BX1), 2.9 * len(SHOW) + 0.9), squeeze=False)
for _k, _b in enumerate(_BX1):
    _s = S2[S2["box"] == _b]
    for _r, (_c, _lab) in enumerate(SHOW):
        ax = AX[_r, _k]
        _d = KDE2[_c][_b]
        _x = _s[_c].to_numpy(float); _x = _x[np.isfinite(_x)]
        if len(_x):
            ax.hist(_x, bins=max(8, min(30, len(_x) // 8)), density=True, color="0.85", edgecolor="0.6", lw=0.6)
            if _d.get("gx") is not None and len(_d.get("gx", [])):
                ax.plot(_d["gx"], _d["dn"], color="k", lw=1.8)
            if np.isfinite(_d.get("dip", np.nan)):
                ax.axvline(_d["dip"], color="#2166ac", ls=":", lw=1.6)
            if np.isfinite(_d.get("med", np.nan)):
                ax.axvline(_d["med"], color="#888888", ls="-", lw=1.0)
            if _c == ACOL:                       # the row the division was actually made on
                for _v in {DIAG2[_b]["cut_lo"], DIAG2[_b]["cut_hi"]}:
                    ax.axvline(_v, color="#111111", ls="--", lw=1.8)
                _bn = np.linspace(_x.min(), _x.max(), 24)
                for _g in GK:
                    _xg = _s.loc[_s["grp"] == _g, _c].to_numpy(float)
                    ax.hist(_xg[np.isfinite(_xg)], bins=_bn, density=True, histtype="step", color=GCOL[_g], lw=2.0)
            ax.text(0.03, 0.96, ("THE DIVISION\n" if _c == ACOL else "")
                                + f"N = {_d['n']}  ({_d['n'] / max(1, len(_s)) * 100:.0f} % defined)\n"
                                f"KDE modes {_d['npk']}"
                                + (f"\nSilverman p = {_d['p']:.3f}" if np.isfinite(_d["p"]) else ""),
                    transform=ax.transAxes, va="top", ha="left", fontsize=8.5,
                    color=("#b2182b" if np.isfinite(_d["p"]) and _d["p"] < P2_SILV_P else "k"))
        if _r == 0:
            ax.set_title(BOX_LABEL.get(_b, _b), fontsize=10)
        ax.set_xlabel(_lab, fontsize=9.5)
        if _c == ACOL:
            for _sp in ax.spines.values():
                _sp.set_edgecolor("#111111"); _sp.set_linewidth(1.8)
        ax.set_ylabel("density" if _k == 0 else "")
fig.suptitle("The candidate axes and where they can be cut" + (f"        [{WIN_LAB}]" if any(
    "pk" in c or "mn" in c or "_w" in c or "comp_c" in c for c, _ in SHOW) else ""), y=0.999, fontsize=11)
fig.tight_layout(rect=[0, 0.05, 1, 0.965], h_pad=3.4)
fig.legend(handles=[Line2D([], [], color=GCOL[g], lw=2.4, label=GLAB[g]) for g in GK]
           + [Line2D([], [], color="k", lw=1.8, label="the KDE of the whole sample"),
              Line2D([], [], color="#2166ac", ls=":", lw=1.8, label="its interior minimum (the dip)"),
              Line2D([], [], color="#888888", lw=1.0, label="the median"),
              Line2D([], [], color="k", ls="--", lw=1.8, label="the cut adopted (P2_CUT)")],
           loc="lower center", ncol=3, fontsize=9, frameon=False)
paper_save(fig, "ba_split"); plt.show()

pd.DataFrame([dict(box=b, axis=ACOL, rule=str(P2_CUT), **{k: v for k, v in DIAG2[b].items()},
                   n_burst=int(((S2["box"] == b) & (S2["grp"] == "burst")).sum()),
                   n_aging=int(((S2["box"] == b) & (S2["grp"] == "aging")).sum()),
                   n_neither=int(((S2["box"] == b) & (S2["grp"] == "")).sum())) for b in _BX1]
             ).to_csv(os.path.join(OUT, "ba_split_rule.csv"), index=False)
S2[KEY + ["uniq", "grp"] + [c for c, _ in SHOW]].to_csv(os.path.join(OUT, "ba_division.csv"), index=False)
print("tables ->", os.path.join(OUT, "ba_split_rule.csv"), "and", os.path.join(OUT, "ba_division.csv"))

## Part 3 — the two groups on the clock

The grid the previous notebook's Part 14 drew, with **this** division: rows are the quantities, columns are the
boxes with the X-ray-off control last, and the abscissa is the time since the truncation, $t-t_{\rm SFT}$.
Each curve is the group median with a bootstrap 16–84 band; an epoch is drawn only where at least `NMIN`
galaxies contribute.

Two markers sit on the axis of every panel — **▼** the group's median jet activation and **|** its median first
X-ray firing, both relative to the truncation. They are what makes the figure readable as a sequence rather
than as two curves: the division was made at ▼, so anything that separates *left of* ▼ was already there
before the jet, and anything that opens only *right of* **|** is downstream of the X-ray channel.

`P3_CLOCK = "jet"` re-zeroes the whole grid on the jet activation instead — the clock the division is defined
on, where the groups part by construction at $t=0$ in the $\Sigma_{\rm H_2}$ row and nowhere else by
construction.

In [ ]:
# ── Part 3 — THE CLOCKS: the two groups through the truncation, one row per quantity, one column per box ──
P3_TRK   = ["ldh2", "lms", "lssfr", "age", "lmdot", "lsig_h2", "comp", "kappa_gas", "lfgas", "lfd", "lfh2"]   # any of P1_TRK; an untracked name now RAISES instead of vanishing
P3_CLOCK = "sft"          # "sft" | "jet" | "x1" — which event the abscissa is zeroed on
P3_UNIQ  = True           # one row per galaxy (an object with four anchors would otherwise count four times)
P3_NMIN  = NMIN
P3_NBOOT = None           # None = NBOOT
P3_MARKS = True           # the median t_jet and t_x1 as marks on each panel's axis
P3_XLIM  = None
P3_RUN   = 3             # a separation counts only if the bands stay disjoint for this many consecutive epochs
_CLAB3 = {"sft": r"$t - t_{\rm SFT}$  [Gyr]", "jet": r"$t - t_{\rm jet}$  [Gyr]", "x1": r"$t - t_{\rm x1}$  [Gyr]"}
_MEV3  = {"sft": ("t_jet", "t_x1"), "jet": ("t_sft", "t_x1"), "x1": ("t_jet", "t_sft")}
_MSYM3 = {"t_jet": ("v", "the jet activation"), "t_x1": ("|", "the first X-ray firing"), "t_sft": ("s", "the truncation")}
_TITL3 = {"sft": "the truncation clock", "jet": "the jet-activation clock", "x1": "the first-X-ray clock"}

_TRK3 = [q for q in P3_TRK if any(q in B1[b]["T"] for b in _BX1)]
_DROP3 = [q for q in P3_TRK if q not in _TRK3]
if _DROP3:      # silently dropping a row is how a quantity "does not plot" with no error to read
    raise KeyError(f"P3_TRK names {_DROP3}, which Part 1 does not track, so those rows would silently "
                   f"vanish.\n  the tracked quantities are {P1_TRK}\n"
                   f"  (a new one is three lines in Part 1: add it to P1_TRK, to P1_LAB and to V)")
ROW3 = []
fig, AX = plt.subplots(len(_TRK3), len(_BX1), figsize=(4.3 * len(_BX1), 2.45 * len(_TRK3)), squeeze=False, sharex=True)
for _j, _b in enumerate(_BX1):
    _F, _T = B1[_b]["F"], B1[_b]["T"]
    _keep = _F["uniq"].to_numpy(bool) if P3_UNIQ else np.ones(len(_F), bool)
    _gm = {g: _keep & (_F["grp"].to_numpy() == g) for g in GK}
    for _i, _q in enumerate(_TRK3):
        ax = AX[_i, _j]
        for _g in GK:
            _A = _T[_q][P3_CLOCK][_gm[_g]]
            _c, _lo, _hi, _n = med_band(_A, nmin=P3_NMIN, nboot=P3_NBOOT)
            ax.fill_between(TG, _lo, _hi, color=GCOL[_g], alpha=0.20, lw=0)
            ax.plot(TG, _c, color=GCOL[_g], lw=2.4)
            for _t, _y, _nn in zip(TG, _c, _n):
                if np.isfinite(_y):
                    ROW3.append(dict(box=_b, quantity=_q, clock=P3_CLOCK, grp=_g, t=_t, med=_y, n=int(_nn)))
        ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.7)
        if P3_MARKS:                      # the OTHER two events, as the median offset of each group
            for _k, _ev in enumerate(_MEV3[P3_CLOCK]):
                _sym = _MSYM3[_ev][0]
                for _g in GK:
                    _d = np.nanmedian(_F.loc[_gm[_g], _ev].to_numpy(float)
                                      - _F.loc[_gm[_g], f"t_{P3_CLOCK}"].to_numpy(float))
                    if np.isfinite(_d):
                        ax.plot([_d], [0.02 + 0.07 * _k], marker=_sym, color=GCOL[_g], ms=6, mew=1.6,
                                transform=ax.get_xaxis_transform(), clip_on=False, zorder=6)
        if _q == "lfgas":
            ax.axhline(np.log10(XRAY_FGAS_MAX), color="#2ca02c", ls=":", lw=1.6)
        if _q == "kappa_gas":
            ax.axhline(KAPPA_ISO, color="0.4", ls=":", lw=1.2)
        if _q == "lfd":
            ax.axhline(DUSTY_CUT, color="#d7191c", ls=":", lw=1.4)
        if _i == 0:
            ax.set_title(BOX_LABEL.get(_b, _b), fontsize=10)
        if _j == 0:
            ax.set_ylabel(P1_LAB.get(_q, _q), fontsize=9.5)
        if _i == len(_TRK3) - 1:
            ax.set_xlabel(_CLAB3[P3_CLOCK])
        if P3_XLIM:
            ax.set_xlim(*P3_XLIM)
_h3 = [Line2D([], [], color=GCOL[g], lw=2.6, label=GLAB[g]) for g in GK]
_h3 += [Line2D([], [], color="0.3", ls="none", marker=_MSYM3[e][0], ms=6, label=_MSYM3[e][1]) for e in _MEV3[P3_CLOCK]]
_h3 += [Line2D([], [], color="#2ca02c", ls=":", lw=1.6, label=rf"the X-ray gate $f_{{\rm gas}} < {XRAY_FGAS_MAX:g}$"),
        Line2D([], [], color="#d7191c", ls=":", lw=1.4, label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$")]
fig.suptitle(f"The two groups on {_TITL3[P3_CLOCK]}"
             + ("  —  the division itself is made at the jet activation (▼)" if P3_CLOCK != "jet" else "")
             + f"   [{'one row per galaxy' if P3_UNIQ else 'anchor rows'}; median and bootstrap 16-84 band]",
             y=0.997, fontsize=11)
fig.tight_layout(rect=[0, 0.035, 1, 0.985])
fig.legend(handles=_h3, loc="lower center", ncol=min(4, len(_h3)), fontsize=9, frameon=False)
paper_save(fig, "ba_clocks"); plt.show()
pd.DataFrame(ROW3).to_csv(os.path.join(OUT, "ba_clocks.csv"), index=False)
print("table ->", os.path.join(OUT, "ba_clocks.csv"))

# ── where do they part?  the first epoch at which the bands stop overlapping, per quantity and box ──
print(f"\nWHERE THE GROUPS PART on the {P3_CLOCK} clock (the first epoch whose bootstrap bands are disjoint, "
      "and the gap at t = 0 and at +2 Gyr):")
print(f"    {'quantity':16s}" + "".join(f"{b:>30s}" for b in _BX1))
for _q in _TRK3:
    _line = f"    {_q:16s}"
    for _b in _BX1:
        _F, _T = B1[_b]["F"], B1[_b]["T"]
        _keep = _F["uniq"].to_numpy(bool) if P3_UNIQ else np.ones(len(_F), bool)
        _st = {g: med_band(_T[_q][P3_CLOCK][_keep & (_F["grp"].to_numpy() == g)], nmin=P3_NMIN, nboot=P3_NBOOT) for g in GK}
        _dis = ((_st["burst"][1] > _st["aging"][2]) | (_st["aging"][1] > _st["burst"][2])) & np.isfinite(_st["burst"][0]) & np.isfinite(_st["aging"][0])
        _run = np.convolve(_dis.astype(int), np.ones(P3_RUN, int), "valid") == P3_RUN     # P3_RUN epochs in a row
        _drawn = np.flatnonzero(np.isfinite(_st["burst"][0]) & np.isfinite(_st["aging"][0]))
        _t0 = TG[int(np.flatnonzero(_run)[0])] if _run.any() else np.nan
        _edge = bool(_run.any() and np.flatnonzero(_run)[0] <= (_drawn[0] if len(_drawn) else 0))
        _g0 = np.interp(0.0, TG, _st["burst"][0] - _st["aging"][0], left=np.nan, right=np.nan)
        _g2 = np.interp(2.0, TG, _st["burst"][0] - _st["aging"][0], left=np.nan, right=np.nan)
        _ts = ("  never" if not np.isfinite(_t0) else ("already" if _edge else f"{_t0:+6.2f}"))
        _line += f"   part {_ts}  d0 {_g0:+5.2f}  d2 {_g2:+5.2f}"
        ROW3.append(dict(box=_b, quantity=_q, clock=P3_CLOCK, grp="gap", t_part=_t0, at_edge=_edge, gap_0=_g0, gap_2=_g2))
    print(_line)
print(f"      (part = the first clock epoch where the bands stay disjoint for {P3_RUN} epochs in a row; "
      "\"already\" = separated from the first epoch the grid can draw, i.e. before the window opens;\n"
      f"       d0 / d2 = {GSH['burst']} minus {GSH['aging']} at t = 0 and +2 Gyr)")
pd.DataFrame(ROW3).to_csv(os.path.join(OUT, "ba_clocks.csv"), index=False)

## Part 3b — the track in the $f_{\rm dust}$–age plane, with the critical points on it

Part 3 draws every quantity against the clock. This one throws the clock away and puts two of them against
each other: the mass-weighted stellar **age** on the abscissa, $\log M_{\rm dust}/M_\star$ on the ordinate,
walked along the clock. A galaxy moves right as it ages and falls as it loses its dust, so a track here is
the whole history as **one curve** — and the critical points marked on it say *where in that curve* the jet
fired, the truncation happened, and the galaxy arrived on the red sequence.

| | |
|---|---|
| the **line** | the median of each group at every clock step — $(\rm age(t),\ f_{\rm dust}(t))$, a parametric curve in $t$, annotated with the clock time every `P3B_DT` Gyr |
| the **band** | the 16–84 spread in $f_{\rm dust}$ at that step |
| the **markers** | the median of the two quantities read **at** each event (Part 1's `<q>_jet`, `_sft`, `_qt`, `_anc`) |
| the **cloud** | every galaxy at its anchor, faint, so the tracks can be read against the population |

**The markers are not points of the line**, and the cell says so: a median taken at a fixed *event* is a
different average from a median taken at a fixed *clock step*, because galaxies reach the event at different
times. Both are printed in the table beneath the figure.

`P3B_X` and `P3B_Y` take any two tracked quantities, so the same cell draws the $f_{\rm H_2}$–age plane or
the dust-to-H₂–$\kappa$ plane by changing two strings; `P3B_TLIM` is the stretch of the clock the track is
drawn over.


In [ ]:
# ── Part 3b — THE TRACK IN THE f_dust-AGE PLANE, with the critical points on it ───────────────────────────
# Part 3 draws every quantity against the clock.  This one throws the clock away and puts two of them against
# each other: the mass-weighted stellar AGE on the abscissa and log M_dust/M* on the ordinate, walked along
# the clock.  A galaxy moves left to right as it ages and falls as it loses its dust, so a track here is the
# whole history as one curve — and the CRITICAL POINTS marked on it say where in that curve the jet fired,
# the truncation happened and the galaxy arrived on the red sequence.
#   the LINE      the median of each group at every clock step (age(t), lfd(t)) — a parametric curve in t
#   the BAND      the 16-84 spread in f_dust at that step, drawn vertically
#   the MARKERS   the median of the two quantities read AT each event (Part 1's <q>_jet, _sft, _qt, _anc),
#                 which is not the same as a point of the median track and is the honest way to place them
#   the CLOUD     every galaxy at its anchor, faint, so the tracks can be seen against the population
P3B_EV    = [("jet", "the jet", "v"), ("sft", "the truncation", "o"), ("qt", "the quenching time", "D"),
             ("anc", "the anchor", "*")]
P3B_PLANES = [("age", "lfd"), ("kappa_gas", "lfd")]   # (abscissa, ordinate), ONE FIGURE EACH, any two
                          # tracked quantities.  The age plane is the history as one curve; the kappa_gas
                          # plane is the chapter's question as a picture — the dynamical state against the
                          # dust, walked along the clock, with the events on it.
P3B_BOXES = [b for b in BOXES if b in _BX1]   # the X-ray-on resolutions; the control is a diagnostic, not a panel
P3B_CLOCK = "sft"         # the clock the track is walked on
P3B_UNIQ  = True
P3B_NMIN  = 5             # a clock step is drawn only with this many galaxies
P3B_TLIM  = (-2.0, 4.0)   # the stretch of the clock drawn [Gyr]
P3B_SCAT  = True          # the faint cloud of every galaxy at its anchor
P3B_ANN   = True          # write the clock time beside the track every P3B_DT Gyr
P3B_DT    = 1.0
_C3B = PFC if "PFC" in globals() else GCOL      # the paper colours (Part 2), the peel's own red and blue
for P3B_X, P3B_Y in P3B_PLANES:        # one figure and one table per plane
    ROW3B = []

    if P3B_X not in P1_TRK or P3B_Y not in P1_TRK:
        raise KeyError(f"P3B_X / P3B_Y must be tracked quantities.  P1_TRK = {P1_TRK}")
    _M3B = (TG >= P3B_TLIM[0]) & (TG <= P3B_TLIM[1])
    _rc3b = plt.rc_context(PAPER_RC); _rc3b.__enter__()
    fig, AX = plt.subplots(1, len(P3B_BOXES), figsize=(5.3 * len(P3B_BOXES), 5.3), squeeze=False, sharey=True)
    for _k, _b in enumerate(P3B_BOXES):
        ax = AX[0, _k]
        _F, _T = B1[_b]["F"], B1[_b]["T"]
        _keep = _F["uniq"].to_numpy(bool) if P3B_UNIQ else np.ones(len(_F), bool)
        if P3B_SCAT:                                   # the population, at the anchor
            _xa, _ya = _F[f"{P3B_X}_anc"].to_numpy(float)[_keep], _F[f"{P3B_Y}_anc"].to_numpy(float)[_keep]
            ax.scatter(_xa, _ya, s=6, color="0.75", alpha=0.35, lw=0, zorder=0)
        for _g in GK:
            _m = _keep & (_F["grp"].to_numpy() == _g)
            if _m.sum() < P3B_NMIN:
                continue
            _X, _Y = _T[P3B_X][P3B_CLOCK][_m], _T[P3B_Y][P3B_CLOCK][_m]
            _n = np.isfinite(_X).sum(axis=0) + 0 * np.isfinite(_Y).sum(axis=0)
            _ok = (np.isfinite(_X) & np.isfinite(_Y)).sum(axis=0) >= P3B_NMIN
            _xm = np.array([np.nanmedian(_X[:, j]) if _ok[j] else np.nan for j in range(len(TG))])
            _ym = np.array([np.nanmedian(_Y[:, j]) if _ok[j] else np.nan for j in range(len(TG))])
            _y1 = np.array([np.nanpercentile(_Y[:, j][np.isfinite(_Y[:, j])], 16) if _ok[j] else np.nan
                            for j in range(len(TG))])
            _y2 = np.array([np.nanpercentile(_Y[:, j][np.isfinite(_Y[:, j])], 84) if _ok[j] else np.nan
                            for j in range(len(TG))])
            _s = _M3B & np.isfinite(_xm) & np.isfinite(_ym)
            _dx = np.diff(_xm[_s])
            if len(_dx) and (np.all(_dx >= 0) or np.all(_dx <= 0)):     # a band needs a single-valued abscissa
                ax.fill_between(_xm[_s], _y1[_s], _y2[_s], color=_C3B[_g], alpha=0.15, lw=0)
            else:                                                       # the track doubles back: two thin lines
                ax.plot(_xm[_s], _y1[_s], color=_C3B[_g], lw=1.0, alpha=0.55, ls=":")
                ax.plot(_xm[_s], _y2[_s], color=_C3B[_g], lw=1.0, alpha=0.55, ls=":")
            ax.plot(_xm[_s], _ym[_s], color=_C3B[_g], lw=2.8, zorder=3)
            if P3B_ANN:                                 # where the clock is, along the curve
                for _t in np.arange(np.ceil(P3B_TLIM[0] / P3B_DT) * P3B_DT, P3B_TLIM[1] + 1e-9, P3B_DT):
                    _j = int(np.argmin(np.abs(TG - _t)))
                    if _s[_j]:
                        ax.annotate(f"{_t:+.0f}", (_xm[_j], _ym[_j]), textcoords="offset points",
                                    xytext=(4, -12), fontsize=14.3, fontweight="bold", color=_C3B[_g], alpha=0.95,
                                    bbox=dict(fc="w", ec="none", alpha=0.6, pad=0.6), zorder=6)
            for _ev, _lab, _mk in P3B_EV:               # the critical points, read AT the event
                _xe, _ye = _F[f"{P3B_X}_{_ev}"].to_numpy(float)[_m], _F[f"{P3B_Y}_{_ev}"].to_numpy(float)[_m]
                _f = np.isfinite(_xe) & np.isfinite(_ye)
                if _f.sum() < P3B_NMIN:
                    continue
                _mx, _my = float(np.median(_xe[_f])), float(np.median(_ye[_f]))
                ax.plot([_mx], [_my], marker=_mk, ms=(16 if _mk == "*" else 10), mfc=_C3B[_g], mec="k", mew=1.0,
                        ls="none", zorder=5)
                ROW3B.append(dict(box=_b, grp=_g, event=_ev, n=int(_f.sum()), x=_mx, y=_my,
                                  x_lo=float(np.percentile(_xe[_f], 16)), x_hi=float(np.percentile(_xe[_f], 84)),
                                  y_lo=float(np.percentile(_ye[_f], 16)), y_hi=float(np.percentile(_ye[_f], 84))))
        ax.axhline(DUSTY_CUT, color="0.35", ls="--", lw=1.3)
        if _k == 0:
            ax.text(0.98, DUSTY_CUT, "dusty above here ", transform=ax.get_yaxis_transform(), fontsize=14.3,
                    va="bottom", ha="right", color="0.35")
        ax.text(0.03, 0.97, BOX_LABEL.get(_b, _b), transform=ax.transAxes, va="top", ha="left", fontsize=17.6,
                fontweight="bold", color="0.15", bbox=dict(fc="w", ec="none", alpha=0.75, pad=2.0), zorder=8)
        ax.set_xlabel(P1_LAB.get(P3B_X, P3B_X))
        if _k == 0:
            ax.set_ylabel(P1_LAB.get(P3B_Y, P3B_Y))
        ax.set_axisbelow(True)
    fig.legend(handles=[Line2D([], [], color=_C3B[g], lw=2.8, label=GLAB[g]) for g in GK]
               + [Line2D([], [], color="0.4", marker=_mk, ls="none", ms=(13 if _mk == "*" else 9), label=_lab)
                  for _ev, _lab, _mk in P3B_EV]
               + [Line2D([], [], color="0.75", marker="o", ls="none", ms=5, label="every galaxy at its anchor"),
                  Line2D([], [], color="0.3", ls="none", label=r"$+n$ along the track: $t - t_{\rm SFT}$ in Gyr")],
               loc="lower center", ncol=4, frameon=False)
    fig.tight_layout(rect=[0, 0.14, 1, 0.995])
    paper_save(fig, f"ba_plane_{P3B_Y}_{P3B_X}"); plt.show()
    _rc3b.__exit__(None, None, None)

    print(f"THE CRITICAL POINTS IN THE ({P3B_X}, {P3B_Y}) PLANE  —  the median of each, read AT the event:")
    print(f"    {'box':9s} {'event':6s}" + "".join(f"{GSH[g]:>26s}" for g in GK))
    for _b in P3B_BOXES:
        for _ev, _lab, _ in P3B_EV:
            _line = f"    {_b:9s} {_ev:6s}"
            for _g in GK:
                _r = [r for r in ROW3B if r["box"] == _b and r["grp"] == _g and r["event"] == _ev]
                _line += (f"  {_r[0]['x']:7.2f} {_r[0]['y']:8.2f}  (N {_r[0]['n']:4d})" if _r
                          else f"{'—':>26s}")
            print(_line)
    print(f"      ({P3B_X}, {P3B_Y}) at each critical point.  The MARKERS are these numbers; the LINE is the\n"
          f"      median of both quantities at each step of the clock, which is a different average and need not\n"
          f"      pass exactly through them.")
    pd.DataFrame(ROW3B).to_csv(os.path.join(OUT, f"ba_plane_{P3B_Y}_{P3B_X}.csv"), index=False)
    print("table ->", os.path.join(OUT, f"ba_plane_{P3B_Y}_{P3B_X}.csv"))

## Part 4 — the observer's question

> A quenched galaxy is caught at one random snapshot and it has $f_{\rm dust} > 10^{-4}$. How likely is it that
> its history was an **aging** one rather than a **starburst**? And the same if what is measured is a
> significantly **rotating** gas disc.

This is not the same question as "do the two groups differ in dust". It is the *inverse* one, the one an
observation actually poses, and it depends on three things at once: how many galaxies each group has, how
long each galaxy stays above the threshold, and when it is looked at.

**One draw per galaxy.** Each galaxy is observed **once** — a snapshot drawn from its own quenched life,
weighted by the time each snapshot represents, so the draw is uniform in cosmic time rather than in snapshot
number. The draw is kept if the galaxy is above the threshold there. Repeating the whole draw `P4_NDRAW`
times gives the probability and its sampling spread. Nothing lets one object enter twice.

**Two estimators, and they mean different things.**
- the **one-draw** probability above, whose expectation is $\sum_{\rm aging} f_i / \sum_{\rm all} f_i$ with
  $f_i$ the *fraction* of galaxy $i$'s quenched life spent above the threshold — one galaxy, one vote;
- the **survey** probability $\sum_{\rm aging} T_i / \sum_{\rm all} T_i$ with $T_i$ the *time* itself — what a
  real snapshot survey returns, where a long-lived object genuinely is caught more often.

Both are printed. The first is the answer to the question as posed; the second is what would be measured, and
the gap between them is the visibility bias.

**The base rate is the thing to compare against**, not 50 %. If aging galaxies are 60 % of the quenched
population to begin with, then "60 % of dusty QGs are aging" says nothing at all.

In [ ]:
# ── Part 4 — THE OBSERVER'S QUESTION: a QG caught once at random with f_dust > 1e-4 — starburst or aging? ──
# One galaxy, one observation.  The snapshot is drawn from the galaxy's own quenched life, weighted by the time
# each snapshot stands for, so the draw is uniform in COSMIC TIME and not in snapshot number.
P4_FDUST  = DUSTY_CUT     # the dust threshold the observation is made at (log M_dust/M*)
P4_KZ     = ROT_SIG       # "significantly rotating" = kappa_gas more than this many s_iso above isotropy
P4_NDRAW  = int(os.environ.get("BA_NDRAW", "2000"))
P4_POST   = True          # only snapshots after the truncation are eligible (a QG, not a still-forming one)
P4_UNIQ   = True
P4_LADDER = [-3.5, -4.0, -4.5, -5.0, -5.5]        # "observable" is a depth: the answer must be quoted with its cut
P4_KLAD   = [0.0, 2.0, 5.0, 8.0]
P4_MINN   = 3            # a probability is quoted only above this many contributing galaxies
P4_SEED   = 4


def _elig4(b):
    """Per galaxy: the eligible snapshots (quenched, and after the truncation), the time weight of each, and the
    two observables read there.  Returns aligned lists over the kept rows of B1[b]['F']."""
    F, R = B1[b]["F"], B1[b]["R"]
    keep = np.flatnonzero((F["uniq"].to_numpy(bool) if P4_UNIQ else np.ones(len(F), bool)) & (F["grp"].to_numpy() != ""))
    E = []
    for i in keep:
        r, t_sft = R[i], float(F["t_sft"].iloc[i])
        m = np.asarray(r["q"], bool)
        if P4_POST and np.isfinite(t_sft):
            m = m & (r["t"] > t_sft)
        if m.sum() < 1:
            continue
        w = np.gradient(r["t"]) if len(r["t"]) > 1 else np.ones(1)
        E.append(dict(i=int(i), grp=F["grp"].iloc[i], box=b, w=np.clip(w[m], 0, None),
                      lfd=r["lfd"][m], kz=r["kz"][m], t=r["t"][m]))
    return E


def _prob4(E, key, thr, above=True, ndraw=None, seed=P4_SEED):
    """P(aging | the observable is past thr at the ONE drawn snapshot), three ways.
      p_draw  the Monte-Carlo answer to the question as posed: one snapshot per galaxy, repeated;
      p_one   its exact expectation, sum_aging f_i / sum_all f_i with f_i the FRACTION of the quenched life above thr;
      p_survey  sum_aging T_i / sum_all T_i — what a snapshot survey returns, where a long-lived object is
                genuinely caught more often (the same source contributes many times, deliberately).
    Also returns the per-group galaxy counts and the expected number of detections."""
    nd = P4_NDRAW if ndraw is None else ndraw
    f, T, g = [], [], []
    for e in E:
        y = e[key]
        ok = np.isfinite(y) & ((y >= thr) if above else (y <= thr))
        wt = float(e["w"].sum())
        if wt <= 0 or not np.isfinite(wt):
            continue
        f.append(float(e["w"][ok].sum()) / wt); T.append(float(e["w"][ok].sum())); g.append(e["grp"])
    f, T, g = np.asarray(f), np.asarray(T), np.asarray(g)
    if not len(f) or f.sum() <= 0:
        return dict(p_draw=np.nan, p_one=np.nan, p_survey=np.nan, n_eff=0.0, lo=np.nan, hi=np.nan,
                    n_burst=0, n_aging=0, base=np.nan, sd=np.nan,
                    frac_burst=np.nan, frac_aging=np.nan)
    ag = (g == "aging")
    p_one = float(f[ag].sum() / f.sum())
    p_srv = float(T[ag].sum() / T.sum()) if T.sum() > 0 else np.nan
    rng = np.random.default_rng(seed)                       # the draw itself: a Bernoulli per galaxy per realisation
    hit = rng.random((nd, len(f))) < f[None, :]
    nag, ntot = (hit & ag[None, :]).sum(axis=1), hit.sum(axis=1)
    ps = np.where(ntot > 0, nag / np.maximum(ntot, 1), np.nan)
    _, lo, hi = wilson(f[ag].sum(), f.sum())                # the interval on the EXPECTED counts
    return dict(p_draw=float(np.nanmean(ps)), sd=float(np.nanstd(ps)), p_one=p_one, p_survey=p_srv,
                n_eff=float(f.sum()), lo=lo, hi=hi, n_burst=int((~ag).sum()), n_aging=int(ag.sum()),
                base=float(ag.sum()) / len(ag), frac_burst=float(np.nanmean(f[~ag])), frac_aging=float(np.nanmean(f[ag])))


E4 = {b: _elig4(b) for b in _BX1}
ROW4, RES4 = [], {}
print(f"THE OBSERVATION: one snapshot per galaxy, drawn uniformly in cosmic time from its quenched life"
      + (" after the truncation" if P4_POST else "") + f";  {P4_NDRAW} realisations.")
print(f"  the dust threshold log M_dust/M* >= {P4_FDUST:g}   |   'rotating' = kappa_gas > 1/3 + {P4_KZ:g} s_iso(N_gas)\n")
_a4 = GSH['aging']                                            # the groups by their own names, not "aging"
print(f"    {'box':10s} {'N gal (b/a)':>14s} {f'base P({_a4})':>16s} | {f'P({_a4} | dusty)':>28s} | {f'P({_a4} | rotating)':>28s}")
for _b in _BX1:
    _E = E4[_b]
    _nb = sum(e["grp"] == "burst" for e in _E); _na = sum(e["grp"] == "aging" for e in _E)
    _base = _na / max(1, _nb + _na)
    _d = _prob4(_E, "lfd", P4_FDUST, True)
    _k = _prob4(_E, "kz", P4_KZ, True)
    RES4[_b] = dict(dust=_d, rot=_k, base=_base)
    _fmt = lambda r: (f"{r['p_draw']:.3f} +-{r['sd']:.3f} [{r['lo']:.2f},{r['hi']:.2f}] n{r['n_eff']:5.1f}"
                      if np.isfinite(r["p_draw"]) and r["n_eff"] >= P4_MINN else f"{'too few':>28s}")
    print(f"    {_b:10s} {_nb:6d}/{_na:<7d} {_base:14.3f} | {_fmt(_d):>28s} | {_fmt(_k):>28s}")
    for _nm, _r in (("dusty", _d), ("rotating", _k)):
        ROW4.append(dict(box=_b, block="main", cond=_nm, thr=P4_FDUST if _nm == "dusty" else P4_KZ, **_r))
print("      (P +- the spread over realisations, [Beta-posterior 68 % band on the expected counts], n = the expected number of\n"
      "       detections, i.e. the effective sample the probability is measured on)")

print("\n  the same numbers the other way round, and what a survey would return instead:")
for _b in _BX1:
    _d, _k, _base = RES4[_b]["dust"], RES4[_b]["rot"], RES4[_b]["base"]
    print(f"    {_b:10s} P(dusty | {GSH['burst']}) = {_d['frac_burst']:.3f}   P(dusty | {_a4}) = {_d['frac_aging']:.3f}"
          f"   ->  one-draw {_d['p_one']:.3f}  vs  survey {_d['p_survey']:.3f}   (base {_base:.3f})")
    print(f"    {'':10s} P(rot   | {GSH['burst']}) = {_k['frac_burst']:.3f}   P(rot   | {_a4}) = {_k['frac_aging']:.3f}"
          f"   ->  one-draw {_k['p_one']:.3f}  vs  survey {_k['p_survey']:.3f}")
print("      P(x | group) is the fraction of that group's quenched life spent past the cut — the duty cycle of the\n"
      "      observable.  The survey number counts the same object once per snapshot it is visible in; the one-draw\n"
      "      number counts it once.  Their difference IS the visibility bias.")

# ── the ladders ─────────────────────────────────────────────────────────────────────────────────────────
LAD4 = {b: {"dust": [], "rot": []} for b in _BX1}
for _b in _BX1:
    for _t in P4_LADDER:
        _r = _prob4(E4[_b], "lfd", _t, True)
        LAD4[_b]["dust"].append((_t, _r)); ROW4.append(dict(box=_b, block="ladder", cond="dusty", thr=_t, **_r))
    for _t in P4_KLAD:
        _r = _prob4(E4[_b], "kz", _t, True)
        LAD4[_b]["rot"].append((_t, _r)); ROW4.append(dict(box=_b, block="ladder", cond="rotating", thr=_t, **_r))

# ── FIG ──────────────────────────────────────────────────────────────────────────────────────────────────
fig, AX = plt.subplots(2, 2, figsize=(11.4, 7.6))
ax = AX[0, 0]
_x = np.arange(len(_BX1))
for _o, (_nm, _c, _mk) in enumerate((("dust", "#d7191c", "o"), ("rot", "#1a9850", "s"))):
    _p = np.array([RES4[b][_nm]["p_draw"] for b in _BX1])
    _e = np.array([[RES4[b][_nm]["p_draw"] - RES4[b][_nm]["lo"] for b in _BX1],
                   [RES4[b][_nm]["hi"] - RES4[b][_nm]["p_draw"] for b in _BX1]])
    ax.errorbar(_x + 0.14 * (_o - 0.5), _p, yerr=np.clip(_e, 0, None), marker=_mk, color=_c, ls="none", ms=8, capsize=3,
                label=rf"$P$({GSH['aging']} | $f_{{\rm dust}} > 10^{{{P4_FDUST:g}}}$)" if _nm == "dust"
                else rf"$P$({GSH['aging']} | $\kappa_{{\rm gas}} > 1/3 + {P4_KZ:g}s_{{\rm iso}}$)")
ax.plot(_x, [RES4[b]["base"] for b in _BX1], marker="_", ls="none", color="k", ms=26, mew=2.2, label=f"the base rate P({GSH['aging']})")
ax.set_xticks(_x); ax.set_xticklabels([b.replace("cis", "m") for b in _BX1]); ax.set_ylim(0, 1)
ax.axhline(0.5, color="0.8", lw=1.0, zorder=0)
ax.set_ylabel(f"probability the history was {GSH['aging'].upper()}"); ax.legend(fontsize=8, frameon=False, loc="lower left")
ax.set_title("(a) one galaxy, one observation", fontsize=10)

ax = AX[0, 1]
for _b in _BX1:
    _t = np.array([t for t, _ in LAD4[_b]["dust"]])
    _p = np.array([r["p_draw"] if r["n_eff"] >= P4_MINN else np.nan for _, r in LAD4[_b]["dust"]])
    ax.plot(_t, _p, marker="o", lw=2.0, ms=5, label=_b.replace("cis", "m"))
    ax.axhline(RES4[_b]["base"], color=ax.lines[-1].get_color(), ls=":", lw=1.0, alpha=0.7)
ax.axvline(P4_FDUST, color="k", ls="--", lw=1.2)
ax.set_xlabel(r"the dust threshold  $\log M_{\rm dust}/M_\star$"); ax.set_ylabel(f"P({GSH['aging']} | above it)")
ax.set_ylim(0, 1); ax.invert_xaxis(); ax.legend(fontsize=8, frameon=False, ncol=2)
ax.set_title("(b) the answer depends on how deep you look\n(dotted = each box's base rate)", fontsize=10)

ax = AX[1, 0]
for _b in _BX1:
    _t = np.array([t for t, _ in LAD4[_b]["rot"]])
    _p = np.array([r["p_draw"] if r["n_eff"] >= P4_MINN else np.nan for _, r in LAD4[_b]["rot"]])
    ax.plot(_t, _p, marker="s", lw=2.0, ms=5, label=_b.replace("cis", "m"))
    ax.axhline(RES4[_b]["base"], color=ax.lines[-1].get_color(), ls=":", lw=1.0, alpha=0.7)
ax.axvline(P4_KZ, color="k", ls="--", lw=1.2)
ax.set_xlabel(r"the rotation cut  $(\kappa_{\rm gas}-1/3)/s_{\rm iso}(N_{\rm gas})$"); ax.set_ylabel(f"P({GSH['aging']} | above it)")
ax.set_ylim(0, 1); ax.legend(fontsize=8, frameon=False, ncol=2)
ax.set_title("(c) and on how strict 'rotating' is", fontsize=10)

ax = AX[1, 1]
for _j, _b in enumerate(_BX1):
    for _g in GK:
        _f = np.array([np.nan if not len(E4[_b]) else 0.0])
        _v = []
        for e in E4[_b]:
            if e["grp"] != _g:
                continue
            w = e["w"]; ok = np.isfinite(e["lfd"]) & (e["lfd"] >= P4_FDUST)
            if w.sum() > 0:
                _v.append(float(w[ok].sum()) / float(w.sum()))
        if len(_v) >= 3:
            ax.scatter([_j + 0.16 * (1 if _g == "burst" else -1)] * len(_v),
                       np.asarray(_v) + np.random.default_rng(_j).normal(0, 0.006, len(_v)),
                       s=5, color=GCOL[_g], alpha=0.28, lw=0)
            _xx = _j + 0.16 * (1 if _g == "burst" else -1)
            ax.plot([_xx], [np.median(_v)], marker="D", color=GCOL[_g], ms=9, mec="k", mew=0.8, zorder=5)
            ax.plot([_xx], [np.mean(_v)], marker="_", color=GCOL[_g], ms=17, mew=2.6, zorder=6)
ax.set_xticks(np.arange(len(_BX1))); ax.set_xticklabels([b.replace("cis", "m") for b in _BX1])
ax.set_ylabel(rf"fraction of the quenched life above $10^{{{P4_FDUST:g}}}$"); ax.set_ylim(-0.03, 1.03)
ax.legend(handles=[Line2D([], [], color=GCOL[g], marker="D", ls="none", ms=8, label=GLAB[g]) for g in GK]
          + [Line2D([], [], color="0.3", marker="_", ls="none", ms=12, mew=2.4, label="the group mean — what (a) is built from")],
          fontsize=8, frameon=False, loc="upper center")
ax.set_title("(d) why: the duty cycle of the observable\n(one point per galaxy; diamond = median, bar = mean)", fontsize=10)
fig.suptitle("What a single observation of a dusty (or rotating) quenched galaxy implies about its history",
             y=0.998, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.965])
paper_save(fig, "ba_observe"); plt.show()
pd.DataFrame(ROW4).to_csv(os.path.join(OUT, "ba_observe.csv"), index=False)
print("table ->", os.path.join(OUT, "ba_observe.csv"))

## Part 5 — how long the dust lasts, and what sets it

The timescale asked for is **not** the first time a branch dips below $f_{\rm dust}=10^{-4}$ — many of them
come back. It is the **last** crossing, after which the galaxy stays below for good: the moment the dust stops
being observable. `t_first`, `t_dark` and the total time above are all measured, and the difference between
the first two is exactly the "not temporarily" qualifier made visible.

**The censoring is the point, not a nuisance.** A branch still above the threshold when we reach its anchor
has not died yet — its lifetime is a *lower limit*, not a missing value, and throwing those away would bias
every median downwards by exactly the galaxies the question is about. Kaplan–Meier uses both kinds, so the
median visibility time and the never-dying plateau are population statements. The log-rank test compares the
two groups using the same information.

**Then the two things a lifetime is made of.** A galaxy stays visible longer either because it starts with
more dust or because it loses it more slowly:
$$t_{\rm dark} \simeq \frac{y_0 - y_{\rm thr}}{|s|},\qquad y=\log_{10} f_{\rm dust}.$$
The **head start** $y_0-y_{\rm thr}$ (the dust at the truncation) and the **initial rate** $s$ are reported
separately per group, and the correlation of $t_{\rm dark}$ with the group is repeated **at fixed head start**
— which is the actual answer to "does it depend on $f_{\rm dust}$ at the SFT?".

In [ ]:
# ── Part 5 — HOW LONG THE DUST LASTS: the time to fall permanently below the threshold, and what sets it ──
P5_THR    = DUSTY_CUT     # the visibility threshold in log M_dust/M*
P5_UNIQ   = True
P5_SLOPE  = 1.0           # Gyr after the truncation over which the INITIAL rate is fitted
P5_MINN   = 10            # a KM curve / a group comparison is drawn only above this many galaxies
P5_NBOOT  = 400
P5_SMAX   = None          # the survival axis [Gyr]; None = the 95th percentile of the follow-up
P5_SEED   = 5
P5_BOXES  = [b for b in BOXES]   # the X-ray-off box is OUT by default: nothing there ever falls below the
                                 # threshold, so its Kaplan-Meier curve is a flat line at 1 and its medians
                                 # are NaN.  Put NOX_BOX back in to see that, it is a statement in itself.
P5_TFLOOR = 0.05          # [Gyr] the smallest lifetime the log axis of the contour panel can show
P5_LEV    = (0.5, 0.9)    # the contours: the smallest regions holding these shares of each group


def _cross5(t, y, thr, t0):
    """ONE branch against ONE threshold, from the truncation to the anchor.  y is log10 f_dust and t is Gyr, so a
    straight line between two samples IS an exponential and the crossing is exact under linear interpolation.
      t_first  the FIRST down-crossing;   t_dark  the LAST one, after which it never returns (the lifetime);
      ev       whether that permanent death was actually seen (False = right-censored at the anchor);
      t_above  the total time above, which counts a return;   y0 the value at the truncation."""
    m = np.isfinite(t) & np.isfinite(y) & (t >= t0)
    t, y = np.asarray(t)[m] - t0, np.asarray(y)[m]
    out = dict(y0=np.nan, head=np.nan, t_first=np.nan, t_dark=np.nan, ev=False, t_above=np.nan,
               n_down=0, n_up=0, t_post=np.nan, vis0=False, n=int(len(t)), s_ini=np.nan, t_ini=np.nan)
    if len(t) < 2:
        return out
    ab = y >= thr
    out.update(y0=float(y[0]), head=float(y[0] - thr), vis0=bool(ab[0]), t_post=float(t[-1]))
    tc = lambda i: float(t[i] + np.clip((y[i] - thr) / (y[i] - y[i + 1]) if abs(y[i] - y[i + 1]) > 1e-12 else 0.0,
                                        0.0, 1.0) * (t[i + 1] - t[i]))
    dn = np.flatnonzero(ab[:-1] & ~ab[1:])
    up = np.flatnonzero((~ab[:-1]) & ab[1:])
    out["n_down"], out["n_up"] = int(len(dn)), int(len(up))
    if not ab[0]:
        out.update(t_first=0.0, t_dark=0.0, ev=True)          # already below at the truncation: no visible life
    elif len(dn):
        out["t_first"] = tc(int(dn[0]))
        if ab[-1]:
            out.update(t_dark=float(t[-1]), ev=False)         # it came back and is still up: CENSORED
        else:
            out.update(t_dark=tc(int(dn[-1])), ev=True)       # the LAST crossing is the permanent one
    else:
        out.update(t_first=float(t[-1]), t_dark=float(t[-1]), ev=False)     # never crossed: censored
    a = 0.0
    for i in range(len(t) - 1):
        dt = t[i + 1] - t[i]
        if ab[i] and ab[i + 1]:
            a += dt
        elif ab[i] or ab[i + 1]:
            fr = float(np.clip((y[i] - thr) / (y[i] - y[i + 1]) if abs(y[i] - y[i + 1]) > 1e-12 else 0.0, 0.0, 1.0))
            a += dt * (fr if ab[i] else (1.0 - fr))
    out["t_above"] = float(a)
    w = t <= P5_SLOPE                                          # the INITIAL rate: a straight line in log10 y
    if w.sum() >= 3 and np.ptp(t[w]) > 0:
        out["s_ini"] = float(np.polyfit(t[w], y[w], 1)[0])
        if out["s_ini"] < 0:
            out["t_ini"] = float(out["head"] / (-out["s_ini"]))
    return out


def _km5(t, ev):
    """Kaplan-Meier with Greenwood's standard error.  A censored branch contributes its time at risk and no event,
    which is exactly what a galaxy still dusty at its anchor is."""
    t, ev = np.asarray(t, float), np.asarray(ev, bool)
    m = np.isfinite(t) & (t >= 0)
    t, ev = t[m], ev[m]
    if not len(t):
        return np.array([0.0]), np.array([1.0]), np.array([0.0]), 0
    o = np.argsort(t); t, ev = t[o], ev[o]
    ts, S, se, s, v, n = [0.0], [1.0], [0.0], 1.0, 0.0, len(t)
    for u in np.unique(t[ev]):
        n_at = int((t >= u).sum()); d = int(((t == u) & ev).sum())
        if n_at <= 0:
            continue
        s *= 1.0 - d / n_at
        v += d / (n_at * max(n_at - d, 1))
        ts.append(float(u)); S.append(s); se.append(s * np.sqrt(v))
    return np.asarray(ts), np.asarray(S), np.asarray(se), n


def _kmmed(t, ev):
    """The median survival: the first time S drops to 0.5.  NaN when the curve never gets there."""
    ts, S, _, _ = _km5(t, ev)
    i = np.flatnonzero(S <= 0.5)
    return float(ts[i[0]]) if len(i) else np.nan


def _logrank(t, ev, g1):
    """The log-rank chi2 (1 dof) and its p, comparing the two groups under censoring."""
    t, ev, g1 = np.asarray(t, float), np.asarray(ev, bool), np.asarray(g1, bool)
    m = np.isfinite(t) & (t >= 0)
    t, ev, g1 = t[m], ev[m], g1[m]
    O, E, V = 0.0, 0.0, 0.0
    for u in np.unique(t[ev]):
        n, n1 = int((t >= u).sum()), int(((t >= u) & g1).sum())
        d, d1 = int(((t == u) & ev).sum()), int(((t == u) & ev & g1).sum())
        if n < 2 or d < 1:
            continue
        O += d1; E += d * n1 / n
        V += d * (n1 / n) * (1 - n1 / n) * (n - d) / max(n - 1, 1)
    if V <= 0:
        return np.nan, np.nan
    chi2 = (O - E) ** 2 / V
    from scipy.stats import chi2 as _c2
    return float(chi2), float(_c2.sf(chi2, 1))


# ── measure every branch ─────────────────────────────────────────────────────────────────────────────────
L5 = []
_BX5 = [b for b in _BX1 if b in P5_BOXES]
for _b in _BX5:
    _F, _R = B1[_b]["F"], B1[_b]["R"]
    _keep = (_F["uniq"].to_numpy(bool) if P5_UNIQ else np.ones(len(_F), bool)) & (_F["grp"].to_numpy() != "")
    for _i in np.flatnonzero(_keep):
        _t0 = float(_F["t_sft"].iloc[_i])
        if not np.isfinite(_t0):
            continue
        _c = _cross5(_R[_i]["t"], _R[_i]["lfd"], P5_THR, _t0)
        _c.update(box=_b, snap=int(_F["snap"].iloc[_i]), gal_id=int(_F["gal_id"].iloc[_i]),
                  grp=_F["grp"].iloc[_i], t_sft=_t0, lfd_sft=float(_F["lfd_sft"].iloc[_i]),
                  log_mstar=float(_F["log_mstar"].iloc[_i]), t_anchor=float(_F["t_anchor"].iloc[_i]),
                  lfd_anc=float(_F["lfd_anc"].iloc[_i]))
        L5.append(_c)
L5 = pd.DataFrame(L5)
ROW5 = []
print(f"THE VISIBILITY TIME after the truncation, at log M_dust/M* = {P5_THR:g}   "
      f"({len(L5)} galaxies; the window is t_SFT -> the anchor)\n")
_gw5 = max(10, max(len(v) for v in GSH.values()) + 1)         # the group column fits the names
print(f"    {'box':10s} {'group':{_gw5}s} {'N':>4s} {'censored':>9s} {'never up':>9s} {'came back':>10s}"
      f" {'med t_dark':>11s} {'med t_first':>12s} {'med t_above':>12s}")
for _b in _BX5:
    for _g in GK:
        _s = L5[(L5["box"] == _b) & (L5["grp"] == _g)]
        if not len(_s):
            continue
        _md = _kmmed(_s["t_dark"].to_numpy(float), _s["ev"].to_numpy(bool))
        _evf = (_s["n_down"].to_numpy(int) > 0) | (~_s["vis0"].to_numpy(bool))    # the FIRST crossing was seen
        _mf = _kmmed(_s["t_first"].to_numpy(float), _evf)
        _row = dict(box=_b, block="census", grp=_g, n=len(_s), cens=float((~_s["ev"]).mean()),
                    neverup=float((~_s["vis0"]).mean()), back=float((_s["n_up"] > 0).mean()),
                    med_dark=_md, med_first=_mf, med_above=float(np.nanmedian(_s["t_above"])),
                    med_head=float(np.nanmedian(_s["head"])), med_s=float(np.nanmedian(_s["s_ini"])),
                    med_tini=float(np.nanmedian(_s["t_ini"])))
        ROW5.append(_row)
        print(f"    {_b:10s} {GSH[_g]:{_gw5}s} {len(_s):4d} {_row['cens'] * 100:8.0f} % {_row['neverup'] * 100:8.0f} %"
              f" {_row['back'] * 100:9.0f} % {_md:11.2f} {_mf:12.2f} {_row['med_above']:12.2f}")
print("      (censored = still above the threshold at the anchor, so t_dark is a lower limit; never up = already\n"
       "       below at the truncation; came back = crossed up again at least once.  All medians are Kaplan-Meier.)")

print(f"\nDOES THE DUST LAST LONGER IN THE {GSH['aging'].upper()}?")
for _b in _BX5:
    _s = L5[L5["box"] == _b]
    if len(_s) < P5_MINN:
        print(f"    {_b:10s} too few"); continue
    _ag = (_s["grp"] == "aging").to_numpy()
    _chi, _p = _logrank(_s["t_dark"].to_numpy(float), _s["ev"].to_numpy(bool), _ag)
    _ma, _mb = _kmmed(_s.loc[_ag, "t_dark"], _s.loc[_ag, "ev"]), _kmmed(_s.loc[~_ag, "t_dark"], _s.loc[~_ag, "ev"])
    if not (np.isfinite(_ma) and np.isfinite(_mb)):
        print(f"    {_b:10s} the survival curve never reaches 1/2 in "
              + ("both groups" if not (np.isfinite(_ma) or np.isfinite(_mb)) else "one group")
              + f" — {float((~_s['ev']).mean()) * 100:.0f} % of the branches are still above "
              f"{P5_THR:g} at their anchor.  In this box the dust does not die inside the window.")
        ROW5.append(dict(box=_b, block="test", med_aging=_ma, med_burst=_mb, logrank_p=_p, n=len(_s),
                         cens=float((~_s["ev"]).mean())))
        continue
    _rng = np.random.default_rng(P5_SEED)
    _bs = []
    for _ in range(P5_NBOOT):
        _k = _rng.integers(0, len(_s), len(_s)); _q = _s.iloc[_k]; _a2 = (_q["grp"] == "aging").to_numpy()
        if _a2.sum() < 3 or (~_a2).sum() < 3:
            continue
        _bs.append(_kmmed(_q.loc[_a2, "t_dark"], _q.loc[_a2, "ev"]) - _kmmed(_q.loc[~_a2, "t_dark"], _q.loc[~_a2, "ev"]))
    _bs = np.asarray(_bs, float); _bs = _bs[np.isfinite(_bs)]
    _d, _dlo, _dhi = _ma - _mb, (np.percentile(_bs, 16) if len(_bs) else np.nan), (np.percentile(_bs, 84) if len(_bs) else np.nan)
    # at fixed head start: is the group still informative once the dust AT the truncation is held constant?
    _rr, _pp = spearmanr(_s["head"], _s["t_dark"], nan_policy="omit")
    _rk = _cellrank(_s.assign(**{"__t": _s["t_dark"]}), "__t", ctrl=["head"], q=3)
    _ac, _pc, _, _ = mw_auc(_rk, _ag)
    _ar, _pr, _, _ = mw_auc(_s["t_dark"].to_numpy(float), _ag)
    print(f"    {_b:10s} median t_dark  {GSH['aging']} {_ma:5.2f}  vs {GSH['burst']} {_mb:5.2f}  Gyr   "
          f"diff {_d:+5.2f} [{_dlo:+.2f},{_dhi:+.2f}]   log-rank p = "
          + (f"{_p:.3g}" if np.isfinite(_p) else "n/a")
          + f"   AUC({GSH['aging']}) {_ar:.2f} raw / {_ac:.2f} at fixed head start   rho(head, t_dark) = {_rr:+.2f}")
    ROW5.append(dict(box=_b, block="test", med_aging=_ma, med_burst=_mb, diff=_d, lo=_dlo, hi=_dhi,
                     logrank_p=_p, auc=_ar, auc_ctrl=_ac, rho_head=_rr, p_head=_pp, n=len(_s)))
print(f"      (AUC > 0.5 = the {GSH['aging'].upper()} group lasts longer.  The controlled column ranks t_dark inside terciles of the\n"
      "       head start y_0 - thr, so it answers 'longer at the SAME dust at the truncation?')")

print("\nWHAT THE LIFETIME IS MADE OF   (head start = the dust at the truncation above the threshold; s = the initial rate)")
print(f"    {'box':10s} {'group':{_gw5}s} {'head [dex]':>11s} {'s [dex/Gyr]':>12s} {'head/|s| [Gyr]':>15s} {'t_dark [Gyr]':>13s}")
for _b in _BX5:
    for _g in GK:
        _r = [r for r in ROW5 if r.get("block") == "census" and r["box"] == _b and r["grp"] == _g]
        if _r:
            _r = _r[0]
            print(f"    {_b:10s} {GSH[_g]:{_gw5}s} {_r['med_head']:11.2f} {_r['med_s']:12.2f} {_r['med_tini']:15.2f} {_r['med_dark']:13.2f}")

# ── FIG ──────────────────────────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12.4, 7.4))
gs = fig.add_gridspec(2, len(_BX5), height_ratios=[1.0, 0.95], hspace=0.38, wspace=0.28)
_smax = P5_SMAX or float(np.nanpercentile(L5["t_post"], 95))
for _j, _b in enumerate(_BX5):
    ax = fig.add_subplot(gs[0, _j])
    for _g in GK:
        _s = L5[(L5["box"] == _b) & (L5["grp"] == _g)]
        if len(_s) < P5_MINN:
            continue
        _ts, _S, _se, _n = _km5(_s["t_dark"].to_numpy(float), _s["ev"].to_numpy(bool))
        ax.step(np.append(_ts, _smax), np.append(_S, _S[-1]), where="post", color=GCOL[_g], lw=2.4)
        ax.fill_between(np.append(_ts, _smax), np.append(_S - _se, (_S - _se)[-1]),
                        np.append(np.minimum(_S + _se, 1), min((_S + _se)[-1], 1)), step="post",
                        color=GCOL[_g], alpha=0.18, lw=0)
    ax.axhline(0.5, color="0.7", lw=1.0, ls=":")
    _cb = L5[L5["box"] == _b]
    if len(_cb) and not _cb["ev"].any():          # a box in which the dust never dies inside the window
        ax.text(0.5, 0.55, f"no branch falls below\n$10^{{{P5_THR:g}}}$ inside its window\n"
                           f"({int((~_cb['ev']).sum())}/{len(_cb)} censored)",
                transform=ax.transAxes, ha="center", va="center", fontsize=9, color="0.35")
    ax.set_xlim(0, _smax); ax.set_ylim(0, 1.02)
    ax.set_title(BOX_LABEL.get(_b, _b), fontsize=10)
    ax.set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]")
    if _j == 0:
        ax.set_ylabel(rf"$P$(still above $10^{{{P5_THR:g}}}$)")
        ax.legend(handles=[Line2D([], [], color=GCOL[g], lw=2.4, label=GLAB[g]) for g in GK],
                  fontsize=7.5, frameon=False, loc="lower left")
ax = fig.add_subplot(gs[1, :2])
# THE SAME PLANE AS A DENSITY.  A point cloud of 3000 galaxies hides its own shape; these are the smallest
# regions holding P5_LEV of each group, over the galaxies whose death was actually SEEN.  The censored ones
# cannot enter a density (their lifetime is a lower bound, not a value) and stay as open triangles.
_hx5 = np.linspace(0, max(0.2, np.nanpercentile(L5["head"], 99)), 24)
_hy5 = np.linspace(np.log10(P5_TFLOOR), np.log10(max(1.0, np.nanpercentile(L5["t_dark"], 99.5))), 24)
for _g in GK:
    _s = L5[(L5["grp"] == _g) & np.isfinite(L5["head"]) & np.isfinite(L5["t_dark"])]
    _e, _c = _s[_s["ev"]], _s[~_s["ev"]]
    if len(_e) >= 15:
        _H, _, _ = np.histogram2d(_e["head"].to_numpy(float),
                                  np.log10(np.clip(_e["t_dark"].to_numpy(float), P5_TFLOOR, None)),
                                  bins=[_hx5, _hy5])
        _K = _H.copy()                               # two 3x3 boxcars: the sample is too small for a fine grid
        for _ in range(2):
            _S2 = np.zeros_like(_K)
            for _dx in (-1, 0, 1):
                for _dy in (-1, 0, 1):
                    _S2 += np.roll(np.roll(_K, _dx, axis=0), _dy, axis=1)
            _K = _S2
        _f = np.sort(_K.ravel())[::-1]
        _cs = np.cumsum(_f) / max(_f.sum(), 1)
        _lv = sorted({float(_f[np.searchsorted(_cs, _l)]) if _l < _cs[-1] else float(_f[-1]) for _l in P5_LEV})
        ax.contour(0.5 * (_hx5[1:] + _hx5[:-1]), 0.5 * (_hy5[1:] + _hy5[:-1]), _K.T,
                   levels=_lv, colors=[GCOL[_g]], linewidths=[1.2, 2.2][:len(_lv)])
    if len(_c):
        ax.scatter(_c["head"], np.log10(np.clip(_c["t_dark"].to_numpy(float), P5_TFLOOR, None)),
                   s=13, marker="^", facecolors="none", edgecolors=GCOL[_g], lw=0.7, alpha=0.45)
for _sl in (0.5, 1.0, 2.0):
    _x = np.linspace(0.02, max(0.1, np.nanpercentile(L5["head"], 99)), 60)
    ax.plot(_x, np.log10(_x / _sl), color="0.6", lw=0.9, ls=":")
    ax.text(_x[-1], np.log10(_x[-1] / _sl), f"  {_sl:g} dex/Gyr", fontsize=7, color="0.45", va="center")
ax.set_xlabel(rf"the head start   $\log M_{{\rm dust}}/M_\star (t_{{\rm SFT}}) - ({P5_THR:g})$  [dex]")
ax.set_ylabel(r"$t_{\rm dark}$  [Gyr]")
ax.set_ylim(_hy5[0], _hy5[-1])
_tk5 = [t for t in (0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10) if _hy5[0] <= np.log10(t) <= _hy5[-1]]
ax.set_yticks(np.log10(_tk5)); ax.set_yticklabels([f"{t:g}" for t in _tk5])
ax.legend(handles=[Line2D([], [], color=GCOL[g], lw=2.2, label=GLAB[g]) for g in GK]
          + [Line2D([], [], color="0.4", marker="^", ls="none", mfc="none", ms=7,
                    label="still visible at the anchor (a lower limit)")],
          fontsize=7.5, frameon=False, loc="lower right")
ax.set_title(f"the lifetime against the head start — the {int(P5_LEV[0] * 100)} and "
             f"{int(P5_LEV[1] * 100)} % contours, with the iso-rate diagonals", fontsize=10)
ax = fig.add_subplot(gs[1, 2:])
_x5 = np.arange(len(_BX5))
for _o, (_c, _lab, _col) in enumerate((("med_head", "head start [dex]", "#4575b4"),
                                       ("med_tini", r"head/|s| [Gyr]", "#fdae61"),
                                       ("med_dark", r"$t_{\rm dark}$ [Gyr]", "#1a9850"))):
    for _gi, _g in enumerate(GK):
        _v = [next((r[_c] for r in ROW5 if r.get("block") == "census" and r["box"] == b and r["grp"] == _g), np.nan)
              for b in _BX5]
        ax.plot(_x5 + 0.26 * (_o - 1) + 0.08 * (_gi - 0.5), _v, marker="o" if _g == "burst" else "s",
                ls="none", ms=7, color=_col, mfc=_col if _g == "burst" else "none", mew=1.6)
ax.set_xticks(_x5); ax.set_xticklabels([b.replace("cis", "m") for b in _BX5])
ax.set_yscale("log")            # the X-ray-off box's head/|s| is ~20x the rest and would flatten every other point
ax.set_ylabel("dex, or Gyr")
ax.legend(handles=[Line2D([], [], color=c, marker="o", ls="none", ms=7, label=l)
                   for c, l in (("#4575b4", "head start [dex]"), ("#fdae61", "head/|s| [Gyr]"), ("#1a9850", r"$t_{\rm dark}$ [Gyr]"))]
          + [Line2D([], [], color="0.3", marker="o", ls="none", ms=7, label=f"{GSH['burst']} (filled)"),
             Line2D([], [], color="0.3", marker="s", ls="none", mfc="none", ms=7, label=f"{GSH['aging']} (open)")],
          fontsize=7.5, frameon=False, ncol=2)
ax.set_title("the two ingredients, group by group", fontsize=10)
fig.suptitle(rf"How long a quenched galaxy stays above $f_{{\rm dust}} = 10^{{{P5_THR:g}}}$ — "
             "the LAST crossing, with the still-visible ones censored, not dropped", y=0.985, fontsize=11)
paper_save(fig, "ba_lifetime"); plt.show()
pd.DataFrame(ROW5).to_csv(os.path.join(OUT, "ba_lifetime.csv"), index=False)
L5.to_csv(os.path.join(OUT, "ba_lifetime_galaxies.csv"), index=False)
print("\ntables ->", os.path.join(OUT, "ba_lifetime{,_galaxies}.csv"))

## Part 6 — the ISM–CGM relation (25 Mpc box only)

The 25 Mpc box is the only one with particle-level products, so it is the only place the gas *outside* the
galaxy can be measured at all. This part **re-reads** the CGM census that `dusty_qgs_kappa_selection` Part 13
already built — `kappa_m25_cgm_nodes.csv`, one row per (galaxy, snapshot) with the CGM split into
cold/warm/hot phases and the mass flux through a 10 kpc shell — and simply **re-splits it on this notebook's
division**. Nothing is re-measured and no particle file is opened.

**It is re-clocked on this notebook's own $t_{\rm SFT}$**, not on the clock Part 13 was run with, so a knob
changed in that notebook cannot silently change what this figure means.

Four questions, in order:

1. **$T_{\rm CGM}$ against time** — mass-weighted and **dust-weighted**. The second is the one that matters
   for grains: a CGM whose mass sits at $10^6$ K can still hold its dust at $10^{4.5}$ K, and only the
   dust-weighted temperature says whether the grains are where sputtering acts.
2. **The inflow/outflow history** — the outflowing gas flux and the *cold* inflow through the shell.
3. **The CGM dust share** — the fraction of the galaxy's total dust that is outside it.
4. **"Are post-starbursts surrounded by a rich CGM?"** — scored in a window after the truncation, and then
   turned around into the conditional an observation would give: **$P$(cold-rich CGM | dusty QG)**, with both
   base rates printed beside it so the lift is readable.

The sample here is small (m25 has ~80 quenched galaxies and the census covers the ones with particle
cut-outs), so every number is quoted with its count and nothing here carries a claim on its own.

In [ ]:
# ── Part 6 — THE ISM-CGM RELATION (m25): the CGM temperature, the flows, and the dust outside the galaxy ──
# A PURE RE-READ of dusty_qgs_kappa_selection Part 13's census, re-split on THIS notebook's division and
# RE-CLOCKED on THIS notebook's t_SFT.  No particle file is opened here.
P6_BOX   = "cis25"
P6_NODES = os.path.join(KSEL, "kappa_m25_cgm_nodes.csv")     # dusty_qgs_kappa_selection Part 13
P6_TB    = np.arange(-3.0, 4.001, 0.5)     # the clock bins [Gyr]; a per-galaxy median is taken inside a bin first
P6_POST  = (0.5, 2.5)                      # the window the "rich CGM?" contrast is scored in
P6_COLD  = None                            # the cold-rich cut on fm_cold (None = the box median at the anchor)
P6_DUSTY = DUSTY_CUT
P6_NMIN  = 3
P6_NBOOT = None
P6_UNIQ  = True
P6_Q = [("lT_cgm_mw", r"$\langle \log T\rangle_{\rm CGM}$, mass-weighted"),
        ("lT_cgm_dw", r"$\langle \log T\rangle_{\rm CGM}$, DUST-weighted"),
        ("fm_cold",   r"cold mass share of the CGM  ($T < 10^5$ K)"),
        ("f_dust_cgm", r"$M_{\rm dust}^{\rm CGM} / (M_{\rm dust}^{\rm CGM} + M_{\rm dust}^{\rm ISM})$"),
        ("lmdot_out_g", r"$\log \dot M_{\rm out}$(gas)  [$M_\odot$ yr$^{-1}$]"),
        ("lmdot_in_gc", r"$\log \dot M_{\rm in}$(cold gas)  [$M_\odot$ yr$^{-1}$]"),
        ("xray_on", "the X-RAY channel on  (share of the group)"),
        ("choked", "the jet on but CHOKED  (no X-ray)")]
# xray_on / choked come straight out of Part 13's census: at every node it records which of SIMBA's AGN
# channels was actually open — the jet criterion met (M_BH > 1e7.5, f_Edd < 0.2) and the X-ray gate
# (f_gas < 0.2) open on top of it, or met and choked.  Averaged over a group they are DUTY CYCLES: the share
# of that group's galaxies with the channel running at that epoch.

if not os.path.exists(P6_NODES):
    print(f"Part 6: {P6_NODES} is missing — dusty_qgs_kappa_selection Part 13 builds it (cluster only, it needs the\n"
          f"        reduced particle cut-outs).  Skipped; every other part is unaffected.")
elif P6_BOX not in _BX1:
    print(f"Part 6: {P6_BOX} is not in this run's boxes ({_BX1}) — skipped.")
else:
    Z6 = pd.read_csv(P6_NODES)
    _F6 = B1[P6_BOX]["F"]
    _keep6 = (_F6["uniq"].to_numpy(bool) if P6_UNIQ else np.ones(len(_F6), bool)) & (_F6["grp"].to_numpy() != "")
    _K6 = _F6.loc[_keep6, ["snap", "gal_id", "grp", "t_sft", "t_anchor", "lfd_anc", "log_mstar", "lsig_h2_jet"]].copy()
    _K6 = _K6.rename(columns={"snap": "anchor_snap"})
    # Part 13's census carries its OWN class column (and its own clock); drop anything this notebook re-supplies
    # so the merge cannot silently produce grp_x / grp_y and leave the figure split on the other notebook's rule.
    _clash6 = [c for c in _K6.columns if c in Z6.columns and c not in ("anchor_snap", "gal_id")]
    if _clash6:
        print(f"Part 6: Part 13's own columns {_clash6} are dropped — this notebook re-supplies them")
        Z6 = Z6.drop(columns=_clash6)
    Z6 = Z6.merge(_K6, on=["anchor_snap", "gal_id"], how="inner")
    Z6["trel"] = Z6["t"].to_numpy(float) - Z6["t_sft"].to_numpy(float)     # RE-CLOCKED here, not by Part 13
    Z6["gid6"] = Z6["anchor_snap"].astype(str) + ":" + Z6["gal_id"].astype(str)
    with np.errstate(all="ignore"):
        if "mdust_cgm" in Z6.columns and "m_star_mem" in Z6.columns:
            Z6["lfd_cgm_star"] = np.log10(np.where((Z6["mdust_cgm"] > 0) & (Z6["m_star_mem"] > 0),
                                                   Z6["mdust_cgm"] / Z6["m_star_mem"], np.nan))
    _ng6 = Z6["gid6"].nunique()
    print(f"Part 6: {len(Z6)} of Part 13's nodes match this notebook's {int(_keep6.sum())} {P6_BOX} galaxies "
          f"-> {_ng6} galaxies with a CGM census  ("
          + ", ".join(f"{GSH[g]} {Z6.loc[Z6['grp'] == g, 'gid6'].nunique()}" for g in GK) + ")")
    print(f"        clock = t - t_SFT of THIS notebook;  nodes span {Z6['trel'].min():+.1f} to {Z6['trel'].max():+.1f} Gyr")

    _TC6 = 0.5 * (P6_TB[:-1] + P6_TB[1:])

    def _track6(D, col):
        """One (galaxy x clock-bin) table: the per-galaxy median inside a bin first, so a galaxy with many nodes
        in one bin does not outvote one with a single node.  Then med_band over galaxies."""
        if col not in D.columns:
            return None
        d = D[["gid6", "trel", col]].copy()
        d["ib"] = np.digitize(d["trel"].to_numpy(float), P6_TB) - 1
        d = d[(d["ib"] >= 0) & (d["ib"] < len(_TC6)) & np.isfinite(d[col].to_numpy(float))]
        if not len(d):
            return None
        P = d.groupby(["gid6", "ib"])[col].median().unstack("ib")
        return P.reindex(columns=np.arange(len(_TC6))).to_numpy(float), P.index.to_numpy()

    ROW6 = []
    _Q6 = [(c, l) for c, l in P6_Q if c in Z6.columns]
    _miss = [c for c, _ in P6_Q if c not in Z6.columns]
    if _miss:
        print(f"        columns absent from the census and therefore not drawn: {_miss}")
    fig, AX = plt.subplots(2, 4, figsize=(17.2, 6.8), sharex=True)
    for _i, (_c, _lab) in enumerate(_Q6[:8]):
        ax = AX[_i // 4, _i % 4]
        for _g in GK:
            _r = _track6(Z6[Z6["grp"] == _g], _c)
            if _r is None:
                continue
            _A, _ids = _r
            _cen, _lo, _hi, _n = med_band(_A, nmin=P6_NMIN, nboot=P6_NBOOT, grid=_TC6)
            ax.fill_between(_TC6, _lo, _hi, color=GCOL[_g], alpha=0.20, lw=0)
            ax.plot(_TC6, _cen, color=GCOL[_g], lw=2.4, marker="o", ms=3.5)
            for _t, _y, _nn in zip(_TC6, _cen, _n):
                if np.isfinite(_y):
                    ROW6.append(dict(block="clock", quantity=_c, grp=_g, t=_t, med=_y, n=int(_nn)))
        ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.7)
        ax.axvspan(*P6_POST, color="0.85", alpha=0.35, lw=0, zorder=0)
        ax.set_ylabel(_lab, fontsize=9)
        if _i // 3 == 1:
            ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]")
    AX[0, 0].legend(handles=[Line2D([], [], color=GCOL[g], lw=2.4, label=GLAB[g]) for g in GK]
                    + [Line2D([], [], color="0.85", lw=8, label=f"the scoring window {P6_POST}")],
                    fontsize=7.5, frameon=False, loc="best")
    fig.suptitle(f"The CGM of the two groups (m25, {_ng6} galaxies) — Part 13's census, re-clocked and re-split",
                 y=0.997, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.965])
    paper_save(fig, "ba_cgm"); plt.show()

    # ── (a) is the post-starburst CGM richer? ────────────────────────────────────────────────────────────
    print(f"\nIS THE CGM RICHER AROUND ONE GROUP?   one value per galaxy = its median over t - t_SFT in "
          f"{P6_POST[0]:+.1f} .. {P6_POST[1]:+.1f} Gyr")
    _W6 = Z6[(Z6["trel"] >= P6_POST[0]) & (Z6["trel"] <= P6_POST[1])]
    _cols6 = [c for c in ["f_dust_cgm", "lfd_cgm_star", "ldg_cgm", "fm_cold", "lT_cgm_dw", "lT_cgm_mw",
                          "lmdot_out_g", "lmdot_in_gc", "f_gas_cgm"] if c in _W6.columns]
    _gw6 = max(11, max(len(v) for v in GSH.values()) + 1)
    print(f"    {'quantity':16s} {GSH['burst']:>{_gw6}s} {GSH['aging']:>{_gw6}s} {'AUC(hi)':>11s} {'p':>9s} "
          f"{'n hi/lo':>10s}")
    for _c in _cols6:
        _g6 = _W6.groupby(["gid6", "grp"])[_c].median().reset_index()
        _y, _m = _g6[_c].to_numpy(float), (_g6["grp"] == "burst").to_numpy()
        _a, _p, _n1, _n0 = mw_auc(_y, _m)
        print(f"    {_c:16s} {np.nanmedian(_y[_m]):{_gw6}.3f} {np.nanmedian(_y[~_m]):{_gw6}.3f} {_a:11.2f} "
              + (f"{_p:9.3g}" if np.isfinite(_p) else f"{'n/a':>9s}") + f" {_n1:4d}/{_n0:<5d}")
        ROW6.append(dict(block="post", quantity=_c, med_burst=np.nanmedian(_y[_m]), med_aging=np.nanmedian(_y[~_m]),
                         auc=_a, p=_p, n_burst=_n1, n_aging=_n0))
    print(f"      (AUC > 0.5 = the {GSH['burst']} group is higher.  With ~40 galaxies a single AUC is suggestive at best;\n"
          "       read it together with the clock figure, which uses every node.)")

    # ── (b) the conditional an observation gives ─────────────────────────────────────────────────────────
    _anc6 = Z6.assign(_d=np.abs(Z6["t"].to_numpy(float) - Z6["t_anchor"].to_numpy(float))) \
               .sort_values("_d").groupby("gid6").head(1)
    _cut6 = float(np.nanmedian(_anc6["fm_cold"])) if P6_COLD is None else float(P6_COLD)
    _cold = _anc6["fm_cold"].to_numpy(float) >= _cut6
    _dust = _anc6["lfd_cat"].to_numpy(float) >= P6_DUSTY if "lfd_cat" in _anc6.columns else \
            _anc6["lfd_anc"].to_numpy(float) >= P6_DUSTY
    _ag6 = (_anc6["grp"] == "aging").to_numpy()
    print(f"\nAT THE ANCHOR ({len(_anc6)} galaxies, the census node nearest each anchor):")
    print(f"    cold-rich  =  cold mass share of the CGM >= {_cut6:.3f}"
          + ("  (the m25 median)" if P6_COLD is None else "")
          + f"      dusty  =  log M_dust/M* >= {P6_DUSTY:g}")
    for _nm, _c, _cnd in (("P(cold-rich CGM | dusty QG)", _cold, _dust), ("P(dusty QG | cold-rich CGM)", _dust, _cold),
                          ("P(cold-rich CGM | dust-poor QG)", _cold, ~_dust), ("P(dusty QG | warm/hot CGM)", _dust, ~_cold)):
        _p, _lo, _hi = wilson(int((_c & _cnd).sum()), int(_cnd.sum()))
        _b0, _, _ = wilson(int(_c.sum()), len(_c))
        print(f"    {_nm:34s} {_p:.2f} [{_lo:.2f}, {_hi:.2f}]   n = {int(_cnd.sum()):3d}"
              f"    (base rate {_b0:.2f})")
        ROW6.append(dict(block="cond", quantity=_nm, p=_p, lo=_lo, hi=_hi, n=int(_cnd.sum()), base=_b0))
    for _g in GK:
        _m = (_anc6["grp"] == _g).to_numpy()
        _p, _lo, _hi = wilson(int((_cold & _m).sum()), int(_m.sum()))
        print(f"    P(cold-rich CGM | {GSH[_g]:9s})       {_p:.2f} [{_lo:.2f}, {_hi:.2f}]   n = {int(_m.sum()):3d}")
        ROW6.append(dict(block="cond", quantity=f"cold|{_g}", p=_p, lo=_lo, hi=_hi, n=int(_m.sum())))
    _ct6 = pd.crosstab(pd.Series(np.where(_dust, "dusty", "dust-poor"), name="ISM"),
                       pd.Series(np.where(_cold, "cold-rich CGM", "warm/hot CGM"), name="CGM"))
    print("\n    the 2x2 the conditionals come from:\n" + _ct6.to_string().replace("\n", "\n    "))
    pd.DataFrame(ROW6).to_csv(os.path.join(OUT, "ba_cgm.csv"), index=False)
    _anc6.to_csv(os.path.join(OUT, "ba_cgm_anchor.csv"), index=False)
    print("\ntables ->", os.path.join(OUT, "ba_cgm{,_anchor}.csv"))

## Part 6b — the grain clocks: growth and sputtering, in the ISM and in the CGM

A **pure re-read** of `dusty_qgs_kappa_selection` Part 17's census — the timescales computed **per particle**
and dust-mass weighted inside each zone of each galaxy at each snapshot — re-split on this notebook's
selection and re-clocked on its own $t_{\rm SFT}$. Nothing is recomputed here.

$$\tau_{\rm grow} = \tau_{\rm ref}\,\frac{a}{0.1\,\mu{\rm m}}
\left(\frac{n_{\rm H}}{n_{\rm ref}}\right)^{-1}\left(\frac{T}{T_{\rm ref}}\right)^{-1/2}
\left(\frac{Z}{Z_\odot}\right)^{-1},\qquad
\tau_{\rm sp} = 0.17\ {\rm Gyr}\;\frac{a}{0.1\,\mu{\rm m}}\;\frac{10^{-27}}{\rho}
\left[\left(\frac{T_0}{T}\right)^{2.5} + 1\right]$$

the first from Hirashita (2000) / Asano+ (2013) in the form Li, Narayanan & Davé (2019) give for SIMBA, the
second Tsai & Mathews (1995) — SIMBA's own sputtering law.

**Two panels are physically empty and are labelled rather than drawn.** The star-forming ISM has no
temperature in SIMBA (it sits on an effective equation of state), so its **sputtering** time means nothing;
its growth time is kept, because that is where the dust is actually made. And above $10^5$ K the sticking of
a metal atom onto a grain collapses, so $\tau_{\rm grow}$ in the **hot CGM** is a formal number and is left
blank. Both conventions are Part 17's own, kept identical so the two notebooks cannot disagree.

**How to read it.** $\log \tau$ is in Gyr: 0 is 1 Gyr (dashed), 1.14 a Hubble time (dotted). $\tau_{\rm sp}$
is the clock of the grain **radius** — the mass goes as $a^3$, so the mass e-folds in $\tau_{\rm sp}/3$,
half a dex faster than the line drawn. Both laws are **linear in the assumed grain size**, so what carries
meaning is the contrast between the groups and the distance from the two reference lines, never the absolute
value.

**ISM and CGM are defined by caesar's member list, not by a radius**: the ISM is the gas particles caesar
assigns to the galaxy, the CGM is everything else within the cut-out's 100 kpc aperture, and the phases are
$T < 10^5$ K (cold), $10^5$–$10^6$ K (warm), $> 10^6$ K (hot).


In [ ]:
# ── Part 6b — THE GRAIN CLOCKS: growth and sputtering, in the ISM and in the CGM, on the clock (m25) ───────
# A PURE RE-READ of dusty_qgs_kappa_selection Part 17's census — the per-PARTICLE timescales, dust-mass
# weighted inside each zone of each galaxy at each snapshot — re-split on THIS notebook's selection and
# RE-CLOCKED on THIS notebook's t_SFT.  Nothing is recomputed here and no particle file is opened.
#
#   tau_grow = P17_TAUG (a/0.1um) (n_H/n_ref)^-1 (T/T_ref)^-1/2 (Z/Zsun)^-1     Hirashita 2000 / Asano+13,
#              in the form Li, Narayanan & Dave (2019) give for SIMBA
#   tau_sp   = 0.17 Gyr (a/0.1um) (1e-27/rho) [(T_0/T)^2.5 + 1]                 Tsai & Mathews (1995),
#              SIMBA's own; it is the timescale of the grain RADIUS, so the MASS e-folds in tau_sp/3
#
# TWO ZONES ARE PHYSICALLY EMPTY AND ARE LABELLED, NOT DRAWN:
#   * the STAR-FORMING ISM has no temperature in SIMBA — it sits on an effective equation of state — so its
#     sputtering time is meaningless.  Its growth time is kept: that is where the dust is actually made.
#   * ABOVE 1e5 K the sticking of a metal atom onto a grain collapses, so tau_grow in the HOT CGM is a
#     formal number and is deliberately left blank.
# Both conventions are Part 17's own, kept here so the two notebooks cannot disagree.
P6B_NODES = os.path.join(KSEL, "kappa_cis25_dust_regrowth_nodes.csv")   # its Part 17
P6B_BOX   = "cis25"
P6B_TB    = np.arange(-3.0, 4.001, 0.5)      # the clock bins [Gyr]; a per-galaxy median is taken in a bin first
P6B_POST  = (0.5, 2.5)                       # the window the table scores
P6B_NMIN  = 1                                # a bin is drawn only with this many galaxies in it
P6B_UNIQ  = True
P6B_HUB   = np.log10(13.8)                   # the dotted line: a Hubble time
P6B_ZONES = [("ism_nsf", "the non-star-forming ISM"), ("ism_sf", "the star-forming ISM"),
             ("cgm_cold", r"the cold CGM ($T < 10^5$ K)"), ("cgm_hot", r"the hot CGM ($T > 10^6$ K)")]
P6B_SKIP  = {("lts", "ism_sf"): "no temperature\n(effective EoS)",
             ("ltg", "cgm_hot"): "formal only\n($T > 10^5$ K)"}

if not os.path.exists(P6B_NODES):
    print(f"Part 6b: {P6B_NODES} is missing — dusty_qgs_kappa_selection Part 17 builds it (cluster only, it needs\n"
          f"         the reduced particle cut-outs).  Skipped; every other part is unaffected.")
elif P6B_BOX not in _BX1:
    print(f"Part 6b: {P6B_BOX} is not in this run's boxes ({_BX1}) — skipped.")
else:
    Z6B = pd.read_csv(P6B_NODES)
    _F6B = B1[P6B_BOX]["F"]
    _k6b = (_F6B["uniq"].to_numpy(bool) if P6B_UNIQ else np.ones(len(_F6B), bool)) & (_F6B["grp"].to_numpy() != "")
    _K6B = _F6B.loc[_k6b, ["snap", "gal_id", "grp", "t_sft"]].rename(columns={"snap": "anchor_snap"})
    Z6B = Z6B.drop(columns=[c for c in ("grp", "t_sft", "cls", "dusty") if c in Z6B.columns])
    Z6B = Z6B.merge(_K6B, on=["anchor_snap", "gal_id"], how="inner")
    Z6B["trel"] = Z6B["t"].to_numpy(float) - Z6B["t_sft"].to_numpy(float)
    Z6B["gid"] = Z6B["anchor_snap"].astype(str) + ":" + Z6B["gal_id"].astype(str)
    _TC6B = 0.5 * (P6B_TB[:-1] + P6B_TB[1:])
    print(f"Part 6b — THE GRAIN CLOCKS ({P6B_BOX}): {len(Z6B)} of Part 17's nodes match "
          f"{Z6B['gid'].nunique()} of this notebook's galaxies  ("
          + ", ".join(f"{GSH[g]} {Z6B.loc[Z6B['grp'] == g, 'gid'].nunique()}" for g in GK) + ")")

    def _trk6b(D, col):
        """(galaxy x clock-bin) medians, so a galaxy with many nodes in one bin does not outvote one with a
        single node — the same rule Part 6 uses."""
        if col not in D.columns:
            return None
        d = D[["gid", "trel", col]].copy()
        d["ib"] = np.digitize(d["trel"].to_numpy(float), P6B_TB) - 1
        d = d[(d["ib"] >= 0) & (d["ib"] < len(_TC6B)) & np.isfinite(d[col].to_numpy(float))]
        if not len(d):
            return None
        P = d.groupby(["gid", "ib"])[col].median().unstack("ib")
        return P.reindex(columns=np.arange(len(_TC6B))).to_numpy(float)

    ROW6B = []
    fig, AX = plt.subplots(2, len(P6B_ZONES), figsize=(3.6 * len(P6B_ZONES), 6.6), squeeze=False, sharex=True)
    for _r, (_pre, _rlab) in enumerate((("ltg", r"$\tau_{\rm grow}$  [Gyr]"),
                                        ("lts", r"$\tau_{\rm sp}$  [Gyr]"))):
        for _c, (_zn, _zlab) in enumerate(P6B_ZONES):
            ax = AX[_r, _c]
            _skip = P6B_SKIP.get((_pre, _zn))
            if _skip is not None:
                ax.text(0.5, 0.5, _skip, transform=ax.transAxes, ha="center", va="center", fontsize=10,
                        color="0.35")
            else:
                for _g in GK:
                    _A = _trk6b(Z6B[Z6B["grp"] == _g], f"{_pre}_{_zn}")
                    if _A is None or not len(_A):
                        continue
                    _cen, _lo, _hi, _n = med_band(_A, nmin=P6B_NMIN, nboot=NBOOT, grid=_TC6B)
                    # the census carries log10 tau; the axis is drawn in GYR and made logarithmic instead,
                    # so the numbers on it are the timescale itself and can be read against 1 Gyr directly
                    ax.fill_between(_TC6B, 10 ** _lo, 10 ** _hi, color=GCOL[_g], alpha=0.20, lw=0)
                    ax.plot(_TC6B, 10 ** _cen, color=GCOL[_g], lw=2.4, marker="o", ms=3.5)
                    for _t, _y, _nn in zip(_TC6B, _cen, _n):
                        if np.isfinite(_y):
                            ROW6B.append(dict(block="clock", quantity=f"{_pre}_{_zn}", grp=_g, t=_t, med=_y,
                                              n=int(_nn)))
                ax.axhline(1.0, color="0.4", ls="--", lw=1.1)          # 1 Gyr
                ax.axhline(10 ** P6B_HUB, color="0.4", ls=":", lw=1.1)  # a Hubble time
                ax.set_yscale("log")
            ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.7)
            if _r == 0:
                ax.set_title(_zlab, fontsize=10)
            if _r == 1:
                ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]")
            ax.set_ylabel(_rlab if _c == 0 else "")
    fig.suptitle("The two clocks of a grain, where the dust actually is — Part 17's per-particle timescales, "
                 "dust-mass weighted, on this notebook's selection", y=0.999, fontsize=11)
    fig.tight_layout(rect=[0, 0.045, 1, 0.965], h_pad=2.2)
    fig.legend(handles=[Line2D([], [], color=GCOL[g], lw=2.4, label=GLAB[g]) for g in GK]
               + [Line2D([], [], color="0.4", ls="--", lw=1.2, label="1 Gyr"),
                  Line2D([], [], color="0.4", ls=":", lw=1.2, label="a Hubble time")],
               loc="lower center", ncol=4, fontsize=9, frameon=False)
    paper_save(fig, "ba_grain_clocks"); plt.show()

    # ── the same, as numbers, in the window after the truncation ─────────────────────────────────────────
    print(f"\nIN THE WINDOW {P6B_POST[0]:+g} .. {P6B_POST[1]:+g} Gyr AFTER THE TRUNCATION  (one value per galaxy = "
          f"its median there):")
    _gw6b = max(12, max(len(v) for v in GSH.values()) + 1)
    print(f"    {'zone':26s} {'clock':12s}" + "".join(f"{GSH[g]:>{_gw6b}s}" for g in GK)
          + f"{'AUC':>8s}{'p':>9s}{'n':>10s}")
    _W6B = Z6B[(Z6B["trel"] >= P6B_POST[0]) & (Z6B["trel"] <= P6B_POST[1])]
    for _zn, _zlab in P6B_ZONES:
        for _pre, _plab in (("ltg", "tau_grow"), ("lts", "tau_sp")):
            _col = f"{_pre}_{_zn}"
            if _col not in _W6B.columns or P6B_SKIP.get((_pre, _zn)) is not None:
                continue
            _g1 = _W6B.groupby(["gid", "grp"])[_col].median().reset_index()
            _m = {g: _g1.loc[_g1["grp"] == g, _col].to_numpy(float) for g in GK}
            _a, _p, _n1, _n0 = mw_auc(_g1[_col].to_numpy(float), (_g1["grp"] == "burst").to_numpy())
            print(f"    {_zlab[:24]:26s} {_plab:12s}"
                  + "".join(f"{np.nanmedian(_m[g]) if len(_m[g]) else np.nan:{_gw6b}.2f}" for g in GK)
                  + (f"{_a:8.2f}{_p:9.1g}" if np.isfinite(_a) else f"{'n/a':>8s}{'':9s}")
                  + f"{_n1:5d}/{_n0:<4d}")
            ROW6B.append(dict(block="post", quantity=_col, zone=_zn, clock=_plab, auc=_a, p=_p,
                              med_burst=np.nanmedian(_m["burst"]) if len(_m["burst"]) else np.nan,
                              med_aging=np.nanmedian(_m["aging"]) if len(_m["aging"]) else np.nan,
                              n_burst=_n1, n_aging=_n0))
    print("      the table stays in log10(tau/Gyr): 0 = 1 Gyr, 1.14 = a Hubble time (the figure is in Gyr).  tau_sp is the RADIUS clock — the MASS\n"
          "      e-folds in tau_sp/3, i.e. 0.5 dex faster than the number printed.  Both laws are LINEAR in\n"
          "      the assumed grain size, so read the CONTRAST between the groups, not the absolute value.")
    pd.DataFrame(ROW6B).to_csv(os.path.join(OUT, "ba_grain_clocks.csv"), index=False)
    print("table ->", os.path.join(OUT, "ba_grain_clocks.csv"))

## Part 6c — where the CGM dust sits, and whether it can come back

**The hypothesis this cell exists to test.** The dust-poor galaxies may have no maintenance heating at all:
their CGM is cold, their dust is out there, nothing much flows either way, and the grains simply sit where
they were put. The dusty ones would then be the ones *actively* heating their CGM while still cycling cold
gas back in, so the dust returns. If that is right, the difference is not **how much** dust is outside — it
is **how far out** it is.

Four measurements of "how far out", all of them Part 13's, none recomputed here:

| | |
|---|---|
| $R_{50}$ | the radius holding half the CGM dust, built from the four shells (0–10, 10–30, 30–60, 60–100 kpc) |
| $f_{\rm close}$ | the share of the CGM dust inside 30 kpc |
| $r_{\rm apo}$ | the mass-weighted median **apocentre** of the CGM dust that is moving outward — where it is going |
| $f_{\rm bound}$ | the share of the CGM dust that is gravitationally **bound** — whether it can come back at all |

The top row puts those four on the clock. The bottom row is the radial profile itself, measured in the window
after the truncation: the dust, the gas, the temperature and the dust-to-gas ratio, shell by shell — so
"the cold CGM is closer in the dusty ones" is a statement you can read off a curve rather than infer.

**What the hypothesis predicts**, and the printed table scores each one: a **smaller** $R_{50}$ and apocentre
and a **larger** bound and close-in share for the dusty group (its dust never got far), a **hotter** inner
CGM (it is being maintained), and a **larger cold inflow**. The X-ray and choked-jet duty cycles are in the
same table, and in Part 6a's clock figure, so "actively heating" is a measured share of the group and not an
assumption.


In [ ]:
# ── Part 6c — WHERE the CGM dust sits, and whether it can come back (m25) ─────────────────────────────────
# The hypothesis this cell exists to test (the user's, 2026-09-11): the dust-poor galaxies have no maintenance
# heating at all — their CGM is cold, their dust is out there, nothing flows either way, and the grains simply
# sit where they were put; the dusty ones are actively heating their CGM but are also cycling cold gas back
# in, so the dust returns.  If that is right the difference is not HOW MUCH dust is outside but HOW FAR OUT
# it is: the dust-poor galaxies' first, strong ejection put it beyond return, the dusty ones keep it close.
#
# Four measurements of "how far out", all of them Part 13's, none recomputed here:
#   R_d50      the radius containing half the CGM dust, from the four shells (0-10, 10-30, 30-60, 60-100 kpc)
#   f_close    the share of the CGM dust inside 30 kpc
#   r_apo      the mass-weighted median APOCENTRE of the CGM dust that is moving outward — where it is going
#   f_bound    the share of the CGM dust that is gravitationally bound — whether it can come back at all
# and the radial profiles themselves in the window after the truncation: the dust, the gas, the temperature
# and the dust-to-gas ratio, shell by shell.
P6C_NODES = P6B_NODES if "P6B_NODES" in dir() else os.path.join(KSEL, "kappa_cis25_dust_regrowth_nodes.csv")
P6C_BOX   = "cis25"
P6C_TB    = np.arange(-3.0, 4.001, 0.5)
P6C_POST  = (0.5, 2.5)                    # the window the profiles and the table are measured in
P6C_NMIN  = 3
P6C_SH    = [(0.0, 10.0, "s10"), (10.0, 30.0, "s30"), (30.0, 60.0, "s60"), (60.0, 100.0, "s100")]
P6C_CLOSE = 30.0                          # [kpc] "close" = inside this radius

if not os.path.exists(P6C_NODES):
    print(f"Part 6c: {P6C_NODES} is missing — skipped.")
elif P6C_BOX not in _BX1:
    print(f"Part 6c: {P6C_BOX} is not in this run's boxes — skipped.")
else:
    Z6C = pd.read_csv(P6C_NODES)
    _F6C = B1[P6C_BOX]["F"]
    _k6c = _F6C["uniq"].to_numpy(bool) & (_F6C["grp"].to_numpy() != "")
    _K6C = _F6C.loc[_k6c, ["snap", "gal_id", "grp", "t_sft"]].rename(columns={"snap": "anchor_snap"})
    Z6C = Z6C.drop(columns=[c for c in ("grp", "t_sft", "cls", "dusty") if c in Z6C.columns])
    Z6C = Z6C.merge(_K6C, on=["anchor_snap", "gal_id"], how="inner")
    Z6C["trel"] = Z6C["t"].to_numpy(float) - Z6C["t_sft"].to_numpy(float)
    Z6C["gid"] = Z6C["anchor_snap"].astype(str) + ":" + Z6C["gal_id"].astype(str)
    _TC6C = 0.5 * (P6C_TB[:-1] + P6C_TB[1:])
    _RE = np.array([0.0] + [b for _, b, _ in P6C_SH])                     # the shell edges, 0 -> 100 kpc
    _RC = np.array([0.5 * (a + b) for a, b, _ in P6C_SH])                 # ... and their centres

    def _prof(D, kind):
        """The four shell masses of each node as an (n, 4) block."""
        return np.column_stack([D[f"m{kind}_{s}"].to_numpy(float) for _, _, s in P6C_SH])

    with np.errstate(all="ignore"):
        _Md, _Mg = _prof(Z6C, "dust"), _prof(Z6C, "gas")
        _cum = np.cumsum(np.nan_to_num(_Md), axis=1)
        _tot = _cum[:, -1]
        _fr = np.where(_tot[:, None] > 0, _cum / np.where(_tot[:, None] > 0, _tot[:, None], 1.0), np.nan)
        # R_d50: where the cumulative dust profile crosses a half, linear in radius inside the shell it does
        _R50 = np.full(len(Z6C), np.nan)
        for _i in range(len(Z6C)):
            if not np.isfinite(_fr[_i]).all() or _tot[_i] <= 0:
                continue
            _y = np.concatenate([[0.0], _fr[_i]])
            _j = int(np.searchsorted(_y, 0.5))
            if 1 <= _j <= 4 and _y[_j] > _y[_j - 1]:
                _R50[_i] = _RE[_j - 1] + (0.5 - _y[_j - 1]) / (_y[_j] - _y[_j - 1]) * (_RE[_j] - _RE[_j - 1])
        Z6C["R_d50"] = _R50
        Z6C["f_close"] = np.where(_tot > 0, np.nan_to_num(_Md[:, 0] + _Md[:, 1]) / np.where(_tot > 0, _tot, 1.0), np.nan)
        Z6C["r_apo"] = Z6C["rapo50_d_cgm"].to_numpy(float) if "rapo50_d_cgm" in Z6C.columns else np.nan
        Z6C["f_bound"] = Z6C["fbnd_d_cgm"].to_numpy(float) if "fbnd_d_cgm" in Z6C.columns else np.nan
    print(f"Part 6c — WHERE THE CGM DUST SITS ({P6C_BOX}): {len(Z6C)} nodes, {Z6C['gid'].nunique()} galaxies  ("
          + ", ".join(f"{GSH[g]} {Z6C.loc[Z6C['grp'] == g, 'gid'].nunique()}" for g in GK) + ")")

    def _trk6c(D, col):
        if col not in D.columns:
            return None
        d = D[["gid", "trel", col]].copy()
        d["ib"] = np.digitize(d["trel"].to_numpy(float), P6C_TB) - 1
        d = d[(d["ib"] >= 0) & (d["ib"] < len(_TC6C)) & np.isfinite(d[col].to_numpy(float))]
        if not len(d):
            return None
        P = d.groupby(["gid", "ib"])[col].median().unstack("ib")
        return P.reindex(columns=np.arange(len(_TC6C))).to_numpy(float)

    ROW6C = []
    _W6C = Z6C[(Z6C["trel"] >= P6C_POST[0]) & (Z6C["trel"] <= P6C_POST[1])]
    fig, AX = plt.subplots(2, 4, figsize=(16.8, 7.2))
    # ── row 1: how far out the dust is, against the clock ────────────────────────────────────────────────
    for _i, (_c, _lab, _logy) in enumerate((("R_d50", r"$R_{50}$ of the CGM dust  [kpc]", False),
                                            ("f_close", rf"share of the CGM dust inside {P6C_CLOSE:g} kpc", False),
                                            ("r_apo", r"$r_{\rm apo}$ of the outgoing dust  [kpc]", True),
                                            ("f_bound", "bound share of the CGM dust", False))):
        ax = AX[0, _i]
        for _g in GK:
            _A = _trk6c(Z6C[Z6C["grp"] == _g], _c)
            if _A is None:
                continue
            _cen, _lo, _hi, _n = med_band(_A, nmin=P6C_NMIN, nboot=NBOOT, grid=_TC6C)
            ax.fill_between(_TC6C, _lo, _hi, color=GCOL[_g], alpha=0.20, lw=0)
            ax.plot(_TC6C, _cen, color=GCOL[_g], lw=2.4, marker="o", ms=3.5)
            for _t, _y, _nn in zip(_TC6C, _cen, _n):
                if np.isfinite(_y):
                    ROW6C.append(dict(block="clock", quantity=_c, grp=_g, t=_t, med=_y, n=int(_nn)))
        if _logy:
            ax.set_yscale("log")
        ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.7)
        ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]"); ax.set_ylabel(_lab, fontsize=9.5)
    # ── row 2: the radial profiles themselves, in the window after the truncation ────────────────────────
    for _i, (_kind, _lab, _logy) in enumerate((("dust", "share of the CGM dust in the shell", False),
                                               ("gas", "share of the CGM gas in the shell", False),
                                               ("T", r"$\langle \log T \rangle$ of the shell", False),
                                               ("dg", r"$\log M_{\rm dust}/M_{\rm gas}$ of the shell", False))):
        ax = AX[1, _i]
        for _g in GK:
            _D = _W6C[_W6C["grp"] == _g]
            if _D["gid"].nunique() < P6C_NMIN:
                continue
            _V = []
            for _gid, _d in _D.groupby("gid"):
                _md, _mg = _prof(_d, "dust"), _prof(_d, "gas")
                with np.errstate(all="ignore"):
                    if _kind == "dust":
                        _v = np.nan_to_num(_md).sum(0) / max(np.nan_to_num(_md).sum(), 1e-30)
                    elif _kind == "gas":
                        _v = np.nan_to_num(_mg).sum(0) / max(np.nan_to_num(_mg).sum(), 1e-30)
                    elif _kind == "T":
                        _v = np.array([np.nanmedian(_d[f"lT_{s}"].to_numpy(float)) for _, _, s in P6C_SH])
                    else:
                        _s1, _s2 = np.nan_to_num(_md).sum(0), np.nan_to_num(_mg).sum(0)
                        _v = np.log10(np.where((_s1 > 0) & (_s2 > 0), _s1 / np.where(_s2 > 0, _s2, 1), np.nan))
                _V.append(_v)
            _V = np.asarray(_V, float)
            _m = np.array([np.nanmedian(_V[:, k]) if np.isfinite(_V[:, k]).sum() >= P6C_NMIN else np.nan
                           for k in range(len(_RC))])
            _q1 = np.array([np.nanpercentile(_V[:, k][np.isfinite(_V[:, k])], 16)
                            if np.isfinite(_V[:, k]).sum() >= P6C_NMIN else np.nan for k in range(len(_RC))])
            _q2 = np.array([np.nanpercentile(_V[:, k][np.isfinite(_V[:, k])], 84)
                            if np.isfinite(_V[:, k]).sum() >= P6C_NMIN else np.nan for k in range(len(_RC))])
            ax.fill_between(_RC, _q1, _q2, color=GCOL[_g], alpha=0.18, lw=0)
            ax.plot(_RC, _m, color=GCOL[_g], lw=2.4, marker="o", ms=5)
            for _r, _y in zip(_RC, _m):
                if np.isfinite(_y):
                    ROW6C.append(dict(block="profile", quantity=_kind, grp=_g, r=_r, med=_y,
                                      n=int(_D["gid"].nunique())))
        for _a, _b, _ in P6C_SH:
            ax.axvline(_b, color="0.85", lw=0.8, zorder=0)
        ax.set_xlabel("radius of the shell  [kpc]"); ax.set_ylabel(_lab, fontsize=9.5)
    fig.suptitle(f"Where the CGM dust is, and whether it can return — {P6C_BOX}, the profiles measured in "
                 f"{P6C_POST[0]:+g} .. {P6C_POST[1]:+g} Gyr after the truncation", y=0.999, fontsize=11)
    fig.tight_layout(rect=[0, 0.045, 1, 0.965], h_pad=2.6)
    fig.legend(handles=[Line2D([], [], color=GCOL[g], lw=2.4, label=GLAB[g]) for g in GK],
               loc="lower center", ncol=2, fontsize=9, frameon=False)
    paper_save(fig, "ba_cgm_where"); plt.show()

    print(f"\nIN {P6C_POST[0]:+g} .. {P6C_POST[1]:+g} Gyr  (one value per galaxy = its median there):")
    _gw6c = max(12, max(len(v) for v in GSH.values()) + 1)
    print(f"    {'quantity':34s}" + "".join(f"{GSH[g]:>{_gw6c}s}" for g in GK) + f"{'AUC':>8s}{'p':>9s}{'n':>10s}")
    for _c, _lab in (("R_d50", "R50 of the CGM dust [kpc]"), ("f_close", f"dust inside {P6C_CLOSE:g} kpc"),
                     ("r_apo", "apocentre of the outgoing dust [kpc]"), ("f_bound", "bound share of that dust"),
                     ("lT_s30", "log T of the 10-30 kpc shell"), ("lT_s100", "log T of the 60-100 kpc shell"),
                     ("xray_on", "the X-ray channel on"), ("choked", "the jet on but choked"),
                     ("lmdot_in_gc", "log cold inflow through 10 kpc"),
                     ("ltg_cgm_cold", "log tau_grow, the cold CGM")):
        if _c not in _W6C.columns:
            continue
        _g1 = _W6C.groupby(["gid", "grp"])[_c].median().reset_index()
        _v = {g: _g1.loc[_g1["grp"] == g, _c].to_numpy(float) for g in GK}
        _a, _p, _n1, _n0 = mw_auc(_g1[_c].to_numpy(float), (_g1["grp"] == "burst").to_numpy())
        print(f"    {_lab:34s}" + "".join(f"{np.nanmedian(_v[g]) if len(_v[g]) else np.nan:{_gw6c}.2f}" for g in GK)
              + (f"{_a:8.2f}{_p:9.1g}" if np.isfinite(_a) else f"{'n/a':>8s}{'':9s}") + f"{_n1:5d}/{_n0:<4d}")
        ROW6C.append(dict(block="post", quantity=_c, auc=_a, p=_p, n_burst=_n1, n_aging=_n0,
                          med_burst=np.nanmedian(_v["burst"]) if len(_v["burst"]) else np.nan,
                          med_aging=np.nanmedian(_v["aging"]) if len(_v["aging"]) else np.nan))
    print("      AUC > 0.5 = the upper (dusty) group is higher.  THE HYPOTHESIS PREDICTS: a SMALLER R50 and\n"
          "      apocentre and a LARGER bound share and close-in dust share for the dusty group (its dust never\n"
          "      got far), a HOTTER inner CGM (it is being maintained) and a LARGER cold inflow.")
    pd.DataFrame(ROW6C).to_csv(os.path.join(OUT, "ba_cgm_where.csv"), index=False)
    print("table ->", os.path.join(OUT, "ba_cgm_where.csv"))

## Part 7 — the 25 Mpc close-up: the critical points, the radii, and the K-S plane

**Ported from `dusty_qgs_kappa_selection` Part 5 and re-split on this notebook's selection.** It is the only
part that reaches outside the `ism_prediction` products: it reads the m25 notebook's own particle-level
tables and the K-S track products, and each block gates on its own files, so the cell runs unchanged off the
cluster and says what is missing.

* **(A) the critical points.** $\kappa_{\rm rot}^{\rm H_2}$, $\kappa_{\rm rot}^{\star}$, sSFR, stellar age,
  $f_{\rm dust}$ and $f_{\rm H_2}$ at ignition → jet → SFT → QT → end, in the two spherical zones of the K-S
  products (core $r < 3.2$ kpc, outskirt 3.2–10 kpc), with the ALMA-C11 detections as bands. **Read it as the
  timing question**: if the selection were a property the galaxy always had, the two curves would already be
  apart at ignition. If they only separate at QT or at the end, it is *made* during the quenching.
* **(B) the radii.** $A_V$, $M_{\rm dust}/M_\star$, $M_{\rm H_2}/M_\star$, stellar age and $\kappa_{\rm rot}$
  per annulus (0–1, 1–3.2, 3.2–10, 10–32 kpc), per-galaxy sightline medians, with the m25 star-forming
  galaxies as the grey reference — so "the difference is in the ISM, not in the stellar structure" is
  something you read off the same figure twice.
* **(C) the K-S plane.** The $R_{50}({\rm H_2})$ tracks SFT → QT → end against Bigiel+08, with the track-end
  regions counted.

**Nothing here is circular.** The groups come from the dust map of Part 1b — a catalogue-level quantity on a
clock — and every quantity drawn here is measured from the *particles*, in apertures, by a different
notebook. The knobs keep the `P7_` prefix (this notebook's Part 5 is the dust lifetime and owns `P5_`), and
`P7_M25` points at the m25 product tree.


In [ ]:
# ── Part 7 — THE 25 Mpc CLOSE-UP of the selection: the critical points, the radii, and the K-S plane ──
# PORTED from dusty_qgs_kappa_selection Part 5 and re-split on THIS notebook's selection (Part 1b's peel).
# It is the only part that reaches outside the ism_prediction products: it reads the m25 notebook's own
# particle-level tables and the K-S track products.  Each block gates on its own file and prints what is
# missing, so the cell runs unchanged off the cluster.
# The only part that reaches outside the ism_prediction products: it reads the m25 notebook's own particle-level tables and the
# K-S track products. Each block gates on its own file and prints what is missing, so the cell runs unchanged off the cluster.
#   (A) THE CRITICAL POINTS   kappa_rot(H2) / kappa_rot(*) / sSFR / stellar age / f_dust / f_H2 at ignition -> jet -> SFT -> QT ->
#       end, in the two spherical zones of the K-S products (core r < 3.2 kpc, outskirt 3.2-10 kpc), with the ALMA-C11
#       detections as bands. The question the sequence answers: is the SELECTION already there at ignition, or is it made?
#   (B) THE RADII             A_V, M_dust/M*, M_H2/M*, stellar age and kappa_rot per annulus (0-1, 1-3.2, 3.2-10, 10-32 kpc),
#       per-galaxy sightline medians, with the m25 star-forming galaxies as the grey reference line.
#   (C) THE K-S PLANE         the R50(H2) tracks SFT -> QT -> end of the two groups against Bigiel+08.
P7_BOX    = "cis25"
P7_NMIN   = 3               # galaxies a drawn point needs — deliberately below the notebook's NMIN: m25 is tens of objects
P7_M25    = globals().get("P7_M25_OVERRIDE", os.path.join(os.getcwd(), "output", P7_BOX))
ANN7  = os.path.join(P7_M25, "tables", "annulus_ism_truth.fits")            # m25 Part 8a: M_dust / M_gas / M_H2 per (annulus, sightline)
APT7  = os.path.join(P7_M25, "tables", "aperture_truth.fits")               # m25 Part 7e: M*, age per (annulus, sightline)
AV7   = os.path.join(P7_M25, "tables", "annulus_av_allincl.fits")           # m25 Part 8b: A_V per (annulus, sightline) — optional panel
KIN7  = os.path.join(P7_M25, "tables", "annulus_kinematics.fits")           # m25 Part 8j0: kappa_rot per spherical zone — optional panels
SEL7  = os.path.join(P7_M25, "tables", "powderday_quenched_selection.fits") # m25 Part 3: pop -> the SF-controls line — optional
KS7   = os.path.join(P7_M25, "ks_tracks_pt", "ks_tracks.fits")              # ks_tracks_quenched_m25 Part 4a (run tag 'pt')
KINST7 = os.path.join(P7_M25, "ks_tracks", "ks_stage_kinematics.fits")      # its Part 3b: kappa + stellar age per spherical zone
OBS7  = os.path.join(os.getcwd(), "obs_data", "almac11", "almac11_gas_dust.csv")
ANNULI7, NSTAR_AP_MIN7 = [("ap1kpc", "0-1"), ("ann3kpc", "1-3.2"), ("ann10kpc", "3.2-10"), ("ann32kpc", "10-32")], 20
P7_STAGES_ALL = ["agn_ign", "jet_on", "sft", "qt", "end"]
P7_ZONES  = [("ap3kpc", "core, $r < 3.2$ kpc"), ("ann10kpc", "outskirt, 3.2-10 kpc")]
P7_QUANT  = [("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$", "kappa"), ("kappa_star", r"$\kappa_{\rm rot}^{\star}$", "kappa"),
             ("ssfr", r"sSFR [yr$^{-1}$]", "log"), ("age_mw_gyr", "stellar age [Gyr]" + "\n" + r"(sim: MASS-weighted)", "lin"),
             ("fdust", r"$M_{\rm dust}/M_\star$", "log"), ("fH2", r"$M_{\rm H_2}/M_\star$", "log")]
P7_NSTAR_MIN, P7_NBOOT = 20, 400
P7_TICK   = {"agn_ign": "ign.", "jet_on": "jet", "sft": "SFT", "qt": "QT", "end": "end"}
# THE AGE AXIS IS NOT LIKE FOR LIKE, and the observed band on it is drawn hatched to say so: the simulation's age is caesar's
# mass-weighted age, the ALMA-C11 age is CIGALE's sfhdelayedbq ONSET age. For a delayed-tau SFH the mass-weighted age sits
# roughly 2 tau below the onset age, and the table carries no tau, so the conversion cannot be done here.
P7_OBS_AGE_LIKE = False

# THE CLASSES ARE THIS NOTEBOOK'S SELECTION, not the kinematic one the original cell was written for:
# Part 1b's peel, looked up by (anchor snapshot, galaxy id) exactly the way that cell looked up its own.
G7 = list(GK)
C7 = dict(GCOL)
L7 = dict(GSH)
_F7 = B1[P7_BOX]["F"] if P7_BOX in B1 else None
KEY7 = ({} if _F7 is None else
        {(int(s_), int(i_)): g_ for s_, i_, g_, u_ in zip(_F7["snap"], _F7["gal_id"], _F7["grp"], _F7["uniq"])
         if u_ and g_ in GK})
rows7 = []


def _fr7(path, cols):
    """A FITS table -> a DataFrame of the named columns, masked values as NaN and byte strings decoded."""
    t = Table.read(path); d = {}
    for c in cols:
        v = t[c]; v = v.filled(np.nan) if hasattr(v, "filled") else v
        v = np.asarray(v)
        d[c] = np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v]) if v.dtype.kind in "SU" else v.astype(float)
    return pd.DataFrame(d)


def _bmed7(v, n=P7_NBOOT, seed=5):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < P7_NMIN:
        return np.nan, np.nan, np.nan, len(v)
    bs = np.median(v[np.random.default_rng(seed).integers(0, len(v), (n, len(v)))], axis=1)
    return float(np.median(v)), float(np.percentile(bs, 16)), float(np.percentile(bs, 84)), len(v)


if P7_BOX not in _BX1:
    print(f"Part 7: {P7_BOX} is not in this run's boxes ({_BX1}) — skipped.")
elif not KEY7:
    print(f"Part 7: no {P7_BOX} galaxy carries a group — Part 1b's peel found none there.")
else:
    print(f"Part 7 — {BOX_LABEL[P7_BOX]}: {len(KEY7)} galaxies carry a group — "
          + ", ".join(f"{L7[c]} {sum(v == c for v in KEY7.values())}" for c in G7))

    # ── (A) the critical-point sequences ──
    if not (os.path.exists(KS7) and os.path.exists(KINST7)):
        print(f"\n(A) critical points: {KS7 if not os.path.exists(KS7) else KINST7} missing — ks_tracks_quenched_m25 "
              "Part 4a (run tag 'pt') and Part 3b build them (cluster); skipped")
    else:
        T7 = Table.read(KS7)
        _s7 = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
        stg, pop, ap, gk = _s7(T7["stage"]), _s7(T7["pop"]), _s7(T7["ap_label"]), _s7(T7["gkey"])
        P7_STAGES = [st for st in P7_STAGES_ALL if st in set(stg[pop == "Q"])]
        _q7 = (pop == "Q") & np.isin(stg, P7_STAGES)
        EP7 = pd.DataFrame(dict(gkey=gk[_q7], stage=stg[_q7], snap=np.asarray(T7["snap"], float)[_q7].astype(int),
                                gx=np.asarray(T7["gx"], float)[_q7].astype(int))).drop_duplicates(["gkey", "stage"])
        _kin7 = Table.read(KINST7)
        K7 = pd.DataFrame({c: (_s7(_kin7[c]) if c == "zone" else np.asarray(_kin7[c], float))
                           for c in ("snap", "gx", "zone", "kappa_H2", "kappa_star", "age_mw_gyr")})
        K7["snap"], K7["gx"] = K7["snap"].astype(int), K7["gx"].astype(int)
        K7 = EP7.merge(K7[K7["zone"].isin([z for z, _ in P7_ZONES])], on=["snap", "gx"], how="inner")
        # the zone ISM at every critical point, from the tracks' projected apertures: core = ap3kpc, outskirt = ap10kpc - ap3kpc
        _m7 = _q7 & np.isin(ap, ["ap3kpc", "ap10kpc"])
        TD7 = pd.DataFrame(dict(gkey=gk[_m7], stage=stg[_m7], ap=ap[_m7],
                                m_dust=np.asarray(T7["m_dust"], float)[_m7], m_H2=np.asarray(T7["m_H2"], float)[_m7],
                                sfr=np.asarray(T7["sfr100"], float)[_m7], m_star=np.asarray(T7["m_star"], float)[_m7],
                                n_star=np.asarray(T7["n_star"], float)[_m7])).drop_duplicates(["gkey", "stage", "ap"])
        _pv7 = TD7.pivot_table(index=["gkey", "stage"], columns="ap", values=["m_dust", "m_H2", "sfr", "m_star", "n_star"], aggfunc="first")
        _ism7 = []
        for (gk_, st_), r in _pv7.iterrows():
            for zone, sel in (("ap3kpc", lambda c: r[(c, "ap3kpc")]), ("ann10kpc", lambda c: r[(c, "ap10kpc")] - r[(c, "ap3kpc")])):
                ms_, ns_ = sel("m_star"), sel("n_star")
                ok_ = np.isfinite(ms_) and ms_ > 0 and ns_ >= P7_NSTAR_MIN
                d = dict(gkey=gk_, stage=st_, zone=zone)
                for c, num in (("fdust", "m_dust"), ("fH2", "m_H2")):
                    v = sel(num)
                    d[c] = (v / ms_) if (ok_ and np.isfinite(v) and v > 0) else np.nan       # zero dust / H2 -> NaN, never a floor
                v = sel("sfr")
                d["ssfr"] = max(v / ms_, SSFR_FLOOR) if (ok_ and np.isfinite(v) and v >= 0) else np.nan
                _ism7.append(d)
        K7 = K7.merge(pd.DataFrame(_ism7), on=["gkey", "stage", "zone"], how="left")
        K7["grp"] = [KEY7.get(tuple(int(v) for v in g_.split("_"))) for g_ in K7["gkey"]]
        K7 = K7[K7["grp"].notna()].reset_index(drop=True)
        print(f"  (A) {K7['gkey'].nunique()} of them are in the K-S tracks; the critical points " + " -> ".join(P7_STAGES) + ": "
              + ", ".join(f"{L7[c]} {K7.loc[(K7['grp'] == c) & (K7['zone'] == 'ap3kpc'), 'gkey'].nunique()}" for c in G7))
        # the observed bands: ALMA-C11 dust detections -> age / f_dust / sSFR, CO detections -> f_H2 (/1.36, bare H2 masses)
        B7 = {}
        if os.path.exists(OBS7):
            _o = pd.read_csv(OBS7)
            _d = _o[_o["det_dust"].astype(bool)]
            if len(_d):
                B7["age_mw_gyr"] = (float((10 ** _d["logage"] / 1e9).min()), float((10 ** _d["logage"] / 1e9).max()))
                B7["fdust"] = (float((10 ** (_d["logMdust"] - _d["logMstar"])).min()), float((10 ** (_d["logMdust"] - _d["logMstar"])).max()))
                B7["ssfr"] = (float((_d["SFR"] / 10 ** _d["logMstar"]).min()), float((_d["SFR"] / 10 ** _d["logMstar"]).max()))
            _c = _o[_o["det_co"].astype(bool)]
            if len(_c):
                B7["fH2"] = (float((_c["MH2_fid"] / 1.36 / 10 ** _c["logMstar"]).min()),
                             float((_c["MH2_fid"] / 1.36 / 10 ** _c["logMstar"]).max()))
        else:
            print(f"  (A) the ALMA-C11 bands need {OBS7} — drawn without them")
        fig, axs = plt.subplots(len(P7_ZONES), len(P7_QUANT), figsize=(2.95 * len(P7_QUANT), 3.4 * len(P7_ZONES)), squeeze=False)
        for zi, (zone, zlab) in enumerate(P7_ZONES):
            for qi, (col, lab, kind) in enumerate(P7_QUANT):
                ax = axs[zi, qi]
                for gi, g in enumerate(G7):
                    off = (gi - (len(G7) - 1) / 2) * 0.16
                    xx, yy, lo, hi = [], [], [], []
                    for i, st_ in enumerate(P7_STAGES):
                        med, l16, h84, n = _bmed7(K7.loc[(K7["grp"] == g) & (K7["zone"] == zone) & (K7["stage"] == st_), col])
                        rows7.append(dict(block="stage", zone=zone, quantity=col, kcls=g, x=st_, n_gal=n, med=med, lo=l16, hi=h84))
                        if np.isfinite(med):
                            xx.append(i + off); yy.append(med); lo.append(med - l16); hi.append(h84 - med)
                    if len(xx):
                        ax.errorbar(xx, yy, yerr=np.array([lo, hi]), color=C7[g], lw=2.2, elinewidth=1.3, capsize=2.5,
                                    marker="o", ms=5.5, mec="k", mew=0.5, ls="-", zorder=5,
                                    label=GLAB[g] if (zi == 0 and qi == 0) else None)
                _mm7 = (col == "age_mw_gyr") and not P7_OBS_AGE_LIKE
                if col in B7 and np.all(np.isfinite(B7[col])):
                    ax.axhspan(B7[col][0], B7[col][1], facecolor="#e7298a", alpha=0.12, lw=0, zorder=0,
                               hatch="///" if _mm7 else None, edgecolor="#e7298a")
                    if _mm7 and zi == 0:
                        ax.text(0.5, 0.985, "obs = CIGALE ONSET age, not mass-weighted", transform=ax.transAxes,
                                ha="center", va="top", fontsize=6.6, color="#8c1b52", fontweight="bold")
                if kind == "log":
                    ax.set_yscale("log")
                if kind == "kappa":
                    ax.axhline(KAPPA_ISO, color="0.3", lw=1.0, ls="--"); ax.set_ylim(0, 1)
                if col == "ssfr":
                    ax.axhline(1e-11, color="0.3", lw=0.9, ls="--")
                if col == "fdust":
                    ax.axhline(10 ** DUSTY_CUT, color="0.4", lw=0.9, ls=":")
                ax.set_xlim(-0.6, len(P7_STAGES) - 0.4)
                ax.set_xticks(range(len(P7_STAGES))); ax.set_xticklabels([P7_TICK.get(s_, s_) for s_ in P7_STAGES], fontsize=8.5)
                ax.set_ylabel((zlab + "\n" if qi == 0 else "") + lab, fontsize=9)
                ax.grid(alpha=0.25, lw=0.5)
        axs[0, 0].legend(frameon=False, fontsize=7.5, loc="best")
        if B7:
            axs[0, 2].legend(handles=[Line2D([], [], color="#e7298a", lw=6, alpha=0.25)], labels=["ALMA-C11 detections (min-max)"],
                             frameon=False, fontsize=7.5, loc="best")
        fig.suptitle(f"{BOX_LABEL[P7_BOX]}: the two groups at the critical points (median + galaxy-bootstrap 16-84; the core / "
                     f"outskirt spherical zones of the K-S products, the ISM from the tracks' projected apertures)", fontsize=10)
        fig.tight_layout(rect=(0, 0, 1, 0.955))
        paper_save(fig, "ba_m25_sequences")
        plt.show()
        print("  READ IT LIKE THIS: if the selection is a PROPERTY the galaxy always had, the curves separate at ignition. "
              "If they only separate at QT / end, the class is MADE during the quenching.")

    # ── (B) the radii ──
    if not (os.path.exists(ANN7) and os.path.exists(APT7)):
        print(f"\n(B) the radial anatomy: {ANN7} or {APT7} missing — the m25 notebook's Parts 8a / 7e (cluster)")
    else:
        K4 = ["snap", "gal_id", "incl", "aperture"]
        RF7 = _fr7(ANN7, K4 + ["M_dust", "M_H2"]).merge(_fr7(APT7, K4 + ["nstar_ap", "mstar", "age_m_star_myr"]), on=K4, how="inner")
        if os.path.exists(AV7):
            RF7 = RF7.merge(_fr7(AV7, K4 + ["A_V"]), on=K4, how="left")
        RF7 = RF7[RF7["aperture"].isin([a for a, _ in ANNULI7])].reset_index(drop=True)
        SF7 = set()
        if os.path.exists(SEL7):
            _s5b = _fr7(SEL7, ["snap", "gal_id", "pop"])
            SF7 = {(int(s_), int(i_)) for s_, i_, p_ in zip(_s5b["snap"], _s5b["gal_id"], _s5b["pop"]) if p_ == "SF"}
        RF7["grp"] = [KEY7.get((int(s_), int(i_)), "star_forming" if (int(s_), int(i_)) in SF7 else None)
                      for s_, i_ in zip(RF7["snap"], RF7["gal_id"])]
        RF7 = RF7[RF7["grp"].notna()].reset_index(drop=True)
        if os.path.exists(KIN7):
            RF7 = RF7.merge(_fr7(KIN7, ["snap", "gal_id", "aperture", "kappa_H2", "kappa_star"]),
                            on=["snap", "gal_id", "aperture"], how="left")
        for _f, _n in ((AV7, "A_V (m25 8b)"), (KIN7, "kappa (m25 8j0)"), (SEL7, "the SF-controls line (m25 Part 3)")):
            if not os.path.exists(_f):
                print(f"  (B) {_n}: {_f} missing — panel / line skipped")
        _ms7 = np.where(RF7["nstar_ap"].to_numpy(float) >= NSTAR_AP_MIN7, RF7["mstar"].to_numpy(float), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            for c, num in (("fdust_ap", "M_dust"), ("fH2_ap", "M_H2")):
                v = RF7[num].to_numpy(float)
                RF7[c] = np.where(np.isfinite(v) & (v > 0) & (_ms7 > 0), v / _ms7, np.nan)
            RF7["age_ap"] = np.where(np.isfinite(_ms7), RF7["age_m_star_myr"].to_numpy(float) / 1e3, np.nan)
            if "A_V" in RF7.columns:
                _av7 = RF7["A_V"].to_numpy(float)
                RF7["A_V"] = np.where(np.isfinite(_av7) & (_av7 > 0), _av7, np.nan)
        QUANT7 = ([("A_V", r"$A_V$ [mag]", "log")] if "A_V" in RF7.columns else []) \
            + [("fdust_ap", r"$M_{\rm dust}/M_\star$", "log"), ("fH2_ap", r"$M_{\rm H_2}/M_\star$", "log"),
               ("age_ap", "stellar age [Gyr]", "lin")] \
            + ([(k, lab, "kappa") for k, lab in (("kappa_H2", r"$\kappa_{\rm rot}^{\rm H_2}$"),
                                                 ("kappa_star", r"$\kappa_{\rm rot}^{\star}$"))] if "kappa_H2" in RF7.columns else [])
        GRP5B = (["star_forming"] if len(SF7) else []) + G7
        C5B = dict(C7, star_forming="0.6")
        L5B = dict(L7, star_forming="m25 star-forming controls")
        _x7 = np.arange(len(ANNULI7))
        _nc7 = (len(QUANT7) + 1) // 2
        rng7 = np.random.default_rng(5)
        NMAX7 = {g: 0 for g in GRP5B}
        fig, axs = plt.subplots(2, _nc7, figsize=(3.5 * _nc7, 7.1), squeeze=False)
        for qi, (col, lab, kind) in enumerate(QUANT7):
            ax = axs[qi // _nc7, qi % _nc7]
            for gi, g in enumerate(GRP5B):
                m_, l_, h_, n_ = [], [], [], []
                for a_, _ in ANNULI7:
                    sub = RF7[(RF7["grp"] == g) & (RF7["aperture"] == a_)]
                    v = sub[col].to_numpy(float); ok = np.isfinite(v)
                    per = {}
                    for s_, i_, vv in zip(sub["snap"].to_numpy(float)[ok], sub["gal_id"].to_numpy(float)[ok], v[ok]):
                        per.setdefault((int(s_), int(i_)), []).append(vv)              # the sightline rows -> one value per galaxy
                    pg = np.array([np.median(vv) for vv in per.values()])
                    if len(pg) >= P7_NMIN:
                        bs = np.median(pg[rng7.integers(0, len(pg), (NBOOT, len(pg)))], axis=1)
                        m_.append(float(np.median(pg))); l_.append(float(np.percentile(bs, 16))); h_.append(float(np.percentile(bs, 84)))
                    else:
                        m_.append(np.nan); l_.append(np.nan); h_.append(np.nan)
                    n_.append(len(pg))
                    rows7.append(dict(block="profile", zone="", quantity=col, kcls=g, x=a_, n_gal=len(pg), med=m_[-1], lo=l_[-1], hi=h_[-1]))
                NMAX7[g] = max(NMAX7[g], max(n_))
                off = (gi - (len(GRP5B) - 1) / 2) * 0.08
                m_, l_, h_ = np.array(m_), np.array(l_), np.array(h_)
                ok_ = np.isfinite(m_)
                if ok_.any():
                    ax.fill_between(_x7[ok_] + off, l_[ok_], h_[ok_], color=C5B[g], alpha=0.15, lw=0)
                    ax.plot(_x7[ok_] + off, m_[ok_], "o-", color=C5B[g], lw=1.3 if g == "star_forming" else 2.1, ms=4.5,
                            mec="k", mew=0.4)
            if kind == "log":
                ax.set_yscale("log")
            if kind == "kappa":
                ax.axhline(KAPPA_ISO, color="0.3", lw=1.0, ls="--"); ax.set_ylim(0, 1)
            if col == "fdust_ap":
                ax.axhline(10 ** DUSTY_CUT, color="0.4", lw=0.9, ls=":")
            ax.set_xticks(_x7); ax.set_xticklabels([r_ for _, r_ in ANNULI7], fontsize=8)
            if qi // _nc7 == 1:
                ax.set_xlabel("annulus [kpc]")
            ax.set_ylabel(lab); ax.grid(alpha=0.25, lw=0.5)
        for qi in range(len(QUANT7), 2 * _nc7):
            axs[qi // _nc7, qi % _nc7].axis("off")
        axs[0, 0].legend(handles=[Line2D([], [], color=C5B[g], marker="o", ms=4.5, mec="k", mew=0.4,
                                         lw=1.3 if g == "star_forming" else 2.1, label=f"{L5B[g]} (N <= {NMAX7[g]})")
                                  for g in GRP5B], frameon=False, fontsize=8)
        fig.suptitle(f"{BOX_LABEL[P7_BOX]}: the radial anatomy of the two groups (+ the SF controls) — per-galaxy sightline medians, "
                     f"bootstrap 16-84 of the class median"
                     + ("" if "kappa_H2" not in RF7.columns else "; kappa in the spherical zones (sightline-independent)"), fontsize=10.5)
        fig.tight_layout()
        paper_save(fig, "ba_m25_profiles")
        plt.show()

    # ── (C) the K-S plane ──
    kl7 = None
    if not os.path.exists(KS7):
        print(f"\n(C) the K-S plane: {KS7} missing — ks_tracks_quenched_m25 Part 4 (run tag 'pt', cluster)")
    else:
        try:
            import ks_tracks_lib as kl7                                  # the B08 / K98 relations and the He factor, verbatim
        except ImportError as e:
            print(f"\n(C) the K-S plane: {KS7} is there but ks_tracks_lib is not importable ({e}) — run the notebook from the repo root")
    if kl7 is not None:
        TR7 = Table.read(KS7)
        _s5c = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
        ap7, pop7, stg7, gk7 = _s5c(TR7["ap_label"]), _s5c(TR7["pop"]), _s5c(TR7["stage"]), _s5c(TR7["gkey"])
        x7, y7, ul7 = np.asarray(TR7["logSigmaH2"], float), np.asarray(TR7["logSigmaSFR"], float), np.asarray(TR7["is_ul"], bool)
        grp7 = np.array([KEY7.get(tuple(int(v) for v in g_.split("_"))) for g_ in gk7], dtype=object)
        STAGES7, MRK7 = ["sft", "qt", "end"], {"sft": "^", "qt": "s", "end": "o"}
        base7 = ap7 == "R50_H2"
        band7 = float(kl7.RELATIONS["B08"]["sig"])
        fig, ax = plt.subplots(figsize=(6.8, 6.3))
        sf_ = base7 & (pop7 == "SF")
        ax.scatter(x7[sf_], y7[sf_], s=7, c="0.65", alpha=0.3, lw=0, zorder=1, rasterized=True)
        xs7 = np.linspace(-1.0, 4.1, 60)
        for rk, lw_, alpha_ in (("B08", 1.4, 1.0), ("K98", 0.9, 0.55)):
            rel = kl7.RELATIONS[rk]
            if rk == "B08":
                ax.fill_between(xs7, rel["A"] + rel["N"] * xs7 - rel["sig"], rel["A"] + rel["N"] * xs7 + rel["sig"],
                                color=rel["color"], alpha=0.10, lw=0, zorder=0)
            ax.plot(xs7, rel["A"] + rel["N"] * xs7, color=rel["color"], lw=lw_, ls=rel["ls"], alpha=alpha_, zorder=2)
        print(f"\n(C) the track-end regions w.r.t. B08 +- {band7:g} dex (the R50_H2 aperture):")
        for g in G7:
            mk = base7 & (pop7 == "Q") & (grp7 == g) & np.isin(stg7, STAGES7)
            for g_ in np.unique(gk7[mk]):
                r_ = np.flatnonzero(mk & (gk7 == g_))
                r_ = r_[np.argsort([STAGES7.index(s_) for s_ in stg7[r_]])]
                ok = np.isfinite(x7[r_]) & np.isfinite(y7[r_])
                if ok.sum() >= 2:
                    ax.plot(x7[r_][ok], y7[r_][ok], "-", color=C7[g], lw=0.8, alpha=0.4, zorder=3)
            medx, medy = [], []
            for st_ in STAGES7:
                ms_ = mk & (stg7 == st_) & ~ul7 & np.isfinite(x7) & np.isfinite(y7)
                if ms_.sum() >= P7_NMIN:
                    medx.append(float(np.median(x7[ms_]))); medy.append(float(np.median(y7[ms_])))
                    ax.scatter(medx[-1], medy[-1], s=95, marker=MRK7[st_], c=C7[g], edgecolors="k", linewidths=0.8, zorder=6)
                    rows7.append(dict(block="ks_track", zone="", quantity="stage_median", kcls=g, x=st_, n_gal=int(ms_.sum()),
                                      med=medy[-1], lo=np.nan, hi=np.nan, logSigmaH2=medx[-1]))
            if len(medx) >= 2:
                ax.plot(medx, medy, "-", color=C7[g], lw=2.6, zorder=5)
            me_ = mk & (stg7 == "end")
            with np.errstate(invalid="ignore"):
                offr = y7[me_] - kl7.relation_y("B08", x7[me_])
            fin_ = np.isfinite(offr) & ~ul7[me_]
            cnt = dict(n=int(len(np.unique(gk7[mk]))), below=int((fin_ & (offr < -band7)).sum()),
                       within=int((fin_ & (np.abs(offr) <= band7)).sum()), above=int((fin_ & (offr > band7)).sum()),
                       censored=int((~fin_).sum()))
            rows7.append(dict(block="ks_region", zone="", quantity="end_vs_B08", kcls=g, x="", n_gal=cnt["n"], med=np.nan,
                              lo=np.nan, hi=np.nan, **{c: cnt[c] for c in ("below", "within", "above", "censored")}))
            print(f"  {L7[g]:26s} N={cnt['n']:3d}: below {cnt['below']}, on {cnt['within']}, above {cnt['above']}, "
                  f"censored/undefined {cnt['censored']}")
        _h7 = [Line2D([], [], color=C7[g], lw=2.4, label=f"{L7[g]} — median track") for g in G7]
        _h7 += [Line2D([], [], ls="", marker=MRK7[st_], ms=8, mfc="0.5", mec="k",
                       label={"sft": "SFT (quench start)", "qt": "QT (quench end)", "end": "track end"}[st_]) for st_ in STAGES7]
        _h7 += [Line2D([], [], ls="", marker="o", ms=5, mfc="0.65", mec="none", label="m25 SF controls"),
                Line2D([], [], color=kl7.RELATIONS["B08"]["color"], lw=1.4, ls=kl7.RELATIONS["B08"]["ls"],
                       label=kl7.RELATIONS["B08"]["label"] + r" $\pm$ scatter"),
                Line2D([], [], color=kl7.RELATIONS["K98"]["color"], lw=0.9, ls=kl7.RELATIONS["K98"]["ls"], alpha=0.55,
                       label=kl7.RELATIONS["K98"]["label"] + " (reference)")]
        ax.legend(handles=_h7, loc="lower right", frameon=False, fontsize=8)
        ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$  [$r<R_{50}({\rm H_2})$, $\times$" + f"{kl7.HE_FACTOR:g} He]")
        ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$")
        ax.set_xlim(-0.5, 3.6); ax.set_ylim(-4.6, 1.1)
        ax.tick_params(direction="in", top=True, right=True)
        ax.set_title(f"{BOX_LABEL[P7_BOX]}: the two groups on the K-S plane", fontsize=10.5)
        fig.tight_layout()
        paper_save(fig, "ba_m25_ks")
        plt.show()

    if rows7:
        _p7 = os.path.join(OUT, "ba_m25_closeup.csv")
        pd.DataFrame(rows7).to_csv(_p7, index=False)
        print(f"\nthe m25 close-up tables -> {_p7}")

## Part 8 — the pictures: one galaxy per group, at each critical point

Parts 1c to 7 score the division. **This part shows it**: one representative galaxy per group, rendered at
**SF peak → jet_on → SFT → QT → anchor → today** (`P8_STAGES`), from the **reduced particle cutouts** — the very
particles the K-S and profile products of Part 7 are measured from — through
`simbanator.visualization.rendering.ReducedRender`.

**The two outer columns** (2026-09-18). *SF peak* is the peak of the branch's own star-formation history, read off
Part 1's series: `P8_SFPEAK = "sfr_max"` (the maximum of the SFR over the whole chain, the default), `"ssfr_pre"`
(Part 1's $t_{\rm pk}$, the sSFR peak in the `P1_PKW` Gyr before the SFT — the burst the notebook scores) or
`"ssfr_max"`. It takes its place in the row's *own* time order, so a peak that falls after the jet is still pushed off
the jet's snapshot the way the other points are. *Today* is the galaxy at $z = 0$ (`P8_Z0SNAP` = 151). The chains end
at their anchor, so the z = 0 galaxy is found by walking every z = 0 galaxy's main-progenitor line — the progen
sidecars `output/cis25/progen_links/` (else the catalogue's `tree_data`) — down to the anchor snapshot: the
lowest-index (most massive) z = 0 galaxy whose line passes through the anchor galaxy is its descendant. A galaxy that
merged *into* a bigger one as the minor partner has no line of its own and its panel says so. The z = 0 cutouts are
**not** part of any plan run so far: the stage is in `P8_SOFT`, so a missing cutout gives an empty panel that names
the (snapshot, index) instead of dropping the galaxy, and the cell writes the plan for every missing one of the whole
selection to `ba_gallery_z0_plan_cis25.hdf5` — one `sbatch --array=0-0 submit_reduced_particles.sh <that plan>` builds
them all (one snapshot), and the next run of this cell draws them. The CSV carries `drawn` / `why` per panel.

Ported from `dusty_qgs_kappa_selection` Part 8. That cell was written against its own Part 6 tables; the
**shim** at the top of this one rebuilds the same three objects out of our own material, so everything below
it is unchanged:

| | |
|---|---|
| `EPO6` | one row per (galaxy, critical point): the **snapshot** nearest this notebook's own $t_{\rm jet}$ / $t_{\rm SFT}$ / $t_{\rm QT}$ / $t_{\rm anchor}$, and the galaxy's index *there* — which is what names the cutout file. The index comes from `kappa_m25_branch_ids.csv`, the branch-id map that notebook wrote |
| `ANC6` | one row per galaxy: which side of the peel it is on, and $(\kappa_{\rm gas}-1/3)/s_{\rm iso}$, the axis the representative is picked closest to its group median in |
| `GRP6` | the two groups, named `sel` (the **dusty QG** side of the dust map) and `rest` (the **dust-poor QG** one) the way that cell names them — the figure prints the Part 1b names |

The composite is stars + gas, Screen-blended and attenuated by the dust column, and since 2026-09-18 with a **white point chosen per
channel** (`P8_RGB_SCALE`): the gas and the dust on one scale across the grid (the `P8_PMAX` percentile of the channel over
every drawn panel, `P8_DYN` decades below it), so a panel whose channel holds a single particle, a gas-free $z = 0$ descendant,
shows it at its true faint surface density instead of stretching it to white and washing the stars out; the stars per panel,
since every panel has thousands and a descendant that has spread out would otherwise fall below the one decade shown; the physical maps (`P8_MAPS`:
gas, $\Sigma_{\rm dust}$, $\Sigma_{\rm H_2}$, temperature) share a log colour scale per field the same way. The mass-weighted
gas **velocity field is drawn as streamlines over the $\Sigma_{\rm H_2}$ map only** (`P8_STREAM_ON`) — the field
itself is always the *full gas* flow (`P8_STREAM_SOURCE`, `P8_STREAM_WEIGHT`), this only chooses the map it is
read against; over all four fields the lines cover the very structure the other maps are there to show, and
`P8_STREAM_DENSITY` (0.55) with `P8_STREAM_LW` (0.7) keeps them a flow rather than a thicket. Face-on, a rotating disc closes its streamlines into circles and one with no
resolved rotation does not — Part 7's κ sequence drawn instead of scored, which is why each row also carries
$\kappa_{\rm rot}$(gas) and $\kappa_{\rm rot}^\star$ beside $\log M_\star$, the age, the **formation redshift** (`P8_ZFORM`, default
`"first"`: the redshift of the chain's first snapshot, when the progenitor first exists; `"age"` = the mass-weighted formation
time $t_{\rm obs} - \langle{\rm age}\rangle_M$, `"half"` = the branch's half-mass time — all three in the CSV) and $\log M_{\rm dust}/M_\star$ (the quantity the selection is made on) on the left.

**Who gets drawn**: set a window on redshift, mass, age and sSFR (`P8_ZRANGE`, `P8_MRANGE`, `P8_AGERANGE`,
`P8_SSFRANGE`) plus a minimum particle count (`P8_MINGAS`, `P8_MINDUST`, `P8_MINSTAR`), and `P8_NPER`
galaxies per group are taken from what survives, ordered by `P8_ORDER` (best-resolved by default). Two
galaxies are never allowed to be the same picture twice: a candidate that shares **any** (snapshot, index)
of its four-epoch chain with one already taken is skipped, and a critical point that lands on a snapshot
another stage of the same galaxy already uses is stepped forward to the next one. `P8_SKIP` shows the next
`P8_NPER`; `P8_PICK` overrides the choice by hand. **The tightest pairs** (`P8_ORDER = "pairs"`, the default): every dusty × dust-poor pair inside the windows is ranked by its
distance in $\log M_\star$, age and $z$ at the observation and the `P8_NPER` tightest are drawn, no galaxy twice, `P8_NGASMIN` the only
resolution floor. **Seeds with partners** (`P8_ORDER = "res"` with `P8_MATCH`): only the `P8_MATCH_SEED` group is drawn by resolution; each of its galaxies then gets, from the other group, the partner closest to it in $\log M_\star$, age and $z$ **at the
observation** (`P8_MATCH_TOL`), so a row and its partner differ in dust and not in mass or age; the pairs and their offsets are
printed. Every number on the row label is the anchor's own catalogue value. `P8_SIZE` and `P8_FS` are the two dials worth touching (`P8_FS` 2.8 since 2026-09-18, the panels growing with it through
`P8_PANEL_IN`). Each panel carries its $z$ and $\delta_{\rm DGR} = \log M_{\rm dust}/M_{\rm H_2}$ (helium-corrected, Part 16b's
definition; `P8_DGR = "dg"` gives $\delta_{\rm DG}$ over the total gas), the symbol alone.

**Two size markers.** The white bar, in the first column only, is the fixed kpc ruler. The amber **circle**, in
every panel, is the galaxy's own half-mass radius *at that epoch* — drawn in the bottom-right corner the way an
ALMA beam is put in a map, so it is a size to hold the picture against (`P8_R50MARK_POS = "centre"` puts the same
circle concentric on the galaxy instead, where the aperture actually is, and a circle too wide for the corner
falls back to that by itself). `P8_R50MARK` chooses the component: `"baryon"` = the member stars + gas, the dust
riding on the gas; also `"star"`, `"gas"`, `"H2"`, `"dust"`, or `None`. It is measured from the particles in the
frame that is drawn — projected, the way an image measures it (`P8_R50MARK_2D`), over the CAESAR members only
(`P8_R50MARK_MEMBERS`) — so read along a row it is the ISM contracting or spreading out, in kpc, beside the
picture that shows it. It goes to `ba_gallery_<box>.csv` as `r50_kpc` / `r50_of`.

**The ISM/CGM boundary** (`P8_ISMEDGE`, worth having once `P8_SIZE` is wider than the galaxy): the ISM is
CAESAR's **member list**, not a radius — Part 6's definition — so its edge is a shape, and what is drawn is the
contour of the *member gas in projection* at the level enclosing `P8_ISMEDGE_ENC` (0.95) of the member gas mass
in the frame. Everything outside the dashed line is CGM. A number instead of `"members"` draws a plain circle of
that radius in kpc, `None` draws nothing, and `P8_ISMEDGE_ON` restricts it to some of the figures. The cell says
so when the contour runs off the frame, which means the box is smaller than the ISM and the line means nothing.

**Every panel on its own, for slides** (`P8_SINGLES`, on by default): besides the grids, each panel of every
figure — the composite and each `P8_MAPS` map — is written alone to `ba_gallery_singles_<box>/` as
`<figure>_<group>_a<anchor>_g<gal>_<column><stage>.png` (`P8_SINGLES_FMT` adds `"pdf"`), drawn exactly as in the
grid but with the kpc ruler on every panel; `P8_SINGLES_RAW` adds `<name>_raw.png`, the bare pixel array
(`P8_XSIZE` pixels, the grid's colour scale, nothing drawn on it); `P8_SINGLES_DECOR = False` drops the text and
the circle; `P8_SINGLES_TITLE` puts the stage name over each. Each map's colour bar is saved alone
(`cbar_<figure>.png`): every panel of one figure shares it. **The table** `ba_gallery_singles_<box>.csv` (same
folder) has one row per figure x galaxy x stage: the file names, the group and its label, the row label's
numbers as columns (`gal_*`) and as text (`row_text`), the epoch (snap, gx, z), the frame and member totals, the
panel's D/G, R50, and the colour scale it was drawn on (`scale_*`: vmin/vmax/cmap for a map, the channels, their
white points and decades for the composite). Panels that are empty in the grid have a row (`drawn = False`,
`why`) and no file.

**This is the one part that needs the reduced cutouts and a working `simbanator` import** — it prints what is
missing and skips cleanly otherwise, so the notebook still runs end to end off the cluster.


In [ ]:
# ── Part 8 — THE PICTURES: one representative galaxy per group, at each critical point ──
# PORTED from dusty_qgs_kappa_selection Part 8 and re-split on THIS notebook's selection.  That cell was
# written against its own Part 6 tables (EPO6 / ANC6 / GRP6); the SHIM below builds the same three objects
# out of Part 1b's peel, the branch-id map and the Part 0 catalogue, so everything after it is unchanged.
# Parts 6 and 7 divide the quenched galaxies in two, by Part 2's selection, and score the division. This part SHOWS it: one
# representative galaxy per class, rendered at sfr_peak -> jet_on -> sft -> qt -> anchor -> z0: the peak of its star-formation
# history, the four critical points the tracks are walked through, and TODAY — the z = 0 descendant of the anchor galaxy
# (2026-09-18: the two outer columns; the chains end at their anchor, so z = 0 is resolved through the progenitor links).
#   FIG 1  the RGB composite: stars + gas, Screen-blended and attenuated by the dust column
#   FIG 2+ one physical map per P8_MAPS field (Sigma_dust, Sigma_H2 by default) on a SHARED log colour scale,
#          with the mass-weighted GAS VELOCITY FIELD as streamlines over the map P8_STREAM_ON names (the H2
#          one by default): face-on, a rotating disc closes its
#          streamlines into circles and one with no resolved rotation does not — the selection drawn instead of scored.
# The renders read the REDUCED PARTICLE CUTOUTS (build_reduced_particles_job.py), i.e. the very particles the KS
# and profile products are measured from, through simbanator.visualization.rendering.ReducedRender — no snapshot
# and no Caesar catalogue, so the whole grid is seconds rather than one full-box load per panel.
# The representative of a class is the galaxy CLOSEST TO ITS CLASS MEDIAN in the axes that define the picture (the kinematic
# significance and the dust), among those whose four epochs all have a reduced file; P8_PICK overrides it.
P8_SIZE    = 35.0             # ◄── HOW BIG EACH IMAGE IS, in kpc a side (physical).  60 = a 60 x 60 kpc box around
                               #     each galaxy.  The cutouts hold particles out to 100 kpc from the centre, so
                               #     anything past ~200 has empty corners and the cell says so.  None = derive it
                               #     from the galaxies themselves (P8_EXTFAC x the median R50(*), inside P8_EXTLIM).
P8_FS      = 3.3              # ◄── HOW BIG THE TEXT IS: one multiplier for every label, title and number drawn here
                               #     (2026-09-18: "almost doubled" from 1.5; the panels grow with it, P8_PANEL_IN).
P8_PANEL_IN = 4.4             # [inch] one panel's width (the height is 1.12 x this); the row-label block beside a row is
                               #     seven lines of 9 x P8_FS pt at P8_LABEL_LS spacing, so the row has to be tall enough to hold it.
P8_LABEL_LS = 1.25            # the line spacing of that block (was 1.6 at P8_FS 1.5)
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════
P8_BOX     = "cis25"
P8_REDUCED = globals().get("P8_REDUCED_OVERRIDE", os.path.join(os.getcwd(), "output", P8_BOX, "reduced_particles"))
P8_PREFIX  = "m25n512"        # the reduced files' name stem: <prefix>_snapNNN_galGGGGGG.h5
P8_STAGES  = ["sfr_peak", "jet_on", "sft", "qt", "anchor", "z0"]   # the columns, in time order: the SFH PEAK, the four
                               #   critical points, and TODAY ("z0": the z = 0 DESCENDANT of the anchor galaxy).  Any subset,
                               #   in any order, is a valid list — the four original ones give the old figure back.
P8_SFPEAK  = "sfr_max"        # WHAT "sfr_peak" IS, on the branch's own history (Part 1's series):
                               #   "sfr_max"  = the MAXIMUM OF THE SFR over the whole chain — the peak of the star-formation
                               #                history, wherever it is (usually well before the jet)
                               #   "ssfr_pre" = Part 1's t_pk: the sSFR peak inside the P1_PKW Gyr before the SFT (the burst
                               #                the notebook scores as b_ssfr / f_burst)
                               #   "ssfr_max" = the sSFR maximum over the whole chain (the earliest snapshot, for most)
P8_Z0SNAP  = 151              # the snapshot "z0" is drawn at (m25n512: 151 = z 0).  The chains END AT THEIR ANCHOR, so the
                               #   z = 0 galaxy is found by walking every z = 0 galaxy's main-progenitor line (the progen
                               #   sidecars output/<box>/progen_links/<prefix>_NNN.hdf5, else the catalogue's own tree_data)
                               #   down to the anchor snapshot: the LOWEST-index (= most massive) z = 0 galaxy whose line
                               #   passes through the anchor galaxy is its descendant.  A galaxy that merged INTO a bigger one
                               #   as the minor partner has no line of its own at z = 0 and its panel says so.
P8_SOFT    = ["z0"]           # stages a galaxy may LACK ON DISK and still be drawn (an empty panel that says what is
                               #   missing).  The z = 0 cutouts do not exist until the plan this cell writes is run:
                               #     sbatch --array=0-0 submit_reduced_particles.sh output/box_resolution/burst_aging/ba_gallery_z0_plan_cis25.hdf5
                               #   (one snapshot, so one task; the plan lists the descendants of EVERY galaxy of the
                               #   selection, not only the drawn ones, so P8_SKIP / P8_PICK never need a second run).
                               #   Every stage not listed here is REQUIRED: a galaxy without its cutout is not drawn.
P8_SPOS    = "faceon"         # "faceon" (looking down the spin axis of P8_FRAME) | "edgeon" (along its disc)
P8_FRAME   = "H2_L"           # what faceon / edgeon refer to: "H2_L" (the angular momentum of the member H2 inside P8_FRAME_RMAX —
                               # the ISM's own spin, 2026-09-04) | "gas_L" | "dust_L" | "star_L" | "stellar" (the stored stellar
                               # principal frame: the inertia tensor of the member stars, nearly degenerate for a spheroid, which is
                               # why faceon / edgeon did nothing visible before). A file without velocities, or with fewer than 20
                               # weighted particles inside the radius, falls back to "stellar" and the cell says so.
P8_FRAME_RMAX = 4.0          # [kpc] the radius the frame's angular momentum is summed inside
P8_FRAME_MEMBERS = True       # ... over the CAESAR members only, whatever P8_MEMBERS says: the CGM must not set the orientation
P8_FRAME_STAGE = None        # ONE frame per row: the spin measured at this critical point orients every panel of the row (None =
                               # each panel its own spin, which jumps from panel to panel when the gas barely rotates). The report
                               # prints each spin's COHERENCE |sum l| / sum |l| (1 = a disc, ~1/sqrt(n) = random): below ~0.3 there
                               # is no disc to orient and any frame is a random tilt — the quenched H2 of most of the sample.
# the frame, in full: P8_SIZE above is the whole story unless it is None, in which case the half-width becomes
# P8_EXTFAC x the median R50(*) of the drawn epochs and P8_EXTLIM is a GUARD ON THAT AUTOMATIC VALUE — raising its
# upper bound does nothing unless the automatic value was already being clipped by it.  The cell prints which decided.
P8_EXTFAC  = 3.5              # P8_SIZE is None: the half-width in units of the median R50(*) of the drawn epochs
P8_EXTLIM  = (8.0, 20.0)      # P8_SIZE is None: the bounds that automatic half-width is clipped to [kpc]
P8_EXT     = None if P8_SIZE is None else 0.5 * float(P8_SIZE)      # the half-width the code works in
P8_SIDE    = "right"          # which margin carries the row labels + numbers: "right" | "left"
P8_MARGIN  = 0.21             # the fraction of the figure width that margin gets (the class names are long)
P8_WSPACE  = 0.015            # the gap between panels, as a fraction of a panel (both directions)
P8_XSIZE   = 400              # render size [pixels] of the composites and maps
P8_MEMBERS = False            # False = the WHOLE 100 kpc cutout (CGM + neighbours), which is also what makes the
                               # smoothing lengths right: with members only, hsml is estimated from the members alone
                               # and the ISM comes out blobbier and smaller than it is. True = catalogue members only.
P8_CHAN    = ("star", "gas", "dust")   # the composite's channels, each painted with its own black-to-hue colormap
P8_DYN     = (1.5, 2.4, 2.4)  # decades shown per channel (log_stretch): more = fainter outskirts visible
P8_PMAX    = 99.9             # the percentile of the POSITIVE pixels that saturates to white, per channel
P8_STRETCH = "dyn"            # "dyn" = log_stretch (fixed dynamic range); "percentile" = RenderRGB's clip, which
                               # saturates a small source into a uniform blob — kept only for comparison
P8_RGB_SCALE = {"star": "panel", "gas": "grid", "dust": "grid"}
                               # WHAT THE COMPOSITE'S WHITE POINT IS, PER CHANNEL (2026-09-18; a plain string applies to all):
                               #   "grid"  = ONE top for the whole figure — the P8_PMAX percentile of that channel's positive
                               #             pixels over EVERY drawn panel, P8_DYN decades below it (the physical maps' rule).
                               #             The gas and the dust want this: a panel whose channel holds ONE particle (a gas-
                               #             free z = 0 descendant) would otherwise stretch it to white and its Screen blend
                               #             would wash the stars out; on the grid scale it sits at its true faint density.
                               #   "panel" = that panel's own top (the renderer's autoscale).  The STARS want this: every
                               #             panel has thousands of them, and on the grid scale a descendant that has spread
                               #             out (twice the R50 of its anchor) falls below the one decade the star channel
                               #             shows and comes out too faint.  Only "dyn" stretches are re-scaled.
# WHO GETS DRAWN — ranges, and how many per group.  (The original cell seeded one class and matched the
# other to it in age and mass; this is the simpler rule the user asked for: a window on each axis, then take
# N per group.  The critical points are automatic either way.)
P8_NPER    = 2                # how many galaxies per group.  The figure has P8_NPER x (the two groups) rows.
P8_ZRANGE  = (0.2, 0.9)      # the anchor redshift          — (lo, hi), either end None = no bound
P8_MRANGE  = (10., 13)     # log M* at the anchor
P8_AGERANGE = (1.5, 10)    # the mass-weighted stellar age at the anchor  [Gyr]
P8_SSFRANGE = (-19, -9)    # log sSFR at the anchor  [yr^-1]
P8_NGASMIN = 50                # the anchor must hold at least this many GAS particles.  SIMBA carries dust ON the
                               # gas particles — there is no dust particle — so this IS the number of dust carriers
                               # the render has to work with, and it is the catalogue's own ngas.
P8_MATCH   = True             # MATCHED PAIRS (2026-09-18): the P8_MATCH_SEED group is drawn first (by P8_ORDER, inside the
                               #   windows) and EVERY one of its galaxies gets, from the other group, the partner closest to it
                               #   in (log M*, age[, z]) AT THE OBSERVATION — so the rows compare like with like and the
                               #   difference between a row of one group and its partner is the dust, not the mass or the age.
                               #   The pairs and their offsets are printed.  False = the two groups drawn independently.
P8_MATCH_SEED = "sel"          # "sel" (the dusty QGs) | "rest" (the dust-poor ones)
P8_MATCH_TOL  = (0.25, 0.75, 0.15)   # the distance is |d log M*| / 0.25 dex + |d age| / 0.75 Gyr + |dz| / 0.15 (None = the
                               #   axis is not used); the best partner is taken even outside these, and the printout says so
P8_ORDER   = "pairs"          # which survivors to take: "pairs" (2026-09-18) = THE TIGHTEST PAIRS: every dusty x dust-poor pair
                               #   inside the windows is ranked by its distance in (log M*, age, z) at the observation
                               #   (P8_MATCH_TOL) and the P8_NPER tightest are drawn, no galaxy twice; P8_NGASMIN is the only
                               #   resolution floor.  "res" = the best resolved seeds (most gas + star particles) with their
                               #   closest partners (P8_MATCH), "median" = the closest to the group median in (the kinematic
                               #   axis, the dust ratio), "random" = a seeded draw from the survivors
P8_DGR     = "dh2"            # THE ratio reported for every galaxy: "dh2" = M_dust/M_H2, "dg" = M_dust/M_gas
P8_ZFORM   = "first"          # THE FORMATION REDSHIFT on the row label (2026-09-18): "first" = the redshift of the chain's FIRST
                               #   snapshot — when the progenitor first exists in the catalogue (the user's choice, the same
                               #   day: a mass-weighted formation redshift is the MEAN of a history and lands after the SF
                               #   peak for a galaxy that kept forming stars, which reads as a contradiction beside that
                               #   column) | "age" = the redshift of the mass-weighted formation time t_obs - <age>_M |
                               #   "half" = the redshift at which the branch's stellar mass first reached half its observed
                               #   value.  All three go to the CSV.  z(t) is read off the chain's own (t, z) pairs where the
                               #   time is inside the chain, else from astropy's Planck15.
P8_ORDSEED = 8                # the seed of "random"
P8_SKIP    = 0                # SKIP this many of the ordering before taking P8_NPER — "show me the next four"
P8_LIST    = 12               # print this many of the ranked pool per group, so a galaxy can be named by hand
                               # in P8_PICK instead of hunting for it with the windows (0 = do not print)
P8_MAPS    = ["gas", "dust", "H2", "temp"]     # the physical maps drawn, one figure each: any key of P8_MAPDEF
#      name -> (kind, label, unit, log?, cmap, symmetric?).  "sigma" = a surface density through ReducedRender.sigma_map;
#      "weighted" = the mass-weighted line-of-sight mean of a per-particle gas quantity through .weighted_map, which is
#      what a temperature (or a velocity, or a metallicity) needs — a temperature does not project as a column.
P8_MAPDEF  = {
    "dust": ("sigma",    r"\Sigma_{\rm dust}",       r"M_\odot\,{\rm kpc^{-2}}",    True,  "magma",   False),
    "H2":   ("sigma",    r"\Sigma_{\rm H_2}",        r"M_\odot\,{\rm kpc^{-2}}",    True,  "viridis", False),
    "HI":   ("sigma",    r"\Sigma_{\rm HI}",         r"M_\odot\,{\rm kpc^{-2}}",    True,  "cividis", False),
    "gas":  ("sigma",    r"\Sigma_{\rm gas}",        r"M_\odot\,{\rm kpc^{-2}}",    True,  "bone",    False),
    "star": ("sigma",    r"\Sigma_\star",            r"M_\odot\,{\rm kpc^{-2}}",    True,  "afmhot",  False),
    "sfr":  ("sigma",    r"\Sigma_{\rm SFR}",        r"M_\odot\,{\rm yr^{-1}\,kpc^{-2}}", True, "inferno", False),
    "temp": ("weighted", r"\langle T \rangle_M",     r"{\rm K}",                    True,  "inferno", False),
    "zgas": ("weighted", r"\langle Z_{\rm gas} \rangle_M", "",                      True,  "cubehelix", False),
    "vlos": ("weighted", r"\langle v_{\rm los} \rangle_M", r"{\rm km\,s^{-1}}",     False, "RdBu_r",  True),
}
P8_STREAM  = True              # overlay the gas velocity field as streamlines
P8_STREAM_ON = ["H2"]         # ... on THESE maps only.  The FIELD is always the full gas flow (P8_STREAM_SOURCE
                               # and P8_STREAM_WEIGHT below, m_gas by default) — this only says which map it is
                               # DRAWN OVER: the molecular one, where the flow is read against the ISM that is
                               # still there.  On all four maps the lines cover the very structure the other
                               # fields exist to show; [] puts them on none, P8_MAPS back on all of them.
P8_NGRID   = 8               # the velocity field grid (per side)
P8_STREAM_DENSITY = 0.55      # HOW MANY streamlines: matplotlib seeds them on its own 30 x 30 base grid scaled by
                               # this.  Below ~0.6 the field reads as a flow; above ~1 it is a thicket.
                               #   (0.5 = half as many lines, 2 = twice) — P8_NGRID only sets the RESOLUTION of the
                               #   velocity field the lines follow, not their number
P8_STREAM_SOURCE = "slab"     # WHICH gas feeds the velocity field: "slab" = every gas particle of the loaded region within
                               #   +-P8_STREAM_DEPTH kpc of the galaxy's mid-plane along the line of sight (enough particles for
                               #   smooth flows, without the CGM in front and behind) | "members" = the CAESAR members only (too few
                               #   for nice flows) | "region" = everything loaded (the CGM and every neighbour in the frame too)
P8_STREAM_DEPTH  = 2.0        # [kpc] the half-thickness of that slab along the line of sight (face-on: the disc plane +- this)
P8_STREAM_BULK   = "members"  # the velocity subtracted before drawing: "members" = the galaxy's own systemic velocity (its member gas,
                               #   its member stars if it has no gas) | "slab" = the mass-weighted mean of the particles used
P8_STREAM_WEIGHT = "m_gas"    # what weights the field: "m_gas" (all the gas) | "m_H2" (the molecular ISM only) | "m_dust"
P8_STREAM_LW = 0.7            # the streamline width
P8_STREAM_ARROW = 0.8         # the arrow size
P8_TAU     = 0.0              # >0 also attenuates every channel by the dust column; with dust drawn as its own
                               # channel that double-counts it, so it is off
P8_CBAR_LEN = 0.46            # the colour bar's LENGTH, as a fraction of the figure width
P8_CBAR_H  = 0.26             # [inch] its THICKNESS — an inch, not a fraction, so it does not thin out as rows
                              #   are added; the figure grows by the bar's whole zone instead of eating the panels
P8_CBAR_GAP = 0.30            # [inch] between the bottom row of panels and the bar
P8_CBAR_PAD = 9               # [pt] between the bar and its label
P8_R50MARK = "baryon"         # A SIZE CIRCLE IN EVERY PANEL, drawn THE WAY A BEAM IS DRAWN and as wide as the
                              #   galaxy's own HALF-MASS RADIUS at that epoch: "baryon" = the member stars + gas
                              #   (the dust rides on the gas), or "star" | "gas" | "H2" | "dust" | None.  The fixed
                              #   kpc ruler stays in the first column — this is the galaxy's own scale, so a row
                              #   shows the ISM contracting or spreading out.
P8_R50MARK_POS = "corner"     # WHERE: "corner" = a filled circle of radius R50 in the bottom-right corner, the way
                              #   an ALMA beam is put in a map | "centre" = the same circle concentric on the galaxy,
                              #   which is the half-mass aperture drawn where it actually is
P8_R50MARK_MEMBERS = True     # measured over the CAESAR MEMBERS only: the galaxy, not the CGM that shares the frame
P8_R50MARK_2D = True          # the PROJECTED half-mass radius in the frame drawn — what an image of it would
                              #   measure; False = the 3D radius, which is the catalogue's convention
P8_R50MARK_COL = "#ffd54f"    # amber, so it is never confused with the white ruler or the ISM contour
P8_R50MARK_FILL = 0.30        # the face alpha of that circle (0 = an outline only)
P8_R50MARK_LW = 1.6
P8_ISMEDGE = "members"        # THE ISM/CGM BOUNDARY drawn on every panel, worth having once the cutout is wider
                              #   than the galaxy: "members" = the outline of the CAESAR MEMBER gas in projection,
                              #   which is the honest version of "where the ISM ends" (the ISM is a member LIST and
                              #   not a radius — Part 6's definition — so its edge is a shape, not a circle);
                              #   a NUMBER = a plain circle of that radius in kpc; None = nothing drawn.
P8_ISMEDGE_ENC = 0.95         # "members": the contour is the level of that projection enclosing this fraction of
                              #   the member gas mass inside the frame — 0.95 is an edge, 0.5 a half-mass isophote
P8_ISMEDGE_SM = 2.0           # [pixel] the member map is smoothed by this before contouring: a boundary, not a coastline
P8_ISMEDGE_ON = None          # which figures carry it: None = all of them, or a list of P8_MAPS keys and/or "rgb"
P8_ISMEDGE_COL = "#00e5ff"    # its colour, width and dash
P8_ISMEDGE_LW = 1.1
P8_ISMEDGE_LS = (0, (5, 3))
P8_CMAP    = {}               # per-field colormap overrides of P8_MAPDEF, e.g. {"temp": "turbo"}
P8_PICK    = {}               # {"sel": [(anchor_snap, gal_id), ...], "rest": [...]} names them by hand; {} = the
                               # match below.
# GALAXIES WORTH LOOKING AT, from the runs behind this notebook (they are coordinates, not groups — the four-way dose x class
# grid they were chosen under is gone):
#   the ROTATING set (2026-09-04), the only one whose H2 has a disc to orient — (116, 61), (116, 119), (116, 42), (110, 71):
#   kappa_rot(H2) 0.77 / 0.64 / 0.69 / 0.49 (the median quenched galaxy sits at 0.2, below the isotropic 1/3, and no frame can
#   align a spin that is noise); age 4.01 / 3.62 / 4.02 / 3.48 Gyr, log M* 10.46 / 10.17 / 10.64 / 10.39, log M_dust/M*
#   -3.38 / -4.74 / -4.26 / -5.31, 201 / 527 / 405 / 853 gas particles, three of them at anchor 116 (z 0.70).
#   the DUST-CONTRAST set, tighter-matched but NOT rotating (kappa 0.1-0.2, age 1.29-1.55 Gyr, log M* 10.34-10.53, dust
#   -3.40 / -4.05 / -5.77 / -4.24) — (100, 45), (90, 43), (95, 60), (90, 31).
P8_PCT     = (2.0, 99.5)      # the shared colour scale of a map grid, as percentiles of its finite pixels
P8_MAPDYN  = 4.0              # ... but a LOG map shows at most this many decades below its top: with the whole region
                               # loaded the empty outskirts drag the low percentile down and wash the galaxy out.
                               # None = the percentile alone
P8_SINGLES = True             # ◄── EVERY PANEL ALSO AS ITS OWN IMAGE (2026-10-12), for building slides by hand: each panel of
                               # every figure (the composite + each P8_MAPS map) goes to OUT/ba_gallery_singles_<box>/
                               # <figure>_<group>_a<anchor>_g<gal>_<column><stage>.<fmt>, drawn as in the grid (the z / ratio
                               # box, the R50 circle, the ISM contour, the streamlines, the group-coloured frame) and with
                               # the kpc ruler on EVERY panel (alone, a panel has no first column to borrow it from); each
                               # map's colour bar is saved alone (cbar_<figure>.<fmt>), and ONE TABLE,
                               # ba_gallery_singles_<box>.csv in the same folder, has a row per file: the galaxy (the row
                               # label's numbers and its text), the epoch, the frame and member totals, the D/G ratio, R50,
                               # the colour scale it was drawn on, and the file names.
P8_SINGLES_FMT = ("png",)     # the formats of the single panels (+ "pdf" for vector; 5 figures x the grid = many files)
P8_SINGLES_RAW = True         # also the BARE image: the pixel array alone at P8_XSIZE pixels, on the same colour scale, nothing
                               # drawn on it (no text, ruler, circle, contour or streamlines) — <name>_raw.png
P8_SINGLES_DECOR = True       # False = the decorated singles carry only the image (+ contour / streamlines), no text or circle
P8_SINGLES_TITLE = False      # True = the stage name ("Quenching", ...) over each single panel

# ══ THE SHIM: the three objects the ported cell expects, built from THIS notebook ══════════════════════════
# The original cell read them from its own Part 6.  Here:
#   EPO6   one row per (galaxy, critical point) with the SNAPSHOT it happens at and the galaxy's index THERE,
#          which is what names the reduced cutout.  The index comes from the branch-id map Part 13 wrote
#          (kappa_m25_branch_ids.csv: anchor_snap, gal_id -> snap, gx), and the snapshot from this notebook's
#          own event times: the snapshot nearest t_jet / t_SFT / t_QT / t_anchor.
#   ANC6   one row per galaxy: its key, the side of the division it is on, and the axis the pick is made on.
#   GRP6   the two groups, named "sel" (the upper side) and "rest" (the lower one) the way that cell names them.
P8_IDS     = os.path.join(KSEL, "kappa_m25_branch_ids.csv")   # dusty_qgs_kappa_selection's branch-id map
P8_GEOMCOL = "kz"             # the axis the representative is picked closest to its group median in
_HAVE8 = True
if P8_BOX not in B1:
    print(f"Part 8: {P8_BOX} is not in this run's boxes — skipped."); _HAVE8 = False
elif not os.path.exists(P8_IDS):
    print(f"Part 8: {P8_IDS} is missing — dusty_qgs_kappa_selection writes it (cluster).  Skipped."); _HAVE8 = False
else:
    _ID8 = pd.read_csv(P8_IDS)
    _GX8 = {(int(a), int(g), int(s)): int(x) for a, g, s, x
            in zip(_ID8["anchor_snap"], _ID8["gal_id"], _ID8["snap"], _ID8["gx"])}
    _C8 = CAT[CAT["box"] == P8_BOX].drop_duplicates(subset=["snap", "gal_id"]).copy()
    _C8 = _C8.set_index([_C8["snap"].astype(int), _C8["gal_id"].astype(int)])
    _FF8 = B1[P8_BOX]["F"]
    _RR8 = B1[P8_BOX]["R"]
    _FF8 = _FF8[_FF8["uniq"].to_numpy(bool) & (_FF8["grp"].to_numpy() != "")]

    def _progen8(sn):
        """The file holding tree_data/progen_galaxy_star for snapshot `sn`: the progen sidecar (the m25 catalogues are
        read-only, find_progen_m25_job.py writes them beside the outputs), else the catalogue itself through simbanator."""
        _p = os.path.join(os.getcwd(), "output", P8_BOX, "progen_links", f"{P8_PREFIX}_{int(sn):03d}.hdf5")
        if os.path.exists(_p):
            return _p
        try:
            from simbanator import Simulation
            from simbanator.analysis.progenitors import progen_tree_file
            return progen_tree_file(Simulation(P8_BOX), int(sn))
        except Exception:
            return None

    def _descendants8(keys, z0snap):
        """{(anchor_snap, gal_id): the galaxy's index at `z0snap`, or None}.  Every z0 galaxy's MAIN-PROGENITOR line is
        walked down, one progen file per snapshot read once, and at each anchor snapshot the descendant of a key is the
        lowest-index (= most massive, caesar's order) z0 galaxy whose line passes through it.  None = no line does: the
        galaxy merged into a bigger one as the minor partner, or its line was lost.  Returns (the map, a reason if the walk
        could not be made)."""
        out = {k: None for k in keys}
        _f0 = _progen8(z0snap)
        if _f0 is None:
            return out, f"no progenitor links at snapshot {int(z0snap)}"
        with h5py.File(_f0, "r") as _f:
            _n0 = int(_f["tree_data/progen_galaxy_star"].shape[0])
        _cur, _sn = np.arange(_n0), int(z0snap)                # where every z0 galaxy's line is at snapshot _sn
        for _a in sorted({int(k[0]) for k in keys}, reverse=True):
            while _sn > _a:
                _fp = _progen8(_sn)
                if _fp is None:
                    return out, f"no progenitor links at snapshot {_sn} — the walk from {int(z0snap)} stops there"
                with h5py.File(_fp, "r") as _f:
                    _pg = _f["tree_data/progen_galaxy_star"][:, 0].astype(int)     # index at _sn - 1 of the main progenitor
                _ok = (_cur >= 0) & (_cur < len(_pg))
                _nxt = np.full_like(_cur, -1); _nxt[_ok] = _pg[_cur[_ok]]
                _cur, _sn = _nxt, _sn - 1
            for k in keys:
                if int(k[0]) == _a:
                    _hits = np.flatnonzero(_cur == int(k[1]))
                    out[k] = int(_hits[0]) if len(_hits) else None
        return out, ""

    _DESC8, _why0 = {}, ""
    if "z0" in P8_STAGES and P8_Z0SNAP:
        _keys8 = sorted({(int(s), int(g)) for s, g in zip(_FF8["snap"], _FF8["gal_id"])})
        _DESC8, _why0 = _descendants8(_keys8, int(P8_Z0SNAP))
        _nres8 = sum(v is not None for v in _DESC8.values())
        print(f"Part 8 — z = 0: {_nres8} of {len(_keys8)} galaxies of the selection have a main-progenitor line reaching "
              f"snapshot {int(P8_Z0SNAP)}" + (f"  ({_why0})" if _why0 else
              f"; the other {len(_keys8) - _nres8} merged into a bigger galaxy as the minor partner (no line of their own)"))

    def _z_of_t8(t, tt, zz):
        """The redshift of cosmic time t [Gyr]: interpolated on the chain's own (t, z) where t is inside it, else Planck15."""
        if not np.isfinite(t):
            return np.nan
        _ok = np.isfinite(tt) & np.isfinite(zz)
        if _ok.sum() >= 2 and tt[_ok].min() <= t <= tt[_ok].max():
            return float(np.interp(t, tt[_ok], zz[_ok]))
        try:
            from astropy.cosmology import Planck15, z_at_value
            import astropy.units as _u
            return float(z_at_value(Planck15.age, max(t, 0.05) * _u.Gyr, zmax=30.0))
        except Exception:
            return np.nan

    _ER8, _AR8, _ncol8, _nmov8 = [], [], [], []
    _nst8 = len([s for s in P8_STAGES if s != "z0"])            # the stages that live on the chain
    for _, _r in _FF8.iterrows():
        _key = (int(_r["snap"]), int(_r["gal_id"]))
        # THE SNAPSHOT OF A CRITICAL POINT IS THE NEAREST ONE OF THIS GALAXY'S OWN CHAIN — every snapshot it
        # exists at, not the ten anchors of the catalogue ladder.  With the ladder the four points collapsed
        # onto two or three snapshots for 27 of 34 galaxies and the row came out as the same picture four
        # times over; the chain is what the cutouts are named after anyway.
        _ser = _RR8[int(_r["ridx"])] if "ridx" in _r else None
        if _ser is None or "snap" not in _ser:
            continue
        _tt8, _ss8 = np.asarray(_ser["t"], float), np.asarray(_ser["snap"], int)
        _o8 = np.argsort(_ss8)
        _tt8, _ss8 = _tt8[_o8], _ss8[_o8]                     # the chain, in snapshot order
        # THE FORMATION REDSHIFT, two ways: the mass-weighted formation time, and the half-mass time of the branch
        _zz8 = np.asarray(_ser["z"], float)[_o8] if "z" in _ser else np.full(len(_tt8), np.nan)
        _age8 = float(_r["age"]) if ("age" in _r and np.isfinite(_r["age"])) else np.nan
        _zf_age8 = _z_of_t8(float(_r["t_anchor"]) - _age8, _tt8, _zz8) if np.isfinite(_age8) else np.nan
        _zf_first8 = float(_zz8[np.isfinite(_zz8)][0]) if np.isfinite(_zz8).any() else _z_of_t8(float(_tt8[0]), _tt8, _zz8)
        _zf_half8 = np.nan
        if "lms" in _ser:
            _lm8 = np.asarray(_ser["lms"], float)[_o8]
            if np.isfinite(_lm8).sum() >= 2 and np.isfinite(_lm8[-1]):
                _hit = np.flatnonzero(np.isfinite(_lm8) & (_lm8 >= _lm8[-1] - np.log10(2.0)))
                if len(_hit):
                    _i = int(_hit[0])
                    _th = (_tt8[_i] if _i == 0 or not np.isfinite(_lm8[_i - 1]) else
                           float(np.interp(_lm8[-1] - np.log10(2.0), [_lm8[_i - 1], _lm8[_i]], [_tt8[_i - 1], _tt8[_i]])))
                    _zf_half8 = _z_of_t8(_th, _tt8, _zz8)
        # THE PEAK OF THE STAR-FORMATION HISTORY, on the branch's own series (P8_SFPEAK)
        _tpk8 = np.nan
        if P8_SFPEAK == "ssfr_pre":
            _tpk8 = float(_r.get("t_pk", np.nan))
        elif "lssfr" in _ser:
            with np.errstate(all="ignore"):
                _y8 = np.asarray(_ser["lssfr"], float)[_o8]
                if P8_SFPEAK == "sfr_max" and "lms" in _ser:
                    _y8 = _y8 + np.asarray(_ser["lms"], float)[_o8]      # log SFR = log sSFR + log M*
            if np.isfinite(_y8).any():
                _tpk8 = float(_tt8[int(np.nanargmax(_y8))])
        # WHERE EACH CRITICAL POINT LANDS, AND THEN NO TWO PANELS OF A ROW ON THE SAME SNAPSHOT.
        # A short quenching event puts the truncation and the quenching time inside one snapshot gap, and the
        # row draws the same picture twice.  A colliding point is pushed FORWARD to the next free snapshot of
        # the chain — the state just after the degenerate one, which is what the column means anyway — with
        # the anchor holding the last place; if that runs off the end they are pushed back from the anchor
        # instead.  Every displacement is counted and printed.
        _ix8, _ORD8, _tv8 = {}, [], {}
        for _st, _tv in (("sfr_peak", _tpk8), ("jet_on", _r["t_jet"]), ("sft", _r["t_sft"]), ("qt", _r["t_qt"]),
                         ("anchor", _r["t_anchor"])):
            if _st in P8_STAGES and np.isfinite(_tv) and len(_tt8):
                _ix8[_st] = int(np.argmin(np.abs(_tt8 - float(_tv)))); _ORD8.append(_st); _tv8[_st] = float(_tv)
        _ORD8.sort(key=lambda s: _tv8[s])                     # in THIS galaxy's time order: the SFH peak can sit after the jet
        _raw8, _top8 = dict(_ix8), len(_ss8) - 1
        for _a8, _b8 in zip(_ORD8[:-1], _ORD8[1:]):           # forward: each point after the one before it
            _ix8[_b8] = max(_ix8[_b8], _ix8[_a8] + 1)
        if _ORD8 and _ix8[_ORD8[-1]] > _top8:                 # ... unless that runs past the end of the chain
            _ix8[_ORD8[-1]] = _top8
            for _a8, _b8 in zip(_ORD8[-2::-1], _ORD8[::-1][:-1]):
                _ix8[_a8] = min(_ix8[_a8], _ix8[_b8] - 1)
        _ch, _mv8 = {}, 0
        for _st in _ORD8:
            _i8 = int(np.clip(_ix8[_st], 0, _top8))
            _mv8 += int(_i8 != _raw8[_st])
            _sn = int(_ss8[_i8])
            _gx = _GX8.get((_key[0], _key[1], _sn))
            if _gx is None:
                continue
            _ch[_st] = (_sn, _gx)
        _nmov8.append(_mv8)
        _ncol8.append(len({v[0] for v in _ch.values()}))
        if "z0" in P8_STAGES and _DESC8.get(_key) is not None:   # TODAY: off the chain, its own snapshot, never collides
            _ch["z0"] = (int(P8_Z0SNAP), int(_DESC8[_key]))
        for _st, (_sn, _gx) in _ch.items():
            _row = dict(key=_key, stage=_st, snap=_sn, gx=_gx, pop="Q")
            _cat = _C8.loc[(_sn, _gx)] if (_sn, _gx) in _C8.index else None
            for _dst, _src in (("ngas_cat", "ngas"), ("nstar_cat", "nstar"), ("z", "z"),
                               ("mdust_cat", "mdust"), ("mh2_cat", "mh2"), ("mgas_cat", "mgas"),
                               ("kappa_gas_cat", "kappa_gas"), ("kappa_star_cat", "kappa_star")):
                _row[_dst] = float(_cat[_src]) if (_cat is not None and _src in _cat.index) else np.nan
            _row["log_mstar_anchor"] = float(_r["log_mstar"]) if "log_mstar" in _r else np.nan
            _ER8.append(_row)
        if "anchor" in _ch:
            _AR8.append(dict(key=_key, hi_geom=bool(_r["grp"] == "burst"), zform_age=_zf_age8, zform_half=_zf_half8, zform_first=_zf_first8,
                             **{P8_GEOMCOL: float(_r["kz_cat"]) if np.isfinite(_r.get("kz_cat", np.nan)) else np.nan}))
    EPO6, ANC6 = pd.DataFrame(_ER8), pd.DataFrame(_AR8)
    # a column that is empty everywhere is worse than an absent one: the automatic extent tests for it
    EPO6 = EPO6.drop(columns=[c for c in ("r50star_cat",) if c in EPO6.columns
                              and not np.isfinite(EPO6[c].to_numpy(float)).any()])
    GRP6 = [("sel", None, GCOL["burst"], "-", GLAB["burst"]), ("rest", None, GCOL["aging"], "-", GLAB["aging"])]
    if not len(ANC6):
        print("Part 8: no galaxy of the selection has an anchor in the branch-id map — skipped."); _HAVE8 = False
    else:
        _nc8 = pd.Series(_ncol8).value_counts().sort_index().to_dict() if _ncol8 else {}
        _nm8 = int(np.sum(np.asarray(_nmov8) > 0)) if _nmov8 else 0
        print(f"Part 8 — the shim: {len(ANC6)} galaxies of the selection carry a chain  ("
              + ", ".join(f"{GSH[g]} {int((ANC6['hi_geom'] == (g == 'burst')).sum())}" for g in GK)
              + f"), {len(EPO6)} (galaxy, epoch) rows over {EPO6['stage'].nunique()} critical points;  "
              + "distinct snapshots per galaxy: " + ", ".join(f"{k} in {v} galaxies" for k, v in _nc8.items())
              + (f";  {_nm8} had a critical point inside another's snapshot and were stepped forward"
                 if _nm8 else "")
              + (f"  <- a galaxy with fewer than {_nst8} STILL repeats a panel: its chain is too short to separate "
                 "them" if any(k < _nst8 for k in _nc8) else ""))

if not _HAVE8:
    print("Part 8: the shim above found nothing to draw — see the line it printed.")
elif not os.path.isdir(P8_REDUCED):
    print(f"Part 8: {P8_REDUCED} is missing — build_reduced_particles_job.py writes the cutouts (cluster).")
else:
    _fs8 = lambda v: float(v) * float(P8_FS)     # one dial for every label in this part
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    import matplotlib.patheffects as pe
    try:
        from simbanator.visualization.rendering import ReducedRender
    except Exception as _e8:
        ReducedRender = None
        print(f"Part 8: simbanator.visualization.rendering.ReducedRender not importable ({_e8}) — "
              f"it needs py-sphviewer + h5py.")
    if ReducedRender is not None:
        # ── where every galaxy is at every critical point, and which of those have a cutout on disk ──
        _EP8 = {(k, s): (int(sn), int(gx)) for k, s, sn, gx
                 in zip(EPO6["key"], EPO6["stage"], EPO6["snap"], EPO6["gx"])}
        _path8 = lambda sn, gx: ReducedRender.epoch_path(P8_REDUCED, sn, gx, P8_PREFIX)

        def _stream8(rr):
            """The velocity field behind the streamlines, from the gas P8_STREAM_SOURCE selects (in rr's CURRENT frame), the bulk
            velocity P8_STREAM_BULK names subtracted -> ((X, Y, VX, VY, den), (n_used, n_in_frame))."""
            g = rr.parts.get("gas", {})
            if "pos" not in g or "vel" not in g:
                return None, (0, 0)
            xyz = rr.project("gas", P8_SPOS); vel = rr.project("gas", P8_SPOS, what="vel")
            w = np.asarray(g.get(P8_STREAM_WEIGHT, g.get("m_gas")), float)
            inframe = (np.abs(xyz[:, 0]) <= _EXT8) & (np.abs(xyz[:, 1]) <= _EXT8)
            memb = np.asarray(g["member"], bool) if "member" in g else np.ones(len(w), bool)
            keep = (memb if P8_STREAM_SOURCE == "members" else
                    (np.abs(xyz[:, 2]) <= float(P8_STREAM_DEPTH)) if P8_STREAM_SOURCE == "slab" else np.ones(len(w), bool))
            keep &= np.isfinite(w) & (w > 0) & np.all(np.isfinite(vel), axis=1)
            # the systemic velocity: the galaxy's own (member gas, else member stars), or the slab's mean
            v0 = None
            if P8_STREAM_BULK == "members":
                mm = memb & np.isfinite(w) & (w > 0)
                if mm.sum() >= 5:
                    v0 = np.average(vel[mm], axis=0, weights=w[mm])
                elif "star" in rr.parts and "vel" in rr.parts["star"]:
                    sv = rr.project("star", P8_SPOS, what="vel"); sw = np.asarray(rr.parts["star"]["m_star"], float)
                    sm = (np.asarray(rr.parts["star"]["member"], bool) if "member" in rr.parts["star"] else np.ones(len(sw), bool)) & (sw > 0)
                    if sm.sum() >= 5:
                        v0 = np.average(sv[sm], axis=0, weights=sw[sm])
            if v0 is None:
                v0 = np.average(vel[keep], axis=0, weights=w[keep]) if keep.sum() else np.zeros(3)
            saved = rr.parts["gas"]
            rr.parts["gas"] = {k: np.asarray(v)[keep] for k, v in saved.items()}          # the selection, in place, briefly
            try:
                X, Y, VX, VY, den = rr.velocity_field(ex=_EXT8, spos=P8_SPOS, weight=P8_STREAM_WEIGHT, ngrid=P8_NGRID,
                                                      subtract_bulk=False)
            finally:
                rr.parts["gas"] = saved
            VX = np.where(den > 0, VX - v0[0], 0.0); VY = np.where(den > 0, VY - v0[1], 0.0)
            return (X, Y, VX, VY, den), (int(np.sum(keep & inframe)), int(np.sum(inframe)))

        def _membmap8(rr):
            """The projected surface density of the CAESAR MEMBER gas alone — the ISM seen in projection.  The ISM is
            a member list and not a radius, so its edge is a SHAPE: P8_ISMEDGE contours this map at the level that
            encloses P8_ISMEDGE_ENC of the member gas mass in the frame, and that contour is the ISM/CGM boundary."""
            g = rr.parts.get("gas", {})
            if "pos" not in g or not len(g["pos"]):
                return None
            memb = np.asarray(g["member"], bool) if "member" in g else np.ones(len(g["pos"]), bool)
            if int(memb.sum()) < 8:
                return None
            saved = rr.parts["gas"]
            rr.parts["gas"] = {k: np.asarray(v)[memb] for k, v in saved.items()}      # the members, in place, briefly
            try:
                return rr.sigma_map("gas", ex=_EXT8, spos=P8_SPOS, xsize=P8_XSIZE)[0]
            finally:
                rr.parts["gas"] = saved

        _R50COMP8 = {"baryon": (("star", "m_star"), ("gas", "m_gas")), "star": (("star", "m_star"),),
                     "gas": (("gas", "m_gas"),), "H2": (("gas", "m_H2"),), "dust": (("gas", "m_dust"),)}

        def _r50bar8(rr):
            """The HALF-MASS RADIUS of what P8_R50MARK names, in the frame that is drawn: the radius holding half of
            that mass, projected the way an image measures it (P8_R50MARK_2D) and over the members only."""
            if not P8_R50MARK:
                return np.nan
            _r, _w = [], []
            for _cp, _ky in _R50COMP8.get(P8_R50MARK, ()):
                _g = rr.parts.get(_cp, {})
                if _ky not in _g or "pos" not in _g:
                    continue
                _m = np.asarray(_g[_ky], float)
                _keep = np.isfinite(_m) & (_m > 0)
                if P8_R50MARK_MEMBERS and "member" in _g:
                    _keep &= np.asarray(_g["member"], bool)
                if not _keep.any():
                    continue
                _xyz = rr.project(_cp, P8_SPOS)[_keep]
                _r.append(np.hypot(_xyz[:, 0], _xyz[:, 1]) if P8_R50MARK_2D
                          else np.sqrt(np.sum(_xyz ** 2, axis=1)))
                _w.append(_m[_keep])
            if not _r:
                return np.nan
            _r, _w = np.concatenate(_r), np.concatenate(_w)
            _o = np.argsort(_r); _c = np.cumsum(_w[_o])
            if not (_c[-1] > 0):
                return np.nan
            return float(_r[_o][min(int(np.searchsorted(_c, 0.5 * _c[-1])), len(_c) - 1)])

        def _enclevel8(m, enc):
            """The contour level of `m` that encloses `enc` of its total mass (the brightest pixels first)."""
            v = np.asarray(m, float).ravel()
            v = v[np.isfinite(v) & (v > 0)]
            if v.size < 16:
                return None
            v = np.sort(v)[::-1]
            c = np.cumsum(v)
            return float(v[min(int(np.searchsorted(c, float(enc) * c[-1])), v.size - 1)])

        def _chain8(key):
            """The (snap, gx) of `key` at every P8_STAGES epoch.  A P8_SOFT stage with no cutout on disk (or no
            descendant at all) is None in the chain and drawn as an empty panel; any other stage missing on disk
            returns None for the whole chain — the galaxy is not drawn."""
            out = {}
            for s_ in P8_STAGES:
                sg = _EP8.get((key, s_))
                ok = sg is not None and os.path.exists(_path8(*sg))
                if not ok and s_ not in P8_SOFT:
                    return None
                out[s_] = sg if ok else None
            return out
        _live8 = lambda ch: {v for v in ch.values() if v is not None}     # the (snap, gx) a chain actually draws

        # ── WHO GETS DRAWN: a window on each axis, then P8_NPER per group ────────────────────────────
        # The gallery only says something about the two groups if the galaxies in it are otherwise comparable,
        # so the windows are the comparison: set them on redshift, mass, age and sSFR and every row of the
        # figure is inside the same box.  Within a window the order is P8_ORDER's — the best resolved by
        # default, because that is the only thing that makes a picture better rather than different.
        _ANCROW8 = EPO6[EPO6["stage"] == "anchor"].drop_duplicates("key").set_index("key")
        _CAND8 = ANC6.copy()
        # the numbers the match, the seed cut and the figure are made of, all at the anchor
        # the age comes from the FULL Part 0 catalogue (CAT: every galaxy above log M* 9.5 per box and anchor), not
        # from QM — QM is CAT after the quenched rule and the mass / age / history cuts, and those are not the K-S
        # products' cuts, so a perfectly drawable anchor row can be missing from QM and have no age at all.
        _SRC8 = ("CAT" if "CAT" in globals() else ("QM" if "QM" in globals() else None))
        _Q8 = globals()[_SRC8][globals()[_SRC8]["box"] == P8_BOX].copy() if _SRC8 else None
        if _Q8 is not None:
            with np.errstate(all="ignore"):
                if "log_mstar" not in _Q8.columns:
                    _mst8 = _Q8["mstar"].to_numpy(float)
                    _Q8["log_mstar"] = np.log10(np.where(_mst8 > 0, _mst8, np.nan))
            _Q8 = _Q8[~_Q8.duplicated(subset=["snap", "gal_id"])]
            _Q8 = _Q8.set_index([_Q8["snap"].astype(int), _Q8["gal_id"].astype(int)])
        if _Q8 is not None and "lssfr" not in _Q8.columns and {"sfr", "mstar"}.issubset(_Q8.columns):
            with np.errstate(all="ignore"):
                _sf8, _mm8 = _Q8["sfr"].to_numpy(float), _Q8["mstar"].to_numpy(float)
                _Q8["lssfr"] = np.log10(np.where((_sf8 > 0) & (_mm8 > 0), _sf8 / np.where(_mm8 > 0, _mm8, 1), np.nan))
        for _c8, _src8 in (("_age", "age"), ("_lm", "log_mstar"), ("_ssfr", "lssfr")):
            _CAND8[_c8] = (_Q8[_src8].reindex(_CAND8["key"]).to_numpy(float)
                             if (_Q8 is not None and _src8 in _Q8.columns) else np.full(len(_CAND8), np.nan))
        if "log_mstar_anchor" in _ANCROW8.columns:            # the KS table always has the mass; QM does not always
            _CAND8["_lm"] = np.where(np.isfinite(_CAND8["_lm"].to_numpy(float)), _CAND8["_lm"].to_numpy(float),
                                      _ANCROW8["log_mstar_anchor"].reindex(_CAND8["key"]).to_numpy(float))
        for _c8, _src8 in (("_ngas", "ngas_cat"), ("_nstar", "nstar_cat"), ("_z", "z"),
                             ("_mdust", "mdust_cat"), ("_mh2", "mh2_cat"), ("_mgas", "mgas_cat"),
                             ("_kgas", "kappa_gas_cat"), ("_kstar", "kappa_star_cat")):
            _CAND8[_c8] = (_ANCROW8[_src8].reindex(_CAND8["key"]).to_numpy(float)
                             if _src8 in _ANCROW8.columns else np.full(len(_CAND8), np.nan))
        with np.errstate(all="ignore"):                          # THE ratio: dust over H2 by default, as in P6_DUST
            _num8 = _CAND8["_mdust"].to_numpy(float)
            _den8 = _CAND8["_mh2" if P8_DGR == "dh2" else "_mgas"].to_numpy(float)
            _CAND8["_dgr"] = np.log10(np.where((_num8 > 0) & (_den8 > 0),
                                                _num8 / np.where(_den8 > 0, _den8, np.nan), np.nan))
        with np.errstate(all="ignore"):
            _md8 = _CAND8["_mdust"].to_numpy(float)
            _CAND8["_lfd"] = np.where(_md8 > 0, np.log10(np.where(_md8 > 0, _md8, 1.0)), np.nan) \
                - _CAND8["_lm"].to_numpy(float)          # log M_dust/M*, the quantity the selection is made on
        _CAND8["_anchor"] = [int(k[0]) for k in _CAND8["key"]]
        _CAND8["_ok"] = [(_chain8(k) is not None) for k in _CAND8["key"]]       # all four epochs on disk
        _CAND8["_rank"] = 0.5 * (_CAND8["_ngas"].rank(pct=True) + _CAND8["_nstar"].rank(pct=True))

        _GNAME8 = [_n8 for _n8, _, _, _, _ in GRP6]                            # "sel" and "rest", the way Part 6 names GRP6
        _GINFO8 = {_n8: (_c8, _lab8) for _n8, _, _c8, _, _lab8 in GRP6}
        def _inclass8(df, name):
            return df["hi_geom"].to_numpy(bool) == (name == "sel")

        # ── WHO GETS DRAWN: the windows, then P8_NPER per group ──────────────────────────────────────
        _base8 = _CAND8[_CAND8["_ok"].to_numpy(bool)].copy()          # all four cutouts on disk
        _WIN8 = [("_z", P8_ZRANGE, "z"), ("_lm", P8_MRANGE, "log M*"), ("_age", P8_AGERANGE, "age [Gyr]"),
                 ("_ssfr", P8_SSFRANGE, "log sSFR")]

        def _win8(df, col, rng):
            """One window; a bound of None is no bound, and a galaxy with no value fails a window that has one."""
            if rng is None or all(r is None for r in rng) or col not in df.columns:
                return np.ones(len(df), bool)
            v = df[col].to_numpy(float)
            m = np.isfinite(v)
            if rng[0] is not None:
                m &= v >= float(rng[0])
            if rng[1] is not None:
                m &= v <= float(rng[1])
            return m

        _keep8, _toll8 = np.ones(len(_base8), bool), []
        for _c8w, _r8w, _n8w in _WIN8:
            _m8w = _win8(_base8, _c8w, _r8w)
            _toll8.append((_n8w, int((_keep8 & ~_m8w).sum())))
            _keep8 &= _m8w
        _mg8 = (_base8["_ngas"].to_numpy(float) >= float(P8_NGASMIN)) if P8_NGASMIN else np.ones(len(_base8), bool)
        _toll8.append((f"n_gas >= {P8_NGASMIN:g}", int((_keep8 & ~_mg8).sum())))
        _keep8 &= _mg8
        _pool8f = _base8[_keep8].copy()

        def _order8(df, name):
            """The survivors of one group, in the order P8_ORDER asks for."""
            d = df[_inclass8(df, name)].copy()
            if not len(d):
                return d
            if P8_ORDER == "median":
                _sc = np.zeros(len(d))
                for _c8o in (P8_GEOMCOL, "_dgr"):
                    v = d[_c8o].to_numpy(float)
                    _s8o = np.nanpercentile(v, 84) - np.nanpercentile(v, 16) if np.isfinite(v).sum() > 3 else np.nan
                    _sc += np.abs(v - np.nanmedian(v)) / (_s8o if np.isfinite(_s8o) and _s8o > 0 else 1.0)
                d["_score"] = np.nan_to_num(_sc, nan=np.inf)
                return d.sort_values("_score")
            if P8_ORDER == "random":
                return d.sample(frac=1.0, random_state=int(P8_ORDSEED))
            return d.sort_values("_rank", ascending=False)          # "res": the best resolved first

        REP8, _miss8, PAIR8 = [], [], []

        def _mkrep8(_n8, _r8):
            """One REP8 row from a pool row."""
            _col8, _lab8 = _GINFO8[_n8]
            return dict(group=_n8, label=_lab8, color=_col8, key=_r8["key"], chain=_chain8(_r8["key"]),
                        seed=False, geom=float(_r8[P8_GEOMCOL]), z=float(_r8["_z"]),
                        age=float(_r8["_age"]), lmstar=float(_r8["_lm"]),
                        ssfr=float(_r8.get("_ssfr", np.nan)), ldgr=float(_r8["_dgr"]),
                        lfd=float(_r8.get("_lfd", np.nan)), kgas=float(_r8.get("_kgas", np.nan)),
                        kstar=float(_r8.get("_kstar", np.nan)),
                        zform=float(_r8.get({"age": "zform_age", "half": "zform_half"}.get(P8_ZFORM, "zform_first"), np.nan)),
                        zform_age=float(_r8.get("zform_age", np.nan)), zform_half=float(_r8.get("zform_half", np.nan)),
                        zform_first=float(_r8.get("zform_first", np.nan)),
                        ngas=float(_r8["_ngas"]), nstar=float(_r8["_nstar"]), cost=np.nan)

        if P8_ORDER == "pairs" and len(_GNAME8) == 2:
            # THE TIGHTEST PAIRS: every (group 1, group 2) pair of the pool, ranked by its distance at the observation
            _A, _Bp = (_pool8f[_inclass8(_pool8f, g)].copy() for g in _GNAME8)
            _tm, _ta, _tz = P8_MATCH_TOL
            _D = np.zeros((len(_A), len(_Bp)))
            for _c8p, _tol in (("_lm", _tm), ("_age", _ta), ("_z", _tz)):
                if _tol:
                    _D = _D + np.abs(_A[_c8p].to_numpy(float)[:, None] - _Bp[_c8p].to_numpy(float)[None, :]) / float(_tol)
            _D = np.where(np.isfinite(_D), _D, np.inf)
            _ord = np.argsort(_D, axis=None)
            _rowsA, _rowsB, _usedA, _usedB = [], [], set(), set()
            for _f in _ord:
                if len(_rowsA) >= int(P8_NPER) + int(P8_SKIP) or not np.isfinite(_D.flat[_f]):
                    break
                _i, _j = np.unravel_index(_f, _D.shape)
                _ra, _rb = _A.iloc[_i], _Bp.iloc[_j]
                if _ra["key"] in _usedA or _rb["key"] in _usedB:
                    continue
                _la, _lb = _live8(_chain8(_ra["key"])), _live8(_chain8(_rb["key"]))
                if (_la & _lb) or any(_la & _live8(_chain8(e["key"])) for e in _rowsA + _rowsB) \
                        or any(_lb & _live8(_chain8(e["key"])) for e in _rowsA + _rowsB):
                    continue                                    # the same object twice, under two names
                _usedA.add(_ra["key"]); _usedB.add(_rb["key"])
                _rowsA.append(_ra); _rowsB.append(_rb)
                _dm, _da, _dz = (float(_rb["_lm"]) - float(_ra["_lm"]), float(_rb["_age"]) - float(_ra["_age"]), float(_rb["_z"]) - float(_ra["_z"]))
                PAIR8.append(dict(seed=_ra["key"], partner=_rb["key"], d_logm=_dm, d_age=_da, d_z=_dz, dist=float(_D[_i, _j]),
                                  inside=((not _tm or abs(_dm) <= _tm) and (not _ta or abs(_da) <= _ta) and (not _tz or abs(_dz) <= _tz))))
            _rowsA, _rowsB, PAIR8 = _rowsA[int(P8_SKIP):], _rowsB[int(P8_SKIP):], PAIR8[int(P8_SKIP):]
            if P8_LIST:
                print(f"  the {min(int(P8_LIST), len(PAIR8))} tightest pairs of the pool ({len(_A)} x {len(_Bp)} candidates), drawn = the first {int(P8_NPER)}"
                      + (f" after skipping {int(P8_SKIP)}" if P8_SKIP else "") + ":")
            for _n8, _rows8 in ((_GNAME8[0], _rowsA), (_GNAME8[1], _rowsB)):
                if not _rows8:
                    _miss8.append(_GINFO8[_n8][1]); continue
                for _r8 in _rows8:
                    REP8.append(_mkrep8(_n8, _r8))
            _seed8 = _GNAME8[0]
        _gorder8 = []
        _seed8 = (P8_MATCH_SEED if P8_MATCH else None) if P8_ORDER != "pairs" else _GNAME8[0]
        if P8_ORDER != "pairs":
            _gorder8 = ([_seed8] + [g for g in _GNAME8 if g != _seed8]) if _seed8 in _GNAME8 else list(_GNAME8)
        for _n8 in _gorder8:
            _col8, _lab8 = _GINFO8[_n8]
            if P8_MATCH and _n8 != _seed8 and any(r["group"] == _seed8 for r in REP8):
                # THE PARTNERS: for every seed row, the closest galaxy of this group at the observation
                _cand = _pool8f[_inclass8(_pool8f, _n8)].copy()
                _tm, _ta, _tz = P8_MATCH_TOL
                _taken8 = []
                for _sr in [r for r in REP8 if r["group"] == _seed8]:
                    _d = np.zeros(len(_cand))
                    if _tm: _d = _d + np.abs(_cand["_lm"].to_numpy(float) - _sr["lmstar"]) / float(_tm)
                    if _ta: _d = _d + np.abs(_cand["_age"].to_numpy(float) - _sr["age"]) / float(_ta)
                    if _tz: _d = _d + np.abs(_cand["_z"].to_numpy(float) - _sr["z"]) / float(_tz)
                    _d = np.where(np.isfinite(_d), _d, np.inf)
                    _ok = np.array([(k not in [e["key"] for e in _taken8]) and
                                    not (_live8(_chain8(k)) & _live8(_chain8(_sr["key"]))) and
                                    not any(_live8(_chain8(k)) & _live8(_chain8(e["key"])) for e in _taken8)
                                    for k in _cand["key"]])
                    _d = np.where(_ok, _d, np.inf)
                    if not np.isfinite(_d).any():
                        print(f"      no partner of '{_lab8}' left for seed {_sr['key']} — that row has none"); continue
                    _j = int(np.argmin(_d)); _pr = _cand.iloc[_j]
                    _taken8.append(_pr)
                    _dm, _da, _dz = (float(_pr["_lm"]) - _sr["lmstar"], float(_pr["_age"]) - _sr["age"], float(_pr["_z"]) - _sr["z"])
                    _inside = ((not _tm or abs(_dm) <= _tm) and (not _ta or abs(_da) <= _ta) and (not _tz or abs(_dz) <= _tz))
                    PAIR8.append(dict(seed=_sr["key"], partner=tuple(_pr["key"]), d_logm=_dm, d_age=_da, d_z=_dz, dist=float(_d[_j]), inside=_inside))
                _rows8 = _taken8
                if not _rows8:
                    _miss8.append(_lab8); continue
                for _r8 in _rows8:
                    REP8.append(dict(group=_n8, label=_lab8, color=_col8, key=_r8["key"], chain=_chain8(_r8["key"]),
                                     seed=False, geom=float(_r8[P8_GEOMCOL]), z=float(_r8["_z"]),
                                     age=float(_r8["_age"]), lmstar=float(_r8["_lm"]),
                                     ssfr=float(_r8.get("_ssfr", np.nan)), ldgr=float(_r8["_dgr"]),
                                     lfd=float(_r8.get("_lfd", np.nan)), kgas=float(_r8.get("_kgas", np.nan)),
                                     kstar=float(_r8.get("_kstar", np.nan)),
                                     zform=float(_r8.get("zform_age" if P8_ZFORM == "age" else "zform_half", np.nan)),
                                     zform_age=float(_r8.get("zform_age", np.nan)), zform_half=float(_r8.get("zform_half", np.nan)),
                                     ngas=float(_r8["_ngas"]), nstar=float(_r8["_nstar"]), cost=np.nan))
                continue
            _hand = [tuple(k) for k in (P8_PICK.get(_n8) or [])]
            _rows8 = []
            for _k8 in _hand:                                        # the hand-named ones first, if drawable
                _h8 = _base8[[k == _k8 for k in _base8["key"]]]
                if len(_h8):
                    _rows8.append(_h8.iloc[0])
                else:
                    print(f"  P8_PICK['{_n8}'] = {_k8} has no cutout at every critical point — ignored")
            _ord8 = _order8(_pool8f, _n8)
            if P8_LIST:
                print(f"  the pool of '{_lab8}', in {P8_ORDER!r} order "
                      f"(skipping {int(P8_SKIP)}, taking {int(P8_NPER)}):")
                for _i8, (_, _c8r) in enumerate(_ord8.head(int(P8_LIST)).iterrows()):
                    _tag = ("  <- drawn" if int(P8_SKIP) <= _i8 < int(P8_SKIP) + int(P8_NPER) else "")
                    print(f"      [{_i8:2d}] {str(_c8r['key']):>12s}  z {_c8r['_z']:.2f}  "
                          f"age {_c8r['_age']:5.2f}  log M* {_c8r['_lm']:6.2f}  "
                          f"log sSFR {float(_c8r.get('_ssfr', np.nan)):7.2f}  "
                          f"{'dh2' if P8_DGR == 'dh2' else 'D/G'} {_c8r['_dgr']:6.2f}  "
                          f"({_c8r['_ngas']:.0f} gas){_tag}")
            for _, _r8 in _ord8.iloc[int(P8_SKIP):].iterrows():
                if len(_rows8) >= int(P8_NPER):
                    break
                if any(_r8["key"] == _e["key"] for _e in _rows8):
                    continue
                # ... AND ITS HISTORY MUST BE ITS OWN.  Two anchor rows of the same physical object resolve to
                # the same cutouts; sharing even one (snapshot, index) with a galaxy already taken means the
                # panel would be the same picture under a different name.
                _ch8 = _chain8(_r8["key"])
                if _ch8 and any(_live8(_ch8) & _live8(_chain8(_e["key"])) for _e in _rows8):
                    print(f"      {str(_r8['key']):>12s} shares its history with one already taken — skipped")
                    continue
                _rows8.append(_r8)
            if not _rows8:
                _miss8.append(_lab8); continue
            for _j8, _r8 in enumerate(_rows8):
                REP8.append(dict(group=_n8, label=_lab8, color=_col8, key=_r8["key"], chain=_chain8(_r8["key"]),
                                  seed=False, geom=float(_r8[P8_GEOMCOL]), z=float(_r8["_z"]),
                                  age=float(_r8["_age"]), lmstar=float(_r8["_lm"]),
                                  ssfr=float(_r8.get("_ssfr", np.nan)), ldgr=float(_r8["_dgr"]),
                                  lfd=float(_r8.get("_lfd", np.nan)), kgas=float(_r8.get("_kgas", np.nan)),
                                  kstar=float(_r8.get("_kstar", np.nan)),
                                  zform=float(_r8.get({"age": "zform_age", "half": "zform_half"}.get(P8_ZFORM, "zform_first"), np.nan)),
                                  zform_age=float(_r8.get("zform_age", np.nan)), zform_half=float(_r8.get("zform_half", np.nan)),
                                  zform_first=float(_r8.get("zform_first", np.nan)),
                                  ngas=float(_r8["_ngas"]), nstar=float(_r8["_nstar"]), cost=np.nan))
        for _lb in _miss8:
            print(f"  no galaxy of '{_lb}' survives the windows with all of {', '.join(P8_STAGES)} on disk "
                  f"— that group is not drawn")
        if (P8_MATCH or P8_ORDER == "pairs") and PAIR8:
            print(f"  MATCHED PAIRS at the observation ("
                  + ("the tightest pairs of the pool, P8_ORDER = 'pairs'" if P8_ORDER == "pairs" else
                     f"seed = '{_GINFO8[_seed8][1]}', partner = the closest of the other group")
                  + f"; the distance = the offsets in log M* / age / z over {P8_MATCH_TOL}):")
            for _p in PAIR8:
                print(f"    {str(_p['seed']):>11s} -> {str(_p['partner']):>11s}   d log M* {_p['d_logm']:+.2f}   d age {_p['d_age']:+.2f} Gyr   "
                      f"dz {_p['d_z']:+.2f}" + ("" if _p["inside"] else "   <- OUTSIDE the tolerances: the best there was"))
        if not REP8:
            print("Part 8: nothing to draw — no group has a galaxy inside the windows with a cutout at every "
                  "critical point.  Widen P8_ZRANGE / P8_MRANGE / P8_AGERANGE / P8_SSFRANGE or lower P8_NGASMIN.")
        # THE PLAN FOR THE CUTOUTS THAT ARE NOT THERE YET: every P8_SOFT (snap, gx) the selection resolves to — all of
        # the pool, not the drawn rows only, so the job runs once — in the format build_reduced_particles_job.py reads.
        _want8 = sorted({_EP8[(k, s_)] for k in _CAND8["key"] for s_ in P8_SOFT if (k, s_) in _EP8})
        _miss0 = [sg for sg in _want8 if not os.path.exists(_path8(*sg))]
        if _want8:
            print(f"  {' / '.join(P8_SOFT)}: {len(_want8) - len(_miss0)} of {len(_want8)} resolved galaxies have their cutout"
                  + (f" — every one drawn as an empty panel until it exists" if _miss0 else ""))
        if _miss0:
            _pp8 = os.path.join(OUT, f"ba_gallery_z0_plan_{P8_BOX}.hdf5")
            with h5py.File(_pp8, "w") as _pf:
                _pf.attrs["sim_name"] = P8_BOX
                _pf.attrs["note"] = f"starburst_vs_aging.ipynb Part 8: the {'/'.join(P8_SOFT)} cutouts of the selection (P8_Z0SNAP {P8_Z0SNAP})"
                _pf.attrs["rmax_kpc"] = 100.0
                _pf.create_dataset("entry_snap", data=np.array([s for s, _ in _miss0], np.int32))
                _pf.create_dataset("entry_gx", data=np.array([g for _, g in _miss0], np.int64))
            print(f"  the {len(_miss0)} missing ({len({s for s, _ in _miss0})} snapshot(s)) are in the plan -> "
                  f"sbatch --array=0-0 submit_reduced_particles.sh {os.path.relpath(_pp8, os.getcwd())}")
        _DGLAB8 = (r"\log\,M_{\rm dust}/M_{\rm H_2}" if P8_DGR == "dh2" else r"\log\,M_{\rm dust}/M_{\rm gas}")
        _DGSHORT8 = (r"\delta_{\rm DGR}" if P8_DGR == "dh2" else r"\delta_{\rm DG}")   # the panel writes the symbol only:
        #   delta_DGR = log M_dust / M_H2 with M_H2 x HE (Part 16b's definition, Part 1's ldh2 convention); delta_DG = log M_dust / M_gas
        _fignote8 = ""
        if REP8:
            _zs8 = [_r["z"] for _r in REP8 if np.isfinite(_r["z"])]
            _rt8 = ", ".join(f"{_n}: {_a}" for _n, _a in
                             (("z", P8_ZRANGE), ("log M*", P8_MRANGE), ("age", P8_AGERANGE),
                              ("log sSFR", P8_SSFRANGE))
                             if _a is not None and any(_x is not None for _x in _a))
            _fignote8 = (f"{P8_NPER} per group, taken by "
                         + {"res": "resolution", "median": "closeness to the group median",
                            "random": "a seeded draw"}.get(P8_ORDER, P8_ORDER)
                         + (f";  {_rt8}" if _rt8 else "")
                         + (f";  $z = {min(_zs8):.2f}$-${max(_zs8):.2f}$" if _zs8 else ""))
        print(f"Part 8 — the pictures, {BOX_LABEL.get(P8_BOX, P8_BOX)}: {len(REP8)} galaxies x "
              f"{len(P8_STAGES)} critical points, {P8_SPOS}, "
              f"{'catalogue members only' if P8_MEMBERS else 'the whole 100 kpc cutout'}.")
        print(f"  the pool: {len(_base8)} galaxies with all four cutouts -> {len(_pool8f)} inside the windows ("
              + ", ".join(f"{_n}: -{_c}" for _n, _c in _toll8 if _c) + ")" if any(c for _, c in _toll8)
              else f"  the pool: {len(_base8)} galaxies with all four cutouts, every one inside the windows")
        print(f"  taken by {P8_ORDER!r}, {P8_NPER} per group.")
        for _r in REP8:
            _f24n = lambda v, f="{:.2f}": (f.format(v) if np.isfinite(v) else "n/a")
            print(f"    {_r['label']:42s} anchor_snap {_r['key'][0]:3d} gal {_r['key'][1]:5d}  "
                  f"z {_r['z']:.2f}  age {_f24n(_r['age'])} Gyr  log M* {_f24n(_r['lmstar'])}  "
                  f"log sSFR {_f24n(_r.get('ssfr', np.nan))}  "
                  f"lfd {_f24n(_r.get('lfd', np.nan))}  kappa_gas {_f24n(_r.get('kgas', np.nan))}  "
                  f"z_form {_f24n(_r.get('zform_first', np.nan))} (first snapshot) / {_f24n(_r.get('zform_age', np.nan))} (mass-weighted) / {_f24n(_r.get('zform_half', np.nan))} (half-mass)  "
                  f"({_r['ngas']:.0f} gas + {_r['nstar']:.0f} star particles)")

        # ── the frame: one half-width for the whole grid, and a line saying which knob set it ──
        if P8_EXT is not None:
            _EXT8 = float(P8_EXT)
            _why8 = (f"P8_SIZE = {2 * _EXT8:g} kpc a side (P8_EXTFAC and P8_EXTLIM are not used)")
        else:
            _r50_8 = (EPO6.groupby(["snap", "gx"])["r50star_cat"].first()      # (snap, gx) repeats across anchors
                       if "r50star_cat" in EPO6.columns else None)
            _rr8 = [float(_r50_8.get(sg, np.nan)) for r in REP8 for sg in r["chain"].values() if sg is not None] \
                if _r50_8 is not None else []
            _rr8 = [v for v in _rr8 if np.isfinite(v) and v > 0]
            _raw8 = round(P8_EXTFAC * float(np.median(_rr8))) if _rr8 else 15.0
            _EXT8 = float(np.clip(_raw8, *P8_EXTLIM))
            _why8 = (f"P8_EXTFAC {P8_EXTFAC:g} x median R50(*) "
                      + (f"{np.median(_rr8):.1f} kpc = {_raw8:g}" if _rr8 else "unavailable -> 15")
                      + (f", CLIPPED by P8_EXTLIM {tuple(P8_EXTLIM)}" if _EXT8 != _raw8 else
                         f"; P8_EXTLIM {tuple(P8_EXTLIM)} did not bind — SET P8_SIZE (kpc a side) for a larger "
                         f"region"))
        print(f"  every panel is {2 * _EXT8:.0f} kpc across (half-width {_EXT8:g} kpc from {_why8}).")

        # ── the renders: one file open per panel, every product taken from it ──
        # THE RENDER CACHE IS KEYED ON THE GALAXY, NOT ON ITS CLASS.  The cell this was ported from drew
        # exactly one galaxy per class, so (class, stage) named a panel uniquely; with P8_NPER > 1 every
        # galaxy of a class wrote into the same slot and every row of that class drew the LAST one rendered.
        _ck8 = lambda r, s_: (r["group"], r["key"], s_)
        _npg8 = ("one galaxy per group" if len(REP8) <= len(_GNAME8)
                 else f"{max(1, len(REP8) // max(1, len(_GNAME8)))} galaxies per group")
        REN8, ROW8, _warn8 = {}, [], False
        _frames8 = []                                            # (class, stage, frame_info) per panel: which frame was used
        for _r in REP8:
            _rowframe = None                                      # the row's common frame: the spin at P8_FRAME_STAGE
            if P8_FRAME_STAGE and P8_FRAME != "stellar" and _r["chain"].get(P8_FRAME_STAGE) is not None:
                _rf = ReducedRender.from_epoch(P8_REDUCED, *_r["chain"][P8_FRAME_STAGE], P8_PREFIX, members_only=P8_MEMBERS,
                                               frame=P8_FRAME, frame_rmax=P8_FRAME_RMAX, frame_members=P8_FRAME_MEMBERS)
                if _rf.frame != "stellar":
                    _rowframe = (_rf.evecs.copy(), dict(_rf.frame_info))
                del _rf
            for _s8 in P8_STAGES:
                if _r["chain"][_s8] is None:                      # a P8_SOFT stage with nothing on disk: an empty panel
                    REN8[_ck8(_r, _s8)] = None
                    _wt = _EP8.get((_r["key"], _s8))
                    ROW8.append(dict(group=_r["group"], label=_r["label"], stage=_s8, anchor_snap=_r["key"][0],
                                     gal_id=_r["key"][1], snap=_wt[0] if _wt else -1, gx=_wt[1] if _wt else -1,
                                     z=np.nan, ext_kpc=_EXT8, drawn=False,
                                     why="cutout not built" if _wt else "no descendant"))
                    continue
                _sn, _gx = _r["chain"][_s8]
                _rr = ReducedRender.from_epoch(P8_REDUCED, _sn, _gx, P8_PREFIX, members_only=P8_MEMBERS,
                                               frame=P8_FRAME, frame_rmax=P8_FRAME_RMAX, frame_members=P8_FRAME_MEMBERS)
                _own = dict(getattr(_rr, "frame_info", {}))
                if _rowframe is not None:
                    _rr.set_evecs(_rowframe[0], label=f"{P8_FRAME}@{P8_FRAME_STAGE}", info=_rowframe[1])
                    _own = dict(_rr.frame_info, own_coherence=_own.get("coherence", np.nan), own_n=_own.get("n_used", 0))
                _frames8.append((f"{_r['group']} {_r['key']}", _s8, _own))
                if np.isfinite(_rr.rmax_kpc) and _EXT8 > _rr.rmax_kpc and not _warn8:
                    print(f"  NOTE the frame half-width {_EXT8:g} kpc runs past the cutout's own rmax_kpc = "
                          f"{_rr.rmax_kpc:g}: the corners are empty because there are no particles there, not because "
                          f"the sky is.")
                    _warn8 = True
                _cell = dict(z=_rr.redshift, ext=[-_EXT8, _EXT8, -_EXT8, _EXT8],
                             rgb=_rr.rgb(ex=_EXT8, spos=P8_SPOS, channels=P8_CHAN, dyn=P8_DYN, pmax=P8_PMAX,
                                         stretch=P8_STRETCH, tau_max=P8_TAU, xsize=P8_XSIZE)[0],
                             tot=_rr.totals(ex=_EXT8, spos=P8_SPOS),
                             memb=_rr.totals(ex=None, spos=P8_SPOS, members=True))
                if P8_STRETCH == "dyn":                                  # the channels as surface densities, for the scaling below
                    _cell["chan"] = {}
                    for _cn8 in P8_CHAN:
                        try:
                            _cell["chan"][_cn8] = _rr.sigma_map(_cn8, ex=_EXT8, spos=P8_SPOS, xsize=P8_XSIZE)[0]
                        except Exception:
                            pass
                for _f8 in P8_MAPS:
                    _cell[f"map_{_f8}"] = (
                        _rr.sigma_map(_f8, ex=_EXT8, spos=P8_SPOS, xsize=P8_XSIZE)[0]
                        if P8_MAPDEF[_f8][0] == "sigma" else
                        _rr.weighted_map(_f8, ex=_EXT8, spos=P8_SPOS, xsize=P8_XSIZE)[0])
                if P8_STREAM:
                    _cell["vel"], _cell["vel_n"] = _stream8(_rr)
                if P8_ISMEDGE == "members":
                    _cell["ism"] = _membmap8(_rr)
                _cell["r50"] = _r50bar8(_rr)
                REN8[_ck8(_r, _s8)] = _cell
                _row = dict(group=_r["group"], label=_r["label"], stage=_s8, anchor_snap=_r["key"][0],
                            gal_id=_r["key"][1], snap=_sn, gx=_gx, z=_rr.redshift, ext_kpc=_EXT8, drawn=True, why="",
                            zform_age=_r.get("zform_age", np.nan), zform_half=_r.get("zform_half", np.nan), zform_first=_r.get("zform_first", np.nan),
                            loaded="members" if P8_MEMBERS else "region",
                            stream_gas=(f"slab +-{P8_STREAM_DEPTH:g} kpc" if P8_STREAM_SOURCE == "slab" else P8_STREAM_SOURCE) if P8_STREAM else "",
                            stream_n_used=_cell.get("vel_n", (0, 0))[0], stream_n_in_frame=_cell.get("vel_n", (0, 0))[1],
                            r50_kpc=_cell.get("r50", np.nan), r50_of=P8_R50MARK or "")
                _row.update({f"frame_{k}": v for k, v in _cell["tot"].items()})       # what is drawn in the panel
                _row.update({f"member_{k}": v for k, v in _cell["memb"].items()})     # the catalogue galaxy itself
                ROW8.append(_row)

        # ── THE COMPOSITE ON ONE SCALE PER CHANNEL ACROSS THE GRID (P8_RGB_SCALE) ──────────────────────────────
        _scale8 = ({c: P8_RGB_SCALE for c in P8_CHAN} if isinstance(P8_RGB_SCALE, str) else
                   {c: P8_RGB_SCALE.get(c, "panel") for c in P8_CHAN})
        _top8 = {}                                               # the grid white points (only on the grid scale)
        if P8_STRETCH == "dyn" and REN8 and any(v == "grid" for v in _scale8.values()):
            _dyn8 = ([float(P8_DYN)] * len(P8_CHAN) if np.isscalar(P8_DYN)
                     else [float(d) for d in P8_DYN] + [3.5] * max(0, len(P8_CHAN) - len(P8_DYN)))
            _cells8g = [c for c in REN8.values() if c is not None and "chan" in c]
            _top8 = {}
            for _cn8 in P8_CHAN:
                _pos = np.concatenate([c["chan"][_cn8][np.isfinite(c["chan"][_cn8]) & (c["chan"][_cn8] > 0)].ravel()
                                       for c in _cells8g if _cn8 in c["chan"]] or [np.array([])])
                _top8[_cn8] = float(np.percentile(_pos, P8_PMAX)) if _pos.size else np.nan

            def _ptop8(m):
                """A panel's own top: the P8_PMAX percentile of its positive pixels (the renderer's rule)."""
                _p = m[np.isfinite(m) & (m > 0)]
                return float(np.percentile(_p, P8_PMAX)) if _p.size else np.nan

            for _c8g in _cells8g:
                _out = None
                for _k8, _cn8 in enumerate(P8_CHAN):
                    if _cn8 not in _c8g["chan"]:
                        continue
                    _hi = _top8[_cn8] if _scale8[_cn8] == "grid" else _ptop8(_c8g["chan"][_cn8])
                    if not (np.isfinite(_hi) and _hi > 0):
                        continue
                    _dn = _dyn8[_k8]; _lo = _hi / 10.0 ** _dn
                    with np.errstate(divide="ignore", invalid="ignore"):
                        _im = np.clip(np.log10(np.clip(np.nan_to_num(_c8g["chan"][_cn8], nan=0.0), _lo, _hi) / _lo) / _dn, 0.0, 1.0)
                    _rgba = ReducedRender.CHANNEL_CMAPS.get(_cn8, cm.Greys_r)(_im)
                    _out = _rgba if _out is None else np.concatenate([1.0 - (1.0 - _out[..., :3]) * (1.0 - _rgba[..., :3]),
                                                                       np.ones_like(_rgba[..., 3:])], axis=-1)   # Screen
                if _out is not None:
                    _c8g["rgb"] = _out
            print("  the composite's white points (P8_RGB_SCALE): "
                  + ", ".join((f"{k} on the GRID scale = {_top8[k]:.2e} Msun/kpc^2" if _scale8[k] == "grid" else f"{k} per panel")
                              for k in P8_CHAN)
                  + f" (the {P8_PMAX:g}th percentile; {tuple(_dyn8[:len(P8_CHAN)])} decades shown) — on the grid scale a panel with "
                  "one gas or dust particle shows it at its true, faint density; per panel the stars keep their own contrast.")
        _stlab8 = {"jet_on": "Jet start", "sft": "Quenching", "qt": "Quenched", "anchor": "Observed", "agn_ign": "ignition",
                    "sfr_peak": "SF peak", "sf_peak": "SF peak", "z0": "Today ($z = 0$)",
                    "post_quench": "post-q", "gas_min": "gas min", "end": "end"}

        _R50LAB8 = {"baryon": r"$R_{50}^{\rm bar}$", "star": r"$R_{50}^\star$", "gas": r"$R_{50}^{\rm gas}$",
                    "H2": r"$R_{50}^{\rm H_2}$", "dust": r"$R_{50}^{\rm dust}$"}.get(P8_R50MARK, r"$R_{50}$")
        _edgewarn8 = [False]

        def _ismedge8(ax, cell, which):
            """The ISM/CGM boundary on one panel, if P8_ISMEDGE asks for it on this figure."""
            if not P8_ISMEDGE or (P8_ISMEDGE_ON is not None and which not in P8_ISMEDGE_ON):
                return
            if isinstance(P8_ISMEDGE, (int, float)) and not isinstance(P8_ISMEDGE, bool):
                ax.add_patch(plt.Circle((0.0, 0.0), float(P8_ISMEDGE), fill=False, color=P8_ISMEDGE_COL,
                                        lw=P8_ISMEDGE_LW, ls=P8_ISMEDGE_LS, zorder=6))
                return
            _m8 = cell.get("ism")
            if _m8 is None:
                return
            try:
                from scipy.ndimage import gaussian_filter
                _ms8 = gaussian_filter(np.where(np.isfinite(_m8), _m8, 0.0), float(P8_ISMEDGE_SM))
            except Exception:
                _ms8 = np.where(np.isfinite(_m8), _m8, 0.0)
            _lv8 = _enclevel8(_ms8, P8_ISMEDGE_ENC)
            if _lv8 is None:
                return
            _in8 = _ms8 >= _lv8
            if (_in8[0].any() or _in8[-1].any() or _in8[:, 0].any() or _in8[:, -1].any()) and not _edgewarn8[0]:
                print(f"  NOTE the ISM contour runs off the frame: the member gas fills the {2 * _EXT8:g} kpc box, so the "
                      f"boundary is outside the picture.  P8_SIZE has to be bigger than the ISM for it to mean anything.")
                _edgewarn8[0] = True
            ax.contour(_ms8, levels=[_lv8], extent=cell["ext"], origin="lower", colors=[P8_ISMEDGE_COL],
                       linewidths=P8_ISMEDGE_LW, linestyles=[P8_ISMEDGE_LS], zorder=6)

        def _dg8of(cell):
            """The panel's member D/G (P8_DGR) and its denominator; NaN unless both masses are there."""
            _mdu8 = float(cell["memb"].get("m_dust", np.nan))
            _mde8 = float(cell["memb"].get("m_H2" if P8_DGR == "dh2" else "m_gas", np.nan)) * (HE if P8_DGR == "dh2" else 1.0)
            # THE RATIO ONLY WHEN BOTH MASSES ARE THERE.  A member list with no H2 at all — the gas-free
            # z = 0 descendants — used to divide by a 1e-30 floor and print ~32 dex (2026-09-18).
            _dg8 = (_mdu8 / _mde8 if (np.isfinite(_mdu8) and np.isfinite(_mde8) and _mdu8 > 0 and _mde8 > 0)
                    else np.nan)
            return _dg8, _mde8

        def _decor8(ax, cell, s, fg, bg, ruler=False, title=False, text=True):
            """What every drawn panel carries on top of its image: the z / ratio box, the R50 circle, the stage name
            (title), the kpc ruler (ruler).  Shared by the grid and by the single panels (P8_SINGLES)."""
            if text:
                _dg8, _mde8 = _dg8of(cell)
                _ann8 = f"$z = {cell['z']:.2f}$" + (
                    f"\n${_DGSHORT8} = {np.log10(_dg8):.2f}$" if np.isfinite(_dg8) else
                    ("\nno H$_2$" if P8_DGR == "dh2" else "\nno gas") if (np.isfinite(_mde8) and _mde8 <= 0) else "")
                ax.text(0.03, 0.97, _ann8, transform=ax.transAxes, ha="left", va="top", linespacing=1.5,
                        fontsize=_fs8(8.5), color="0.1", zorder=12,
                        bbox=dict(boxstyle="round,pad=0.32", facecolor="white", alpha=0.82, edgecolor="none"))
            _r50p8 = float(cell.get("r50", np.nan))          # the galaxy's OWN scale, panel by panel
            if text and P8_R50MARK and np.isfinite(_r50p8) and _r50p8 > 0:
                # the half-mass radius as a CIRCLE, put in the corner the way a beam is put in a map (or
                # concentric on the galaxy, which is where the aperture actually is), in DATA units — the
                # panel is square in kpc, so a circle stays a circle
                _pad8 = 0.055 * 2 * _EXT8
                _off8 = (2 * _r50p8) > 0.80 * 2 * _EXT8          # too wide to sit in a corner
                _ctr8 = (P8_R50MARK_POS == "centre") or _off8    # ... so it goes where it actually is
                _cxy8 = (0.0, 0.0) if _ctr8 else (_EXT8 - _pad8 - _r50p8, -_EXT8 + _pad8 + _r50p8)
                ax.add_patch(plt.Circle(
                    _cxy8, _r50p8, lw=P8_R50MARK_LW, zorder=11, edgecolor=P8_R50MARK_COL,
                    ls="-" if not _ctr8 else (0, (6, 3)),
                    facecolor=mcolors.to_rgba(P8_R50MARK_COL, 0.0 if _ctr8 else float(P8_R50MARK_FILL))))
                if _ctr8:                                         # concentric: the number goes to the corner
                    ax.text(0.955, 0.045, f"{_R50LAB8} $= {_r50p8:.1f}$ kpc", transform=ax.transAxes,
                            color=P8_R50MARK_COL, ha="right", va="bottom", fontsize=_fs8(8), zorder=12,
                            path_effects=[pe.withStroke(linewidth=2.2, foreground=bg)])
                else:                                             # beam-style: the number over the circle
                    ax.text(0.955, 0.5 + (_cxy8[1] + _r50p8) / (2 * _EXT8) + 0.012,
                            f"{_R50LAB8} $= {_r50p8:.1f}$ kpc", transform=ax.transAxes,
                            color=P8_R50MARK_COL, ha="right", va="bottom", fontsize=_fs8(8), zorder=12,
                            path_effects=[pe.withStroke(linewidth=2.2, foreground=bg)])
            if title:
                ax.set_title(_stlab8.get(s, s), fontsize=_fs8(12))
            if ruler and text:
                _bar = 5.0 if _EXT8 <= 25 else (10.0 if _EXT8 <= 60 else 25.0)
                _f24b = _bar / (2 * _EXT8)                   # its length as a fraction of the panel
                ax.plot([0.06, 0.06 + _f24b], [0.065, 0.065], "-", transform=ax.transAxes, color=fg, lw=2.6,
                        zorder=12, path_effects=[pe.withStroke(linewidth=4.2, foreground=bg)])
                ax.text(0.06 + 0.5 * _f24b, 0.105, f"{_bar:.0f} kpc", transform=ax.transAxes, color=fg,
                        ha="center", va="bottom", fontsize=_fs8(8), zorder=12,
                        path_effects=[pe.withStroke(linewidth=2.2, foreground=bg)])

        def _num8of(r, plain=False):
            """The numbers of a row's label block: mathtext for the figure, plain text (plain) for the singles table."""
            _v8 = lambda k: (f"{r[k]:.2f}" if np.isfinite(r.get(k, np.nan)) else "n/a")
            if plain:
                return [f"log M* = {_v8('lmstar')}", f"age = {_v8('age')} Gyr", f"z_form = {_v8('zform')}",
                        f"log Mdust/M* = {_v8('lfd')}", f"kappa_rot(gas) = {_v8('kgas')}", f"kappa_rot* = {_v8('kstar')}"]
            return [rf"$\log M_\star = $ {_v8('lmstar')}", rf"age $= $ {_v8('age')} Gyr",
                    rf"$z_{{\rm form}} = $ {_v8('zform')}",
                    rf"$\log M_{{\rm dust}}/M_\star = $ {_v8('lfd')}",
                    rf"$\kappa_{{\rm rot}}$(gas) $= $ {_v8('kgas')}",
                    rf"$\kappa_{{\rm rot}}^\star = $ {_v8('kstar')}"]

        # ── THE SINGLE PANELS (P8_SINGLES): every panel of every figure as its own file + one table of them ──
        SNG8 = []                                                # one row per (figure, galaxy, stage): the singles table
        _SDIR8 = os.path.join(OUT, f"ba_gallery_singles_{P8_BOX}")
        _ROWIX8 = {(w["group"], w["anchor_snap"], w["gal_id"], w["stage"]): w for w in ROW8}

        def _singles8(draw, which, meta, cbar=None, cbar_label=None, fg="w", bg="k"):
            """Each panel of one figure alone, the figure's colour bar alone, and their rows in SNG8."""
            if not (P8_SINGLES and REP8):
                return
            os.makedirs(_SDIR8, exist_ok=True)
            _cbf8 = ""
            if cbar is not None:                                 # the colour bar on its own, as in the grid
                _cbf8 = f"cbar_{which}"
                fig = plt.figure(figsize=(float(P8_PANEL_IN) * 1.6, float(P8_CBAR_H) + 0.05))
                _cax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
                _cb8 = fig.colorbar(cbar, cax=_cax, orientation="horizontal")
                _cb8.ax.tick_params(labelsize=_fs8(9.5), length=3.5, pad=2.5)
                _cb8.outline.set_linewidth(0.8)
                _cax.set_title(cbar_label, fontsize=_fs8(11), pad=float(P8_CBAR_PAD))
                for _ext in P8_SINGLES_FMT:
                    fig.savefig(os.path.join(_SDIR8, f"{_cbf8}.{_ext}"), dpi=PAPER_DPI, bbox_inches="tight")
                plt.close(fig)
            _n8 = 0
            for _r in REP8:
                for _j, _s8 in enumerate(P8_STAGES):
                    _cell = REN8[_ck8(_r, _s8)]
                    _stem = f"{which}_{_r['group']}_a{int(_r['key'][0]):03d}_g{int(_r['key'][1]):06d}_{_j}{_s8}"
                    _w = _ROWIX8.get((_r["group"], _r["key"][0], _r["key"][1], _s8), {})
                    _row = dict(figure=which, file="", file_raw="", file_cbar=_cbf8 + f".{P8_SINGLES_FMT[0]}" if _cbf8 else "",
                                group=_r["group"], label=_r["label"], color=_r["color"],
                                anchor_snap=_r["key"][0], gal_id=_r["key"][1],
                                column=_j, stage=_s8, stage_label=_stlab8.get(_s8, _s8).replace("$", ""),
                                row_text="; ".join(_num8of(_r, plain=True)))
                    _row.update({f"gal_{k}": _r.get(k, np.nan) for k in
                                 ("z", "lmstar", "age", "ssfr", "ldgr", "lfd", "kgas", "kstar", "zform",
                                  "zform_age", "zform_half", "zform_first", "ngas", "nstar", "geom")})
                    _row.update({k: v for k, v in _w.items() if k not in ("group", "label", "stage", "anchor_snap", "gal_id")})
                    _row.update(side_kpc=2 * _EXT8, pixels=P8_XSIZE, spos=P8_SPOS, frame=P8_FRAME,
                                frame_stage=P8_FRAME_STAGE or "", members_only=P8_MEMBERS)
                    _row.update({f"scale_{k}": getattr(v, "name", v) for k, v in meta.items()})
                    if _cell is None:                            # nothing to draw: the row says why, no file
                        SNG8.append(_row); continue
                    _dg8, _ = _dg8of(_cell)
                    _row[f"panel_log_{'dh2' if P8_DGR == 'dh2' else 'dg'}"] = np.log10(_dg8) if np.isfinite(_dg8) else np.nan
                    fig = plt.figure(figsize=(float(P8_PANEL_IN), float(P8_PANEL_IN)))
                    ax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
                    ax.set_xticks([]); ax.set_yticks([])
                    for _sp in ax.spines.values():
                        _sp.set_color(_r["color"]); _sp.set_linewidth(2.0)
                    draw(ax, _cell, _r, _s8)
                    _decor8(ax, _cell, _s8, fg, bg, ruler=True, title=P8_SINGLES_TITLE, text=P8_SINGLES_DECOR)
                    for _ext in P8_SINGLES_FMT:
                        fig.savefig(os.path.join(_SDIR8, f"{_stem}.{_ext}"), dpi=PAPER_DPI, bbox_inches="tight")
                    plt.close(fig)
                    _row["file"] = f"{_stem}.{P8_SINGLES_FMT[0]}"
                    if P8_SINGLES_RAW:                           # the pixel array alone, on the grid's colour scale
                        if which == "rgb":
                            _px = np.clip(np.nan_to_num(np.asarray(_cell["rgb"], float), nan=0.0), 0.0, 1.0)
                            plt.imsave(os.path.join(_SDIR8, f"{_stem}_raw.png"), _px, origin="lower")
                        else:
                            _m = _cell[f"map_{which}"]
                            with np.errstate(divide="ignore", invalid="ignore"):
                                _d = np.log10(np.where(_m > 0, _m, np.nan)) if meta["log"] else np.asarray(_m, float)
                            _d = np.where(np.isfinite(_d), _d, meta["vmin"])      # empty = the colormap's low end
                            plt.imsave(os.path.join(_SDIR8, f"{_stem}_raw.png"), _d, origin="lower",
                                       cmap=meta["cmap"], vmin=meta["vmin"], vmax=meta["vmax"])
                        _row["file_raw"] = f"{_stem}_raw.png"
                    SNG8.append(_row); _n8 += 1
            print(f"  singles: {_n8} {which} panels -> {_SDIR8}/{which}_*"
                  + (f" (+ {_cbf8})" if _cbf8 else ""))

        def _grid8(draw, title, fname, cbar=None, cbar_label=None, fg="w", bg="k", which=None, meta=None):
            """The 4 x 4 frame every figure of this part shares: classes down, critical points across."""
            if not REP8:
                return
            _cells8 = [c for c in REN8.values() if c is not None]      # the drawn panels; None = an empty one
            if P8_STREAM and _cells8 and "vel_n" in _cells8[0]:
                _nu = [c["vel_n"] for c in _cells8]
                print(f"  streamlines: the {P8_STREAM_WEIGHT}-weighted velocity field of "
                      + {"slab": f"the gas within +-{P8_STREAM_DEPTH:g} kpc of the mid-plane along the line of sight (P8_STREAM_SOURCE 'slab')",
                         "members": "the galaxy's CAESAR member gas only", "region": "EVERY gas particle in the frame (CGM and neighbours included)"}[P8_STREAM_SOURCE]
                      + f", the {'galaxy' if P8_STREAM_BULK == 'members' else 'selection'}'s bulk velocity subtracted"
                      + f" — per panel {min(n for n, _ in _nu)}-{max(n for n, _ in _nu)} particles used of {min(m for _, m in _nu)}-{max(m for _, m in _nu)} in the frame")
            if P8_R50MARK:
                _r50all8 = [float(c.get("r50", np.nan)) for c in _cells8]
                _r50all8 = [v for v in _r50all8 if np.isfinite(v)]
                print(f"  the amber CIRCLE in EVERY panel has the radius {_R50LAB8.replace('$', '')}, the "
                      + ("projected " if P8_R50MARK_2D else "3D ")
                      + f"half-mass radius of the member {P8_R50MARK} ("
                      + ("stars + gas" if P8_R50MARK == "baryon" else P8_R50MARK)
                      + ") AT THAT EPOCH, measured from the particles in the drawn frame"
                      + (f" — {min(_r50all8):.1f}-{max(_r50all8):.1f} kpc over the grid" if _r50all8 else "")
                      + (" — drawn in the bottom-right corner the way a beam is, so it is a size to compare against"
                         if P8_R50MARK_POS != "centre" else " — drawn concentric on the galaxy, where the aperture is")
                      + ".  The white bar in the first column is the fixed kpc ruler.")
            if P8_ISMEDGE and not isinstance(P8_ISMEDGE, (int, float)):
                print(f"  the dashed {P8_ISMEDGE_COL} contour is the ISM/CGM BOUNDARY: the level of the CAESAR MEMBER gas in "
                      f"projection that encloses {100 * P8_ISMEDGE_ENC:.0f} % of the member gas mass in the frame.  The ISM is a "
                      f"member LIST and not a radius (Part 6), so its edge is a shape; everything outside the line is CGM.")
            elif P8_ISMEDGE:
                print(f"  the dashed {P8_ISMEDGE_COL} circle is drawn at r = {float(P8_ISMEDGE):g} kpc by hand (P8_ISMEDGE).")
            if _frames8:                                             # the frame every panel was drawn in
                _nfb = sum(1 for _, _, fi in _frames8 if fi.get("frame") == "stellar")
                print(f"  projection frame P8_FRAME = {P8_FRAME}" + (f" measured at {P8_FRAME_STAGE} for the whole row" if P8_FRAME_STAGE else "")
                      + f": {len(_frames8) - _nfb} of {len(_frames8)} panels in that frame"
                      + (f", {_nfb} fell back to the stellar frame" if _nfb else "") + " (tilt = angle between that spin and the stellar minor axis):")
                for _lb8, _st8, fi in _frames8:
                    print(f"    {str(_lb8)[:34]:34s} {_st8:7s} {fi.get('frame', '?'):10s} n={fi.get('n_used', 0):5d}"
                          + (f"  coherence {fi['coherence']:.2f}" if np.isfinite(fi.get("coherence", np.nan)) else "")
                          + (f"  (this epoch's own spin: coherence {fi['own_coherence']:.2f}, n={fi['own_n']})" if "own_coherence" in fi else "")
                          + (f"  tilt {fi['tilt_vs_stellar_deg']:5.1f} deg" if "tilt_vs_stellar_deg" in fi else f"  ({fi.get('reason', '')})"))
            # the colour bar gets its OWN band at the foot of the figure, measured in inches (ticks below the bar,
            # label above it, a gap up to the panels) and the figure GROWS by it, so the panels never pay for it
            _cbtick8 = (_fs8(9.5) + 6.0) / 72.0
            _cblab8 = (_fs8(11.0) + float(P8_CBAR_PAD) + 5.0) / 72.0
            _cbin8 = (float(P8_CBAR_GAP) + float(P8_CBAR_H) + _cbtick8 + _cblab8) if cbar is not None else 0.0
            fig, axs = plt.subplots(len(REP8), len(P8_STAGES),
                                    figsize=(float(P8_PANEL_IN) * len(P8_STAGES) / (1 - P8_MARGIN),
                                             1.12 * float(P8_PANEL_IN) * len(REP8) + 0.9 + _cbin8),
                                    squeeze=False)
            _fh8 = float(fig.get_figheight())
            for _i, _r in enumerate(REP8):
                for _j, _s8 in enumerate(P8_STAGES):
                    ax = axs[_i][_j]
                    _cell = REN8[_ck8(_r, _s8)]
                    ax.set_xticks([]); ax.set_yticks([])
                    for _sp in ax.spines.values():
                        _sp.set_color(_r["color"]); _sp.set_linewidth(2.0)
                    if _cell is None:                                  # the empty panel: what is missing, in words
                        _wt = _EP8.get((_r["key"], _s8))
                        ax.set_facecolor("0.12"); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
                        ax.text(0.5, 0.5, (f"cutout not built yet\nsnap {_wt[0]}, gal {_wt[1]}\n(run the plan)" if _wt
                                           else "no $z = 0$ descendant\n(merged into a\nlarger galaxy)"),
                                transform=ax.transAxes, ha="center", va="center", fontsize=_fs8(9), color="0.75",
                                linespacing=1.5)
                        if _i == 0:
                            ax.set_title(_stlab8.get(_s8, _s8), fontsize=_fs8(12))
                        continue
                    draw(ax, _cell, _r, _s8)
                    _decor8(ax, _cell, _s8, fg, bg, ruler=(_j == 0), title=(_i == 0))
            #fig.suptitle(title, fontsize=_fs8(13) if "\n" not in title else 11.5)
            _rt8 = P8_SIDE == "right"
            fig.tight_layout(rect=(0 if _rt8 else P8_MARGIN, _cbin8 / _fh8,
                                   1 - P8_MARGIN if _rt8 else 1, 0.985), pad=0.6)
            fig.subplots_adjust(wspace=P8_WSPACE, hspace=P8_WSPACE)     # images: the grid wants almost no gutter
            # THE DESCRIPTION SITS BESIDE ITS OWN ROW: the name and the numbers are ONE block, and the block's
            # centre is put on the row's centre (a line of text is fontsize x linespacing / 72 inches tall, and
            # the figure height turns that into the fraction fig.text works in), so every row reads across.
            _tl8 = lambda n, fs, ls: n * float(fs) * float(ls) / (72.0 * _fh8)
            for _i, _r in enumerate(REP8):                      # written after the layout, so nothing is clipped
                _bb = (axs[_i][-1] if _rt8 else axs[_i][0]).get_position()
                _x, _ha = (_bb.x1 + 0.012, "left") if _rt8 else (_bb.x0 - 0.012, "right")
                _y = _bb.y0 + 0.5 * _bb.height
                _num8 = _num8of(_r)
                _lab8x = _r["label"].replace(" $+$ ", "\n")
                _hlab8 = _tl8(_lab8x.count("\n") + 1, _fs8(12), 1.15)     # the name
                _hnum8 = _tl8(len(_num8), _fs8(9), float(P8_LABEL_LS))    # the numbers under it
                _hgap8 = _tl8(0.35, _fs8(12), 1.0)                        # and the air between the two
                _tby8 = _y + 0.5 * (_hlab8 + _hgap8 + _hnum8)             # the top of the block, so its MIDDLE is at _y
                fig.text(_x, _tby8, _lab8x, ha=_ha, va="top", fontsize=_fs8(12),
                         color=_r["color"], linespacing=1.15, fontweight="bold")
                fig.text(_x, _tby8 - _hlab8 - _hgap8, "\n".join(_num8), ha=_ha,
                         va="top", fontsize=_fs8(9), color="0.2", linespacing=float(P8_LABEL_LS))
            if cbar is not None:
                _bot8 = min(axs[-1][_j].get_position().y0 for _j in range(len(P8_STAGES)))   # ride under the panels,
                _cby8 = max(_cbtick8 / _fh8,                                    # not at the very bottom of the page
                            _bot8 - (float(P8_CBAR_GAP) + _cblab8 + float(P8_CBAR_H)) / _fh8)
                _cax = fig.add_axes([0.5 - 0.5 * float(P8_CBAR_LEN), _cby8,
                                     float(P8_CBAR_LEN), float(P8_CBAR_H) / _fh8])
                _cb8 = fig.colorbar(cbar, cax=_cax, orientation="horizontal")
                _cb8.ax.tick_params(labelsize=_fs8(9.5), length=3.5, pad=2.5)
                _cb8.outline.set_linewidth(0.8)
                _cax.set_title(cbar_label, fontsize=_fs8(11), pad=float(P8_CBAR_PAD))
            paper_save(fig, fname)
            plt.show()
            if which is not None:
                _singles8(draw, which, meta or {}, cbar=cbar, cbar_label=cbar_label, fg=fg, bg=bg)

        # ── FIGURE 1: the composite ──
        def _drawrgb8(ax, cell, r, s):
            ax.imshow(cell["rgb"], extent=cell["ext"], origin="lower")
            _ismedge8(ax, cell, "rgb")
            if r is REP8[0] and s == P8_STAGES[0]:                 # name the channels once, in their own colours
                for _k8, _nm8 in enumerate(P8_CHAN):
                    ax.text(0.965, 0.955 - 0.08 * _k8, _nm8, transform=ax.transAxes, ha="right", va="top",
                            fontsize=_fs8(9.5), weight="bold",
                            color=ReducedRender.CHANNEL_CMAPS.get(_nm8, cm.Greys_r)(0.88),
                            path_effects=[pe.withStroke(linewidth=2.2, foreground="k")])

        _grid8(_drawrgb8,
                rf"Part 8 — {_npg8} through its own quenching ({BOX_LABEL.get(P8_BOX, P8_BOX)}, "
                rf"{P8_SPOS}, {' $+$ '.join(P8_CHAN)}, {P8_STRETCH} stretch)"
                + ("\n" + _fignote8 if _fignote8 else ""),
                f"ba_gallery_rgb_{P8_BOX}", which="rgb",
                meta=dict(channels=" + ".join(P8_CHAN), stretch=P8_STRETCH, dyn_dex=str(tuple(np.atleast_1d(P8_DYN))),
                          pmax=P8_PMAX, tau_max=P8_TAU,
                          **{f"{c}_scale": _scale8[c] for c in P8_CHAN},
                          **{f"{c}_white_Msun_kpc2": _top8.get(c, np.nan) for c in P8_CHAN}))

        # ── FIGURE 2+: the physical maps, one per field, shared colour scale, velocity field on top ──
        for _f8 in (P8_MAPS if REP8 else []):
            _kind8, _lab8, _unit8, _log8, _cm8, _sym8 = P8_MAPDEF[_f8]
            _cm8 = P8_CMAP.get(_f8, _cm8)
            _all8 = np.concatenate([REN8[_ck8(r, s)][f"map_{_f8}"].ravel()
                                     for r in REP8 for s in P8_STAGES if REN8[_ck8(r, s)] is not None])
            _all8 = _all8[np.isfinite(_all8)]
            if _log8:
                _all8 = _all8[_all8 > 0]
            if not len(_all8):
                print(f"  the {_f8} maps are empty everywhere — figure skipped")
                continue
            if _sym8:                                     # a diverging quantity: one symmetric scale about zero
                _q8 = float(np.percentile(np.abs(_all8), P8_PCT[1])); _vmn, _vmx = -_q8, _q8
            else:
                _vmn, _vmx = (float(np.percentile(_all8, P8_PCT[0])), float(np.percentile(_all8, P8_PCT[1])))
                if _log8:
                    _vmn, _vmx = np.log10(_vmn), np.log10(_vmx)
                    if P8_MAPDYN is not None:
                        _vmn = max(_vmn, _vmx - float(P8_MAPDYN))
            _pre8 = r"\log\," if _log8 else ""
            _cbl8 = "$" + _pre8 + _lab8 + "$" + ("  [$" + _unit8 + "$]" if _unit8 else "")
            _sm8 = cm.ScalarMappable(norm=mcolors.Normalize(_vmn, _vmx), cmap=_cm8)

            def _draw8(ax, cell, r, s, _f=_f8, _lg=_log8, _c=_cm8, _lo=_vmn, _hi=_vmx):
                ax.set_facecolor(plt.get_cmap(_c)(0.0))    # empty pixels take the colormap's own low end, so the
                _m = cell[f"map_{_f}"]                     # white streamlines and labels read everywhere
                with np.errstate(divide="ignore", invalid="ignore"):
                    _d = np.log10(np.where(_m > 0, _m, np.nan)) if _lg else np.where(np.isfinite(_m), _m, np.nan)
                ax.imshow(_d, extent=cell["ext"], origin="lower", vmin=_lo, vmax=_hi, cmap=_c)
                if P8_STREAM and (_f in P8_STREAM_ON) and cell.get("vel") is not None:
                    _X, _Y, _VX, _VY, _den = cell["vel"]
                    if np.any(np.hypot(_VX, _VY) > 0):
                        ax.streamplot(_X, _Y, _VX, _VY, color="w", density=P8_STREAM_DENSITY, linewidth=P8_STREAM_LW, arrowsize=P8_STREAM_ARROW,
                                      zorder=4)
                _ismedge8(ax, cell, _f)
                ax.set_xlim(cell["ext"][0], cell["ext"][1]); ax.set_ylim(cell["ext"][2], cell["ext"][3])

            _grid8(_draw8,
                    "Part 8 — $" + _lab8 + "$ through the critical points, " + _npg8 + " "
                    + f"({BOX_LABEL.get(P8_BOX, P8_BOX)}, {P8_SPOS}"
                    + ((f", the gas velocity field within $\\pm${P8_STREAM_DEPTH:g} kpc of the mid-plane as streamlines)" if P8_STREAM_SOURCE == "slab" else
                        ", the member gas velocity field as streamlines)" if P8_STREAM_SOURCE == "members" else ", the region's gas velocity field as streamlines)")
                       if (P8_STREAM and _f8 in P8_STREAM_ON) else ")")
                    + ("\n" + _fignote8 if _fignote8 else ""),
                    f"ba_gallery_{_f8}_{P8_BOX}", cbar=_sm8, cbar_label=_cbl8,
                    fg="w", bg="k", which=_f8,
                    meta=dict(quantity=_lab8, unit=_unit8, log=bool(_log8), vmin=_vmn, vmax=_vmx,
                              cmap=_cm8, label=_cbl8))

        if ROW8:
            _p8 = os.path.join(OUT, f"ba_gallery_{P8_BOX}.csv")
            pd.DataFrame(ROW8).to_csv(_p8, index=False)
            print(f"  what was drawn, panel by panel (ids, epochs, masses inside the frame) -> {_p8}")
        if SNG8:
            _ps8 = os.path.join(_SDIR8, f"ba_gallery_singles_{P8_BOX}.csv")
            pd.DataFrame(SNG8).to_csv(_ps8, index=False)
            print(f"  singles table ({len(SNG8)} rows, one per figure x galaxy x stage) -> {_ps8}")

## Part 8s — the SEDs of the gallery: with and without dust, at every epoch drawn

The powderday SEDs of the very `(snap, galaxy)` pairs Part 8 renders, in the **two arms of the m25 powderday campaign**:
`dust_on` (`dusty_simdust`, SIMBA's own dust) and `dust_off` (`nodust_1e-12`, the dust mass x 1e-12 = the intrinsic stellar
SED), from the same parameter masters, so the gallery's SEDs and the campaign's 532 are one set. Each epoch is looked for
first in the campaign (`output/cis25/sed_quenched_regions`: only the 10 anchors z = 0.3-2 were run there, so only gallery
**anchors** that were in that sample can be found), then in this part's own tree (`output/cis25/sed_ba_gallery`).

**What is written** (`OUT/ba_gallery_seds_<box>/`), one file per epoch, named like the singles
(`sed_<group>_a<anchor>_g<gal>_<column><stage>.{fits,csv}`): the rest-frame wavelength (ascending, + `wav_um_obs`), and
for every arm x aperture (the campaign's 1 / 3.2 / 10 / 32 / 100 kpc ladder, read from the file) x sightline (`i0p0`
... `i135p270`) the $\nu L_\nu$ [erg/s] with its Monte-Carlo error, + the sightline median. A missing arm is NaN, so
every file has one schema; the FITS header and the CSV's `#` lines carry the epoch and where each arm came from.
`ba_gallery_seds_<box>.csv` is the index: one row per epoch, the source of each arm (`campaign` / `gallery` /
`missing`), the files, and at `P8S_AP` (32 kpc ~ the 35 kpc frame) $L_{\rm bol}$ and $L_{\rm IR}$ of both arms,
$L_{\rm dust} = \int_{\lambda > 3\,\mu m} (L_{\rm on} - L_{\rm off})$, and $A_V$ / $A_{\rm FUV} = -2.5 \log(L_{\rm on} / L_{\rm off})$
at 0.55 / 0.153 µm (the sightline median and the 16-84 of the four). The figure is the gallery's grid of SEDs
(`ba_gallery_seds_<box>`, dust on in the group colour with the sightline band, dust off dashed), and every panel alone
(`sed_<...>.png`) for slides. Nothing is written inside the panels (z, $A_V$, the missing epochs are all in the index CSV); `P8S_PANEL_W` / `P8S_PANEL_H` set the panel shape, `P8S_ROWLAB` the row names beside the grid, `P8S_SINGLE_LEGEND` a legend in the single panels.

**The missing epochs** (every non-anchor stage, and an anchor that was not in the campaign's sample): set
`P8S_STAGE = True` **once, on the cluster**. It cuts their Stage-0 particle files (the campaign's 100 kpc sphere and
particle types, into the campaign's own `filtered_particles`, existing files skipped) and writes the SLURM masters of both
arms under `output/cis25/sed_ba_gallery/` (job names `bag_*`). Then submit
`output/cis25/sed_ba_gallery/{dusty_simdust,nodust_1e-12}/powderday_sed_out/submit_all_snaps.sh` and re-run this cell
(with `P8S_STAGE` back to `False`). Resubmitting only fills the gaps. Reading the SEDs needs `hyperion` (the powderday
kernel); the cell says so and stops otherwise. It reads `ROW8` when Part 8 ran in the session, else `ba_gallery_<box>.csv`.


In [ ]:
# ── Part 8s — THE SEDs OF THE GALLERY: dust on and dust off, at every epoch Part 8 draws ──
# The powderday SEDs of the very (snap, galaxy) pairs the gallery renders: the dust_on arm (SIMBA's own dust) and the
# dust_off arm (dust mass x 1e-12 = the intrinsic stars), both from the m25 powderday campaign's parameter masters, so
# the gallery's SEDs and the campaign's 532 are ONE set.  Each epoch is looked for, in order, in
#   P8S_TREES[0]  the campaign (output/<box>/sed_quenched_regions — the 10 anchors z = 0.3-2, 532 galaxies)
#   P8S_TREES[1]  this part's own tree (output/<box>/sed_ba_gallery — whatever P8S_STAGE staged and the cluster ran)
# and what is found is written apart, one file per epoch, so a slide can be built from it by hand:
#   OUT/ba_gallery_seds_<box>/sed_<group>_a<anchor>_g<gal>_<column><stage>.{fits,csv}   (the names of the singles)
#       wav_um (REST frame, ascending), nu_Hz, wav_um_obs (= wav_um x (1 + z)), then per arm (dust_on / dust_off),
#       aperture (ap1kpc ... ap100kpc, read from the file) and sightline (i0p0 ... i135p270, the campaign's four):
#       <arm>_<ap>_<incl> = nu L_nu [erg/s] and <arm>_<ap>_<incl>_err (the Monte-Carlo error), + <arm>_<ap>_med (the
#       median over the sightlines).  An arm that has not run is NaN, never dropped, so every file has one schema.
#   OUT/ba_gallery_seds_<box>/ba_gallery_seds_<box>.csv   one row per epoch: where each arm was found (or 'missing'),
#       the files, and at P8S_AP: L_bol and L_IR(8-1000 um) of both arms, A_V and A_FUV = -2.5 log(on / off) at 0.55 /
#       0.153 um, and L_dust = the integral of (on - off) beyond 3 um (the dust emission alone: the dust_off arm still has
#       the stellar Rayleigh-Jeans tail past 8 um, so L_IR(on) alone overstates it) — the sightline median and the 16-84
#   OUT/ba_gallery_seds_<box>.{png,pdf}   (no text inside the panels: z, A_V, ... are in the index CSV) the gallery's grid (galaxies down, critical points across) of the SEDs, and
#       every panel alone in the same folder (sed_<...>.png) for the slides
# P8S_STAGE = True (CLUSTER, once): the epochs with an arm missing get (1) a Stage-0 particle cutout (extract_particles,
# the campaign's radius / particle types, skipped when the file exists — the folder is the campaign's own) and (2) the
# SLURM masters of both arms under P8S_TREES[1] (MakeSED, the campaign's paramf per arm).  Then, on the cluster:
#     bash output/<box>/sed_ba_gallery/dusty_simdust/powderday_sed_out/submit_all_snaps.sh
#     bash output/<box>/sed_ba_gallery/nodust_1e-12/powderday_sed_out/submit_all_snaps.sh
# and re-run this cell: the new SEDs are picked up from P8S_TREES[1].  A job exits early on a galaxy whose .rtout.sed
# exists, so resubmitting only fills the gaps.
P8S_BOX    = globals().get("P8_BOX", "cis25")
P8S_TREES  = [os.path.join(os.getcwd(), "output", P8S_BOX, "sed_quenched_regions"),   # the campaign (read only)
              os.path.join(os.getcwd(), "output", P8S_BOX, "sed_ba_gallery")]         # this part's own runs
P8S_ARMS   = {"dust_on":  dict(run_tag="dusty_simdust", paramf="parameters_master.py"),
              "dust_off": dict(run_tag="nodust_1e-12",  paramf="parameters_master-nodust.py")}
P8S_AP     = "ap32kpc"        # the aperture drawn and summarised (ap1kpc / ap3kpc / ap10kpc / ap32kpc / ap100kpc): 32 kpc ~
                               # the gallery's 35 kpc frame.  Every aperture is in the per-epoch files whatever this says.
P8S_INCL   = "median"         # the sightline drawn: "median" (+ the four as a band) or one of i0p0 / i45p90 / i90p180 / i135p270
P8S_FMT    = ("fits", "csv")  # the per-epoch files
P8S_WLIM   = (0.09, 900.0)    # [um, rest frame] the x range drawn (ends short of 1e3: its tick label would run into the
                               # next panel's 1e-1)
P8S_YDEX   = 5.0              # decades drawn below the brightest curve of the figure
P8S_PANEL_W = 5.6             # [inch] one panel's width: wide, so the 0.1-1000 um range (four decades) is not squeezed
P8S_PANEL_H = 3.0             # [inch] its height
P8S_ROWLAB = True             # the galaxy's name + (anchor, id) beside each row, OUTSIDE the panels (False = none)
P8S_SINGLE_LEGEND = False     # a legend inside each single panel (the grid's legend sits under the figure, outside)
P8S_SINGLES = True            # every panel also alone (sed_<...>.png), as the gallery's P8_SINGLES
P8S_STAGE  = False            # ◄── CLUSTER, ONCE: cut the particle files + write the SLURM masters of the missing epochs
P8S_RCUT   = 100.0            # [pkpc] the Stage-0 sphere (== the campaign's R_CUTOUT_KPC: the 100 kpc aperture is the box)
P8S_PTYPES = ("PartType0", "PartType4", "PartType5")
P8S_PARTITION = "INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL"
P8S_SELFILE = "selection_ba_gallery"   # MakeSED's selection HDF5 (under P8S_TREES[1]/<run_tag>/target_selection/)
_LSUN8S = 3.828e33

# ── the epochs: what Part 8 drew (in memory), else what it wrote ──
_rows8s = [dict(r) for r in globals().get("ROW8", [])]
if not _rows8s:
    _csv8s = os.path.join(OUT, f"ba_gallery_{P8S_BOX}.csv")
    _rows8s = pd.read_csv(_csv8s).to_dict("records") if os.path.exists(_csv8s) else []
_stg8s = list(globals().get("P8_STAGES", []))
for _w in _rows8s:                                   # the stage order the gallery uses (any stage it lacks goes last)
    if _w["stage"] not in _stg8s:
        _stg8s.append(_w["stage"])
_EPS8S = [w for w in _rows8s if int(w.get("gx", -1)) >= 0 and int(w.get("snap", -1)) >= 0]
_gal8s = list(dict.fromkeys((w["group"], int(w["anchor_snap"]), int(w["gal_id"])) for w in _rows8s))
_lab8s = {(w["group"], int(w["anchor_snap"]), int(w["gal_id"])): w.get("label", w["group"]) for w in _rows8s}
_col8s = {k: v[0] for k, v in globals().get("_GINFO8", {}).items()}
_stlab8s = globals().get("_stlab8", {})
_SED8S = os.path.join(OUT, f"ba_gallery_seds_{P8S_BOX}")


def _sedpath8s(tree, arm, snap, gx):
    return os.path.join(tree, P8S_ARMS[arm]["run_tag"], "powderday_sed_out", f"snap_{snap:03d}", f"gal_{gx}",
                        f"snap{snap:03d}.galaxy{gx:06d}.rtout.sed")


def _find8s(arm, snap, gx):
    """The first tree that holds this arm's FINISHED SED of (snap, gx): (path, 'campaign' | 'gallery'), else
    (None, 'running') when the file is write-locked (hyperion holds the .rtout.sed open for the whole run),
    (None, 'incomplete') when it opens but has no peeled SED yet (a crashed / killed job), or (None, 'missing')."""
    _why = "missing"
    for _t, _nm in zip(P8S_TREES, ("campaign", "gallery")):
        _p = _sedpath8s(_t, arm, snap, gx)
        if not os.path.exists(_p):
            continue
        try:
            with h5py.File(_p, "r") as _h:
                if "Input/Output/Peeled/group_00001" in _h or "Peeled/group_00001" in _h:
                    return _p, _nm
            _why = "incomplete"
        except BlockingIOError:
            _why = "running"
        except OSError:
            _why = "incomplete"
    return None, _why


if not _EPS8S:
    print("Part 8s: no gallery epochs (run Part 8 first, or its CSV is missing) — skipped.")
else:
    _HAVE8S = True
    try:
        from hyperion.model import ModelOutput
    except Exception as _e:
        print(f"Part 8s: hyperion is not importable here ({type(_e).__name__}) — the SEDs cannot be read; run on the "
              f"cluster's powderday kernel."); _HAVE8S = False
    _where8s = {(int(w["snap"]), int(w["gx"])): {a: _find8s(a, int(w["snap"]), int(w["gx"])) for a in P8S_ARMS}
                for w in _EPS8S}
    _miss8s = sorted(k for k, v in _where8s.items() if any(p is None for p, _ in v.values()))
    print(f"Part 8s: {len(_where8s)} distinct (snap, gal) epochs in the gallery; both arms found for "
          f"{len(_where8s) - len(_miss8s)} ("
          + ", ".join(f"{sum(1 for v in _where8s.values() if v[a][1] == n)} {a} in the {n}"
                      for a in P8S_ARMS for n in ("campaign", "gallery"))
          + f"); {len(_miss8s)} still to run" + (f": {_miss8s}" if _miss8s else "") + ".")
    _busy8s = {k: [a for a, (p, n) in v.items() if n in ("running", "incomplete")] for k, v in _where8s.items()}
    _busy8s = {k: v for k, v in _busy8s.items() if v}
    if _busy8s:
        print("  of these, a file exists but is not readable yet (write-locked = job still running, or no peeled SED = "
              "a killed job: delete it and resubmit): "
              + ", ".join(f"{k}:{'/'.join(f'{a}={_where8s[k][a][1]}' for a in v)}" for k, v in _busy8s.items()))

    # ── STAGING (cluster): the Stage-0 cutouts + the SLURM masters of the missing epochs ──
    if P8S_STAGE and _miss8s:
        from simbanator.io.simba import Simulation
        from simbanator.analysis import extract_particles
        from simbanator.sed.makesed import MakeSED
        _sim8s = Simulation(P8S_BOX)
        _pref8s = _sim8s.file_format.split("_{")[0]
        _hyd8s = os.path.join(os.getcwd(), "output", _sim8s.name, "filtered_particles")
        _sn8s = np.array([s for s, _ in _miss8s], int); _gx8s = np.array([g for _, g in _miss8s], int)
        for _s in np.unique(_sn8s):
            _ids = np.unique(_gx8s[_sn8s == _s])
            print(f"  snap {_s:3d}: Stage-0 cutouts of {len(_ids)} galaxies {_ids.tolist()}")
            _cs = _sim8s.load_catalog(snap=int(_s))
            extract_particles(_cs, _sim8s.get_snapshot_file(int(_s)), int(_s), galaxy_ids=_ids, radius=P8S_RCUT,
                              ptypes=P8S_PTYPES, sim_name=_sim8s.name, prefix=_pref8s, overwrite=False, verbose=1)
            del _cs
        for _a, _cfg in P8S_ARMS.items():
            _ms = MakeSED(_sim8s, nnodes=1, model_run_name=f"bag_{_cfg['run_tag']}", hydro_dir_base=_hyd8s,
                          selection_file=P8S_SELFILE, output_dir=P8S_TREES[1], run_tag=_cfg["run_tag"])
            _ms.selection_gals(snaps=_sn8s, galaxyID=_gx8s)
            _ms.create_master("cluster", "region", radius=P8S_RCUT, partition=P8S_PARTITION, prefix=_pref8s,
                              paramf=_cfg["paramf"], snaps_to_run=None)
            print(f"  {_a}: masters written -> bash {os.path.join(_ms.model_dir_base, 'submit_all_snaps.sh')}")
        print("  submit both scripts on the cluster, then re-run this cell.")
    elif _miss8s:
        print("  P8S_STAGE = True (on the cluster) cuts their particle files and writes the SLURM masters.")

    # ── READ + WRITE: one file per epoch, every arm x aperture x sightline ──
    _SUM8S, _CUR8S = [], {}
    if _HAVE8S:
        os.makedirs(_SED8S, exist_ok=True)
        _apl8s = _incl8s = None
        for _w in _EPS8S:
            _s, _g = int(_w["snap"]), int(_w["gx"])
            _key = (_w["group"], int(_w["anchor_snap"]), int(_w["gal_id"]))
            _j = _stg8s.index(_w["stage"])
            _stem = f"sed_{_w['group']}_a{_key[1]:03d}_g{_key[2]:06d}_{_j}{_w['stage']}"
            _z = float(_w.get("z", np.nan))
            _dat, _src = {}, {}
            for _a in P8S_ARMS:
                _p, _src[_a] = _where8s[(_s, _g)][_a]
                if _p is None:
                    continue
                with h5py.File(_p, "r") as _h:              # the aperture ladder and the sightlines, from the file itself
                    _ga = _h["Input/Output/Peeled/group_00001"]
                    _nap = int(_ga.attrs["n_ap"])
                    _rap = np.geomspace(float(_ga.attrs["ap_min"]), float(_ga.attrs["ap_max"]), _nap) / 3.0857e21
                    _ang = _ga["angles"][:]
                _apl = [f"ap{(round(r) if r >= 2 else round(r, 1)):g}kpc" for r in _rap]
                _inc = [f"i{float(t):g}p{float(ph):g}" for t, ph in zip(_ang["theta"], _ang["phi"])]
                _apl8s, _incl8s = _apl8s or _apl, _incl8s or _inc
                _m = ModelOutput(_p)
                for _k, _al in enumerate(_apl):
                    _sd = tuple(_m.get_sed(inclination="all", aperture=_k, uncertainties=True))
                    _wv, _vv, _uu = (_sd + (None,))[:3]
                    _o = np.argsort(_wv)                        # hyperion gives wavelength DESCENDING
                    _dat[(_a, _al)] = (np.asarray(_wv)[_o], np.asarray(_vv)[:, _o],
                                       (np.asarray(_uu)[:, _o] if _uu is not None else np.full_like(np.asarray(_vv)[:, _o], np.nan)))
            _row = dict(group=_w["group"], label=_w.get("label", ""), stage=_w["stage"], column=_j,
                        anchor_snap=_key[1], gal_id=_key[2], snap=_s, gx=_g, z=_z,
                        **{f"src_{a}": _src[a] for a in P8S_ARMS})
            if _dat:
                _wav = next(iter(_dat.values()))[0]
                _cols = {"wav_um": _wav, "nu_Hz": 2.99792458e14 / _wav, "wav_um_obs": _wav * (1 + (_z if np.isfinite(_z) else 0.0))}
                for _a in P8S_ARMS:
                    for _al in _apl8s:
                        _v, _u = (_dat[(_a, _al)][1:] if (_a, _al) in _dat else
                                  (np.full((len(_incl8s), len(_wav)), np.nan),) * 2)
                        for _i, _il in enumerate(_incl8s):
                            _cols[f"{_a}_{_al}_{_il}"] = _v[_i]; _cols[f"{_a}_{_al}_{_il}_err"] = _u[_i]
                        _cols[f"{_a}_{_al}_med"] = np.median(_v, axis=0)
                _meta = dict(group=_w["group"], stage=_w["stage"], anchor=_key[1], gal_id=_key[2], snap=_s, gx=_g,
                             z=_z, box=P8S_BOX, units="nu L_nu [erg/s], rest-frame wavelength [um]",
                             arms=",".join(f"{a}={P8S_ARMS[a]['run_tag']}:{_src[a]}" for a in P8S_ARMS),
                             apertures=",".join(_apl8s), sightlines=",".join(_incl8s))
                _files = []
                if "fits" in P8S_FMT:
                    from astropy.table import Table as _T8s
                    _t = _T8s(_cols); _t.meta.update({k.upper(): str(v) for k, v in _meta.items()})
                    _t.write(os.path.join(_SED8S, f"{_stem}.fits"), overwrite=True); _files.append(f"{_stem}.fits")
                if "csv" in P8S_FMT:
                    with open(os.path.join(_SED8S, f"{_stem}.csv"), "w") as _fh:
                        _fh.write("".join(f"# {k} = {v}\n" for k, v in _meta.items()))
                        pd.DataFrame(_cols).to_csv(_fh, index=False)
                    _files.append(f"{_stem}.csv")
                _row["files"] = " ".join(_files)
                _CUR8S[(_key, _w["stage"])] = (_wav, _dat, _z)
                # the numbers of the summary, at P8S_AP, per sightline -> median and 16-84
                _lnw = np.log(_wav)

                def _band8s(lo, hi, v):
                    _m8 = (_wav >= lo) & (_wav <= hi)
                    _x8, _y8 = _lnw[_m8], v[:, _m8]
                    return np.sum(0.5 * (_y8[:, 1:] + _y8[:, :-1]) * np.diff(_x8), axis=1)

                def _at8s(w0, v):
                    return np.array([np.interp(np.log(w0), _lnw, vi) for vi in v])

                _q8s = lambda x: (np.nanmedian(x), np.nanpercentile(x, 16), np.nanpercentile(x, 84))
                for _a in P8S_ARMS:
                    if (_a, P8S_AP) not in _dat:
                        continue
                    _v = _dat[(_a, P8S_AP)][1]
                    for _nm, _x in (("Lbol", _band8s(0, np.inf, _v)), ("LIR", _band8s(8.0, 1000.0, _v))):
                        _row[f"log_{_nm}_{_a}_Lsun"], _row[f"log_{_nm}_{_a}_p16"], _row[f"log_{_nm}_{_a}_p84"] = \
                            _q8s(np.log10(np.clip(_x, 1e-30, None) / _LSUN8S))
                if all((a, P8S_AP) in _dat for a in ("dust_on", "dust_off")):
                    _von, _voff = _dat[("dust_on", P8S_AP)][1], _dat[("dust_off", P8S_AP)][1]
                    _row["log_Ldust_Lsun"], _row["log_Ldust_p16"], _row["log_Ldust_p84"] = _q8s(np.log10(np.clip(
                        _band8s(3.0, np.inf, _von - _voff), 1e-30, None) / _LSUN8S))   # the dust emission (7d0's definition)
                    for _nm, _w0 in (("AV", 0.55), ("AFUV", 0.153)):
                        with np.errstate(divide="ignore", invalid="ignore"):
                            _x = -2.5 * np.log10(_at8s(_w0, _von) / _at8s(_w0, _voff))
                        _row[_nm], _row[f"{_nm}_p16"], _row[f"{_nm}_p84"] = _q8s(_x)
            for _k in ("member_m_dust", "member_m_H2", "member_m_gas", "member_m_star", "r50_kpc"):
                if _k in _w:
                    _row[_k] = _w[_k]
            _SUM8S.append(_row)
        if _SUM8S:
            _ps8s = os.path.join(_SED8S, f"ba_gallery_seds_{P8S_BOX}.csv")
            pd.DataFrame(_SUM8S).to_csv(_ps8s, index=False)
            print(f"  {len(_CUR8S)} of {len(_SUM8S)} epochs have an SED -> one file each in {_SED8S}/sed_*."
                  f"{{{','.join(P8S_FMT)}}}; the index -> {_ps8s}")
            print(f"  at {P8S_AP} (sightline median [16-84 of the four]):")
            for _r in _SUM8S:
                if "AV" in _r:
                    print(f"    {_r['group']:5s} a{_r['anchor_snap']:03d} g{_r['gal_id']:<4d} {_r['stage']:8s} z {_r['z']:.2f}:  "
                          f"A_V {_r['AV']:.2f} [{_r['AV_p16']:.2f}, {_r['AV_p84']:.2f}]  A_FUV {_r['AFUV']:.2f}  "
                          f"log L_dust {_r.get('log_Ldust_Lsun', np.nan):.2f} Lsun")

    # ── THE FIGURE: the gallery's grid, galaxies down and critical points across ──
    if _CUR8S:
        _ymax8s = max(np.nanmax(_d[1]) for _, _dd, _ in _CUR8S.values() for (_a, _al), _d in _dd.items()
                      if _al == P8S_AP and np.isfinite(np.nanmax(_d[1]))) / _LSUN8S

        def _sedpanel8s(ax, key, stage, solo=False):
            """One SED panel: dust_on in the group colour (the sightline band behind it), dust_off dashed."""
            _c = _col8s.get(key[0], "C0")
            ax.set_xscale("log"); ax.set_yscale("log")
            ax.set_xlim(*P8S_WLIM); ax.set_ylim(_ymax8s / 10 ** P8S_YDEX, 2.0 * _ymax8s)
            _cur = _CUR8S.get((key, stage))
            if _cur is None:
                return                                  # nothing drawn: the index CSV says which epochs are missing
            _wav, _dd, _z = _cur
            for _a, _ls, _cc in (("dust_off", "--", "0.35"), ("dust_on", "-", _c)):
                if (_a, P8S_AP) not in _dd:
                    continue
                _v = _dd[(_a, P8S_AP)][1] / _LSUN8S
                _y = np.median(_v, axis=0) if P8S_INCL == "median" else _v[_incl8s.index(P8S_INCL)]
                if _a == "dust_on" and P8S_INCL == "median":
                    ax.fill_between(_wav, _v.min(axis=0), _v.max(axis=0), color=_c, alpha=0.25, lw=0)
                ax.plot(_wav, _y, _ls, color=_cc, lw=1.6 if _a == "dust_on" else 1.2,
                        label=("with dust" if _a == "dust_on" else "without dust"))
            if solo:
                ax.set_xlabel(r"$\lambda_{\rm rest}$ [$\mu$m]"); ax.set_ylabel(r"$\nu L_\nu$ [L$_\odot$]")

        with plt.rc_context(globals().get("PAPER_RC", {})):
            _nr, _nc = len(_gal8s), len(_stg8s)
            fig, axs = plt.subplots(_nr, _nc, figsize=(P8S_PANEL_W * _nc + (2.2 if P8S_ROWLAB else 0.4), P8S_PANEL_H * _nr + 0.9),
                                    sharex=True, sharey=True, squeeze=False)
            for _i, _key in enumerate(_gal8s):
                for _j, _st in enumerate(_stg8s):
                    ax = axs[_i][_j]
                    _sedpanel8s(ax, _key, _st)
                    if _i == 0:
                        ax.set_title(_stlab8s.get(_st, _st), fontsize=14)
                    if _i == _nr - 1:
                        ax.set_xlabel(r"$\lambda_{\rm rest}$ [$\mu$m]", fontsize=14)
                    if _j == 0:
                        ax.set_ylabel(r"$\nu L_\nu$ [L$_\odot$]", fontsize=14)
                    ax.tick_params(labelsize=11)
                if P8S_ROWLAB:
                    axs[_i][-1].text(1.04, 0.5, _lab8s.get(_key, _key[0]).replace(" $+$ ", "\n") + f"\n({_key[1]}, {_key[2]})",
                                     transform=axs[_i][-1].transAxes, ha="left", va="center", fontsize=12,
                                     color=_col8s.get(_key[0], "k"), fontweight="bold")
            _hl = [h for a in axs.ravel() for h in a.get_legend_handles_labels()[0]][:2]
            if _hl:
                fig.legend(_hl, [h.get_label() for h in _hl], loc="lower center", ncol=2, fontsize=12, frameon=False)
            fig.tight_layout(rect=(0, 0.6 / fig.get_figheight(), 1 - (2.0 if P8S_ROWLAB else 0.0) / fig.get_figwidth(), 1)); fig.subplots_adjust(wspace=0.04, hspace=0.06)
            paper_save(fig, f"ba_gallery_seds_{P8S_BOX}")
            plt.show()
            if P8S_SINGLES:
                _n = 0
                for _key in _gal8s:
                    for _j, _st in enumerate(_stg8s):
                        if (_key, _st) not in _CUR8S:
                            continue
                        fig, ax = plt.subplots(figsize=(P8S_PANEL_W, P8S_PANEL_H))
                        _sedpanel8s(ax, _key, _st, solo=True)
                        if P8S_SINGLE_LEGEND:
                            ax.legend(fontsize=10, frameon=False, loc="lower center")
                        fig.savefig(os.path.join(_SED8S, f"sed_{_key[0]}_a{_key[1]:03d}_g{_key[2]:06d}_{_j}{_st}.png"),
                                    dpi=PAPER_DPI, bbox_inches="tight")
                        plt.close(fig); _n += 1
                print(f"  {_n} single SED panels -> {_SED8S}/sed_*.png")


## Part 9 — the observer's forecast: disky quenched galaxies and their dust, redshift by redshift

**This part is not about the selection.** Every other cell divides the quenched population; this one asks
what a survey would *find* in it. Two numbers per redshift, and their product:

| | |
|---|---|
| $P(\rm disky \mid QG)$ | how often a quenched galaxy has a **disky stellar morphology** at all |
| $P(f_{\rm dust} > 10^{-4} \mid \rm disky\ QG)$ | and how often that disky quenched galaxy is also **dusty** |
| the two multiplied | the chance that one quenched galaxy drawn at that redshift is **both** |

plus panel (a): the $f_{\rm dust}$ of every disky quenched galaxy, galaxy by galaxy, with the median and the
16–84 range wherever there are at least `P9_NMIN` of them.

**The sample is the catalogue at every anchor of the ladder** — ten redshifts from $z = 2.02$ to $0.31$,
every galaxy in the box at each of them, not the branch sample the rest of the notebook works on. "Disky" is
$\kappa_{\rm rot}^\star$ above `P9_KSTAR`: the stellar rotation the catalogue measures over hundreds of star
particles, so the isotropic noise on it is $\sim$0.02 and a fixed line is the honest cut. The **threshold
ladder** printed before the figure shows what other choices would give — the answer depends strongly on it,
and saying "disky" without saying the cut means nothing.

**Is $\kappa_\star$ the right requirement?** It is the right *axis*, and it is **not a morphology**.
$\kappa_{\rm rot}^\star$ measures rotational support; an observer classifies from *light*, and a
bulge-dominated disc — an S0 — is a disc in an image while sitting at $\kappa_\star \sim 0.35$–$0.50$, below
any "rotation-dominated" line. Panel (a) is therefore the first thing the cell draws: $P(\rm dusty)$ as a
**continuous function of $\kappa_\star$**, with the adopted cut and its alternatives marked. It rises
monotonically — 4 % at $\kappa_\star \approx 0.3$, 7 % at 0.38, 12 % at 0.42, 18 % at 0.48, 26 % at 0.52,
43 % above 0.55 — and the two boxes agree, so the axis is right and the threshold is a choice you can see
rather than a hidden assumption. Panels (d)–(f) draw the alternatives (`P9_KALT`) as thin lines beside the
adopted cut for the same reason.

**What is *not* a substitute: the stellar size.** The mass–size residual is uncorrelated with $\kappa_\star$
($\rho = +0.05$) and runs the *other* way — $P(\rm dusty)$ is 13 % for the compact half of the quenched
population against 3 % for the extended one — so "extended = disky" would select the opposite galaxies. The
cell prints that as a warning rather than offering it as an option.

**Two knobs worth understanding before reading the numbers.** `P9_RULE` chooses the quenched definition:
`"ssfr"` (the physical one — a mass floor and sSFR below `PASSIVE_FACTOR`/$t_H$) or `"notebook"` (Part 0's,
which *also* demands 50 gas and 500 star particles and a dust-to-H₂ floor). The second removes the
gas-poorest quenched galaxies — the dust-poor end — so it biases $P(\rm dusty)$ **up**; both counts are
printed. And `P9_BOXES` puts the 50 Mpc box first, with the 100 Mpc one beside it: at $\kappa_\star > 0.5$ a
disky quenched galaxy is a **1 % object**, so m50 has only a handful per redshift and m100 is what resolves
the trend.


In [ ]:
# ── Part 9 — THE OBSERVER'S FORECAST: disky quenched galaxies and their dust, redshift by redshift ────────
# THIS PART IS NOT ABOUT THE SELECTION.  Every other cell divides the quenched population; this one asks what
# a survey would find in it: P(disky | QG), P(f_dust > cut | disky QG), and the two multiplied.
#
# IS kappa_* THE RIGHT REQUIREMENT?  It is the right AXIS and it is not a morphology.  kappa_rot(*) measures
# ROTATIONAL SUPPORT; an observer classifies from LIGHT, and a bulge-dominated disc — an S0 — is a disc in an
# image while sitting at kappa_* ~ 0.35-0.50, below any "rotation-dominated" line.  Panel (a) is therefore the
# first thing this cell draws: P(dusty) as a CONTINUOUS function of kappa_*, with the adopted cut marked and
# the alternatives beside it.  It rises monotonically from 4 % at kappa_* ~ 0.3 to ~29 % above 0.5, so the
# axis is the right one and the threshold is a choice the reader can see rather than a hidden assumption.
# WHAT IS NOT A SUBSTITUTE: the stellar size.  The mass-size residual is uncorrelated with kappa_* (rho 0.05)
# and runs the OTHER way — the COMPACT quenched galaxies are the dusty ones — so "extended = disky" would
# select the opposite population.  The cell prints that too, as a warning rather than an option.
P9_BOXES  = ["cis50", "cis100"]   # the user's box first; the 100 Mpc one has 8x the volume and is drawn beside it
P9_KSTAR  = 0.5           # THE ADOPTED CUT: "disky" = kappa_rot(*) above this
P9_KALT   = [0.40, 0.45]  # ... drawn as thin lines beside it, because the answer depends on where it is put
P9_FDUST  = DUSTY_CUT     # "dusty" = log M_dust/M* above this
P9_RULE   = "ssfr"        # the quenched rule: "ssfr" (mass + sSFR) | "notebook" (Part 0's, which also demands
                          # 50 gas + 500 star particles and a dust/H2 floor, and so cuts the dust-poor end)
P9_MASS   = MASS_FLOOR    # the mass floor of the sample [log M*]
P9_NMIN   = 5             # a point is drawn only with this many disky QGs behind it
P9_KNMIN  = 25            # ... and a kappa_* bin of panel (a) needs this many: the lowest bin holds
                          # a handful of oddities and would put a spurious upturn at the left edge
# A BULGE-TO-TOTAL RATIO INSTEAD?  Not a measured one — nothing in the catalogue decomposes the light.  What
# CAN be written down is the two-component reading of kappa_* itself: if a galaxy is a disc with
# kappa_disc and a bulge that is isotropic (kappa = 1/3), then kappa_* = (1-B/T) kappa_disc + (B/T)/3 and
#     B/T  =  (kappa_disc - kappa_*) / (kappa_disc - 1/3)
# Panel (a) carries that as a SECOND AXIS on top.  It is a relabelling, not a new measurement: B/T is a
# monotone function of kappa_* here, so it cannot rank galaxies differently — it only puts the threshold in
# units an observer recognises (kappa_* = 0.50 is B/T = 0.66, 0.40 is 0.85).  A real B/T needs a bulge-disc
# decomposition of the mock images, which is the m25 imaging products' business, not this catalogue's.
P9_BT      = True         # draw the implied B/T axis on panel (a)
P9_KDISC   = 0.85         # the kappa_rot a pure disc is given in that reading
P9_KB     = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.75]    # the kappa_* bins of panel (a)
P9_FLOOR  = FLOOR         # zero dust is drawn at this log, never dropped
P9_JIT    = 0.010         # the jitter of the individual galaxies in panel (b), in redshift
ROW9 = []

_C9 = CAT[CAT["box"].isin([b for b in P9_BOXES if (CAT["box"] == b).any()])].copy()
if not len(_C9):
    print(f"Part 9: none of {P9_BOXES} is in the catalogue — skipped.")
else:
    with np.errstate(all="ignore"):
        _ms9 = _C9["mstar"].to_numpy(float)
        _C9["lm9"] = np.log10(np.where(_ms9 > 0, _ms9, np.nan))
        _C9["ssfr9"] = _C9["sfr"].to_numpy(float) / np.where(_ms9 > 0, _ms9, np.nan)
        _fd9 = _C9["mdust"].to_numpy(float) / np.where(_ms9 > 0, _ms9, np.nan)
        _C9["lfd9"] = np.where(_fd9 > 0, np.log10(np.where(_fd9 > 0, _fd9, 1.0)), P9_FLOOR)
        _C9["lre9"] = np.log10(np.where(_C9["r_half_star"].to_numpy(float) > 0,
                                        _C9["r_half_star"].to_numpy(float), np.nan))
        _q9 = (_C9["lm9"].to_numpy(float) > P9_MASS) & (_C9["ssfr9"].to_numpy(float)
                                                        < PASSIVE_FACTOR / (_C9["t_H_gyr"].to_numpy(float) * 1e9))
        _qn9 = _q9 & (_C9["ngas"].to_numpy(float) >= NGAS_MIN) & (_C9["nstar"].to_numpy(float) >= NSTAR_MIN) \
            & (_C9["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * _C9["mh2"].to_numpy(float))
    _C9["qg9"] = _qn9 if P9_RULE == "notebook" else _q9
    _C9["disk9"] = _C9["kappa_star"].to_numpy(float) > P9_KSTAR
    _C9["dusty9"] = _C9["lfd9"].to_numpy(float) > P9_FDUST
    _Q9 = _C9[_C9["qg9"]].copy()
    print(f"PART 9 — THE OBSERVER'S FORECAST.  quenched = {P9_RULE!r} (log M* > {P9_MASS:g}, sSFR < "
          f"{PASSIVE_FACTOR:g}/t_H"
          + (f", {NGAS_MIN} gas + {NSTAR_MIN} star particles, dust/H2 floor" if P9_RULE == "notebook" else "")
          + f"),  disky = kappa_* > {P9_KSTAR:g},  dusty = log M_dust/M* > {P9_FDUST:g}")
    print(f"  {int(_q9.sum())} quenched rows under 'ssfr' against {int(_qn9.sum())} under 'notebook' — the "
          f"second is the first minus its gas-poorest, i.e. minus part of the dust-poor end.")

    # ── is kappa_* the right requirement?  the continuum, and what is NOT a substitute ───────────────────
    print(f"\n  IS kappa_* THE RIGHT REQUIREMENT?  P(dusty) against it, pooled over redshift:")
    print(f"      {'kappa_* bin':>14s}" + "".join(f"{b:>22s}" for b in P9_BOXES))
    _KC9 = 0.5 * (np.asarray(P9_KB[:-1]) + np.asarray(P9_KB[1:]))
    for _i in range(len(P9_KB) - 1):
        _line = f"      {P9_KB[_i]:.2f}-{P9_KB[_i + 1]:.2f}".rjust(20)
        for _b in P9_BOXES:
            _s = _Q9[(_Q9["box"] == _b)]
            _m = (_s["kappa_star"].to_numpy(float) > P9_KB[_i]) & (_s["kappa_star"].to_numpy(float) <= P9_KB[_i + 1])
            _line += (f"{int(_m.sum()):8d}{np.mean(_s['dusty9'].to_numpy(bool)[_m]) * 100:9.0f} %"
                      if _m.sum() else f"{0:8d}{'n/a':>11s}")
        print(_line)
    print("      N, then the share of them that are dusty.  It rises monotonically: the axis is right, the\n"
          "      THRESHOLD is the choice — and a bulge-dominated disc sits at 0.35-0.50, not above 0.5.")
    if P9_BT:
        _bt = lambda k: float(np.clip((P9_KDISC - k) / (P9_KDISC - KAPPA_ISO), 0, 1))
        print(f"      IN B/T, under a {P9_KDISC:g} disc + an isotropic bulge:  "
              + ",  ".join(f"kappa_* {k:.2f} = B/T {_bt(k):.2f}" for k in (0.35, 0.40, 0.45, 0.50, 0.55))
              + f"\n      — a RELABELLING of the same axis (B/T is monotone in kappa_* here), not a new "
                f"measurement: a real\n      B/T needs a bulge-disc decomposition of the images.")
    with np.errstate(all="ignore"):
        _mb9 = pd.cut(_Q9["lm9"], np.arange(P9_MASS, 12.75, 0.25))
        _Q9["dre9"] = _Q9["lre9"] - _Q9.groupby(_mb9, observed=False)["lre9"].transform("median")
        _rho9 = _Q9[["kappa_star", "dre9"]].corr(method="spearman").iloc[0, 1]
        _lo9 = _Q9["dre9"].to_numpy(float) < -0.1
        _hi9 = _Q9["dre9"].to_numpy(float) > 0.1
    print(f"\n  AND WHAT IS NOT A SUBSTITUTE: the stellar size.  rho(kappa_*, the mass-size residual) = "
          f"{_rho9:+.2f}, and P(dusty) is {np.nanmean(_Q9['dusty9'].to_numpy(bool)[_lo9]) * 100:.0f} % for the "
          f"COMPACT half ({int(_lo9.sum())} below -0.1 dex) against "
          f"{np.nanmean(_Q9['dusty9'].to_numpy(bool)[_hi9]) * 100:.0f} % for the extended one "
          f"({int(_hi9.sum())} above +0.1) — 'extended = disky' would select the opposite population.")

    # ── the figure ──────────────────────────────────────────────────────────────────────────────────────
    _CB9 = dict(zip(P9_BOXES, ["#1f78b4", "#e66101", "#33a02c", "#6a3d9a"]))
    _MK9 = dict(zip(P9_BOXES, ["o", "s", "^", "v"]))
    plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
    fig, AX = plt.subplots(2, 3, figsize=(16.2, 8.4))
    _pc = lambda ax: ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v * 100:.0f}%"))

    # (a) the continuum: P(dusty) against kappa_*, with the cut and its alternatives
    ax = AX[0, 0]
    for _b in P9_BOXES:
        _s = _Q9[_Q9["box"] == _b]
        _p, _l, _h, _n = [], [], [], []
        for _i in range(len(P9_KB) - 1):
            _m = (_s["kappa_star"].to_numpy(float) > P9_KB[_i]) & (_s["kappa_star"].to_numpy(float) <= P9_KB[_i + 1])
            _v = wilson(int(_s["dusty9"].to_numpy(bool)[_m].sum()), int(_m.sum())) if _m.sum() >= P9_KNMIN \
                else (np.nan,) * 3
            _p.append(_v[0]); _l.append(_v[1]); _h.append(_v[2]); _n.append(int(_m.sum()))
            ROW9.append(dict(block="kappa", box=_b, k_lo=P9_KB[_i], k_hi=P9_KB[_i + 1], n=int(_m.sum()),
                             p_dusty=_v[0]))
        ax.fill_between(_KC9, _l, _h, color=_CB9[_b], alpha=0.18, lw=0)
        ax.plot(_KC9, _p, color=_CB9[_b], lw=2.2, marker=_MK9[_b], ms=5, label=BOX_LABEL.get(_b, _b))
    for _k, _st in [(P9_KSTAR, "-")] + [(_k, ":") for _k in P9_KALT]:
        ax.axvline(_k, color="0.35", ls=_st, lw=1.4 if _st == "-" else 1.0)
        ax.text(_k, -0.02, f"{_k:g}", transform=ax.get_xaxis_transform(), ha="center", va="top",
                fontsize=8.5, color="0.35")
    ax.set_xlabel(r"$\kappa_{\rm rot}^\star$"); ax.set_ylabel(rf"$P(f_{{\rm dust}} > 10^{{{P9_FDUST:g}}})$")
    if P9_BT:                       # the same axis in the units an observer would use
        _bt9 = lambda k: np.clip((P9_KDISC - np.asarray(k, float)) / (P9_KDISC - KAPPA_ISO), 0.0, 1.0)
        _tax = ax.secondary_xaxis("top", functions=(lambda k: _bt9(k),
                                                    lambda b: P9_KDISC - np.asarray(b, float) * (P9_KDISC - KAPPA_ISO)))
        _tax.set_xlabel(rf"implied $B/T$   (a $\kappa_{{\rm disc}} = {P9_KDISC:g}$ disc + an isotropic bulge)",
                        fontsize=9)
    ax.set_title("(a) is $\\kappa_\\star$ the right requirement?", fontsize=10.5, loc="left")
    _pc(ax)

    # (b) the kappa_* distribution of the quenched population, redshift by redshift
    ax = AX[0, 1]
    for _b in P9_BOXES:
        _s = _Q9[_Q9["box"] == _b]
        _z = sorted(_s["z"].unique())
        _q = np.array([np.nanpercentile(_s.loc[_s["z"] == z_, "kappa_star"], [16, 50, 84]) for z_ in _z])
        ax.fill_between(_z, _q[:, 0], _q[:, 2], color=_CB9[_b], alpha=0.15, lw=0)
        ax.plot(_z, _q[:, 1], color=_CB9[_b], lw=2.2, marker=_MK9[_b], ms=5)
    ax.axhline(P9_KSTAR, color="0.35", lw=1.4)
    ax.set_xlabel("redshift"); ax.set_ylabel(r"$\kappa_{\rm rot}^\star$ of the QGs")
    ax.set_title("(b) how much they rotate at all\n(median, 16-84)", fontsize=10.5, loc="left")

    # (c) the dust of the disky QGs, galaxy by galaxy
    ax = AX[0, 2]
    for _b in P9_BOXES:
        _s = _Q9[(_Q9["box"] == _b) & _Q9["disk9"]]
        _rng = np.random.default_rng(9)
        for _z, _g in _s.groupby("z"):
            _v = _g["lfd9"].to_numpy(float)
            _x = float(_z) + _rng.normal(0, P9_JIT, len(_v))
            ax.scatter(_x, _v, s=12, color=_CB9[_b], alpha=0.45, lw=0)
            if len(_v) >= P9_NMIN:
                ax.plot([float(_z)], [np.median(_v)], marker="_", ms=16, mew=2.6, color=_CB9[_b])
    ax.axhline(P9_FDUST, color="0.35", ls="--", lw=1.4)
    ax.text(0.01, P9_FDUST, " dusty above here", transform=ax.get_yaxis_transform(), fontsize=8.5, va="bottom",
            color="0.35")
    ax.set_xlabel("redshift"); ax.set_ylabel(r"$\log M_{\rm dust}/M_\star$")
    ax.set_title("(c) every disky QG, one point each\n(bar = median)", fontsize=10.5, loc="left")

    # ── the three probabilities, redshift by redshift ────────────────────────────────────────────────────
    for _b in P9_BOXES:
        _s = _Q9[_Q9["box"] == _b]
        _z9, _P = [], {k: ([], [], []) for k in ("disky", "dusty", "joint", "nondisky")}
        _ALT = {_k: [] for _k in P9_KALT}
        for _zv, _g in _s.groupby("z"):
            _d = _g["disk9"].to_numpy(bool); _u = _g["dusty9"].to_numpy(bool)
            _z9.append(float(_zv))
            for _k9, (_num, _den) in (("disky", (_d.sum(), len(_g))),
                                      ("dusty", ((_d & _u).sum(), _d.sum())),
                                      ("joint", ((_d & _u).sum(), len(_g))),
                                      ("nondisky", ((~_d & _u).sum(), (~_d).sum()))):
                _v = wilson(int(_num), int(_den)) if (_den and (_k9 != "dusty" or _d.sum() >= P9_NMIN)) \
                    else (np.nan,) * 3
                for _a, _bq in zip(_P[_k9], _v):
                    _a.append(_bq)
            for _k in P9_KALT:
                _da = _g["kappa_star"].to_numpy(float) > _k
                _ALT[_k].append((float(_da.mean()), float(np.mean(_u[_da])) if _da.sum() >= P9_NMIN else np.nan,
                                 float((_da & _u).mean())))
            ROW9.append(dict(block="z", box=_b, z=float(_zv), n_qg=len(_g), n_disky=int(_d.sum()),
                             n_disky_dusty=int((_d & _u).sum()), p_disky=_P["disky"][0][-1],
                             p_dusty_given_disky=_P["dusty"][0][-1], p_joint=_P["joint"][0][-1],
                             p_dusty_given_nondisky=_P["nondisky"][0][-1],
                             med_lfd_disky=float(np.nanmedian(_g["lfd9"].to_numpy(float)[_d])) if _d.sum() else np.nan))
        _z9 = np.asarray(_z9)
        for _ax, _k9, _ttl in ((AX[1, 0], "disky", "(d) how often a QG is disky"),
                               (AX[1, 1], "dusty", "(e) and that disky QG dusty"),
                               (AX[1, 2], "joint", "(f) the two together")):
            _m, _l, _h = (np.asarray(x, float) for x in _P[_k9])
            _ax.fill_between(_z9, _l, _h, color=_CB9[_b], alpha=0.18, lw=0)
            _ax.plot(_z9, _m, color=_CB9[_b], lw=2.2, marker=_MK9[_b], ms=5, label=BOX_LABEL.get(_b, _b))
            for _j, _k in enumerate(P9_KALT):                       # the alternative cuts, thin
                _ax.plot(_z9, [v[{"disky": 0, "dusty": 1, "joint": 2}[_k9]] for v in _ALT[_k]],
                         color=_CB9[_b], lw=0.9, ls=[":", "--"][_j % 2], alpha=0.75)
            _ax.set_title(_ttl, fontsize=10.5, loc="left")
        AX[1, 1].plot(_z9, np.asarray(_P["nondisky"][0], float), color="0.55", lw=1.2, ls="-", alpha=0.9)
    for _ax, _yl in ((AX[1, 0], r"$P(\rm disky \mid QG)$"),
                     (AX[1, 1], rf"$P(\rm dusty \mid disky\ QG)$"),
                     (AX[1, 2], r"$P(\rm disky\ and\ dusty \mid QG)$")):
        _ax.set_xlabel("redshift"); _ax.set_ylabel(_yl); _ax.set_ylim(0, None); _pc(_ax)
    for _ax in AX.ravel():
        _ax.grid(axis="y", color="0.9", lw=0.8)
        _ax.set_axisbelow(True)
        if _ax is not AX[0, 0]:
            _ax.invert_xaxis()
    fig.legend(handles=[Line2D([], [], color=_CB9[b], marker=_MK9[b], lw=2.2, ms=5, label=BOX_LABEL.get(b, b))
                        for b in P9_BOXES]
               + [Line2D([], [], color="0.35", lw=1.4, label=rf"the adopted cut $\kappa_\star > {P9_KSTAR:g}$"),
                  Line2D([], [], color="0.5", lw=0.9, ls=":", label="the alternatives "
                         + ", ".join(f"{k:g}" for k in P9_KALT)),
                  Line2D([], [], color="0.55", lw=1.2, label="(e) the NON-disky QGs, for contrast")],
               loc="lower center", ncol=5, fontsize=9, frameon=False)
    fig.suptitle("What a survey would find: disky quenched galaxies, and the dust in them", y=0.995,
                 fontsize=12)
    fig.tight_layout(rect=[0, 0.05, 1, 0.96], h_pad=2.8, w_pad=2.2)
    paper_save(fig, "ba_disky_forecast"); plt.show()
    plt.rcParams.update({"axes.spines.top": True, "axes.spines.right": True})

    print(f"\n  {'box':9s} {'z':>5s} {'N QG':>6s} {'disky':>6s} {'P(disky)':>10s} {'dusty|disky':>16s} "
          f"{'dusty|not':>10s} {'joint':>8s}")
    for _r in [r for r in ROW9 if r.get("block") == "z"]:
        print(f"  {_r['box']:9s} {_r['z']:5.2f} {_r['n_qg']:6d} {_r['n_disky']:6d} {_r['p_disky'] * 100:9.1f} %"
              + (f" {_r['p_dusty_given_disky'] * 100:7.0f} % [{_r['n_disky_dusty']:2d}/{_r['n_disky']:<3d}]"
                 if np.isfinite(_r["p_dusty_given_disky"]) else f"{'n/a':>16s}")
              + (f" {_r['p_dusty_given_nondisky'] * 100:9.0f} %" if np.isfinite(_r["p_dusty_given_nondisky"])
                 else f"{'n/a':>10s}") + f" {_r['p_joint'] * 100:7.2f} %")
    pd.DataFrame(ROW9).to_csv(os.path.join(OUT, "ba_disky_forecast.csv"), index=False)
    print("table ->", os.path.join(OUT, "ba_disky_forecast.csv"))

## Part 10 — paper figure 1: where the two dust modes are, and where they are not

**One quantity, three ways.** Everything in this figure is $\log M_{\rm dust}/M_\star$ — the quantity Part 1b
peeled — over the galaxies of the peel.

| | |
|---|---|
| **row 1** | **the selection itself.** The map the two groups were cut out of, one panel per resolution: the dust against the time since the jet fired, every column scaled to its own peak, with Part 1b's two **regions** drawn on it and each group's median track walked through them. |
| **row 2** | the same dust as a plain distribution, read at the **five events of a galaxy's own history** — the jet, the first X-ray firing, the two ends of the quenching transition ($t_{q,\,\rm start}=t_{\rm SFT}$, $t_{q,\,\rm end}=t_{\rm QT}$) and the anchor, which is where an observer catches it ($t_{\rm obs}$). |
| **row 3** | and read at **five redshifts**, at the ladder's own snapshots — the nearest one to each target, labelled with its true $z$, never interpolated. Part 1 carries the redshift the histories were written with, so this row needs no cosmology of its own. |

**What counts as two modes is Part 1b's own rule**, applied to a one-dimensional sample: a peak has to stand
`P1B_PROM` above the floor of the density, and two peaks are two modes only if the valley between them falls
to `P1B_DEEP` of the lower one — a shoulder is one mode. The figure and the selection therefore cannot
disagree about the word. Every panel is annotated with the reference box's mode count and, where there is
one, the valley's position and its **depth** as a fraction of the lower peak (small = deep = a real gap);
the same numbers for every box, plus the dusty share and the floor share, go to
`ba_paper_dust_modes.csv`. `BA_PSILV=1` adds Silverman's bootstrap $p$ to that table — it is the whole cost
of the cell and says nothing the depth does not.

**The censoring floor is never a mode.** Zero dust is stored at `FLOOR`, and a delta function at a storage
floor is not a population: those galaxies are counted, drawn as the grey strip, printed as "no dust %" and
kept out of the KDE. It is what `P1B_FLOORS` does to the map, for the same reason.

**Read row 3 knowing what its sample is.** A chain ends at its own anchor, so the $z=3$ panel holds the
*progenitors of the whole sample* — almost none of them quenched yet — while the $z=0.3$ panel holds only
those still in the quenched sample there. Both numbers are printed on the panel: how many galaxies, and what
share of them is already quenched. `PF1_ZQ = True` restricts every panel to the quenched ones instead, which
is the observer's population but empties the high-redshift end.

**Knobs.** `PF1_BOXES` (row 1; add `NOX_BOX` to draw the X-ray-off control as a further panel), `PF1_EPOCHS`
and `PF1_ZS` (which events and which redshifts, and how they are labelled), `PF1_ZQ`, `PF1_Q` (the quantity
— anything in `P1_TRK`, though only the peeled one makes row 1 the selection), `PF1_BW` (the KDE bandwidth
of rows 2–3), `PF1_FTOL` (how close to `FLOOR` still counts as the floor), `PF1_NMIN`, `PF1_MED` (the median
tracks over the map), `PF1_CUT` (the `DUSTY_CUT` line), `PF1_BOXCOL` / `PF1_BOXLS`. The paper colours `PFC`
are defined here and Part 11 reuses them: the peel's red and blue, so a region in row 1 and a track in the
next figure are the same colour.

In [ ]:
# ── Part 10 — PAPER FIGURE 1: the selection on the map, and WHEN the two modes appear ─────────────────────
# One figure, three rows, one quantity — log M_dust/M*, the one Part 1b peeled — over the galaxies of the peel.
#   ROW 1  THE SELECTION.  The map the two groups were cut out of, one panel per resolution: the distribution
#          of the dust against the time since the jet fired, each column scaled to its own peak, with the two
#          REGIONS drawn on it and each group's median track walked through them.
#   ROW 2  the same quantity as a plain distribution, read at the FIVE EVENTS of a galaxy's own history —
#          the jet, the first X-ray firing, the two ends of the quenching transition, and the anchor.
#   ROW 3  and read at five REDSHIFTS, on the ladder's own snapshots (no interpolation and no cosmology of
#          this notebook's own: Part 1 carries the redshift the histories were written with).
# The two rows answer one question.  The map has two ridges; a distribution has two modes only where and when
# it has them, and rows 2 and 3 say WHEN — how late in a galaxy's own history, and how late in cosmic time.
PF1_STEM   = "ba_paper_dust_modes"
PF1_BOXES  = [b for b in BOXES if b in _BX1]        # the X-ray-on resolutions: rows 1-3
PF1_NOX    = False         # ROW 1 ONLY: the X-ray-off box as a further map panel — the SAME cells, calibrated on
                           # the X-ray-on boxes, laid over a run in which the channel does not exist.  The count
                           # it prints for the dust-poor region is the control: there is nobody in it.
PF1_Q      = P1D_ON        # the quantity the whole figure is about; the peel's own, so row 1 IS the selection
PF1_UNIQ   = P1B_UNIQ      # galaxies, not anchors — the peel's convention
PF1_EPOCHS = [("jet", r"$t_{\rm jet}$"), ("x1", r"$t_{\rm X\!-\!ray}$"), ("sft", r"$t_{q,\,\rm start}$"),
              ("qt", r"$t_{q,\,\rm end}$"), ("anc", r"$t_{\rm obs}$")]   # (the P1 epoch key, how it is labelled)
PF1_ZS     = [3.0, 1.5, 1.0, 0.7, 0.3]     # row 3: the nearest LADDER snapshot to each is used and its true z printed
PF1_ZQ     = False         # row 3: True = only the branches that are already QUENCHED at that redshift
                           # (Part 1's own per-snapshot flag), False = every branch alive there, progenitors included
PF1_ZSPLIT = False         # row 3: draw the ALREADY-QUENCHED subset of the reference box as a second, thin
                           # curve inside the same panel.  It is the control on the whole row: a second mode
                           # that is in the full sample and NOT in this one is the quenched / still-forming
                           # MIXTURE being seen, not two modes of the quenched population.  (Nothing to draw
                           # when PF1_ZQ is already True.)
PF1_KDE    = False         # rows 2-3: the per-box KDE curves over the histogram (the diagnostic version).  The
                           # paper version is the HISTOGRAM of the reference box alone and the division line —
                           # only m100n1024 has the statistics, and the mode counts stay in the table.
PF1_ANNOT  = False         # rows 2-3: the mode count / N / shares written in the panel (diagnostic)
PF1_COUNTS = True          # row 1: the two group counts in the corner of each map
PF1_ROWLAB = {"map": "the selection", "epoch": "critical points", "redshift": "redshifts"}   # the boxed
                           # label at the left of each row (None = none)
PF1_BW     = None          # the KDE bandwidth of rows 2 and 3 (None = Scott)
PF1_FTOL   = 0.05          # dex: a value this close to FLOOR is ON the censoring floor — no dust at all.  It is
                           # counted and reported separately and never smeared into the KDE, which is what
                           # Part 1b's P1B_FLOORS does to the map for exactly the same reason.
PF1_NMIN   = 20            # no curve, no mode count, below this many galaxies
PF1_MED    = True          # row 1: each group's median track over the map
PF1_CUT    = True          # the observable threshold DUSTY_CUT as a line in rows 2 and 3
PF1_NBIN   = 34            # the histogram of the reference box behind the curves
PF1_BOXCOL = {"cis100": "#111111", "cis50": "#1b9e77", "cis25": "#7570b3", "cis50nox": "#a6761d"}
PF1_BOXLS  = {"cis100": "-", "cis50": "--", "cis25": ":", "cis50nox": "-."}
PF1_SILV   = bool(int(os.environ.get("BA_PSILV", "0")))   # ALSO run Silverman's bootstrap on every panel
                           # (env BA_PSILV=1).  It is the whole cost of this cell and says nothing the valley
                           # depth does not; the mode count and the depth are always printed.
from matplotlib.patches import Patch      # the one legend key Part 0 has no handle for
if "PEEL" not in globals():
    raise RuntimeError("Part 10 draws Part 1b's own map and regions — run Part 1b (and Part 2, for the group "
                       "keys) first.")
_BX10 = [b for b in PF1_BOXES if b in PEEL]
if not _BX10:
    raise RuntimeError(f"none of PF1_BOXES = {PF1_BOXES} was peeled; Part 1b ran on {list(PEEL)}")
_REF10 = _BX10[0]          # the box whose histogram is drawn behind rows 2-3 and whose modes are annotated
_BXMAP10 = _BX10 + ([NOX_BOX] if (PF1_NOX and NOX_BOX in PEEL and NOX_BOX not in _BX10) else [])   # row 1

# THE COLOURS: the paper figures use the PEEL's colours (the regions of row 1 and the tracks of Part 11 are
# then the same red and the same blue), not Part 0's orange / purple.
PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
PFL = {g: GLAB.get(g, g) for g in GK}


def _pf_modes(v, bw=None):
    """Part 1b's own mode rule applied to a 1D sample: the KDE scaled to its peak, the peaks that stand
    P1B_PROM above its floor, the interior minimum between the two highest and HOW DEEP it is as a fraction of
    the lower of the two (P1B_DEEP or below = a valley, above = a shoulder and therefore ONE mode).
    Returns (n, n_modes, dip, depth, gx, dens)."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < PF1_NMIN or np.std(v) <= 0:
        return len(v), 0, np.nan, np.nan, np.array([]), np.array([])
    try:
        kde = gaussian_kde(v, bw_method=bw)
    except Exception:
        return len(v), 0, np.nan, np.nan, np.array([]), np.array([])
    gx = np.linspace(v.min() - 0.5 * v.std(), v.max() + 0.5 * v.std(), KDE_NGRID)
    d = kde(gx)
    dn = d / max(float(d.max()), 1e-300)
    pk = np.flatnonzero((dn[1:-1] > dn[:-2]) & (dn[1:-1] >= dn[2:])) + 1
    pk = pk[dn[pk] - dn.min() >= P1B_PROM]
    if len(pk) < 2:
        return len(v), len(pk), np.nan, 1.0, gx, d
    a, b = sorted(pk[np.argsort(dn[pk])[-2:]])
    lo = a + int(np.argmin(dn[a:b + 1]))
    depth = float(dn[lo] / min(dn[a], dn[b]))
    if depth > P1B_DEEP:                      # a shoulder, not a valley: one mode, the way Part 1b counts them
        return len(v), 1, np.nan, depth, gx, d
    return len(v), len(pk), float(gx[lo]), depth, gx, d


def _pf_split(v):
    """A sample cut into the values ON the censoring floor and the rest; (the rest, the floor share, n)."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if not len(v):
        return v, np.nan, 0
    fl = v <= FLOOR + PF1_FTOL
    return v[~fl], float(fl.mean()), len(v)


# ── row 3: the ladder's own snapshots.  Part 1 carries the redshift of every step of every series, so the
# population "at z = 1" is the galaxies of the sample that have the ladder snapshot nearest z = 1 — read there,
# never interpolated, and the snapshot's TRUE redshift is what the panel is labelled with. ──
_ZMAP10 = {}
for _b in _BX10:
    _sz = {}
    for _S in B1[_b]["R"]:
        for _s, _z in zip(np.asarray(_S["snap"], int), np.asarray(_S["z"], float)):
            if np.isfinite(_z):
                _sz.setdefault(int(_s), float(_z))
    _ZMAP10[_b] = _sz
    if not _sz:
        print(f"  !! {_b}: the series carry no redshift (histories written before the 'z' column) — row 3 "
              f"will be empty for it; re-run Part 1 against histories that have it.")


def _pf_atz(box, zt):
    """(the values of PF1_Q at the ladder snapshot nearest zt, that snapshot, its true redshift) for one box."""
    sz = _ZMAP10.get(box) or {}
    if not sz:
        return np.array([]), -1, np.nan
    ss = np.array(sorted(sz)); zz = np.array([sz[int(s)] for s in ss])
    k = int(np.argmin(np.abs(zz - zt)))
    s_want, z_true = int(ss[k]), float(zz[k])
    F, R = B1[box]["F"], B1[box]["R"]
    keep = F["uniq"].to_numpy(bool) if PF1_UNIQ else np.ones(len(F), bool)
    out, qq = [], []
    for i in np.flatnonzero(keep):
        S = R[int(F["ridx"].to_numpy()[i])]
        j = np.flatnonzero(np.asarray(S["snap"], int) == s_want)
        if not len(j):
            continue                                   # the chain does not reach that snapshot
        _q = bool(np.asarray(S["q"])[j[0]])
        if PF1_ZQ and not _q:
            continue                                   # ... and is not quenched there, if that is asked for
        out.append(float(np.asarray(S[PF1_Q], float)[j[0]])); qq.append(_q)
    return np.asarray(out, float), np.asarray(qq, bool), s_want, z_true


# ── the samples of rows 2 and 3, collected before anything is drawn: the x range is then the DATA's ───────
_S2 = FR[FR["uniq"]] if PF1_UNIQ else FR
SMP10 = {}                     # (row, panel) -> {box: values};  QSH10 the quenched share of a redshift panel
QSH10, ZTRUE10 = {}, {}
for _ep, _lab in PF1_EPOCHS:
    SMP10[("epoch", _lab)] = {b: _S2.loc[_S2["box"] == b, f"{PF1_Q}_{_ep}"].to_numpy(float) for b in _BX10}
QONLY10 = {}                   # the already-quenched subset of the reference box, panel by panel
for _zt in PF1_ZS:
    _v, _q, _s, _z = {}, {}, {}, {}
    for _b in _BX10:
        _v[_b], _qq, _s[_b], _z[_b] = _pf_atz(_b, _zt)
        _q[_b] = float(_qq.mean()) if len(_qq) else np.nan
        if _b == _REF10:
            _qsub = _v[_b][_qq] if len(_qq) == len(_v[_b]) else np.array([])
    _zr = np.nanmedian([x for x in _z.values() if np.isfinite(x)]) if _z else np.nan
    _lab = rf"$z = {_zr:.2f}$" if np.isfinite(_zr) else f"z = {_zt:g}"
    SMP10[("redshift", _lab)] = _v
    QSH10[_lab], ZTRUE10[_zt] = _q, (_s, _z, _lab)
    QONLY10[_lab] = _qsub if (PF1_ZSPLIT and not PF1_ZQ) else np.array([])
_pool10 = np.concatenate([v for d in SMP10.values() for v in d.values() if len(v)] + [np.array([FLOOR])])
_pool10 = _pool10[np.isfinite(_pool10)]
_XLO = min(FLOOR - 0.35, float(np.nanpercentile(_pool10, 0.2)))
_XHI = float(np.nanpercentile(_pool10, 99.8)) + 0.15
_GX10 = np.linspace(_XLO, _XHI, KDE_NGRID)

# ── the figure ────────────────────────────────────────────────────────────────────────────────────────────
_n1, _n2 = len(_BXMAP10), max(len(PF1_EPOCHS), len(PF1_ZS))
_NC = int(np.lcm(_n1, _n2))
ROW10 = []


def _pf_stats(v_all, box, title, row, qsh=None, tag=None):
    """The statistics of one sample into ROW10 — every box, whether or not it is drawn."""
    v, ffl, n = _pf_split(v_all)
    nn, npk, dip, dep, _, _ = _pf_modes(v, PF1_BW)
    _p = np.nan
    if PF1_SILV and nn >= PF1_NMIN:
        _p = _silverman(v, k=1, nboot=200)[0]
    ROW10.append(dict(row=row, box=(tag or box), panel=title, n=n, n_offfloor=nn, floor_frac=ffl, n_modes=npk,
                      dip=dip, valley_depth=dep, silverman_p=_p,
                      frac_dusty=float(np.nanmean(np.asarray(v_all, float) >= DUSTY_CUT)) if n else np.nan,
                      frac_quenched=(qsh or {}).get(box, np.nan) if tag is None else 1.0))
    return v, n


def _pf_panel(ax, smp, title, row, qsh=None, qonly=None):
    """One distribution panel: the histogram of the reference box, the division line, the floor.  The KDE
    curves, the quenched-only control and the in-panel statistics are diagnostics (PF1_KDE / PF1_ZSPLIT /
    PF1_ANNOT) and off in the paper version."""
    _hi, _ref = 0.0, None
    for _b in _BX10:
        v, n = _pf_stats(np.asarray(smp.get(_b, np.array([])), float), _b, title, row, qsh)
        if n < PF1_NMIN:
            continue
        if _b == _REF10:
            _h, _, _ = ax.hist(v, bins=np.linspace(_XLO, _XHI, PF1_NBIN + 1), density=True, color="0.78",
                               edgecolor="0.55", lw=0.5, zorder=1)
            _hi = max(_hi, float(np.max(_h)) if len(_h) else 0.0)
            _ref = ROW10[-1]
        if PF1_KDE and len(v) >= PF1_NMIN and np.std(v) > 0:
            y = gaussian_kde(v, bw_method=PF1_BW)(_GX10)
            ax.plot(_GX10, y, color=PF1_BOXCOL.get(_b, "0.3"), ls=PF1_BOXLS.get(_b, "-"), lw=1.8, zorder=3)
            _hi = max(_hi, float(np.nanmax(y)))
    if qonly is not None and len(qonly) >= PF1_NMIN:        # the control: the same box, quenched galaxies only
        _vq, _nq = _pf_stats(qonly, _REF10, title, row, tag=f"{_REF10} (quenched only)")
        if PF1_ZSPLIT and len(_vq) >= PF1_NMIN and np.std(_vq) > 0:
            _y2 = gaussian_kde(_vq, bw_method=PF1_BW)(_GX10)
            ax.plot(_GX10, _y2, color="#b2182b", lw=1.3, alpha=0.9, zorder=5)
            _hi = max(_hi, float(np.nanmax(_y2)))
    ax.axvspan(_XLO, FLOOR + PF1_FTOL, color="0.86", alpha=0.6, lw=0, zorder=0)
    if PF1_CUT:
        ax.axvline(DUSTY_CUT, color=P1D_COL[P1D_NAMES[0]], ls="--", lw=1.8, zorder=4)
    ax.text(0.04, 0.95, title, transform=ax.transAxes, va="top", ha="left", fontsize=17.6, fontweight="bold",
            color="0.15", zorder=6)
    if PF1_ANNOT and _ref is not None:
        ax.text(0.97, 0.95, (f"{int(_ref['n_modes'])} mode" + ("s" if _ref["n_modes"] != 1 else "")
                             + (f"\nvalley {_ref['dip']:.1f} ({_ref['valley_depth']:.2f})" if np.isfinite(_ref["dip"]) else "")
                             + f"\nN {int(_ref['n'])}, dusty {_ref['frac_dusty'] * 100:.0f} %"
                             + (f"\nquenched {_ref['frac_quenched'] * 100:.0f} %" if np.isfinite(_ref["frac_quenched"]) else "")),
                transform=ax.transAxes, va="top", ha="right", fontsize=11.7, color="0.35",
                bbox=dict(fc="w", ec="none", alpha=0.75, pad=1.2), zorder=6)
    ax.set_xlim(_XLO, _XHI)
    ax.set_ylim(0, max(_hi * 1.32, 1e-3))
    ax.set_yticks([])
    for _s in ("top", "right"):
        ax.spines[_s].set_visible(False)


with plt.rc_context(PAPER_RC):
    fig = plt.figure(figsize=(3.6 * _n2 + 1.0, 13.0))
    gs = fig.add_gridspec(3, _NC, height_ratios=[1.45, 1.0, 1.0], hspace=0.34, wspace=1.4,
                          left=0.09, right=0.988, top=0.985, bottom=0.145)
    AXM = [fig.add_subplot(gs[0, (_NC // _n1) * k:(_NC // _n1) * (k + 1)]) for k in range(_n1)]
    AXE = [fig.add_subplot(gs[1, (_NC // _n2) * k:(_NC // _n2) * (k + 1)]) for k in range(len(PF1_EPOCHS))]
    AXZ = [fig.add_subplot(gs[2, (_NC // _n2) * k:(_NC // _n2) * (k + 1)]) for k in range(len(PF1_ZS))]

    # ── ROW 1 — the map and the two regions ──────────────────────────────────────────────────────────────
    for _k, _b in enumerate(_BXMAP10):
        ax, _P = AXM[_k], PEEL[_b]
        ax.set_facecolor("k")
        ax.pcolormesh(TG, _YC, np.ma.masked_invalid(_P["stages"][0]["dens"]), cmap=P1B_CMAP, shading="nearest",
                      vmin=0, vmax=1, rasterized=True)
        for _i, _g in enumerate(P1D_NAMES[:2]):
            _M = _P["masks"][_i]
            if _M is None:
                continue
            ax.contourf(TG, _YC, _M.astype(float), levels=[0.5, 1.5], colors=[P1D_COL[_g]], alpha=0.26)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=["w"], linewidths=3.4, alpha=0.85)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=[P1D_COL[_g]], linewidths=2.0)
        if PF1_MED:
            for _g, _nm in zip(GK, P1D_NAMES[:2]):
                _m = _P["grp"] == _nm
                if _m.sum() < 3:
                    continue
                _md = np.array([np.nanmedian(_P["A"][_m, j]) if np.isfinite(_P["A"][_m, j]).sum() >= 3 else np.nan
                                for j in range(len(TG))])
                ax.plot(TG, _md, color="w", lw=3.2, alpha=0.75, solid_capstyle="round")
                ax.plot(TG, _md, color=PFC[_g], lw=2.0, solid_capstyle="round")
        ax.axvline(0.0, color="w", ls="--", lw=1.0, alpha=0.65)
        ax.text(0.03, 0.96, BOX_LABEL.get(_b, _b) + ("  (the control)" if _b == NOX_BOX else ""),
                transform=ax.transAxes, va="top", ha="left", fontsize=17.6, fontweight="bold", color="w",
                bbox=dict(fc="k", ec="none", alpha=0.45, pad=2.0))
        if PF1_COUNTS:
            ax.text(0.03, 0.05, "   ".join(f"{int((_P['grp'] == n).sum())} {s}" for n, s in
                                           zip(P1D_NAMES[:2], ("dusty", "dust-poor"))),
                    transform=ax.transAxes, va="bottom", ha="left", fontsize=14.3, color="w",
                    bbox=dict(fc="k", ec="none", alpha=0.45, pad=1.8))
        ax.set_xlabel(r"$t-t_{\rm jet}$  [Gyr]")
        ax.set_xlim(TG[0], TG[-1]); ax.set_ylim(_YE[0], _YE[-1])
        if _k == 0:
            ax.set_ylabel(P1_LAB.get(PF1_Q, PF1_Q).split("  [")[0])
        else:
            ax.set_yticklabels([])
        ROW10.append(dict(row="map", box=_b, panel="jet clock",
                          n_dusty=int((_P["grp"] == P1D_NAMES[0]).sum()),
                          n_pure=int((_P["grp"] == P1D_NAMES[1]).sum())))

    # ── ROWS 2 and 3 — the distributions ─────────────────────────────────────────────────────────────────
    for _k, (_ep, _lab) in enumerate(PF1_EPOCHS):
        _pf_panel(AXE[_k], SMP10[("epoch", _lab)], _lab, "epoch")
        AXE[_k].set_xlabel(P1_LAB.get(PF1_Q, PF1_Q).split("  [")[0] if _k == len(PF1_EPOCHS) // 2 else "")
    AXE[0].set_ylabel("galaxies")
    for _k, _zt in enumerate(PF1_ZS):
        _lab = ZTRUE10[_zt][2]
        _pf_panel(AXZ[_k], SMP10[("redshift", _lab)], _lab, "redshift", qsh=QSH10.get(_lab),
                  qonly=QONLY10.get(_lab))
        AXZ[_k].set_xlabel(P1_LAB.get(PF1_Q, PF1_Q).split("  [")[0] if _k == len(PF1_ZS) // 2 else "")
    AXZ[0].set_ylabel("galaxies")

    # ── the boxed label at the left of each row ──────────────────────────────────────────────────────────
    if PF1_ROWLAB:
        for _axs, _key in ((AXM, "map"), (AXE, "epoch"), (AXZ, "redshift")):
            if not PF1_ROWLAB.get(_key):
                continue
            _pos = _axs[0].get_position()
            fig.text(0.022, 0.5 * (_pos.y0 + _pos.y1), PF1_ROWLAB[_key], rotation=90, va="center", ha="center",
                     fontsize=18.2, fontweight="bold", color="0.25",
                     bbox=dict(boxstyle="round,pad=0.5", fc="#f3efe6", ec="0.5", lw=1.0))

    _h10 = [Line2D([], [], color=P1D_COL[P1D_NAMES[0]], lw=2.4, label=f"the {P1D_NAMES[0]} region"),
            Line2D([], [], color=P1D_COL[P1D_NAMES[1]], lw=2.4, label=f"the {P1D_NAMES[1]} region"),
            Line2D([], [], color="0.3", lw=2.0, label="the median track of each group"),
            Line2D([], [], color=P1D_COL[P1D_NAMES[0]], ls="--", lw=1.8,
                   label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$, the dusty / dust-poor division"),
            Patch(fc="0.86", ec="none", label="no dust at all (the storage floor)")]
    if PF1_KDE:
        _h10 += [Line2D([], [], color=PF1_BOXCOL.get(b, "0.3"), ls=PF1_BOXLS.get(b, "-"), lw=1.8,
                        label=BOX_LABEL.get(b, b)) for b in _BX10]
    if PF1_ZSPLIT and not PF1_ZQ:
        _h10 += [Line2D([], [], color="#b2182b", lw=1.3, label="row 3: the already-quenched ones alone")]
    fig.legend(handles=_h10, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.53, 0.0), fontsize=16)
    paper_save(fig, PF1_STEM); plt.show()

P10 = pd.DataFrame(ROW10)
P10.to_csv(os.path.join(OUT, f"{PF1_STEM}.csv"), index=False)
print(f"\nWHEN THE DISTRIBUTION IS DOUBLE  —  {P1_LAB.get(PF1_Q, PF1_Q).split('  [')[0]}, "
      f"{'galaxies' if PF1_UNIQ else 'anchor rows'}, Part 1b's own mode rule "
      f"(a peak stands {P1B_PROM:g} above the floor of the density; two peaks are two modes only if the valley "
      f"between them falls to {P1B_DEEP:g} of the lower one):")
for _row, _ttl in (("epoch", "at the events of a galaxy's own history"), ("redshift", "at fixed redshift")):
    print(f"  {_ttl}:")
    _sub = P10[P10["row"] == _row]
    for _p in _sub["panel"].unique():
        _line = f"    {_p:>16s} "
        for _b in _BX10:
            _r = _sub[(_sub["panel"] == _p) & (_sub["box"] == _b)]
            if not len(_r) or int(_r["n"].iloc[0]) < PF1_NMIN:
                _line += f"  {_b} n/a"; continue
            _r = _r.iloc[0]
            _line += (f"  {_b} {int(_r['n']):4d}: {int(_r['n_modes'])} mode"
                      + (f" (valley {_r['dip']:+.2f}, depth {_r['valley_depth']:.2f})" if np.isfinite(_r["dip"])
                         else f" (deepest {_r['valley_depth']:.2f})" if np.isfinite(_r["valley_depth"]) else "")
                      + f", no dust {_r['floor_frac'] * 100:.0f} %, dusty {_r['frac_dusty'] * 100:.0f} %")
        print(_line)
print(f"  the redshift row reads the LADDER SNAPSHOT nearest each target and is labelled with its true z: "
      + ", ".join(f"z={_z:g} -> snap {list(ZTRUE10[_z][0].values())[0]} "
                  f"(z = {list(ZTRUE10[_z][1].values())[0]:.2f})" for _z in PF1_ZS if _z in ZTRUE10))
print(f"  it takes every branch alive at that snapshot"
      + (" that is ALREADY QUENCHED there (PF1_ZQ)" if PF1_ZQ else
         " — progenitors included, since a galaxy quenched at z = 0.3 was a star-forming galaxy at z = 3 and "
         "the point of the row is when the second mode appears at all (PF1_ZQ = True asks for the quenched ones only)")
      + f";  the {'modes' if PF1_SILV else 'mode count and the valley depth'} are annotated for {_REF10}"
      + ("" if PF1_SILV else " (BA_PSILV=1 adds Silverman's bootstrap p to the table)") + ".")
print("table ->", os.path.join(OUT, f"{PF1_STEM}.csv"))

## Part 10b — paper figure 1b: the selection alone

Part 10's first row on its own: one panel per X-ray-on box, the distribution of $\log M_{\rm dust}/M_\star$
at every step of the **jet clock** — Part 1b's own map, the per-epoch KDE of the galaxies' tracks with each
column scaled to its own peak — and on it the **two regions of the peel**: the dusty one (stage 1, the upper
ridge) and the dust-poor one (stage 2, the brightest blob once the dusty tracks are removed). The regions
are Part 1b's cells, calibrated on the two big boxes pooled and laid over every box unchanged, which is why
the same outline sits on all three panels. Nothing else is drawn: the median tracks and the distributions at
the events stay in Part 10.

**Knobs.** `PFS_BOXES`, `PFS_NOX` (the X-ray-off box as a further panel — the same cells over a run without
the channel), `PFS_MED` (the median track of each group back on the map), `PFS_COUNTS` (the two group counts
in the corner), `PFS_CBAR` (the colour bar), `PFS_ALPHA` (the fill of the regions; 0 = outline only),
`PFS_TLIM` / `PFS_YLIM` (the stretch of the clock and of the ordinate; `None` = the map's own), `PFS_CUT`
(the observable threshold as a line — it is not the selection).

In [ ]:
# ── Part 10b — PAPER FIGURE 1b: THE SELECTION alone — the jet-aligned dust map of each resolution, with the
#    two regions the groups were cut out of ─────────────────────────────────────────────────────────────────
# Part 10's first row on its own, for the paper: one panel per X-ray-on box, the distribution of
# log M_dust/M* at every step of the jet clock (Part 1b's own map — the per-epoch KDE of the galaxies'
# tracks, each column scaled to its own peak, so a thin epoch is as bright as a crowded one), and on it the
# two REGIONS of the peel: the dusty one (stage 1, the upper ridge) and the dust-poor one (stage 2, the
# brightest blob once the dusty tracks are removed).  The regions are Part 1b's cells, calibrated on the two
# big boxes pooled and laid over every box unchanged — which is why the same outline is on all three panels.
# Nothing else is drawn: no median tracks (PFS_MED puts them back), no distributions at the events (Part 10).
PFS_STEM   = "ba_paper_selection"
PFS_BOXES  = [b for b in BOXES if b in PEEL]      # the X-ray-on resolutions, one panel each
PFS_NOX    = False       # the X-ray-off box as a further panel: the same cells over a run without the channel
                         # (the count it prints for the dust-poor region is the control: there is nobody in it)
PFS_MED    = False       # each group's median track over the map (Part 10 draws it; the paper version is the map alone)
PFS_COUNTS = True        # the two group counts in the corner of each panel
PFS_CBAR   = True        # a colour bar: the density of each epoch column, scaled to that column's peak
PFS_ALPHA  = 0.26        # the fill of the regions (0 = outline only)
PFS_TLIM   = None        # None = the whole clock TG; or (t_lo, t_hi) in Gyr
PFS_YLIM   = None        # None = the map's own range (Part 1b's _YE); or (y_lo, y_hi)
PFS_CUT    = False       # the observable threshold DUSTY_CUT as a horizontal line (it is not the selection)
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
if "PEEL" not in globals():
    raise RuntimeError("Part 10b draws Part 1b's own map and regions — run Part 1b (and Part 2, for the colours) first.")
_BXS = [b for b in PFS_BOXES if b in PEEL]
if not _BXS:
    raise RuntimeError(f"none of PFS_BOXES = {PFS_BOXES} was peeled; Part 1b ran on {list(PEEL)}")
_BXS += [NOX_BOX] if (PFS_NOX and NOX_BOX in PEEL and NOX_BOX not in _BXS) else []
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
_TLS = PFS_TLIM if PFS_TLIM else (float(TG[0]), float(TG[-1]))
_YLS = PFS_YLIM if PFS_YLIM else (float(_YE[0]), float(_YE[-1]))
_CTXS = {"sft": r"t-t_{\rm SFT}", "jet": r"t-t_{\rm jet}", "x1": r"t-t_{\rm x1}", "qt": r"t-t_{\rm QT}"}

ROW10B = []
with plt.rc_context(PAPER_RC):
    fig, AX = plt.subplots(1, len(_BXS), figsize=(5.9 * len(_BXS) + (1.1 if PFS_CBAR else 0.0), 6.0),
                           squeeze=False, sharey=True)
    for _k, _b in enumerate(_BXS):
        ax, _P = AX[0, _k], PEEL[_b]
        ax.set_facecolor("k")
        _im = ax.pcolormesh(TG, _YC, np.ma.masked_invalid(_P["stages"][0]["dens"]), cmap=P1B_CMAP,
                            shading="nearest", vmin=0, vmax=1, rasterized=True)
        _ext = {}
        for _i, _g in enumerate(P1D_NAMES[:2]):
            _M = _P["masks"][_i]
            if _M is None:
                continue
            if PFS_ALPHA > 0:
                ax.contourf(TG, _YC, _M.astype(float), levels=[0.5, 1.5], colors=[P1D_COL[_g]], alpha=PFS_ALPHA)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=["w"], linewidths=3.4, alpha=0.85)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=[P1D_COL[_g]], linewidths=2.0)
            _jj, _ii = np.flatnonzero(_M.any(axis=0)), np.flatnonzero(_M.any(axis=1))
            _ext[_g] = (float(TG[_jj[0]]), float(TG[_jj[-1]]), float(_YE[_ii[0]]), float(_YE[_ii[-1] + 1]))
        if PFS_MED:
            for _g, _nm in zip(GK, P1D_NAMES[:2]):
                _m = _P["grp"] == _nm
                if _m.sum() < 3:
                    continue
                _md = np.array([np.nanmedian(_P["A"][_m, j]) if np.isfinite(_P["A"][_m, j]).sum() >= 3 else np.nan
                                for j in range(len(TG))])
                ax.plot(TG, _md, color="w", lw=3.2, alpha=0.75, solid_capstyle="round")
                ax.plot(TG, _md, color=PFC[_g], lw=2.0, solid_capstyle="round")
        ax.axvline(0.0, color="w", ls="--", lw=1.0, alpha=0.65)
        if PFS_CUT:
            ax.axhline(DUSTY_CUT, color="w", ls=":", lw=1.2, alpha=0.8)
        ax.text(0.03, 0.96, BOX_LABEL.get(_b, _b) + ("  (the control)" if _b == NOX_BOX else ""),
                transform=ax.transAxes, va="top", ha="left", fontsize=17.6, fontweight="bold", color="w",
                bbox=dict(fc="k", ec="none", alpha=0.45, pad=2.0))
        _nd, _npr = int((_P["grp"] == P1D_NAMES[0]).sum()), int((_P["grp"] == P1D_NAMES[1]).sum())
        if PFS_COUNTS:
            ax.text(0.03, 0.05, f"{_nd} dusty   {_npr} dust-poor", transform=ax.transAxes, va="bottom", ha="left",
                    fontsize=14.3, color="w", bbox=dict(fc="k", ec="none", alpha=0.45, pad=1.8))
        ax.set_xlabel(f"${_CTXS.get(P1B_CLOCK, P1B_CLOCK)}$  [Gyr]")
        ax.set_xlim(*_TLS); ax.set_ylim(*_YLS)
        if _k == 0:
            ax.set_ylabel(P1_LAB.get(P1D_ON, P1D_ON).split("  [")[0])
        ROW10B.append(dict(box=_b, clock=P1B_CLOCK, quantity=P1D_ON, n_dusty=_nd, n_pure=_npr,
                           n_intermediate=int((_P["grp"] == P1D_NAMES[2]).sum()), n_all=int(len(_P["grp"])),
                           **{f"{s}_{k}": v for s, g in (("dusty", P1D_NAMES[0]), ("pure", P1D_NAMES[1]))
                              for k, v in zip(("t_lo", "t_hi", "y_lo", "y_hi"), _ext.get(g, (np.nan,) * 4))}))
    fig.subplots_adjust(left=0.07, right=(0.90 if PFS_CBAR else 0.985), bottom=0.24, top=0.975, wspace=0.08)
    if PFS_CBAR:
        _pos = AX[0, -1].get_position()
        _cax = fig.add_axes([_pos.x1 + 0.012, _pos.y0, 0.014, _pos.y1 - _pos.y0])
        _cb = fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap=P1B_CMAP), cax=_cax)
        _cb.set_label("galaxies per epoch,\nscaled to the epoch's peak", fontsize=15)
        _cb.set_ticks([0, 0.5, 1])
    _h10b = [Line2D([], [], color=P1D_COL[P1D_NAMES[0]], lw=2.4, label=f"the {P1D_NAMES[0]} region"),
             Line2D([], [], color=P1D_COL[P1D_NAMES[1]], lw=2.4, label=f"the {P1D_NAMES[1]} region")]
    if PFS_MED:
        _h10b += [Line2D([], [], color="0.3", lw=2.0, label="the median track of each group")]
    if PFS_CUT:
        _h10b += [Line2D([], [], color="0.3", ls=":", lw=1.4, label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$")]
    fig.legend(handles=_h10b, loc="lower center", ncol=min(4, len(_h10b)), frameon=False,
               bbox_to_anchor=(0.5 * (0.07 + (0.90 if PFS_CBAR else 0.985)), 0.02))
    paper_save(fig, PFS_STEM); plt.show()

P10B = pd.DataFrame(ROW10B)
P10B.to_csv(os.path.join(OUT, f"{PFS_STEM}.csv"), index=False)
print(f"THE SELECTION  —  the {P1D_ON} map on the {P1B_CLOCK} clock, {'galaxies' if P1B_UNIQ else 'anchor rows'}, "
      f"the regions calibrated on {' + '.join(_CALB) if _CALB else 'each box alone'} and laid over every box:")
for _, _r in P10B.iterrows():
    print(f"  {BOX_LABEL.get(_r['box'], _r['box']):34s} {int(_r['n_dusty']):4d} dusty  {int(_r['n_pure']):4d} dust-poor  "
          f"{int(_r['n_intermediate']):4d} intermediate  of {int(_r['n_all'])};  "
          f"dusty region t = [{_r['dusty_t_lo']:+.2f}, {_r['dusty_t_hi']:+.2f}] Gyr, "
          f"log f_dust = [{_r['dusty_y_lo']:.2f}, {_r['dusty_y_hi']:.2f}];  "
          f"dust-poor region t = [{_r['pure_t_lo']:+.2f}, {_r['pure_t_hi']:+.2f}], "
          f"log f_dust = [{_r['pure_y_lo']:.2f}, {_r['pure_y_hi']:.2f}]")
print(f"  a galaxy is in a region when its track spends {P1D_NHIT} epochs inside it (P1D_NHIT), a crosser of both "
      f"{'goes to the intermediates' if P1D_PURE else 'stays in the first'} (P1D_PURE); the map's colour is each "
      f"epoch's own KDE scaled to its peak (P1B_BW = {P1B_BW:g} x Scott).")
print("table ->", os.path.join(OUT, f"{PFS_STEM}.csv"))

## Part 10c — paper figure 1c: the peel, stage by stage

Part 10b shows the two regions on every box's map; this is **how they were found**, in the box that carries
the statistics: one row, the three stages of Part 1b's peel on the jet-aligned dust map of m100n1024. Stage 1
is the whole map and the most populated bright region above the valley, the dusty QGs; stage 2 the same map
with those tracks removed and the most populated bright blob of what is left, the dust-poor QGs; stage 3 the
residual, the intermediates. Every column keeps stage 1's scaling and is dimmed by the fraction of its
galaxies still in play, Part 1b's convention, so a region that empties looks empty.

**Knobs.** `PFP_BOX`, `PFP_STAGES` (which stages, in order), `PFP_DIM` (dim by the share still in play, or
rescale each stage), `PFP_MED` (the median of what is still in play), `PFP_TEXT` (the two-line note per
panel), `PFP_CBAR`, `PFP_ALPHA`, `PFP_TLIM` / `PFP_YLIM`.

The colour bar is named by `PFP_CBLAB` and the two peeled regions by `PFP_REGLAB` (2026-09-19: *counts normalized by peak
at epoch*, *dusty branch*, *dust-poor branch*), in the legend and in the in-panel notes alike.


In [ ]:
# ── Part 10c — PAPER FIGURE 1c: THE PEEL, stage by stage, in the 100 Mpc box ─────────────────────────────
# Part 10b shows the two regions on every box's map; this is HOW they were found, in the box that carries
# the statistics: one row, the three stages of Part 1b's peel on the jet-aligned dust map of m100n1024.
#   STAGE 1  the whole map; the most populated bright region ABOVE the valley       -> the dusty QGs
#   STAGE 2  the same map with the dusty tracks REMOVED; the most populated bright blob of what is left,
#            below the valley                                                        -> the dust-poor QGs
#   STAGE 3  the residual: the galaxies that never entered either region             -> the intermediates
# Part 1b's drawing convention, verbatim: every column keeps stage 1's scaling (each epoch's KDE scaled to its
# own peak) and is DIMMED by the fraction of its galaxies still in play, so a region that empties looks empty
# instead of renormalising itself back to full brightness (PFP_DIM = False rescales each stage instead).
PFP_STEM   = "ba_paper_peel"
PFP_BOX    = "cis100"        # the box shown (the regions themselves are calibrated on P1D_CAL, m100 + m50 pooled)
PFP_STAGES = [0, 1, 2]       # the stages drawn, in order: 0 = the whole map, 1 = the dusty tracks removed, 2 = the residual
PFP_DIM    = True            # dim each stage's columns by the share of its galaxies still in play (Part 1b's convention)
PFP_MED    = False           # the median track of what is still in play at each stage (Part 1b's cyan line)
PFP_TEXT   = True            # a two-line note per panel: what the map holds, what the region takes
PFP_CBAR   = True            # the colour bar
PFP_ALPHA  = 0.26            # the fill of the region found at that stage (0 = outline only)
PFP_TLIM   = None            # None = the whole clock TG; or (t_lo, t_hi) in Gyr
PFP_YLIM   = None            # None = the map's own range (Part 1b's _YE); or (y_lo, y_hi)
PFP_FS     = 2.0             # EVERY piece of text in this figure, as a multiple of PAPER_RC (2026-09-18: "at least
                             #   double" — 2 = 36/42/36 pt ticks/labels/legend; the panels grow with it, PFP_PANEL)
PFP_PANEL  = (8.6, 9.0)      # [inch] one panel, wide x high (the room the doubled labels need; was 5.9 x 6.0)
PFP_CBLAB  = "counts normalized by peak at epoch"    # the colour bar's name (2026-09-19, the user's wording)
PFP_REGLAB = ["dusty branch", "dust-poor branch"]    # what the two peeled regions are CALLED on this figure — the legend
                             #   and the in-panel notes (the stages keep their order; None = Part 1b's own group names)
from matplotlib.patches import Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
if "PEEL" not in globals():
    raise RuntimeError("Part 10c draws Part 1b's own stages — run Part 1b first.")
if PFP_BOX not in PEEL:
    raise KeyError(f"PFP_BOX = {PFP_BOX!r} was not peeled; Part 1b ran on {list(PEEL)}")
_P10C = PEEL[PFP_BOX]
_STG = [k for k in PFP_STAGES if 0 <= int(k) < len(_P10C["stages"])]
if not _STG:
    raise ValueError(f"PFP_STAGES = {PFP_STAGES}: the peel has stages 0-{len(_P10C['stages']) - 1}")
_TLP = PFP_TLIM if PFP_TLIM else (float(TG[0]), float(TG[-1]))
_YLP = PFP_YLIM if PFP_YLIM else (float(_YE[0]), float(_YE[-1]))
_CTXP = {"sft": r"t-" + r"t_{q,\,\rm start}", "jet": r"t-t_{\rm jet}", "x1": r"t-t_{\rm x1}", "qt": r"t-" + r"t_{q,\,\rm end}"}
_fsP = lambda v: float(v) * float(PFP_FS)      # one dial for every hard-coded size below
_NIN = {g: int((_P10C["grp"] == g).sum()) for g in _G3}          # the three groups' counts
_RLAB = {g: (PFP_REGLAB[i] if (PFP_REGLAB and i < len(PFP_REGLAB)) else g) for i, g in enumerate(_G3)}
_NALL = int(len(_P10C["grp"]))
_n1 = np.maximum(_P10C["stages"][0]["n"].astype(float), 1.0)     # the galaxies per column at stage 1

ROW10C = []
with plt.rc_context({**PAPER_RC, **{k: _fsP(PAPER_RC[k]) for k in ("font.size", "axes.labelsize", "xtick.labelsize",
                                                                     "ytick.labelsize", "legend.fontsize", "axes.titlesize")}}):
    fig, AX = plt.subplots(1, len(_STG), figsize=(PFP_PANEL[0] * len(_STG) + (1.6 if PFP_CBAR else 0.0), PFP_PANEL[1]),
                           squeeze=False, sharey=True)
    for _i, _k in enumerate(_STG):
        ax, _S = AX[0, _i], _P10C["stages"][_k]
        ax.set_facecolor("k")
        _sh = _S["dens"] * ((_S["n"] / _n1)[None, :] if PFP_DIM else 1.0)
        ax.pcolormesh(TG, _YC, np.ma.masked_invalid(_sh), cmap=P1B_CMAP, shading="nearest", vmin=0, vmax=1,
                      rasterized=True)
        _M, _ext = _S["mask"], (np.nan,) * 4
        if _M is not None and _k < 2:
            _col = P1D_COL[_G3[_k]]
            if PFP_ALPHA > 0:
                ax.contourf(TG, _YC, _M.astype(float), levels=[0.5, 1.5], colors=[_col], alpha=PFP_ALPHA)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=["w"], linewidths=3.4, alpha=0.85)
            ax.contour(TG, _YC, _M.astype(float), levels=[0.5], colors=[_col], linewidths=2.0)
            _jj, _ii = np.flatnonzero(_M.any(axis=0)), np.flatnonzero(_M.any(axis=1))
            _ext = (float(TG[_jj[0]]), float(TG[_jj[-1]]), float(_YE[_ii[0]]), float(_YE[_ii[-1] + 1]))
        _rw = _S["rows"]
        if PFP_MED and int(_rw.sum()) >= 3:         # the median of what is still in play
            _sub = _P10C["A"][_rw]
            _md = np.array([np.nanmedian(_sub[:, j]) if np.isfinite(_sub[:, j]).sum() >= 3 else np.nan for j in range(len(TG))])
            ax.plot(TG, _md, color="w", lw=3.2, alpha=0.75, solid_capstyle="round")
            ax.plot(TG, _md, color="#67e8f9", lw=2.0, solid_capstyle="round")
        ax.axvline(P1D_T0, color="w", ls="--", lw=_fsP(1.0), alpha=0.65)
        if _i == 0:
            ax.text(0.03, 0.96, BOX_LABEL.get(PFP_BOX, PFP_BOX), transform=ax.transAxes, va="top", ha="left",
                    fontsize=_fsP(17.6), fontweight="bold", color="w", bbox=dict(fc="k", ec="none", alpha=0.45, pad=2.0))
        _nplay = int(_rw.sum())
        _note = {0: (f"all {_nplay} galaxies", f"{_RLAB[_G3[0]]}: {_NIN[_G3[0]]}"),
                 1: (f"{_RLAB[_G3[0]]} removed  ({_nplay} left)", f"{_RLAB[_G3[1]]}: {_NIN[_G3[1]]}"),
                 2: (f"both removed  ({_nplay} left)", f"{_RLAB[_G3[2]]}: {_NIN[_G3[2]]}")}.get(_k, (f"stage {_k + 1}", ""))
        if PFP_TEXT:
            ax.text(0.03, 0.05, "\n".join(s for s in _note if s), transform=ax.transAxes, va="bottom", ha="left",
                    fontsize=_fsP(0.8 * 14.3), color="w", bbox=dict(fc="k", ec="none", alpha=0.45, pad=1.8))   # the notes: 0.8 x the dial, or they run into the next panel
        ax.set_xlabel(f"${_CTXP.get(P1B_CLOCK, P1B_CLOCK)}$  [Gyr]")
        ax.set_xlim(*_TLP); ax.set_ylim(*_YLP)
        if _i == 0:
            ax.set_ylabel(P1_LAB.get(P1D_ON, P1D_ON).split("  [")[0])
        ROW10C.append(dict(box=PFP_BOX, stage=_k + 1, clock=P1B_CLOCK, quantity=P1D_ON, n_in_play=_nplay,
                           region=(_G3[_k] if _k < 2 and _M is not None else "-"),
                           n_region=(_NIN[_G3[_k]] if _k < 2 else _NIN[_G3[2]]),
                           **dict(zip(("t_lo", "t_hi", "y_lo", "y_hi"), _ext))))
    _lP, _rP = 0.04 + 0.025 * PFP_FS, (0.905 - 0.022 * PFP_FS if PFP_CBAR else 0.985)   # the margins grow with the text
    fig.subplots_adjust(left=_lP, right=_rP, bottom=0.10 + 0.09 * PFP_FS, top=0.975, wspace=0.08)
    if PFP_CBAR:
        _pos = AX[0, -1].get_position()
        _cax = fig.add_axes([_pos.x1 + 0.010, _pos.y0, 0.012, _pos.y1 - _pos.y0])
        _cb = fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap=P1B_CMAP), cax=_cax)
        _cb.set_label(PFP_CBLAB + ("\n(dimmed by the share still in play)" if PFP_DIM else ""),
                      fontsize=_fsP(0.8 * (15 if not PFP_DIM else 13.5)))
        _cb.set_ticks([0, 0.5, 1]); _cb.ax.tick_params(length=_fsP(3.5), width=_fsP(0.8))
    _h10c = [Line2D([], [], color=P1D_COL[_G3[0]], lw=_fsP(2.4), label=_RLAB[_G3[0]]),
             Line2D([], [], color=P1D_COL[_G3[1]], lw=_fsP(2.4), label=_RLAB[_G3[1]])]
    if PFP_MED:
        _h10c += [Line2D([], [], color="#67e8f9", lw=2.0, label="the median of what is still in play")]
    fig.legend(handles=_h10c, loc="lower center", ncol=min(3, len(_h10c)), frameon=False,
               bbox_to_anchor=(0.5 * (_lP + _rP), 0.012), handlelength=2.2)
    paper_save(fig, PFP_STEM); plt.show()

P10C = pd.DataFrame(ROW10C)
P10C.to_csv(os.path.join(OUT, f"{PFP_STEM}.csv"), index=False)
print(f"THE PEEL, STAGE BY STAGE, in {BOX_LABEL.get(PFP_BOX, PFP_BOX)}  —  the {P1D_ON} map on the {P1B_CLOCK} clock, "
      f"{'galaxies' if P1B_UNIQ else 'anchor rows'}; the regions calibrated on {' + '.join(_CALB) if _CALB else PFP_BOX}:")
for _, _r in P10C.iterrows():
    print(f"  stage {int(_r['stage'])}: {int(_r['n_in_play']):5d} galaxies in play"
          + (f";  the {_r['region']} region takes {int(_r['n_region'])} "
             f"(t = [{_r['t_lo']:+.2f}, {_r['t_hi']:+.2f}] Gyr, log f_dust = [{_r['y_lo']:.2f}, {_r['y_hi']:.2f}])"
             if _r["region"] != "-" else f";  the residual = {int(_r['n_region'])} {_G3[2]}"))
print(f"  bright = within {P1D_BRIGHT:g} of the peak of its own side of the valley (P1D_BRIGHT); a track joins a region at "
      f"{P1D_NHIT} epochs inside (P1D_NHIT); a crosser of both {'goes to the intermediates' if P1D_PURE else 'stays in the first'} "
      f"(P1D_PURE); the columns are {'dimmed by the share still in play (PFP_DIM)' if PFP_DIM else 'rescaled at every stage'}.")
print("table ->", os.path.join(OUT, f"{PFP_STEM}.csv"))

## Part 11 — paper figure 2: the two groups through the truncation, with the statistics behind them

Part 3 draws every tracked quantity on the clock, one column per box, as a diagnostic. This is the paper
version: **six** quantities, on the truncation clock, for one reference box, with the other resolutions
behind them as thin lines — and, the point of the cell, **the number of galaxies behind every point of every
track drawn in the back of its own panel**, on one scale for the whole figure, so an epoch carried by a
handful of galaxies cannot be read as if it were carried by a thousand.

| | |
|---|---|
| $\log M_\star$, the age | **what the two groups are** — whether they are the same galaxies at all |
| $\kappa_{\rm rot}$(gas), $-\log R_{50,\rm gas}/R_{50,\star}$ | what their **gas** is doing: rotating or not, contracted or extended |
| $\log M_{\rm dust}/M_{\rm gas}$, $\log M_{\rm dust}/M_{\rm metals}$ | whether the dust difference is a difference in the **reservoir** or in the **depletion** — D/M is D/G with the enrichment divided out, so a split that is only a metallicity split does not survive into it |

**The two events are on the figure.** Each group's median jet activation (▼) and median first X-ray firing
(|) are marked on every panel, in the group's own colour, so the order of the events is read off the figure
and not out of the text; the numbers are printed under it. The band is the bootstrap 16–84 of the median,
the same estimator Part 3 uses, and an epoch is drawn only above `PF2_NMIN` galaxies.

**Dust-to-metal needs the metallicity**, which Part 1 reads from the histories' `zgas` — a histories cache
built before that column existed leaves `ldtm` empty and the panel says so by being blank. The metal budget
is $Z_{\rm gas}M_{\rm gas} + M_{\rm dust}$ (SIMBA's grains are metals taken *out of* the gas phase);
`P1_ZDUST = True` in Part 1 says `zgas` already counts them.

**Knobs.** `PF2_TRK` (the six quantities — anything in `P1_TRK`; an untracked name raises rather than
drawing a blank), `PF2_CLOCK` (`sft` / `jet` / `x1` / `qt`), `PF2_BOX` (the reference box: bands, counts and
event marks) and `PF2_THIN` (the resolutions drawn as thin medians behind it), `PF2_HIST` / `PF2_HFRAC` (the
counts in the back, and how much of the panel the largest of them takes), `PF2_MARKS`, `PF2_HLINE` (the
reference lines and their captions), `PF2_NCOL`, `PF2_NMIN`, `PF2_XLIM`, `PF2_TSHOW` (the clock points the
printed gap table is read at).

In [ ]:
# ── Part 11 — PAPER FIGURE 2: the two groups through the truncation, with the statistics behind them ──────
# Part 3 draws every tracked quantity on the clock, one column per box, as a diagnostic.  This is the paper
# version of it: SIX quantities, the ones the argument is made of, on the truncation clock, for the
# reference box, with the other resolutions behind them as thin lines and — the point of the cell — THE
# NUMBER OF GALAXIES behind every point of every track drawn in the back of its own panel, so an epoch where
# a curve is carried by a handful of galaxies cannot be read as if it were carried by a thousand.
#   the stellar mass and the age    say WHAT the two groups are (they are not the same galaxies at all)
#   kappa_gas and the compactness   say what their GAS is doing
#   D/G and D/M                     say whether the dust difference is a difference in the reservoir or in
#                                   the depletion — D/M divides the enrichment out of D/G
# The median jet activation and the median first X-ray firing of each group are marked on every panel, so the
# order of the events is on the figure and not only in the text.
PF2_STEM  = "ba_paper_tracks"
PF2_TRK   = ["lms", "age", "kappa_gas", "comp", "ldg", "ldtm"]   # any of P1_TRK; an untracked name RAISES
PF2_CLOCK = "sft"          # "sft" | "jet" | "x1" | "qt" — any clock Part 1 gridded (P1_CLOCKS)
PF2_BOX   = next((b for b in BOXES if b in _BX1), None)   # the reference box: bands, counts and marks
PF2_THIN  = [b for b in BOXES if b in _BX1 and b != PF2_BOX]      # the other resolutions, as thin medians
PF2_UNIQ  = True           # one row per galaxy
PF2_NMIN  = NMIN
PF2_NBOOT = None           # None = NBOOT
PF2_NCOL  = 3
PF2_HIST  = True           # the galaxy count behind each track — the whole point of the cell
PF2_HFRAC = 0.30           # the share of the panel height the LARGEST count in the whole figure takes, so the
                           # back of every panel is on ONE scale and thin epochs are thin everywhere
PF2_MARKS = True           # the median jet / first-X-ray position of each group on the clock
PF2_HLINE = {"kappa_gas": (KAPPA_ISO, r"isotropic $\kappa=1/3$"), "comp": (0.0, r"$R_{50,\rm gas}=R_{50,\star}$")}
PF2_XLIM  = None           # None = the whole clock TG
PF2_TSHOW = (0.0, 1.0, 2.0)     # the clock points the printed gap table is read at
PF2_FS    = 2.0            # EVERY piece of text, as a multiple of the notebook's base sizes (11 pt font, 9.5 pt ticks;
                           #   2026-09-18: "at least double").  The panels grow with it (PF2_PANEL) or the labels collide.
PF2_PANEL = (7.2, 5.6)     # [inch] one panel, wide x high (was 4.6 x 3.05 at PF2_FS = 1)
PF2_YLAB  = {"age": r"stellar age  [Gyr]"}   # per-quantity y-labels for THIS figure (P1_LAB otherwise): the mass-weighted
                           #   age's full label is taller than a panel at PF2_FS = 2 — "mass-weighted" goes to the caption
PF2_TITLE = False          # the figure title (the paper figures carry none; True puts the old one back)
_fs2 = lambda v: float(v) * float(PF2_FS)
# the quenching times are named for what they are: t_{q,start} = the SFT (the onset), t_{q,end} = the QT; the anchor is t_obs
_CLAB11 = {"sft": r"$t - t_{q,\,\rm start}$  [Gyr]", "jet": r"$t - t_{\rm jet}$  [Gyr]",
           "x1": r"$t - t_{\rm x1}$  [Gyr]", "qt": r"$t - t_{q,\,\rm end}$  [Gyr]"}
_MEV11  = {"sft": ("t_jet", "t_x1"), "jet": ("t_sft", "t_x1"), "x1": ("t_jet", "t_sft"),
           "qt": ("t_jet", "t_x1")}
_MSYM11 = {"t_jet": ("v", (0, (4, 2)), "the jet activation"), "t_x1": ("|", (0, (1, 1.8)), "the first X-ray firing"),
           "t_sft": ("s", (0, (5, 2)), r"$t_{q,\,\rm start}$, the quenching onset")}
if PF2_BOX is None:
    raise RuntimeError(f"none of {BOXES} was read by Part 1 (it has {_BX1})")
_MISS11 = [q for q in PF2_TRK if q not in P1_TRK]
if _MISS11:
    raise KeyError(f"PF2_TRK names {_MISS11}, which Part 1 does not track, so those panels would be blank.\n"
                   f"  the tracked quantities are {P1_TRK}\n"
                   f"  (a new one is three lines in Part 1: add it to P1_TRK, to P1_LAB and to V)")
if PF2_CLOCK not in P1_CLOCKS:
    raise KeyError(f"PF2_CLOCK = {PF2_CLOCK!r} is not one of the clocks Part 1 gridded: {P1_CLOCKS}")
from matplotlib.patches import Patch
if "PFC" not in globals():          # Part 10 defines the paper colours; stand alone if it has not been run
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}


def _pf2_rows(box):
    F = B1[box]["F"]
    keep = F["uniq"].to_numpy(bool) if PF2_UNIQ else np.ones(len(F), bool)
    return F, {g: keep & (F["grp"].to_numpy() == g) for g in GK}


# ── the tracks, and the counts behind them, computed before anything is drawn (the count axis is global) ──
TRK11, CNT11, ROW11 = {}, {}, []
for _b in [PF2_BOX] + list(PF2_THIN):
    _F, _gm = _pf2_rows(_b)
    for _q in PF2_TRK:
        for _g in GK:
            _c, _lo, _hi, _n = med_band(B1[_b]["T"][_q][PF2_CLOCK][_gm[_g]], nmin=PF2_NMIN, nboot=PF2_NBOOT)
            TRK11[(_b, _q, _g)] = (_c, _lo, _hi, _n)
            CNT11[(_b, _q, _g)] = np.where(_n >= PF2_NMIN, _n, 0)
            for _t, _y, _l, _h, _nn in zip(TG, _c, _lo, _hi, _n):
                if np.isfinite(_y):
                    ROW11.append(dict(box=_b, quantity=_q, clock=PF2_CLOCK, grp=_g, group=GSH.get(_g, _g),
                                      t=_t, med=_y, lo=_l, hi=_h, n=int(_nn)))
_NMAX11 = max([float(np.nanmax(CNT11[(PF2_BOX, q, g)])) for q in PF2_TRK for g in GK] + [1.0])

_nr11 = int(np.ceil(len(PF2_TRK) / PF2_NCOL))
_RC11 = {"font.size": _fs2(11), "axes.labelsize": _fs2(11), "xtick.labelsize": _fs2(9.5), "ytick.labelsize": _fs2(9.5),
         "legend.fontsize": _fs2(9), "axes.titlesize": _fs2(11), "axes.grid": False,
         "xtick.major.size": _fs2(3.5), "ytick.major.size": _fs2(3.5), "axes.linewidth": _fs2(0.8)}
_ctx11 = plt.rc_context(_RC11); _ctx11.__enter__()
fig, AX = plt.subplots(_nr11, PF2_NCOL, figsize=(PF2_PANEL[0] * PF2_NCOL, PF2_PANEL[1] * _nr11 + 0.4 * PF2_FS), squeeze=False)
for _i, _q in enumerate(PF2_TRK):
    ax = AX[_i // PF2_NCOL, _i % PF2_NCOL]
    if PF2_HIST:                       # the counts, in the back and on one scale for the whole figure
        axb = ax.twinx()
        axb.set_zorder(ax.get_zorder() - 1); ax.patch.set_visible(False)
        for _g in GK:
            axb.fill_between(TG, 0, CNT11[(PF2_BOX, _q, _g)], step="mid", color=PFC[_g], alpha=0.15, lw=0)
            axb.step(TG, CNT11[(PF2_BOX, _q, _g)], where="mid", color=PFC[_g], alpha=0.45, lw=_fs2(0.7))
        axb.set_ylim(0, _NMAX11 / PF2_HFRAC); axb.set_yticks([]); axb.set_ylabel("")
        for _s in axb.spines.values():
            _s.set_visible(False)
    for _b in PF2_THIN:                # the other resolutions: the median alone, no band
        for _g in GK:
            ax.plot(TG, TRK11[(_b, _q, _g)][0], color=PFC[_g], lw=_fs2(1.15), alpha=0.85,
                    ls=(0, (4, 2)) if _b == PF2_THIN[0] else (0, (1, 1.6)), zorder=3)
    for _g in GK:
        _c, _lo, _hi, _n = TRK11[(PF2_BOX, _q, _g)]
        ax.fill_between(TG, _lo, _hi, color=PFC[_g], alpha=0.22, lw=0, zorder=2)
        ax.plot(TG, _c, color=PFC[_g], lw=_fs2(2.5), zorder=4)
    ax.axvline(0.0, color="k", ls="--", lw=_fs2(1.0), alpha=0.65, zorder=1)
    if _q in PF2_HLINE:
        ax.axhline(PF2_HLINE[_q][0], color="0.35", ls=":", lw=_fs2(1.2), zorder=1)
    if PF2_MARKS:
        _F, _gm = _pf2_rows(PF2_BOX)
        for _k11, _ev in enumerate(_MEV11[PF2_CLOCK]):
            _sym, _ls, _ = _MSYM11[_ev]
            for _g in GK:
                _d = np.nanmedian(_F.loc[_gm[_g], _ev].to_numpy(float)
                                  - _F.loc[_gm[_g], f"t_{PF2_CLOCK}"].to_numpy(float))
                if not np.isfinite(_d):
                    continue
                ax.axvline(_d, color=PFC[_g], ls=_ls, lw=_fs2(1.1), alpha=0.5, zorder=1)
                ax.plot([_d], [1.0 + 0.055 * _k11], marker=_sym, color=PFC[_g], ms=_fs2(6.5), mew=_fs2(1.6),
                        transform=ax.get_xaxis_transform(), clip_on=False, zorder=7)
                if _i == 0:      # the numbers behind the marks, once
                    ROW11.append(dict(box=PF2_BOX, quantity="(events)", clock=PF2_CLOCK, grp=_g,
                                      group=GSH.get(_g, _g), t=float(_d), med=np.nan, lo=np.nan, hi=np.nan,
                                      n=int(np.isfinite(_F.loc[_gm[_g], _ev].to_numpy(float)).sum()),
                                      event=_ev))
    ax.set_ylabel(PF2_YLAB.get(_q, P1_LAB.get(_q, _q)), fontsize=_fs2(12.3))
    ax.set_xlim(*(PF2_XLIM if PF2_XLIM else (TG[0], TG[-1])))
    if _i // PF2_NCOL == _nr11 - 1:
        ax.set_xlabel(_CLAB11.get(PF2_CLOCK, PF2_CLOCK))
    if _q in PF2_HLINE:
        ax.text(0.985, PF2_HLINE[_q][0], " " + PF2_HLINE[_q][1], transform=ax.get_yaxis_transform(),
                ha="right", va="bottom", fontsize=_fs2(9.1), color="0.35")
for _i in range(len(PF2_TRK), _nr11 * PF2_NCOL):
    AX[_i // PF2_NCOL, _i % PF2_NCOL].axis("off")
_h11 = [Line2D([], [], color=PFC[g], lw=_fs2(2.6), label=f"{GLAB.get(g, g)}  "
                f"({int((B1[PF2_BOX]['F']['uniq'].to_numpy(bool) & (B1[PF2_BOX]['F']['grp'].to_numpy() == g)).sum())} "
                f"in {BOX_LABEL.get(PF2_BOX, PF2_BOX).split(' ')[0]})") for g in GK]
_h11 += [Line2D([], [], color="0.3", ls="none", marker=_MSYM11[e][0], ms=_fs2(6.5), mew=_fs2(1.6), label=_MSYM11[e][2])
         for e in _MEV11[PF2_CLOCK]]
_h11 += [Line2D([], [], color="0.45", lw=_fs2(1.15), ls=(0, (4, 2)) if k == 0 else (0, (1, 1.6)),
                label=BOX_LABEL.get(b, b).split(" ")[0]) for k, b in enumerate(PF2_THIN)]
_h11 += [Patch(fc="0.7", alpha=0.5, ec="none",
               label=f"in the back: galaxies behind the track (peak {int(_NMAX11)})")]
if PF2_TITLE:
    fig.suptitle(f"The two groups through the truncation in {BOX_LABEL.get(PF2_BOX, PF2_BOX)} — median and "
                 f"bootstrap 16-84 band, one row per galaxy, an epoch drawn only above {PF2_NMIN} galaxies",
                 y=0.995, fontsize=_fs2(14.3))
_leg11 = 0.030 + 0.020 * PF2_FS * (2 if len(_h11) > 4 else 1)        # the room two legend rows need, in figure units
fig.tight_layout(rect=[0, _leg11, 1, 0.975 if PF2_TITLE else 0.995], h_pad=3.2, w_pad=1.2)   # h_pad: the event marks sit above each panel
fig.legend(handles=_h11, loc="lower center", ncol=min(4, len(_h11)), fontsize=_fs2(11.1), frameon=False,
           handlelength=2.4, columnspacing=1.6)
paper_save(fig, PF2_STEM); plt.show()
_ctx11.__exit__(None, None, None)

P11 = pd.DataFrame(ROW11)
P11.to_csv(os.path.join(OUT, f"{PF2_STEM}.csv"), index=False)
print(f"\nTHE TWO TRACKS on the {PF2_CLOCK} clock  —  {GSH['burst']} minus {GSH['aging']}, at "
      + ", ".join(f"{t:+g}" for t in PF2_TSHOW) + " Gyr, and the galaxies behind each:")
for _q in PF2_TRK:
    _line = f"    {P1_LAB.get(_q, _q).split('  [')[0][:34]:36s}"
    for _b in [PF2_BOX] + list(PF2_THIN):
        _d = [np.interp(t, TG, TRK11[(_b, _q, "burst")][0] - TRK11[(_b, _q, "aging")][0],
                        left=np.nan, right=np.nan) for t in PF2_TSHOW]
        _nn = [int(np.nanmax(CNT11[(_b, _q, g)])) for g in GK]
        _line += f"  {_b}: " + " ".join(f"{v:+5.2f}" if np.isfinite(v) else "   -- " for v in _d) + f" [N {_nn[0]}/{_nn[1]}]"
    print(_line)
_F11, _gm11 = _pf2_rows(PF2_BOX)
print(f"  the events, as the median offset from t_{PF2_CLOCK} in {PF2_BOX} "
      f"(what the marks on every panel are):")
for _ev in _MEV11[PF2_CLOCK]:
    print(f"    {_MSYM11[_ev][2]:26s} " + "   ".join(
        f"{GSH.get(g, g)} {np.nanmedian(_F11.loc[_gm11[g], _ev].to_numpy(float) - _F11.loc[_gm11[g], f't_{PF2_CLOCK}'].to_numpy(float)):+.2f} Gyr"
        f" (on {int(np.isfinite(_F11.loc[_gm11[g], _ev].to_numpy(float)).sum())})" for g in GK))
print("table ->", os.path.join(OUT, f"{PF2_STEM}.csv"))

## Part 12 — paper figure 3: the gas at the onset

Part 11 draws six quantities on the truncation clock for the reference box; this cell is the three gas
quantities the argument rests on, **one row per resolution** because $\kappa_{\rm gas}$ is the one quantity
here that a particle count can fake. Each panel is the **median track of each group on the clock with its
bootstrap 16–84 band** (`med_band`, Part 0), and on it the **critical points**: the jet activation, the
quenching onset and the observation (`PF3_EV`), each drawn at the group's median offset from the clock and at
the median of the quantity **read at that event** (Part 1's `<q>_jet`, `<q>_sft`, `<q>_anc` — a snapshot
within `P1_TOL` of the event or nothing), with the 16–84 range of that read as a bar. A marker is not a point
of the line — a median at a fixed event is a different average from one at a fixed clock step, Part 3b's
convention — so both are drawn and both are printed. `PF3_STYLE = "violins"` gives back the earlier version,
the distributions at the events alone.

Every separation is scored twice in the printout and the CSV — the raw AUC toward the dusty group, and the
same statistic on the value **ranked inside (epoch × mass) cells** (`_auc_ctrl`, Part 0), which is the number
to quote: the two groups differ in cosmic time and in stellar mass whatever they are divided on.

**Knobs.** `PF3_STYLE` (`"tracks"` | `"violins"`), `PF3_Q` (the quantities, any of `P1_TRK`), `PF3_CLOCK`
(tracks: the clock, any of `P1_CLOCKS`), `PF3_EV` (the events and their labels), `PF3_MARK` (one marker per
event), `PF3_TLIM` (the stretch of the clock drawn), `PF3_XERR` (a horizontal bar = the 16–84 spread of the
event's own offset), `PF3_NBOOT`, `PF3_LEAD` (the lead-time panel; `None` drops it), `PF3_LEADLOG`,
`PF3_BOXES`, `PF3_YLIM` (violins: the percentile range of each axis), `PF3_HLINE` (the reference lines),
`PF3_NMIN`, `PF3_AUC`.

In [ ]:
# ── Part 12 — PAPER FIGURE 3: the gas at the ONSET — the two groups' whole tracks through the jet activation
#    and the truncation, with the critical points on them ─────────────────────────────────────────────────
# Part 11 draws six quantities on the truncation clock for the reference box; this cell is the three GAS
# quantities the argument rests on, one row per resolution because kappa_gas is the one quantity here that a
# particle count can fake.  Each panel is the MEDIAN TRACK of each group on the clock with its bootstrap
# 16-84 band (med_band, Part 0), and ON it the CRITICAL POINTS: every event of PF3_EV drawn at the group's
# median offset from the clock, at the median of the quantity READ AT that event (Part 1's <q>_jet, _qt,
# _anc — a snapshot within P1_TOL of the event or nothing; the clock's own event is the dashed line), with the 16-84 range of that read as a bar.  The
# markers are NOT points of the line — a median at a fixed event is a different average from one at a fixed
# clock step (Part 3b's convention) — which is why both are drawn and both are printed.
#   PF3_STYLE = "violins" gives back the earlier version: the distributions at the events alone, one violin
#   per (event, group), no clock.
# The separation is scored twice in the PRINTOUT and the CSV — raw, and ranked inside (epoch x mass) cells,
# the notebook's convention — and is not written on the figure (PF3_AUC puts it back).  The lead time of the
# jet lives in Part 14's ladder (PF3_LEAD puts it back here, as a further panel in either style).
PF3_STEM  = "ba_paper_onset"
PF3_STYLE = "tracks"        # "tracks" = the median track + band on the clock with the critical points on it;
                            # "violins" = the distributions at the events alone
PF3_Q     = ["kappa_gas", "lsig_h2", "comp"]     # any of P1_TRK: tracked on the clock AND read at the events
PF3_CLOCK = "sft"           # tracks: the clock (any of P1_CLOCKS) — the truncation; the events sit at each group's
                            # MEDIAN offset from it, and the clock's own event is the dashed line, not a marker
PF3_EV    = [("jet", "$t_{\\rm jet}$\njet\nactivation"), ("qt", "$t_{q,\\,\\rm end}$\nquenching\nend"),
             ("anc", "$t_{\\rm obs}$\nobserved")]     # the critical points (tracks: markers; violins: columns).
                            # The event keys are P1_EV's: jet, sft, qt, x1, anc ("tq" is not one)
PF3_EVLAB = {"jet": "$t_{\\rm jet}$\njet\nactivation", "sft": "$t_{q,\\,\\rm start}$\nquenching\nonset",
             "qt": "$t_{q,\\,\\rm end}$\nquenching\nend", "x1": "$t_{\\rm X\\!-\\!ray}$\nfirst X-ray\nfiring",
             "anc": "$t_{\\rm obs}$\nobserved"}      # the label of every event Part 12b may ask for (PF3_EV's win)
PF3_MARK  = {"jet": "v", "sft": "s", "anc": "o", "x1": "P", "qt": "D"}   # tracks: one marker per event
PF3_TSTEP = None            # tracks: the clock step [Gyr] (None = Part 0's TG step).  The tracks are re-gridded from
                            # the raw snapshot series onto PF3_TLIM, so the range may reach beyond TG (-3 .. +4)
PF3_TLIM  = (-4.0, 4.0)     # tracks: the stretch of the clock drawn [Gyr] (the dusty group's jet is 2.3-2.9 Gyr
                            # before the truncation); a marker past it is not drawn and the printout says so
PF3_NAMES = False           # the run's real name at the left of each row (the rows are in BOXES order regardless)
PF3_STRIP = ["anc"]         # tracks: the events of PF3_EV drawn in a NARROW STRIP beside the clock instead of on
                            # it — the observation has no clock position (the anchors spread over gigayears, the
                            # dusty group's median sits past the end of the clock), so it is read the way the
                            # violins read it: the two groups side by side, at no time.  [] puts every event on
                            # the clock.
PF3_STRIPW = 0.16           # tracks: the width of that strip, as a fraction of the panel
PF3_HLPOS = {"kappa_gas": "left", "comp": "left-below"}   # tracks: where each reference-line label sits —
                            # "left" | "right", with "-below" to hang it under the line (default above it); a
                            # plain string applies to every quantity
PF3_XERR  = False           # tracks: a thin horizontal bar through each marker = the 16-84 spread of the
                            # event's own offset on the clock (the median offset is where the marker sits)
PF3_NBOOT = None            # tracks: the bootstrap behind the band (None = NBOOT)
PF3_SEP   = ["anc"]         # violins: a vertical line is drawn BEFORE each of these events
PF3_LEAD  = None            # ("lag_sft", label) adds the lead-time panel back; it is a rung of Part 14 instead
PF3_AUC   = False           # write the AUCs in the panels (they are always printed and in the CSV)
PF3_BOXES = [b for b in BOXES if b in _BX1]      # one row per resolution
PF3_UNIQ  = True
PF3_NMIN  = 5               # a violin, a marker, or a clock step of a track needs this many galaxies
PF3_YLIM  = (1.0, 99.0)     # violins: the percentile range of each panel's axis, over the events and groups
PF3_LAB   = {"kappa_gas": "gas rotational support\n" + r"$\kappa_{\rm gas}$",
             "lsig_h2": "H$_2$ surface density\n" + r"$\log \Sigma_{\rm H_2}$  [M$_\odot$ kpc$^{-2}$]",
             "comp": "gas compactness\n" + r"$-\log R_{50,\rm gas}/R_{50,\star}$",
             "lsig_gas": "gas surface density\n" + r"$\log \Sigma_{\rm gas}$",
             "lfgas": "gas fraction\n" + r"$\log M_{\rm gas}/M_\star$"}
PF3_HLINE = {"kappa_gas": (KAPPA_ISO, r"isotropic, $\kappa = 1/3$"), "comp": (0.0, r"$R_{50,\rm gas} = R_{50,\star}$")}
PF3_LEADLOG = False
_CLAB12 = {"sft": r"$t - t_{q,\,\rm start}$  [Gyr]", "jet": r"$t - t_{\rm jet}$  [Gyr]",
           "x1": r"$t - t_{\rm x1}$  [Gyr]", "qt": r"$t - t_{q,\,\rm end}$  [Gyr]"}   # the names the legend uses
from matplotlib.patches import Patch
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
_MISS12 = [q for q in PF3_Q if q not in P1_TRK]
if _MISS12:
    raise KeyError(f"PF3_Q names {_MISS12}, which Part 1 does not track.  P1_TRK = {P1_TRK}")
if PF3_STYLE not in ("tracks", "violins"):
    raise ValueError(f"PF3_STYLE = {PF3_STYLE!r}: 'tracks' or 'violins'")
if PF3_STYLE == "tracks" and PF3_CLOCK not in P1_CLOCKS:
    raise KeyError(f"PF3_CLOCK = {PF3_CLOCK!r} is not one of the clocks Part 1 gridded: {P1_CLOCKS}")
_EV12 = [e for e, _ in PF3_EV]
PF3_EVLAB = {**PF3_EVLAB, **dict(PF3_EV)}         # PF3_EV's own labels win
_EVL12 = {e: (l.split("\n")[0] + (", " + " ".join(l.split("\n")[1:]) if "\n" in l else "")) for e, l in PF3_EVLAB.items()}
_EVTXT12 = {"jet": "t_jet", "sft": "t_q,start", "anc": "t_obs", "x1": "t_x1", "qt": "t_q,end"}   # plain text
_INS12 = [e for e in _EV12 if e in PF3_STRIP] if PF3_STYLE == "tracks" else []   # the events in the strip
_ONC12 = [e for e in _EV12 if e not in _INS12]                                    # the events on the clock


def _pf3_rows(box):
    F = B1[box]["F"]
    keep = F["uniq"].to_numpy(bool) if PF3_UNIQ else np.ones(len(F), bool)
    return F, {g: keep & (F["grp"].to_numpy() == g) for g in GK}


def _pf3_violin(ax, x, v, color, w=0.38):
    """One half-transparent violin with its 16-84 bar and median dot; returns the count drawn."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < PF3_NMIN or np.std(v) <= 0:
        return len(v)
    parts = ax.violinplot([v], positions=[x], widths=w, showextrema=False, showmedians=False)
    for b in parts["bodies"]:
        b.set_facecolor(color); b.set_edgecolor(color); b.set_alpha(0.32); b.set_linewidth(0.8)
    q1, q2, q3 = np.percentile(v, [16, 50, 84])
    ax.plot([x, x], [q1, q3], color=color, lw=3.4, solid_capstyle="round", zorder=4)
    ax.plot([x], [q2], marker="o", ms=6.5, mfc="w", mec=color, mew=1.9, ls="none", zorder=5)
    return len(v)


def _pf3_score(F, both, m1, col):
    """(AUC raw, p, AUC at fixed epoch + mass, p) of one column toward the dusty group."""
    D = F.loc[both, ["t_anchor", "log_mstar", col]]
    a, p, n1, n0 = mw_auc(D[col].to_numpy(float), m1[both])
    ac, pc, _, _ = _auc_ctrl(D, col, m1[both])
    return a, p, ac, pc, int(n1 + n0)


def _pf3_point(F, m, q, ev, t0, floor=None, nmin=None):
    """One critical point of one group: (the median offset of the event on the clock, its 16-84, the median of
    q read AT the event, its 16-84, the count) — over the galaxies that have BOTH the event and the read.  A
    value at or below `floor` is censoring (a storage floor), not a value, and is left out."""
    nmin = PF3_NMIN if nmin is None else nmin
    dt = (F[P1_EV[ev]].to_numpy(float) - t0)[m]
    v = F[f"{q}_{ev}"].to_numpy(float)[m]
    ok = np.isfinite(dt) & np.isfinite(v) & ((v > floor) if floor is not None else True)
    if ok.sum() < nmin:
        return (np.nan,) * 6 + (int(ok.sum()),)
    x1, x2, x3 = np.percentile(dt[ok], [16, 50, 84])
    y1, y2, y3 = np.percentile(v[ok], [16, 50, 84])
    return x2, x1, x3, y2, y1, y3, int(ok.sum())


def _pf3_stack(box, q, m, t0, grid):
    """The (n_gal, n_t) stack of q for the rows m of the box, re-gridded from the RAW snapshot series onto `grid`
    around each galaxy's own clock zero t0 — the same interpolation Part 1 used for TG, on any range."""
    F, R = B1[box]["F"], B1[box]["R"]
    ri = F["ridx"].to_numpy(int)
    rows = np.flatnonzero(np.asarray(m, bool))
    A = np.full((len(rows), len(grid)), np.nan)
    for i, r in enumerate(rows):
        if np.isfinite(t0[r]) and ri[r] < len(R):
            S = R[ri[r]]
            A[i] = to_grid(np.asarray(S["t"], float) - t0[r], np.asarray(S[q], float), grid=grid)
    return A


def _pf3_track_panel(ax, axs, box, q, F, gm, t0, floor=None, nmin=None, nboot=None, hline=None, ylabel=None,
                     xlabel=False, striplab=False, clock=None, events=None, strip=None, tlim=None, tstep=None):
    """ONE PANEL of the tracks style (Parts 12 and 12b): each group's median track + bootstrap 16-84 band of q
    on the clock whose zero is t0 (per galaxy), over the galaxies of gm; the on-clock `events` as markers at
    the group's median offset (at the median read AT the event, 16-84 bar), and the `strip` events in `axs`
    (None = no strip).  The tracks are re-gridded from the raw series onto `tlim` at `tstep`, so the range may
    reach beyond TG.  A value at or below `floor` is censoring and is masked out of the track and the reads.
    `clock` / `events` / `strip` / `tlim` / `tstep` default to Part 12's own (PF3_CLOCK, PF3_EV, PF3_STRIP,
    PF3_TLIM, TG's step).  Returns (the track rows for the CSV, the markers that fell outside tlim)."""
    nmin = PF3_NMIN if nmin is None else nmin
    clock = PF3_CLOCK if clock is None else clock
    events = _EV12 if events is None else list(events)
    tlim = PF3_TLIM if tlim is None else tuple(tlim)
    tstep = float(TG[1] - TG[0]) if (tstep is None and PF3_TSTEP is None) else float(PF3_TSTEP if tstep is None else tstep)
    ins = ([e for e in events if e in (PF3_STRIP if strip is None else strip)]) if axs is not None else []
    G = np.arange(tlim[0], tlim[1] + 1e-9, tstep)
    rows, off, ys = [], [], []
    for g in GK:
        A = _pf3_stack(box, q, gm[g], t0, G)
        if floor is not None:
            A = np.where(A > floor, A, np.nan)
        cen, lo, hi, n = med_band(A, nmin=nmin, nboot=nboot, grid=G)
        s = np.isfinite(cen)
        if s.any():
            ax.fill_between(G[s], lo[s], hi[s], color=PFC[g], alpha=0.22, lw=0, zorder=2)
            ax.plot(G[s], cen[s], color=PFC[g], lw=2.8, zorder=4)
            ys += [lo[s], hi[s]]
        rows += [dict(box=box, quantity=q, clock=clock, grp=g, group=GSH.get(g, g), t=float(t), med=float(y),
                      lo=float(l), hi=float(h), n=int(nn)) for t, y, l, h, nn in zip(G, cen, lo, hi, n) if np.isfinite(y)]
        for e in events:                   # the critical points
            x2, x1, x3, y2, y1, y3, _ = _pf3_point(F, gm[g], q, e, t0, floor=floor, nmin=nmin)
            if not np.isfinite(x2):
                continue
            if e in ins:                   # in the strip: side by side, at no clock position
                xs = (-0.24, 0.24)[GK.index(g)]
                axs.plot([xs, xs], [y1, y3], color=PFC[g], lw=3.2, solid_capstyle="round", zorder=6)
                axs.plot([xs], [y2], marker=PF3_MARK.get(e, "o"), ms=12, mfc="w", mec=PFC[g], mew=2.4, ls="none", zorder=7)
                ys.append(np.array([y1, y3]))
                continue
            if not (tlim[0] <= x2 <= tlim[1]):
                off.append((box, q, g, e, x2))
                continue
            ax.plot([x2, x2], [y1, y3], color=PFC[g], lw=3.2, solid_capstyle="round", zorder=6)
            if PF3_XERR:
                ax.plot([max(x1, tlim[0]), min(x3, tlim[1])], [y2, y2], color=PFC[g], lw=1.3, alpha=0.75, zorder=5)
            ax.plot([x2], [y2], marker=PF3_MARK.get(e, "o"), ms=12, mfc="w", mec=PFC[g], mew=2.4, ls="none", zorder=7)
            ys.append(np.array([y1, y3]))
    ax.axvline(0.0, color="k", ls="--", lw=1.1, alpha=0.6, zorder=1)
    pool = np.concatenate([np.asarray(y, float).ravel() for y in ys]) if ys else np.array([])
    pool = pool[np.isfinite(pool)]
    if len(pool):
        lo12, hi12 = float(pool.min()), float(pool.max())
        pad = 0.12 * (hi12 - lo12)
        ax.set_ylim(lo12 - pad, hi12 + (1.6 if PF3_AUC else 1.0) * pad)
    if hline is not None:
        ax.axhline(hline[0], color="0.4", ls=":", lw=1.4, zorder=1)
        hp = (PF3_HLPOS.get(q, "left") if isinstance(PF3_HLPOS, dict) else str(PF3_HLPOS)).lower()
        ax.text(0.03 if hp.startswith("left") else 0.97, hline[0], hline[1], transform=ax.get_yaxis_transform(),
                ha="left" if hp.startswith("left") else "right", va="top" if hp.endswith("below") else "bottom",
                fontsize=19.5, color="0.25", bbox=dict(fc="w", ec="none", alpha=1.0, pad=0.8), zorder=8)
    ax.set_xlim(*tlim)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(_CLAB12.get(clock, clock))
    else:
        ax.set_xticklabels([])
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    if axs is not None:                    # the strip: the panel's y range, no y axis, a dashed wall on its left
        axs.set_ylim(*ax.get_ylim()); axs.set_xlim(-0.6, 0.6)
        axs.set_yticks([]); axs.set_xticks([0.0])
        axs.set_xticklabels(["\n".join(PF3_EVLAB.get(e, e) for e in ins)] if striplab else [""])
        axs.tick_params(axis="x", length=0)
        if hline is not None:
            axs.axhline(hline[0], color="0.4", ls=":", lw=1.4, zorder=1)
        axs.axvline(-0.58, color="0.55", ls="--", lw=1.3, zorder=1)
        for sp in ("top", "right", "left"):
            axs.spines[sp].set_visible(False)
    return rows, off


def _pf3_auc_text(ax, axs, rows, box, q, events=None, strip=None):
    """PF3_AUC: the AUCs written in the panel — one line per on-clock event, stacked in the corner, and the strip
    events' at the top of the strip (a diagnostic; off in the paper version)."""
    ln = []
    events = _EV12 if events is None else list(events)
    strip = _INS12 if strip is None else list(strip)
    for e in events:
        r = [x for x in rows if x["box"] == box and x["quantity"] == q and x["event"] == e and x["grp"] == "auc"]
        if not r or not np.isfinite(r[0]["auc"]):
            continue
        if e in strip and axs is not None:
            axs.text(0.5, 0.985, f"{r[0]['auc']:.2f} | {r[0]['auc_ctrl']:.2f}", transform=axs.transAxes, ha="center",
                     va="top", fontsize=13, color="0.3", bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.0))
        else:
            ln.append(f"{_EVTXT12.get(e, e)}: AUC {r[0]['auc']:.2f} | {r[0]['auc_ctrl']:.2f}")
    if ln:
        ax.text(0.02, 0.985, "\n".join(ln), transform=ax.transAxes, ha="left", va="top", fontsize=12.5, color="0.3",
                bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.0), zorder=9)


# ── THE STATISTICS AT THE EVENTS, whatever is drawn: median [16-84] per group, the AUCs, the offsets ──────
ROW12, ROWT12 = [], []
_QALL12 = PF3_Q + ([PF3_LEAD[0]] if PF3_LEAD else [])
_TCOL12 = P1_EV[PF3_CLOCK] if PF3_STYLE == "tracks" else None
for _b in PF3_BOXES:
    _F, _gm = _pf3_rows(_b)
    _both = _gm["burst"] | _gm["aging"]
    _t0 = _F[_TCOL12].to_numpy(float) if _TCOL12 else np.zeros(len(_F))
    for _q in PF3_Q:
        for _e in _EV12:
            _col = f"{_q}_{_e}"
            _v = _F[_col].to_numpy(float)
            for _g in GK:
                _vv = _v[_gm[_g]]; _vv = _vv[np.isfinite(_vv)]
                _x2, _x1, _x3, _, _, _, _np = _pf3_point(_F, _gm[_g], _q, _e, _t0)
                ROW12.append(dict(box=_b, quantity=_q, event=_e, grp=_g, group=GSH.get(_g, _g), n=int(len(_vv)),
                                  p16=float(np.percentile(_vv, 16)) if len(_vv) else np.nan,
                                  med=float(np.median(_vv)) if len(_vv) else np.nan,
                                  p84=float(np.percentile(_vv, 84)) if len(_vv) else np.nan,
                                  clock=PF3_CLOCK if _TCOL12 else "-", n_point=_np,
                                  dt_med=_x2, dt_p16=_x1, dt_p84=_x3))
            _a, _p, _ac, _pc, _n = _pf3_score(_F, _both, _gm["burst"], _col)
            ROW12.append(dict(box=_b, quantity=_q, event=_e, grp="auc", group="AUC dusty vs dust-poor", n=_n,
                              auc=_a, p=_p, auc_ctrl=_ac, p_ctrl=_pc))
    if PF3_LEAD:
        _lc = PF3_LEAD[0]
        _v = _F[_lc].to_numpy(float)
        for _g in GK:
            _vv = _v[_gm[_g]]
            if PF3_LEADLOG:
                _vv = np.log10(np.where(_vv > 0, _vv, np.nan))
            _f = _vv[np.isfinite(_vv)]
            ROW12.append(dict(box=_b, quantity=_lc, event="-", grp=_g, group=GSH.get(_g, _g), n=int(len(_f)),
                              p16=float(np.percentile(_f, 16)) if len(_f) else np.nan,
                              med=float(np.median(_f)) if len(_f) else np.nan,
                              p84=float(np.percentile(_f, 84)) if len(_f) else np.nan))
        _a, _p, _ac, _pc, _n = _pf3_score(_F, _both, _gm["burst"], _lc)
        ROW12.append(dict(box=_b, quantity=_lc, event="-", grp="auc", group="AUC dusty vs dust-poor", n=_n,
                          auc=_a, p=_p, auc_ctrl=_ac, p_ctrl=_pc))


def _pf3_lead_panel(ax, F, gm):
    """The lead-time panel, the same in either style: one violin per group."""
    _lc, _ll = PF3_LEAD
    _v = F[_lc].to_numpy(float)
    for _g, _x in zip(GK, (0.0, 1.0)):
        _vv = _v[gm[_g]]
        if PF3_LEADLOG:
            _vv = np.log10(np.where(_vv > 0, _vv, np.nan))
        _pf3_violin(ax, _x, _vv, PFC[_g], w=0.6)
    ax.axhline(0.0, color="0.4", ls=":", lw=1.4)
    ax.set_xticks([0, 1]); ax.set_xticklabels([GSH.get(g, g).replace(" ", "\n") for g in GK]); ax.set_xlim(-0.6, 1.6)
    ax.set_ylabel(("log " if PF3_LEADLOG else "") + _ll)
    ax.set_axisbelow(True)


def _pf3_boxlabels(fig, AX):
    """The run's real name at the left of each row, in figure coordinates (placed after the layout); PF3_NAMES."""
    if not PF3_NAMES:
        return
    for _r, _b in enumerate(PF3_BOXES):
        _pos = AX[_r, 0].get_position()
        fig.text(0.014, 0.5 * (_pos.y0 + _pos.y1), BOX_LABEL.get(_b, _b), rotation=90, va="center", ha="center",
                 fontsize=27.3, fontweight="bold", color="0.2")


_nc12 = len(PF3_Q) + (1 if PF3_LEAD else 0)
_OFF12 = []          # the markers that fell outside PF3_TLIM, for the printout
with plt.rc_context({**PAPER_RC, "axes.labelsize": 23, "xtick.labelsize": 20, "ytick.labelsize": 20,
                     "legend.fontsize": 20}):
    if PF3_STYLE == "tracks":
        # ── THE TRACKS: one panel per (box, quantity) through _pf3_track_panel, the strips as insets ─────
        fig, AX = plt.subplots(len(PF3_BOXES), _nc12, figsize=((7.6 if _INS12 else 7.0) * _nc12 + 0.6,
                                                                 4.6 * len(PF3_BOXES) + 1.6), squeeze=False)
        for _r, _b in enumerate(PF3_BOXES):
            _F, _gm = _pf3_rows(_b)
            _t0 = _F[_TCOL12].to_numpy(float)
            _last = _r == len(PF3_BOXES) - 1
            for _c, _q in enumerate(PF3_Q):
                ax = AX[_r, _c]
                # the strip hangs off the panel's right edge, in the panel's own coordinates
                axs = ax.inset_axes([1.07, 0.0, PF3_STRIPW, 1.0]) if _INS12 else None
                _rows, _off = _pf3_track_panel(ax, axs, _b, _q, _F, _gm, _t0, nboot=PF3_NBOOT, hline=PF3_HLINE.get(_q),
                                               ylabel=PF3_LAB.get(_q, P1_LAB.get(_q, _q)), xlabel=_last, striplab=_last)
                ROWT12 += _rows; _OFF12 += _off
                if PF3_AUC:
                    _pf3_auc_text(ax, axs, ROW12, _b, _q)
            if PF3_LEAD:
                _pf3_lead_panel(AX[_r, -1], _F, _gm)
        # tight_layout does not see the insets: the column pad leaves room for the strip when there is one
        fig.tight_layout(rect=[0.045 if PF3_NAMES else 0.0, 0.10, 1, 0.995], h_pad=1.6, w_pad=(5.8 if _INS12 else 2.4))
        _pf3_boxlabels(fig, AX)
        _h12 = [Patch(fc=PFC[g], alpha=0.5, ec=PFC[g], label=GLAB.get(g, g)) for g in GK]
        _h12 += [Line2D([], [], color="0.3", lw=2.8, label="median track, bootstrap 16-84 band")]
        if PF3_CLOCK not in _EV12:
            _h12 += [Line2D([], [], color="k", ls="--", lw=1.4, label=_EVL12.get(PF3_CLOCK, PF3_CLOCK) + " (the line)")]
        _h12 += [Line2D([], [], color="0.3", marker=PF3_MARK.get(e, "o"), mfc="w", mew=2.2, ms=11, lw=3.2,
                        label=_EVL12.get(e, e)) for e in _EV12]
        fig.legend(handles=_h12, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0),
                   title="markers: the median value read at the event, and its 16-84 range", title_fontsize=18)
    else:
        # ── THE VIOLINS: the distributions at the events alone ───────────────────────────────────────────
        fig, AX = plt.subplots(len(PF3_BOXES), _nc12, figsize=(7.8 * _nc12 + 0.6, 4.6 * len(PF3_BOXES) + 1.4),
                               squeeze=False)
        for _r, _b in enumerate(PF3_BOXES):
            _F, _gm = _pf3_rows(_b)
            for _c, _q in enumerate(PF3_Q):
                ax = AX[_r, _c]
                _pool = np.concatenate([_F[f"{_q}_{e}"].to_numpy(float)[_gm["burst"] | _gm["aging"]] for e in _EV12])
                _pool = _pool[np.isfinite(_pool)]
                for _k, _e in enumerate(_EV12):
                    _v = _F[f"{_q}_{_e}"].to_numpy(float)
                    for _g, _dx in zip(GK, (-0.21, 0.21)):
                        _pf3_violin(ax, _k + _dx, _v[_gm[_g]], PFC[_g])
                    _r12 = [r for r in ROW12 if r["box"] == _b and r["quantity"] == _q and r["event"] == _e
                            and r["grp"] == "auc"]
                    if PF3_AUC and _r12 and np.isfinite(_r12[0]["auc"]):
                        ax.text(_k, 0.985, f"AUC {_r12[0]['auc']:.2f} | {_r12[0]['auc_ctrl']:.2f}",
                                transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=13, color="0.3",
                                bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.0))
                for _sev in PF3_SEP:
                    if _sev in _EV12:
                        ax.axvline(_EV12.index(_sev) - 0.5, color="0.55", ls="--", lw=1.3, zorder=1)
                if _q in PF3_HLINE:
                    ax.axhline(PF3_HLINE[_q][0], color="0.4", ls=":", lw=1.4, zorder=1)
                    ax.text(0.96, PF3_HLINE[_q][0], PF3_HLINE[_q][1], transform=ax.get_yaxis_transform(),
                            ha="right", va="bottom", fontsize=19.5, color="0.25",
                            bbox=dict(fc="w", ec="none", alpha=1.0, pad=0.8), zorder=6)
                if len(_pool):
                    _lo12, _hi12 = np.percentile(_pool, PF3_YLIM)
                    _pad = 0.14 * (_hi12 - _lo12)
                    ax.set_ylim(_lo12 - _pad, _hi12 + (1.6 if PF3_AUC else 1.0) * _pad)
                ax.set_xticks(range(len(PF3_EV)))
                ax.set_xticklabels([l for _, l in PF3_EV] if _r == len(PF3_BOXES) - 1 else [""] * len(PF3_EV))
                ax.set_xlim(-0.65, len(PF3_EV) - 0.35)
                ax.set_ylabel(PF3_LAB.get(_q, P1_LAB.get(_q, _q)))
                ax.set_axisbelow(True)
                for _s in ("top", "right"):
                    ax.spines[_s].set_visible(False)
            if PF3_LEAD:
                _pf3_lead_panel(AX[_r, -1], _F, _gm)
        fig.tight_layout(rect=[0.045 if PF3_NAMES else 0.0, 0.05, 1, 0.995], h_pad=1.8, w_pad=2.2)
        _pf3_boxlabels(fig, AX)
        fig.legend(handles=[Patch(fc=PFC[g], alpha=0.5, ec=PFC[g], label=GLAB.get(g, g)) for g in GK]
                   + [Line2D([], [], color="0.3", marker="o", mfc="w", ms=8, lw=3.8, label="median and 16-84 range")],
                   loc="lower center", ncol=3, frameon=False)
    paper_save(fig, PF3_STEM); plt.show()

P12 = pd.DataFrame(ROW12)
P12.to_csv(os.path.join(OUT, f"{PF3_STEM}.csv"), index=False)
if ROWT12:
    pd.DataFrame(ROWT12).to_csv(os.path.join(OUT, f"{PF3_STEM}_tracks.csv"), index=False)
print(f"THE GAS AT THE ONSET  —  median [16-84] per group READ AT EACH EVENT, and the AUC toward the {GSH['burst']} "
      f"(raw | at fixed epoch + mass):")
for _b in PF3_BOXES:
    print(f"  {BOX_LABEL.get(_b, _b)}")
    for _q in _QALL12:
        for _e in (_EV12 if _q in PF3_Q else ["-"]):
            _s = P12[(P12["box"] == _b) & (P12["quantity"] == _q) & (P12["event"] == _e)]
            _r = {r["grp"]: r for _, r in _s.iterrows()}
            if "auc" not in _r:
                continue
            print(f"    {_q:10s} at {_e:4s}  "
                  + "  ".join(f"{GSH.get(g, g)} {_r[g]['med']:+6.2f} [{_r[g]['p16']:+.2f},{_r[g]['p84']:+.2f}] (N {int(_r[g]['n'])})"
                              for g in GK if g in _r)
                  + f"   AUC {_r['auc']['auc']:.2f} | {_r['auc']['auc_ctrl']:.2f}")
if PF3_STYLE == "tracks":
    print(f"  the tracks are on the {PF3_CLOCK} clock ({_CLAB12.get(PF3_CLOCK, PF3_CLOCK).split('  [')[0]}), drawn over "
          f"{PF3_TLIM[0]:+g} to {PF3_TLIM[1]:+g} Gyr where at least {PF3_NMIN} galaxies contribute "
          f"(bootstrap {NBOOT if PF3_NBOOT is None else PF3_NBOOT}); the markers sit at each group's MEDIAN offset of the event from it:")
    for _b in PF3_BOXES:
        _s = P12[(P12["box"] == _b) & (P12["quantity"] == PF3_Q[0]) & (P12["grp"] != "auc")]
        print(f"    {_b:9s} " + "   ".join(
            f"{_EVTXT12.get(e, e)} {GSH.get(g, g)} "
            f"{_s[(_s['event'] == e) & (_s['grp'] == g)]['dt_med'].iloc[0]:+.2f} "
            f"[{_s[(_s['event'] == e) & (_s['grp'] == g)]['dt_p16'].iloc[0]:+.2f},{_s[(_s['event'] == e) & (_s['grp'] == g)]['dt_p84'].iloc[0]:+.2f}] Gyr"
            + (" (in the strip)" if e in PF3_STRIP else "")
            for e in _EV12 for g in GK if len(_s[(_s["event"] == e) & (_s["grp"] == g)])))
    if _OFF12:
        print(f"  !! {len(_OFF12)} marker(s) fell outside PF3_TLIM and are not drawn: "
              + "; ".join(f"{b} {q} {GSH.get(g, g)} {e} at {x:+.2f} Gyr" for b, q, g, e, x in _OFF12[:8])
              + (" ..." if len(_OFF12) > 8 else "") + "  (widen PF3_TLIM to see them)")
    print("  a marker is the median of the quantity read AT the event — not a point of the median track, which is the\n"
          "  median at a fixed clock step over whoever is there; the two agree where the event's offsets are narrow.")
print("  (an AUC of 0.5 is no separation; 0.98 means almost every dusty galaxy is above almost every dust-poor one.\n"
      "   The second number ranks the quantity inside terciles of anchor epoch and stellar mass first.  Not on the figure.)")
print("table ->", os.path.join(OUT, f"{PF3_STEM}.csv")
      + (f"  and the tracks -> {os.path.join(OUT, f'{PF3_STEM}_tracks.csv')}" if ROWT12 else ""))

## Part 12b — paper figure 3b: the onset in the 25 Mpc box alone

Part 12's high-resolution companion: **m25n512 only**, one panel per quantity — the three gas quantities of
Part 12 plus the **sSFR**, the **stellar mass** and the **dust-to-metal ratio**. It follows Part 12's style
(`PF3B_STYLE = None`): as **tracks**, each panel is the median track of each group with its bootstrap 16–84
band and the critical points on it, the observation in the strip beside the clock (Part 12's own panel
helper, so the two figures cannot disagree); as **violins**, the distributions at the events, the last set
apart. The clock, the events and the range are its own: the tracks are aligned at the **truncation**
(`PF3B_CLOCK = "sft"`, the dashed line), the markers are the **jet activation** and the **end of the quenching
transition** (`PF3B_EV`; the truncation itself is the line, not a marker), no reference lines (`PF3B_HLINE`),
and the range runs 6 Gyr back (`PF3B_TLIM`) so the dusty group's jet, a median 2.9 Gyr before the truncation
in this box, is on the figure. The tracks are re-gridded from the raw snapshot series, so the range is not
bound to Part 0's clock `TG`. A value sitting on a storage floor (the sSFR of a galaxy with no star formation,
`SSFR_FLOOR`) is censoring, not a value: it is kept out of the track, the violin and the event reads, and its
count is printed. It reuses Part 12's helpers and its event list, so Part 12 has to have run.

**Knobs.** `PF3B_STYLE` (`None` = follow `PF3_STYLE`), `PF3B_CLOCK`, `PF3B_EV`, `PF3B_STRIP`, `PF3B_TLIM`,
`PF3B_TSTEP` (each `None` = Part 12's), `PF3B_HLINE` (`dict(PF3_HLINE)` puts Part 12's lines back), `PF3B_Q`,
`PF3B_BOX`, `PF3B_NCOL`, `PF3B_LAB`, `PF3B_FLOORS`, `PF3B_NMIN` (3 — the box has 14 and 20 galaxies),
`PF3B_NBOOT`; the markers and the strip width are Part 12's (`PF3_MARK`, `PF3_STRIPW`).

The legend names the events by symbol alone (`PF3B_EVSHORT`) under one title, `PF3B_LEGTITLE` = *markers: median + 16th to
84th percentile* (2026-09-19); the clock is $t - t_{q,\,\rm start}$, the quenching onset.


In [ ]:
# ── Part 12b — PAPER FIGURE 3b: the onset in the 25 Mpc box alone, six quantities ────────────────────────
# Part 12 over the resolutions; this is its high-resolution companion: m25n512 only, one panel per quantity,
# the three gas quantities of Part 12 plus the sSFR, the stellar mass and the dust-to-metal ratio.  It follows
# Part 12's STYLE (PF3B_STYLE = None): as TRACKS each panel is the median track + bootstrap 16-84 band of
# each group with the critical points on it and the observation in the strip beside the clock (Part 12's
# _pf3_track_panel, verbatim); as VIOLINS it is the distributions at the events, the last set apart.  THE CLOCK,
# THE EVENTS AND THE RANGE ARE ITS OWN: by default the tracks are aligned at the TRUNCATION (PF3B_CLOCK = "sft",
# the dashed line), the markers are the jet activation and the end of the quenching transition (PF3B_EV; the
# truncation itself is the line, not a marker), and the range runs 6 Gyr back so the dusty group's jet — a
# median 2.9 Gyr before the truncation in this box — is on the figure (PF3B_TLIM; the tracks are re-gridded from
# the raw series, so the range is not bound to Part 0's TG).  A value sitting on a storage floor (PF3B_FLOORS:
# the sSFR of a galaxy with no star formation) is censoring, not a value: it is kept out of the track, the
# violin and the event reads, and its count is printed.  It reuses Part 12's helpers, so Part 12 has to have run.
PF3B_STEM  = "ba_paper_onset_m25"
PF3B_STYLE = None           # None = Part 12's PF3_STYLE; or "tracks" | "violins" on its own
PF3B_BOX   = "cis25"
PF3B_CLOCK = "sft"          # the clock the m25 tracks are aligned on (any of P1_CLOCKS); None = Part 12's
PF3B_EV    = ["jet", "qt", "anc"]   # the critical points drawn (keys of P1_EV / PF3_EVLAB); None = Part 12's.
                            # The clock's own event needs no marker: it is the vertical line at 0
PF3B_STRIP = ["anc"]        # of those, the ones drawn in the strip beside the clock (no clock position)
PF3B_EVSHORT = True         # THE LEGEND NAMES THE EVENTS BY SYMBOL ONLY (2026-09-19): t_jet, t_q,end, t_obs — not
                            #   "t_jet, jet activation".  False = Part 12's full descriptions.
PF3B_LEGTITLE = "markers: median + 16th to 84th percentile"    # the legend's title
PF3B_TLIM  = (-6.0, 4.0)    # the stretch of the clock [Gyr]; may reach beyond TG (the tracks are re-gridded)
PF3B_TSTEP = None           # the clock step (None = TG's 0.25 Gyr)
PF3B_Q     = ["kappa_gas", "comp", "lsig_h2", "lssfr", "lms", "ldtm"]
PF3B_NCOL  = 3
PF3B_LAB   = {"kappa_gas": r"$\kappa_{\rm gas}$", "comp": r"$-\log R_{50,\rm gas}/R_{50,\star}$",
              "lsig_h2": r"$\log \Sigma_{\rm H_2}$  [M$_\odot$ kpc$^{-2}$]", "lssfr": r"$\log$ sSFR  [yr$^{-1}$]",
              "lms": r"$\log M_\star$  [M$_\odot$]", "ldtm": r"$\log M_{\rm dust}/M_{\rm metals}$"}   # the quantity alone
PF3B_NAME  = False          # the run's name in the first panel
PF3B_HLINE = {}             # the reference lines (dict(PF3_HLINE) puts Part 12's back); none in this figure
PF3B_FLOORS = {"lssfr": np.log10(SSFR_FLOOR) + 0.5}     # a value at a storage floor is censoring, not a value: it
                                                       # is kept out of the track / violin and its share is printed
PF3B_NMIN  = 3              # the box has 14 and 20 galaxies: a violin, a marker or a clock step needs this many
PF3B_NBOOT = None           # tracks: the bootstrap behind the band (None = NBOOT)
if "_pf3_track_panel" not in globals():
    raise RuntimeError("Part 12b reuses Part 12's helpers — run Part 12 first.")
_MISS12B = [q for q in PF3B_Q if q not in P1_TRK]
if _MISS12B:
    raise KeyError(f"PF3B_Q names {_MISS12B}, which Part 1 does not track.  P1_TRK = {P1_TRK}")
_STY12B = PF3_STYLE if PF3B_STYLE is None else PF3B_STYLE
if _STY12B not in ("tracks", "violins"):
    raise ValueError(f"PF3B_STYLE = {PF3B_STYLE!r}: None, 'tracks' or 'violins'")
_CLK12B = PF3_CLOCK if PF3B_CLOCK is None else PF3B_CLOCK
if _CLK12B not in P1_CLOCKS:
    raise KeyError(f"PF3B_CLOCK = {PF3B_CLOCK!r} is not one of the clocks Part 1 timed: {P1_CLOCKS}")
_EV12B = list(_EV12 if PF3B_EV is None else PF3B_EV)
_BAD12B = [e for e in _EV12B if e not in P1_EV]
if _BAD12B:
    raise KeyError(f"PF3B_EV names {_BAD12B}; the events are {sorted(P1_EV)}")
_INS12B = [e for e in _EV12B if e in (PF3_STRIP if PF3B_STRIP is None else PF3B_STRIP)] if _STY12B == "tracks" else []
_TL12B = tuple(PF3_TLIM if PF3B_TLIM is None else PF3B_TLIM)
if PF3B_BOX not in _BX1:
    print(f"Part 12b: {PF3B_BOX} is not in this run's boxes ({_BX1}) — skipped.")
else:
    _F, _gm = _pf3_rows(PF3B_BOX)
    _both = _gm["burst"] | _gm["aging"]
    _t0 = _F[P1_EV[_CLK12B]].to_numpy(float)
    ROW12B, ROWT12B, _OFF12B = [], [], []
    # ── the statistics at the events, whatever is drawn: median [16-84] per group with the floor share, the
    #    AUCs raw | at fixed epoch + mass, and the events' offsets on the clock ──────────────────────────────
    for _q in PF3B_Q:
        _fl = PF3B_FLOORS.get(_q)
        for _e in _EV12B:
            _col = f"{_q}_{_e}"
            _v = _F[_col].to_numpy(float)
            _vv = np.where(np.isfinite(_v) & ((_v > _fl) if _fl is not None else True), _v, np.nan)
            for _g in GK:
                _x = _vv[_gm[_g]]; _f = _x[np.isfinite(_x)]
                _nfl = int(np.sum(np.isfinite(_v[_gm[_g]]) & ~np.isfinite(_vv[_gm[_g]])))
                _x2, _x1, _x3, _, _, _, _np = _pf3_point(_F, _gm[_g], _q, _e, _t0, floor=_fl, nmin=PF3B_NMIN)
                ROW12B.append(dict(box=PF3B_BOX, quantity=_q, event=_e, grp=_g, group=GSH.get(_g, _g), n=int(len(_f)),
                                   n_floor=_nfl, p16=float(np.percentile(_f, 16)) if len(_f) else np.nan,
                                   med=float(np.median(_f)) if len(_f) else np.nan,
                                   p84=float(np.percentile(_f, 84)) if len(_f) else np.nan,
                                   clock=_CLK12B, n_point=_np, dt_med=_x2, dt_p16=_x1, dt_p84=_x3))
            _a, _p, _ac, _pc, _n = _pf3_score(_F.assign(**{_col: _vv}), _both, _gm["burst"], _col)
            ROW12B.append(dict(box=PF3B_BOX, quantity=_q, event=_e, grp="auc", group="AUC dusty vs dust-poor", n=_n,
                               n_floor=0, auc=_a, p=_p, auc_ctrl=_ac, p_ctrl=_pc))
    _nr = int(np.ceil(len(PF3B_Q) / PF3B_NCOL))
    _nmin_keep = PF3_NMIN
    PF3_NMIN = PF3B_NMIN                     # the violin helper reads it; restored below
    with plt.rc_context({**PAPER_RC, "axes.labelsize": 23, "xtick.labelsize": 18 if _STY12B == "violins" else 20,
                         "ytick.labelsize": 20, "legend.fontsize": 20}):
        if _STY12B == "tracks":
            # ── THE TRACKS: one panel per quantity through Part 12's helper, the strip as an inset ────────
            fig, AX = plt.subplots(_nr, PF3B_NCOL, figsize=((7.6 if _INS12B else 7.0) * PF3B_NCOL + 0.4, 4.8 * _nr + 1.6),
                                   squeeze=False)
            for _qi, _q in enumerate(PF3B_Q):
                ax = AX[_qi // PF3B_NCOL, _qi % PF3B_NCOL]
                axs = ax.inset_axes([1.07, 0.0, PF3_STRIPW, 1.0]) if _INS12B else None
                _last = _qi // PF3B_NCOL == _nr - 1
                _rows, _off = _pf3_track_panel(ax, axs, PF3B_BOX, _q, _F, _gm, _t0, floor=PF3B_FLOORS.get(_q),
                                               nmin=PF3B_NMIN, nboot=PF3B_NBOOT, hline=PF3B_HLINE.get(_q),
                                               ylabel=PF3B_LAB.get(_q, P1_LAB.get(_q, _q)), xlabel=_last, striplab=_last,
                                               clock=_CLK12B, events=_EV12B, strip=_INS12B, tlim=_TL12B, tstep=PF3B_TSTEP)
                ROWT12B += _rows; _OFF12B += _off
                if PF3_AUC:
                    _pf3_auc_text(ax, axs, ROW12B, PF3B_BOX, _q, events=_EV12B, strip=_INS12B)
            for _qi in range(len(PF3B_Q), _nr * PF3B_NCOL):
                AX[_qi // PF3B_NCOL, _qi % PF3B_NCOL].axis("off")
            if PF3B_NAME:
                AX[0, 0].text(0.03, 0.97, BOX_LABEL.get(PF3B_BOX, PF3B_BOX), transform=AX[0, 0].transAxes, va="top",
                              ha="left", fontsize=23.4, fontweight="bold", color="0.2")
            fig.tight_layout(rect=[0, 0.13, 1, 0.995], h_pad=1.6, w_pad=(5.8 if _INS12B else 2.4))
            _h12b = [Patch(fc=PFC[g], alpha=0.5, ec=PFC[g], label=GLAB.get(g, g) + f"  (N = {int(_gm[g].sum())})") for g in GK]
            _h12b += [Line2D([], [], color="0.3", lw=2.8, label="median track, bootstrap 16-84 band")]
            _nm12b = ((lambda e: PF3_EVLAB.get(e, e).split("\n")[0]) if PF3B_EVSHORT else (lambda e: _EVL12.get(e, e)))
            _h12b += [Line2D([], [], color="k", ls="--", lw=1.4, label=_nm12b(_CLK12B) + " (the line)")]
            _h12b += [Line2D([], [], color="0.3", marker=PF3_MARK.get(e, "o"), mfc="w", mew=2.2, ms=11, lw=3.2,
                             label=_nm12b(e)) for e in _EV12B]
            fig.legend(handles=_h12b, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0),
                       title=PF3B_LEGTITLE, title_fontsize=18)
        else:
            # ── THE VIOLINS: the distributions at the three events, the observation set apart ─────────────
            fig, AX = plt.subplots(_nr, PF3B_NCOL, figsize=(7.8 * PF3B_NCOL + 0.4, 4.8 * _nr + 1.4), squeeze=False)
            for _qi, _q in enumerate(PF3B_Q):
                ax = AX[_qi // PF3B_NCOL, _qi % PF3B_NCOL]
                _fl = PF3B_FLOORS.get(_q)
                _pool = []
                for _k, _e in enumerate(_EV12B):
                    _v = _F[f"{_q}_{_e}"].to_numpy(float)
                    _vv = np.where(np.isfinite(_v) & ((_v > _fl) if _fl is not None else True), _v, np.nan)
                    _pool.append(_vv[_both])
                    for _g, _dx in zip(GK, (-0.21, 0.21)):
                        _pf3_violin(ax, _k + _dx, _vv[_gm[_g]], PFC[_g])
                    _r12b = [r for r in ROW12B if r["quantity"] == _q and r["event"] == _e and r["grp"] == "auc"]
                    if PF3_AUC and _r12b and np.isfinite(_r12b[0]["auc"]):
                        ax.text(_k, 0.985, f"AUC {_r12b[0]['auc']:.2f} | {_r12b[0]['auc_ctrl']:.2f}",
                                transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=13, color="0.3",
                                bbox=dict(fc="w", ec="none", alpha=0.8, pad=1.0))
                for _sev in PF3_SEP:
                    if _sev in _EV12B:
                        ax.axvline(_EV12B.index(_sev) - 0.5, color="0.55", ls="--", lw=1.3, zorder=1)
                if _q in PF3B_HLINE:
                    ax.axhline(PF3B_HLINE[_q][0], color="0.4", ls=":", lw=1.4, zorder=1)
                    ax.text(0.96, PF3B_HLINE[_q][0], PF3B_HLINE[_q][1], transform=ax.get_yaxis_transform(),
                            ha="right", va="bottom", fontsize=19.5, color="0.25",
                            bbox=dict(fc="w", ec="none", alpha=1.0, pad=0.8), zorder=6)
                _pool = np.concatenate(_pool); _pool = _pool[np.isfinite(_pool)]
                if len(_pool):
                    _lo, _hi = np.percentile(_pool, PF3_YLIM); _pad = 0.14 * (_hi - _lo)
                    ax.set_ylim(_lo - _pad, _hi + (1.6 if PF3_AUC else 1.0) * _pad)
                ax.set_xticks(range(len(_EV12B)))
                ax.set_xticklabels([PF3_EVLAB.get(e, e) for e in _EV12B] if _qi // PF3B_NCOL == _nr - 1 else [""] * len(_EV12B))
                ax.set_xlim(-0.65, len(_EV12B) - 0.35)
                ax.set_ylabel(PF3B_LAB.get(_q, P1_LAB.get(_q, _q)))
                for _s in ("top", "right"):
                    ax.spines[_s].set_visible(False)
            for _qi in range(len(PF3B_Q), _nr * PF3B_NCOL):
                AX[_qi // PF3B_NCOL, _qi % PF3B_NCOL].axis("off")
            if PF3B_NAME:
                AX[0, 0].text(0.03, 0.97, BOX_LABEL.get(PF3B_BOX, PF3B_BOX), transform=AX[0, 0].transAxes, va="top",
                              ha="left", fontsize=23.4, fontweight="bold", color="0.2")
            fig.tight_layout(rect=[0, 0.05, 1, 0.995], h_pad=1.8, w_pad=2.2)
            fig.legend(handles=[Patch(fc=PFC[g], alpha=0.5, ec=PFC[g], label=GLAB.get(g, g) + f"  (N = {int(_gm[g].sum())})") for g in GK]
                       + [Line2D([], [], color="0.3", marker="o", mfc="w", ms=8, lw=3.8, label="median and 16-84 range")],
                       loc="lower center", ncol=3, frameon=False)
        paper_save(fig, PF3B_STEM); plt.show()
    PF3_NMIN = _nmin_keep
    P12B = pd.DataFrame(ROW12B)
    P12B.to_csv(os.path.join(OUT, f"{PF3B_STEM}.csv"), index=False)
    if ROWT12B:
        pd.DataFrame(ROWT12B).to_csv(os.path.join(OUT, f"{PF3B_STEM}_tracks.csv"), index=False)
    print(f"THE ONSET IN {BOX_LABEL.get(PF3B_BOX, PF3B_BOX)}  —  median [16-84] READ AT EACH EVENT, AUC toward the "
          f"{GSH['burst']} (raw | at fixed epoch + mass):")
    for _q in PF3B_Q:
        for _e in _EV12B:
            _s = P12B[(P12B["quantity"] == _q) & (P12B["event"] == _e)]
            _r = {r["grp"]: r for _, r in _s.iterrows()}
            if "auc" not in _r:
                continue
            print(f"    {_q:10s} at {_e:4s}  "
                  + "  ".join(f"{GSH.get(g, g)} {_r[g]['med']:+6.2f} [{_r[g]['p16']:+.2f},{_r[g]['p84']:+.2f}] (N {int(_r[g]['n'])}"
                              + (f", {int(_r[g]['n_floor'])} at the floor" if _r[g]["n_floor"] else "") + ")" for g in GK if g in _r)
                  + f"   AUC {_r['auc']['auc']:.2f} | {_r['auc']['auc_ctrl']:.2f}")
    if _STY12B == "tracks":
        _s = P12B[(P12B["quantity"] == PF3B_Q[0]) & (P12B["grp"] != "auc")]
        print(f"  the tracks are on the {_CLK12B} clock over {_TL12B[0]:+g} to {_TL12B[1]:+g} Gyr (re-gridded from the raw "
              f"series) where at least {PF3B_NMIN} galaxies contribute (bootstrap {NBOOT if PF3B_NBOOT is None else PF3B_NBOOT}); "
              f"the markers sit at each group's MEDIAN offset of the event: "
              + "   ".join(f"{_EVTXT12.get(e, e)} {GSH.get(g, g)} {_s[(_s['event'] == e) & (_s['grp'] == g)]['dt_med'].iloc[0]:+.2f} Gyr"
                           + (" (in the strip)" if e in _INS12B else "")
                           for e in _EV12B for g in GK if len(_s[(_s["event"] == e) & (_s["grp"] == g)])))
        if _OFF12B:
            print(f"  !! {len(_OFF12B)} marker(s) fell outside PF3B_TLIM and are not drawn: "
                  + "; ".join(f"{q} {GSH.get(g, g)} {e} at {x:+.2f} Gyr" for _, q, g, e, x in _OFF12B[:8]) + "  (widen PF3B_TLIM)")
        print("  a marker is the median of the quantity read AT the event, not a point of the median track; a value at a storage\n"
              "  floor (PF3B_FLOORS) is left out of both and counted above.")
    print("table ->", os.path.join(OUT, f"{PF3B_STEM}.csv")
          + (f"  and the tracks -> {os.path.join(OUT, f'{PF3B_STEM}_tracks.csv')}" if ROWT12B else ""))

## Part 13 — paper figure 4: the ledger of the surviving dust

Where does the dust a dusty quenched galaxy is *seen* with come from? Three books per galaxy, from Part 1's
raw series and nothing else:

| | |
|---|---|
| **the class** | over the leg after the truncation, against the observable threshold `DUSTY_CUT`: **kept** (never fell below it), **regrown** (fell below and is back above it at the anchor), **faded** (below it at the anchor) |
| **the rebound** | $\log M_{\rm dust}$ at the anchor minus its post-truncation **minimum** — how much dust was made or brought in after the low point, in dust *mass*, so a merger's stars do not dilute it |
| **the ex-situ growth** | $\log M_\star$ at the anchor minus $\log M_\star$ at `PF4_FROM` ($t_{q,\rm end}$ by default). A galaxy below the passive line forms almost nothing in situ, so stellar mass gained after the quenching time is mass that **arrived**: the merger axis, with no merger tree needed. The largest single-step rise is kept too (one event, not a trickle). |

Panel (c) then asks whether the rebound tracks the arrivals: if the surviving dust is merger-fed the two
correlate and the regrowers sit right of the merger line; if it is regrown or recycled in place they do not.
Panel (d) is the gas book, $\Delta \log M_{\rm gas}$ over the same leg — re-accretion, if any.

**Read it knowing what the class is measured against.** The groups are Part 1b's, cut on the *jet-clock map*;
the class here is the plain observable at the anchor. A dusty group that is mostly *faded* by its anchor is
not a contradiction — it is the lifetime of Part 5 seen from the other end.

**Knobs.** `PF4_FROM` (`"qt"` or `"sft"`), `PF4_JUMP` (dex of $\log M_\star$ growth that counts as a merger;
0.04 = 10 %, a 1:10 event), `PF4_REB` (dex of rebound that counts as regrowth), `PF4_BOXES`, `PF4_NMIN`,
`PF4_CLS` (the classes and the alpha they are drawn at).

In [ ]:
# ── Part 13 — PAPER FIGURE 4: THE LEDGER OF THE SURVIVING DUST — kept, regrown, or brought in ────────────
# Where does the dust a dusty quenched galaxy is seen with come from?  Three books, per galaxy, from Part 1's
# raw series and nothing else:
#   THE CLASS     over the leg after the truncation, against the observable threshold DUSTY_CUT:
#                   kept     never fell below it;
#                   regrown  fell below it and is back above it at the anchor;
#                   faded    below it at the anchor.
#   THE REBOUND   log M_dust at the anchor minus its post-truncation MINIMUM — how much dust was made or
#                 brought in after the low point (in M_dust, not per unit M*, so a merger's stars do not
#                 dilute it).
#   THE EX-SITU GROWTH   log M* at the anchor minus log M* at PF4_FROM.  A galaxy below the passive line forms
#                 almost nothing in situ, so stellar mass gained after the quenching time is mass that ARRIVED:
#                 this is the merger axis, with no merger tree needed.  The largest single-step rise is kept
#                 too (one event, not a trickle).
# The figure then asks whether the rebound tracks the arrivals — if the surviving dust is merger-fed the two
# correlate and the regrowers sit to the right of the merger line; if it is regrown or recycled in place they
# do not.
PF4_STEM  = "ba_paper_origin"
PF4_BOXES = [b for b in BOXES if b in _BX1]
PF4_FROM  = "qt"          # the ex-situ growth is measured from this event to the anchor: "qt" (default — the
                          # galaxy is on the red sequence, in-situ growth is off) or "sft" (the whole leg)
PF4_JUMP  = 0.04          # dex of log M* gained after PF4_FROM that counts as a merger (10^0.04 - 1 = 10 %,
                          # a 1:10 event — the boundary between a minor merger and accretion noise)
PF4_REB   = 0.5           # dex of log M_dust rebound from the post-truncation minimum that counts as regrowth
PF4_UNIQ  = True
PF4_NMIN  = 5
PF4_CLS   = [("kept", 1.0), ("regrown", 0.55), ("faded", 0.18)]     # the class and the alpha it is drawn at
from matplotlib.patches import Patch
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
_EVT4 = {"qt": "t_qt", "sft": "t_sft"}
if PF4_FROM not in _EVT4:
    raise KeyError(f"PF4_FROM = {PF4_FROM!r}; use 'qt' or 'sft'")

L13 = []
for _b in PF4_BOXES:
    _F, _R = B1[_b]["F"], B1[_b]["R"]
    _keep = (_F["uniq"].to_numpy(bool) if PF4_UNIQ else np.ones(len(_F), bool)) & (_F["grp"].to_numpy() != "")
    for _i in np.flatnonzero(_keep):
        _S = _R[int(_F["ridx"].iloc[_i])]
        _t = np.asarray(_S["t"], float)
        _t0, _tf = float(_F["t_sft"].iloc[_i]), float(_F[_EVT4[PF4_FROM]].iloc[_i])
        if not (np.isfinite(_t0) and np.isfinite(_tf)):
            continue
        _lfd, _lms = np.asarray(_S["lfd"], float), np.asarray(_S["lms"], float)
        _lmg = np.asarray(_S["lfgas"], float) + _lms
        _post = (_t >= _t0 - 1e-9) & np.isfinite(_lfd) & np.isfinite(_lms)
        if _post.sum() < 2:
            continue
        _ip = np.flatnonzero(_post)
        _ld = _lfd[_ip] + _lms[_ip]                          # log M_dust along the leg (the floor is a value)
        _k = int(np.argmin(_lfd[_ip]))
        _r = dict(box=_b, snap=int(_F["snap"].iloc[_i]), gal_id=int(_F["gal_id"].iloc[_i]),
                  grp=_F["grp"].iloc[_i], t_sft=_t0, t_from=_tf, t_anchor=float(_t[_ip[-1]]),
                  lfd_sft=float(_lfd[_ip[0]]), lfd_min=float(_lfd[_ip[_k]]), lfd_anc=float(_lfd[_ip[-1]]),
                  t_min=float(_t[_ip[_k]] - _t0), n_post=int(len(_ip)),
                  reb_md=float(_ld[-1] - _ld[_k]),               # the rebound, in dust mass
                  reb_fd=float(_lfd[_ip[-1]] - _lfd[_ip[_k]]),   # ... and per unit stellar mass
                  net_md=float(_ld[-1] - _ld[0]))
        _r["cls"] = ("kept" if _r["lfd_min"] >= DUSTY_CUT else
                     "regrown" if _r["lfd_anc"] >= DUSTY_CUT else "faded")
        # the ex-situ book: from PF4_FROM to the anchor
        _ok = (_t >= _tf - 1e-9) & np.isfinite(_lms)
        _j = np.flatnonzero(_ok)
        if len(_j) >= 2:
            _r["dlms"] = float(_lms[_j[-1]] - _lms[_j[0]])
            _r["jump_max"] = float(np.max(np.diff(_lms[_j])))
            _r["n_jump"] = int((np.diff(_lms[_j]) >= PF4_JUMP).sum())
            _r["dlmg"] = float(_lmg[_j[-1]] - _lmg[_j[0]]) if np.isfinite(_lmg[_j[-1]]) and np.isfinite(_lmg[_j[0]]) else np.nan
            _r["t_leg"] = float(_t[_j[-1]] - _t[_j[0]])
        else:
            _r.update(dlms=np.nan, jump_max=np.nan, n_jump=0, dlmg=np.nan, t_leg=np.nan)
        _r["merger"] = bool(np.isfinite(_r["dlms"]) and _r["dlms"] >= PF4_JUMP)
        L13.append(_r)
L13 = pd.DataFrame(L13)
if not len(L13):
    raise RuntimeError("Part 13: no branch had a post-truncation leg — did Parts 1 and 2 run?")

# ── the figure: one row per resolution ────────────────────────────────────────────────────────────────────
ROW13 = []
fig, AX = plt.subplots(len(PF4_BOXES), 4, figsize=(15.2, 3.1 * len(PF4_BOXES) + 0.8), squeeze=False)
_flab = {"qt": r"t_{q,\,\rm end}", "sft": r"t_{q,\,\rm start}"}[PF4_FROM]
for _r, _b in enumerate(PF4_BOXES):
    _D = L13[L13["box"] == _b]
    # (a) the classes
    ax = AX[_r, 0]
    for _y, _g in enumerate(GK):
        _d = _D[_D["grp"] == _g]
        if len(_d) < PF4_NMIN:
            continue
        _left = 0.0
        for _cl, _al in PF4_CLS:
            _f = float((_d["cls"] == _cl).mean())
            ax.barh(_y, _f, left=_left, color=PFC[_g], alpha=_al, edgecolor="w", lw=0.8, height=0.62)
            if _f >= 0.06:
                ax.text(_left + _f / 2, _y, f"{_f * 100:.0f}", ha="center", va="center", fontsize=10.4,
                        color="w" if _al > 0.4 else "0.2")
            ROW13.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), block="class", cls=_cl, frac=_f, n=len(_d)))
            _left += _f
        ax.text(1.01, _y, f"N {len(_d)}", va="center", fontsize=9.75, color="0.35")
    ax.set_yticks(range(len(GK))); ax.set_yticklabels([GSH.get(g, g) for g in GK], fontsize=11.1)
    ax.set_xlim(0, 1.0); ax.set_ylim(-0.6, len(GK) - 0.4)
    ax.set_xlabel(rf"share of the group, after $t_{{q,\,\rm start}}$" if _r == len(PF4_BOXES) - 1 else "")
    if _r == 0:
        ax.set_title(rf"kept / regrown / faded  (at $f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$)", fontsize=13)
    # (b) the ex-situ growth
    ax = AX[_r, 1]
    for _x, _g in enumerate(GK):
        _v = _D.loc[_D["grp"] == _g, "dlms"].to_numpy(float); _v = _v[np.isfinite(_v)]
        if len(_v) < PF4_NMIN:
            continue
        _pr = ax.violinplot([_v], positions=[_x], widths=0.62, showextrema=False, showmedians=False)
        for _bd in _pr["bodies"]:
            _bd.set_facecolor(PFC[_g]); _bd.set_edgecolor(PFC[_g]); _bd.set_alpha(0.32)
        _q = np.percentile(_v, [16, 50, 84])
        ax.plot([_x, _x], [_q[0], _q[2]], color=PFC[_g], lw=3, solid_capstyle="round")
        ax.plot([_x], [_q[1]], marker="o", ms=5.5, mfc="w", mec=PFC[_g], mew=1.7, ls="none", zorder=5)
        _fm = float((_v >= PF4_JUMP).mean())
        ax.text(_x, 0.985, f"{_fm * 100:.0f} % merged", transform=ax.get_xaxis_transform(), ha="center",
                va="top", fontsize=9.75, color=PFC[_g], bbox=dict(fc="w", ec="none", alpha=0.8, pad=0.8))
        ROW13.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), block="exsitu", med=_q[1], p16=_q[0], p84=_q[2],
                          frac_merger=_fm, n=len(_v)))
    ax.axhline(PF4_JUMP, color="0.35", ls=":", lw=1.2)
    ax.text(0.99, PF4_JUMP, f" 1:{int(round(1 / (10 ** PF4_JUMP - 1)))} merger", transform=ax.get_yaxis_transform(),
            ha="right", va="bottom", fontsize=9.1, color="0.35")
    ax.set_xticks(range(len(GK))); ax.set_xticklabels([GSH.get(g, g) for g in GK], fontsize=11.1)
    ax.set_xlim(-0.6, len(GK) - 0.4)
    _vp = _D["dlms"].to_numpy(float); _vp = _vp[np.isfinite(_vp)]
    if len(_vp):          # a branch that LOST a large share of its stars after quenching is a tree swap, not a
        # measurement: the axis stops at the 2nd percentile and never below -0.15 dex (the violin runs off it)
        ax.set_ylim(max(-0.15, min(-0.02, np.percentile(_vp, 2))), max(PF4_JUMP * 2.5, np.percentile(_vp, 99)) * 1.35)
    ax.set_ylabel(rf"$\Delta \log M_\star$ after ${_flab}$  [dex]", fontsize=12.3)
    if _r == 0:
        ax.set_title("the ex-situ growth (what arrived)", fontsize=13)
    # (c) the rebound against the arrivals
    ax = AX[_r, 2]
    for _g in GK:
        _d = _D[(_D["grp"] == _g) & np.isfinite(_D["dlms"]) & np.isfinite(_D["reb_md"])]
        if len(_d) < PF4_NMIN:
            continue
        _rg = _d["cls"] == "regrown"
        ax.scatter(_d.loc[~_rg, "dlms"], _d.loc[~_rg, "reb_md"], s=11, facecolors="none", edgecolors=PFC[_g],
                   lw=0.7, alpha=0.55)
        ax.scatter(_d.loc[_rg, "dlms"], _d.loc[_rg, "reb_md"], s=16, color=PFC[_g], alpha=0.85, lw=0)
        _rho, _p = spearmanr(_d["dlms"], _d["reb_md"], nan_policy="omit")
        _mr = _d.loc[_rg, "merger"]
        ROW13.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), block="rebound", rho=float(_rho), p=float(_p),
                          n=len(_d), n_regrown=int(_rg.sum()),
                          frac_regrown_merger=float(_mr.mean()) if _rg.any() else np.nan,
                          med_reb_md=float(np.median(_d["reb_md"])),
                          frac_reb=float((_d["reb_md"] >= PF4_REB).mean())))
        ax.text(0.02, 0.97 - 0.09 * GK.index(_g), rf"$\rho$ = {_rho:+.2f} (N {len(_d)})", transform=ax.transAxes,
                fontsize=10.4, color=PFC[_g], va="top", bbox=dict(fc="w", ec="none", alpha=0.75, pad=0.8), zorder=7)
    ax.axvline(PF4_JUMP, color="0.35", ls=":", lw=1.2); ax.axhline(PF4_REB, color="0.35", ls=":", lw=1.2)
    ax.axhline(0.0, color="0.75", lw=0.8)
    _vx = _D["dlms"].to_numpy(float); _vx = _vx[np.isfinite(_vx)]
    _vy = _D["reb_md"].to_numpy(float); _vy = _vy[np.isfinite(_vy)]
    if len(_vx) and len(_vy):
        ax.set_xlim(max(-0.15, min(-0.02, np.percentile(_vx, 2))), max(PF4_JUMP * 3, np.percentile(_vx, 99.5)) * 1.1)
        ax.set_ylim(min(-0.1, np.percentile(_vy, 0.5)), max(PF4_REB * 1.5, np.percentile(_vy, 99.5)) * 1.1)
    ax.set_xlabel(rf"$\Delta \log M_\star$ after ${_flab}$  [dex]" if _r == len(PF4_BOXES) - 1 else "")
    ax.set_ylabel(r"rebound of $\log M_{\rm dust}$  [dex]", fontsize=12.3)
    if _r == 0:
        ax.set_title("does the regrowth track the arrivals?", fontsize=13)
    # (d) the gas book
    ax = AX[_r, 3]
    for _x, _g in enumerate(GK):
        _v = _D.loc[_D["grp"] == _g, "dlmg"].to_numpy(float); _v = _v[np.isfinite(_v)]
        if len(_v) < PF4_NMIN:
            continue
        _pr = ax.violinplot([_v], positions=[_x], widths=0.62, showextrema=False, showmedians=False)
        for _bd in _pr["bodies"]:
            _bd.set_facecolor(PFC[_g]); _bd.set_edgecolor(PFC[_g]); _bd.set_alpha(0.32)
        _q = np.percentile(_v, [16, 50, 84])
        ax.plot([_x, _x], [_q[0], _q[2]], color=PFC[_g], lw=3, solid_capstyle="round")
        ax.plot([_x], [_q[1]], marker="o", ms=5.5, mfc="w", mec=PFC[_g], mew=1.7, ls="none", zorder=5)
        ROW13.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), block="gas", med=_q[1], p16=_q[0], p84=_q[2],
                          frac_regas=float((_v > 0).mean()), n=len(_v)))
    ax.axhline(0.0, color="0.35", ls=":", lw=1.2)
    ax.set_xticks(range(len(GK))); ax.set_xticklabels([GSH.get(g, g) for g in GK], fontsize=11.1)
    ax.set_xlim(-0.6, len(GK) - 0.4)
    _vp = _D["dlmg"].to_numpy(float); _vp = _vp[np.isfinite(_vp)]
    if len(_vp):
        _lo, _hi = np.percentile(_vp, [1, 99]); _pad = 0.15 * (_hi - _lo)
        ax.set_ylim(_lo - _pad, _hi + 1.5 * _pad)
    ax.set_ylabel(rf"$\Delta \log M_{{\rm gas}}$ after ${_flab}$  [dex]", fontsize=12.3)
    if _r == 0:
        ax.set_title("the gas book (re-accretion?)", fontsize=13)
    AX[_r, 0].text(-0.36, 0.5, BOX_LABEL.get(_b, _b).split(" ")[0], transform=AX[_r, 0].transAxes,
                   rotation=90, va="center", ha="center", fontsize=13.7, fontweight="bold")
fig.suptitle(f"The ledger of the surviving dust — kept, regrown or brought in; the ex-situ growth measured from "
             f"$t_{{q,\\,\\rm end}}$ to the anchor" if PF4_FROM == "qt" else
             "The ledger of the surviving dust — kept, regrown or brought in; the ex-situ growth measured over the "
             "whole leg after the truncation", y=0.995, fontsize=14.3)
fig.tight_layout(rect=[0.015, 0.05, 1, 0.97], h_pad=1.8, w_pad=1.5)
fig.legend(handles=[Patch(fc=PFC[g], alpha=0.6, ec=PFC[g], label=GLAB.get(g, g)) for g in GK]
           + [Patch(fc="0.3", alpha=a, label=c) for c, a in PF4_CLS]
           + [Line2D([], [], color="0.3", marker="o", ls="none", ms=5, label="a regrower (panel c, filled)"),
              Line2D([], [], color="0.3", marker="o", mfc="none", ls="none", ms=5, label="kept or faded (open)")],
           loc="lower center", ncol=4, fontsize=11.1, frameon=False)
paper_save(fig, PF4_STEM); plt.show()

P13 = pd.DataFrame(ROW13)
P13.to_csv(os.path.join(OUT, f"{PF4_STEM}.csv"), index=False)
L13.to_csv(os.path.join(OUT, f"{PF4_STEM}_galaxies.csv"), index=False)
print(f"THE LEDGER  —  the leg after the truncation, against f_dust = 10^{DUSTY_CUT:g}; the ex-situ growth from "
      f"{_EVT4[PF4_FROM]} to the anchor, a merger = {PF4_JUMP:g} dex (1:{int(round(1 / (10 ** PF4_JUMP - 1)))}):")
for _b in PF4_BOXES:
    for _g in GK:
        _c = {r["cls"]: r["frac"] for _, r in P13[(P13["box"] == _b) & (P13["grp"] == _g) & (P13["block"] == "class")].iterrows()}
        _e = P13[(P13["box"] == _b) & (P13["grp"] == _g) & (P13["block"] == "exsitu")]
        _rb = P13[(P13["box"] == _b) & (P13["grp"] == _g) & (P13["block"] == "rebound")]
        _gs = P13[(P13["box"] == _b) & (P13["grp"] == _g) & (P13["block"] == "gas")]
        if not _c:
            continue
        print(f"  {_b:7s} {GSH.get(_g, _g):14s} kept {_c.get('kept', 0) * 100:3.0f} %  regrown {_c.get('regrown', 0) * 100:3.0f} %  "
              f"faded {_c.get('faded', 0) * 100:3.0f} %"
              + (f"   merged {_e['frac_merger'].iloc[0] * 100:3.0f} % (median dlogM* {_e['med'].iloc[0]:+.3f})" if len(_e) else "")
              + (f"   rho(rebound, arrivals) {_rb['rho'].iloc[0]:+.2f}, regrowers merged {_rb['frac_regrown_merger'].iloc[0] * 100:.0f} %"
                 if len(_rb) and np.isfinite(_rb['frac_regrown_merger'].iloc[0]) else
                 (f"   rho(rebound, arrivals) {_rb['rho'].iloc[0]:+.2f}" if len(_rb) else ""))
              + (f"   gas rose in {_gs['frac_regas'].iloc[0] * 100:.0f} %" if len(_gs) else ""))
print("  READ IT AS: a dusty group that is mostly KEPT with a low merger share owes its dust to never having lost it;\n"
      "  a high regrown share whose rebound tracks the arrivals owes it to mergers; a high regrown share that does\n"
      "  not is regrowth or recycling in place (Parts 6/6c say from where).")
print("tables ->", os.path.join(OUT, f"{PF4_STEM}{{,_galaxies}}.csv"))

## Part 14 — paper figure 5: the timescale ladder

Nothing here is measured. The cell puts side by side, per group and on **one logarithmic axis**, the clocks
the earlier parts already built, read inside the window after the truncation, so that "which of these is
the dust's lifetime made of?" is answered by looking:

| section | rung | from |
|---|---|---|
| growth | $\tau_{\rm grow}$ in the non-star-forming ISM, the star-forming ISM, the cold CGM | Part 6b (m25) |
| destruction | $\tau_{\rm sp}/3$ — the **mass** e-folding, which is what a lifetime compares to — in the non-SF ISM, the cold CGM, the hot CGM | Part 6b (m25) |
| supply | the turnover of the CGM gas by the cold inflow, $M_{\rm gas}(<100\,{\rm kpc})/\dot M_{\rm in}$ | Part 6c (m25) |
| the lifetime | $t_{\rm dark}$, the Kaplan–Meier median with its quartiles, per box | Part 5 |
| the window | how long the galaxies are followed after the truncation — every lifetime is censored by it | Part 1 |

Each rung is the per-galaxy median inside `PF5_POST`, then the median and 16–84 over the galaxies of the
group; the two medians are printed in the right-hand column. A rung whose source did not run (Parts 6b/6c
need the m25 particle census, cluster only) is left blank and says so, and the rest of the ladder is drawn.

Both grain laws are linear in the assumed grain size, so read the **contrast between the groups and the
order of the rungs**, not the third digit. An open arrow on $t_{\rm dark}$ means the late quartile never
fell inside the window.

**Knobs.** `PF5_POST` (the window; Part 6b's by default), `PF5_BOXES` (the $t_{\rm dark}$ and follow-up rows),
`PF5_M25`, `PF5_NMIN`, `PF5_HUB`.

In [ ]:
# ── Part 14 — PAPER FIGURE 5: THE TIMESCALE LADDER — growth, destruction, supply, the dust lifetime and the
#    evolutionary timescales of the 25 Mpc box, on one axis ───────────────────────────────────────────────
# Nothing here is measured.  The cell puts side by side, per group, the clocks the earlier parts already
# built, so that "which of these is the dust's lifetime made of?" can be read off one axis:
#   GROWTH        tau_grow in the non-star-forming ISM, the star-forming ISM and the cold CGM   (Part 6b)
#   DESTRUCTION   tau_sp / 3 — the MASS e-folding, which is what a lifetime compares to — in the non-SF ISM,
#                 the cold CGM and the hot CGM                                                  (Part 6b)
#   SUPPLY        the turnover of the CGM gas by the cold inflow, M_gas(< 100 kpc) / Mdot_in   (Part 6c)
#   LIFETIME      t_dust, the time from the truncation to the dust falling permanently below the threshold
#                 (Part 5's t_dark: the Kaplan-Meier median and quartiles)
#   EVOLUTIONARY TIMESCALES   the time since quenching (t_obs - t_q,start), the duration of the quenching
#                 transition (t_q,end - t_q,start) and the jet lead time (t_q,start - t_jet)    (Part 1)
# THE GRAIN CLOCKS AND THE TURNOVER ARE READ OVER A GALAXY'S WHOLE HISTORY (PF5_WHEN = "all": every node of it
# in the particle census, ~22 per galaxy) — characteristic timescales of these galaxies, not of one moment
# after the truncation.  PF5_WHEN = "post" reads them inside PF5_POST after the truncation instead, and the
# printout gives both, so the choice is visible.
PF5_STEM  = "ba_paper_timescales"
PF5_BOX   = "cis25"         # the only box with the particle census; the lifetime and the evolutionary rungs
                            # are its too, so the whole ladder is one population
PF5_WHEN  = "all"           # "all" = every node of the history | "post" = the window PF5_POST after the truncation
PF5_POST  = P6B_POST if "P6B_POST" in dir() else (0.5, 2.5)
PF5_NMIN  = 3
PF5_HUB   = 13.8
PF5_SEC   = [("growth", "GROWTH"), ("destruction", "DESTRUCTION"), ("supply", "SUPPLY"),
             ("lifetime", "LIFETIME"), ("evolution", "EVOLUTIONARY TIMESCALES")]
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
if PF5_BOX not in _BX1:
    raise RuntimeError(f"Part 14: {PF5_BOX} is not in this run's boxes ({_BX1})")
_LOG3 = np.log10(3.0)
_Z6B = globals().get("Z6B"); _Z6C = globals().get("Z6C")


def _pf5_pergal(Z, col, shift=0.0, when=None):
    """One number per galaxy: its median of `col` (log10 Gyr) over the nodes selected by `when`, shifted."""
    when = PF5_WHEN if when is None else when
    if Z is None or col not in Z.columns:
        return None
    W = Z[np.isfinite(Z[col].to_numpy(float))]
    if when == "post":
        W = W[(W["trel"] >= PF5_POST[0]) & (W["trel"] <= PF5_POST[1])]
    if not len(W):
        return None
    g = W.groupby(["gid", "grp"])[col].median().reset_index()
    return {gk: g.loc[g["grp"] == gk, col].to_numpy(float) + shift for gk in GK}


def _pf5_summ(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < PF5_NMIN:
        return np.nan, np.nan, np.nan, len(v)
    return float(np.median(v)), float(np.percentile(v, 16)), float(np.percentile(v, 84)), len(v)


def _pf5_km(t, ev):
    """The Kaplan-Meier median and the times by which a quarter and three quarters have fallen."""
    ts, S, _, n = _km5(t, ev)
    q = []
    for lev in (0.75, 0.5, 0.25):
        i = np.flatnonzero(S <= lev)
        q.append(float(ts[i[0]]) if len(i) else np.nan)
    return q[1], q[0], q[2], n


# ── the supply rung: the CGM gas inside 100 kpc over the cold inflow through 10 kpc, per node ─────────────
if _Z6C is not None and "lmdot_in_gc" in _Z6C.columns and all(f"mgas_{s}" in _Z6C.columns for _, _, s in P6C_SH):
    with np.errstate(all="ignore"):
        _mg = np.nansum(np.column_stack([_Z6C[f"mgas_{s}"].to_numpy(float) for _, _, s in P6C_SH]), axis=1)
        _md = 10 ** _Z6C["lmdot_in_gc"].to_numpy(float)                      # Msun / yr
        _Z6C["lt_turn"] = np.log10(np.where((_mg > 0) & (_md > 0), _mg / _md / 1e9, np.nan))   # Gyr

# ── the rungs ────────────────────────────────────────────────────────────────────────────────────────────
LAD, CMP14 = [], []            # the ladder, and the all-vs-post comparison for the printout
def _add(sec, lab, src, col=None, shift=0.0, note=""):
    per = _pf5_pergal(src, col, shift) if col else src
    LAD.append((sec, lab, {g: _pf5_summ(per[g]) if per else (np.nan,) * 3 + (0,) for g in GK}, note))
    if col and src is not None:
        for _w in ("all", "post"):
            _pw = _pf5_pergal(src, col, shift, when=_w)
            CMP14.append(dict(row=lab, when=_w, **{f"med_{g}": (_pf5_summ(_pw[g])[0] if _pw else np.nan) for g in GK}))

_add("growth", r"$\tau_{\rm grow}$, non-SF ISM", _Z6B, "ltg_ism_nsf", note="Part 6b")
_add("growth", r"$\tau_{\rm grow}$, SF ISM", _Z6B, "ltg_ism_sf", note="Part 6b")
_add("growth", r"$\tau_{\rm grow}$, cold CGM", _Z6B, "ltg_cgm_cold", note="Part 6b")
_add("destruction", r"$\tau_{\rm sp}/3$, non-SF ISM", _Z6B, "lts_ism_nsf", -_LOG3, "Part 6b")
_add("destruction", r"$\tau_{\rm sp}/3$, cold CGM", _Z6B, "lts_cgm_cold", -_LOG3, "Part 6b")
_add("destruction", r"$\tau_{\rm sp}/3$, hot CGM", _Z6B, "lts_cgm_hot", -_LOG3, "Part 6b")
_add("supply", "CGM turnover\n" + r"$M_{\rm gas}(<100\,{\rm kpc})/\dot M_{\rm in}$", _Z6C, "lt_turn", note="Part 6c")
# the lifetime: Kaplan-Meier, Part 5's L5
_per = None
if "L5" in globals() and len(L5):
    _per = {}
    for _g in GK:
        _s = L5[(L5["box"] == PF5_BOX) & (L5["grp"] == _g)]
        _s = _s[np.isfinite(_s["t_dark"].to_numpy(float)) & (_s["t_dark"].to_numpy(float) > 0)]
        if len(_s) < PF5_NMIN:
            _per[_g] = (np.nan, np.nan, np.nan, len(_s)); continue
        _m, _lo, _hi, _n = _pf5_km(_s["t_dark"].to_numpy(float), _s["ev"].to_numpy(bool))
        _per[_g] = (np.log10(_m) if _m > 0 else np.nan, np.log10(_lo) if _lo > 0 else np.nan,
                    np.log10(_hi) if _hi > 0 else np.nan, _n)
    LAD.append(("lifetime", "dust lifetime\n" + r"$t_{\rm dust}$", _per, "Part 5 (KM median, quartiles)"))
else:
    LAD.append(("lifetime", "dust lifetime\n" + r"$t_{\rm dust}$", {g: (np.nan,) * 3 + (0,) for g in GK}, "Part 5"))
# the evolutionary timescales, from the branch table of the same box
_F14 = B1[PF5_BOX]["F"]; _k14 = _F14["uniq"].to_numpy(bool)
NEG14 = {}
for _col, _lab in (("t_post", "time since quenching\n" + r"$t_{\rm obs}-t_{q,\,\rm start}$"),
                   ("tq_span", "quenching duration\n" + r"$t_{q,\,\rm end}-t_{q,\,\rm start}$"),
                   ("lag_sft", "jet lead time\n" + r"$t_{q,\,\rm start}-t_{\rm jet}$")):
    _per, _neg = {}, {}
    for _g in GK:
        _v = _F14.loc[_k14 & (_F14["grp"].to_numpy() == _g), _col].to_numpy(float)
        _v = _v[np.isfinite(_v)]
        _neg[_g] = float((_v <= 0).mean()) if len(_v) else np.nan       # a log axis cannot show these
        _per[_g] = np.log10(_v[_v > 0])
    NEG14[_lab] = _neg
    _add("evolution", _lab, _per, note="Part 1")

# ── the figure ────────────────────────────────────────────────────────────────────────────────────────────
_ypos, _ylab, _ysec, _y = [], [], [], 0.0
for _sk, _sl in PF5_SEC:
    _rows = [r for r in LAD if r[0] == _sk]
    if not _rows:
        continue
    _ysec.append((_y, len(_rows), _sl))
    for _r in _rows:
        _ypos.append(_y); _ylab.append(_r[1]); _y += 1.0
    _y += 0.75
_fmt = lambda v: (f"{v:.0f}" if v >= 100 else f"{v:.2g}")
_XMIN14 = min([10 ** r[2][g][1] for r in LAD for g in GK if np.isfinite(r[2][g][1])]
              + [10 ** r[2][g][0] for r in LAD for g in GK if np.isfinite(r[2][g][0])] + [0.3]) * 0.4
_XMAX14 = max([10 ** r[2][g][2] for r in LAD for g in GK if np.isfinite(r[2][g][2])]
              + [10 ** r[2][g][0] for r in LAD for g in GK if np.isfinite(r[2][g][0])] + [PF5_HUB * 3]) * 1.6
ROW14 = []
with plt.rc_context(PAPER_RC):
    fig, ax = plt.subplots(figsize=(10.6, 0.62 * len(_ypos) + 2.6))
    _k = 0
    for _sk, _sl in PF5_SEC:
        for _r in [r for r in LAD if r[0] == _sk]:
            _yy = _ypos[_k]; _k += 1
            _any = False
            for _g, _dy in zip(GK, (-0.19, 0.19)):        # the dusty lane on top (the y axis is inverted)
                _m, _lo, _hi, _n = _r[2][_g]
                ROW14.append(dict(section=_sk, row=_r[1].replace("\n", " "), grp=_g, group=GSH.get(_g, _g),
                                  log_med=_m, log_lo=_lo, log_hi=_hi, med_gyr=10 ** _m if np.isfinite(_m) else np.nan,
                                  n=_n, source=_r[3], when=(PF5_WHEN if _sk in ("growth", "destruction", "supply") else "-")))
                if not np.isfinite(_m):
                    continue
                _any = True
                _x0 = 10 ** _lo if np.isfinite(_lo) else 10 ** _m
                if np.isfinite(_hi):
                    ax.plot([_x0, 10 ** _hi], [_yy + _dy, _yy + _dy], color=PFC[_g], lw=3.6, alpha=0.6,
                            solid_capstyle="round")
                else:                                     # the late quartile never came: open-ended
                    ax.annotate("", xy=(_XMAX14 * 0.8, _yy + _dy), xytext=(_x0, _yy + _dy),
                                arrowprops=dict(arrowstyle="->", color=PFC[_g], lw=2.4, alpha=0.6))
                ax.plot([10 ** _m], [_yy + _dy], marker="o", ms=8, mfc=PFC[_g], mec="w", mew=1.3, ls="none", zorder=5)
            if _any:                                      # the two medians, in a column at the right
                for _g, _dy in zip(GK, (-0.19, 0.19)):
                    _m = _r[2][_g][0]
                    ax.text(1.012, _yy + _dy, (r"$\mathbf{" + _fmt(10 ** _m) + "}$") if np.isfinite(_m) else "–",
                            transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=15, color=PFC[_g])
            else:
                ax.text(np.sqrt(_XMIN14 * _XMAX14), _yy, f"not available here ({_r[3]} did not run)", ha="center",
                        va="center", fontsize=13.7, color="0.5", style="italic")
    for _y0, _n, _sl in _ysec:
        ax.axhspan(_y0 - 0.5, _y0 + _n - 0.5, color="0.955", zorder=0)
        ax.text(0.004, _y0 - 0.46, _sl, transform=ax.get_yaxis_transform(), fontsize=13, color="0.45",
                va="bottom", ha="left", fontweight="bold")
    ax.axvline(1.0, color="0.4", ls="--", lw=1.2); ax.axvline(PF5_HUB, color="0.4", ls=":", lw=1.2)
    ax.text(1.0, -0.86, " 1 Gyr", fontsize=14.3, color="0.4", ha="left", va="bottom", clip_on=False)
    ax.text(PF5_HUB, -0.86, " a Hubble time", fontsize=14.3, color="0.4", ha="left", va="bottom", clip_on=False)
    ax.text(1.012, -0.72, "median [Gyr]", transform=ax.get_yaxis_transform(), fontsize=13.7, color="0.45",
            ha="left", va="top")
    ax.set_xscale("log"); ax.set_xlim(_XMIN14, _XMAX14)
    ax.set_yticks(_ypos); ax.set_yticklabels(_ylab, fontsize=16.2)
    ax.set_ylim(-0.8, max(_ypos) + 1.1); ax.invert_yaxis()
    ax.set_xlabel("timescale  [Gyr]")
    ax.set_axisbelow(True)
    for _s in ("top", "right"):
        ax.spines[_s].set_visible(False)
    fig.legend(handles=[Line2D([], [], color=PFC[g], marker="o", lw=3.6, alpha=0.8, label=GLAB.get(g, g)) for g in GK]
               + [Line2D([], [], color="0.3", lw=3.6, alpha=0.5,
                         label=r"16-84 over the galaxies (quartiles for $t_{\rm dust}$)")],
               loc="lower center", ncol=3, frameon=False)
    fig.tight_layout(rect=[0, 0.07, 0.92, 1])
    paper_save(fig, PF5_STEM); plt.show()

P14 = pd.DataFrame(ROW14)
P14.to_csv(os.path.join(OUT, f"{PF5_STEM}.csv"), index=False)
print(f"THE LADDER ({BOX_LABEL.get(PF5_BOX, PF5_BOX)})  —  medians in Gyr [16-84]; the grain clocks and the turnover over "
      + ("EVERY node of a galaxy's history" if PF5_WHEN == "all" else f"{PF5_POST[0]:+g} .. {PF5_POST[1]:+g} Gyr after the truncation") + ":")
for _sk, _sl in PF5_SEC:
    _rows = P14[P14["section"] == _sk]
    if not len(_rows):
        continue
    print(f"  {_sl.lower()}")
    for _lab in _rows["row"].unique():
        _line = f"    {_lab.replace('$', '').replace(chr(92), '')[:52]:54s}"
        for _g in GK:
            _r = _rows[(_rows["row"] == _lab) & (_rows["grp"] == _g)].iloc[0]
            _line += (f"  {GSH.get(_g, _g)} {_r['med_gyr']:7.3g} [{10 ** _r['log_lo'] if np.isfinite(_r['log_lo']) else np.nan:.2g}, "
                      f"{10 ** _r['log_hi'] if np.isfinite(_r['log_hi']) else np.nan:.2g}] (N {int(_r['n'])})"
                      if np.isfinite(_r["med_gyr"]) else f"  {GSH.get(_g, _g)} n/a")
        print(_line)
if CMP14:
    _C = pd.DataFrame(CMP14)
    print("  the same grain clocks over the WHOLE history vs inside the post-truncation window (medians, Gyr):")
    for _lab in _C["row"].unique():
        _a = _C[(_C["row"] == _lab) & (_C["when"] == "all")].iloc[0]; _p = _C[(_C["row"] == _lab) & (_C["when"] == "post")].iloc[0]
        print(f"    {_lab.replace(chr(10), ' ').replace('$', '').replace(chr(92), '')[:44]:46s}"
              + "  ".join(f"{GSH.get(g, g)} all {10 ** _a[f'med_{g}']:.3g} / post {10 ** _p[f'med_{g}']:.3g}" for g in GK))
if _Z6B is not None:
    _nn = _Z6B.groupby("gid").size()
    print(f"  the census holds {len(_Z6B)} nodes for {len(_nn)} of these galaxies (median {_nn.median():.0f} per galaxy, "
          f"clock span {np.nanmedian(_Z6B.groupby('gid')['trel'].max() - _Z6B.groupby('gid')['trel'].min()):.1f} Gyr).")
_negs = {k: v for k, v in NEG14.items() if any(np.isfinite(x) and x > 0 for x in v.values())}
if _negs:
    print("  share of galaxies a log axis cannot show (the interval is zero or negative): "
          + "; ".join(f"{k.split(chr(10))[0]}: " + ", ".join(f"{GSH.get(g, g)} {v[g] * 100:.0f} %" for g in GK) for k, v in _negs.items()))
print("  tau_sp is drawn as the MASS clock (tau_sp / 3); both grain laws are linear in the assumed grain size, so read the\n"
      "  contrast between groups and the ORDER of the rungs, not the third digit.  An arrow on the lifetime means the late\n"
      "  quartile never fell inside the follow-up (the first evolutionary rung).")
print("table ->", os.path.join(OUT, f"{PF5_STEM}.csv"))

## Part 15 — paper figure 6: the timeline

The closing figure: for each group and each resolution, **when things happen relative to the truncation** —
the jet activation, the first X-ray firing, the end of the transition ($t_{q,\rm end}$), the moment the dust
falls permanently below the threshold ($t_{\rm dark}$, Part 5) and where the galaxy is observed (the anchor).
Each event is its median with the 16–84 spread over the galaxies, Kaplan–Meier quartiles for $t_{\rm dark}$
because it is censored; an open arrow means the late quartile never came inside the window. It is the sequence
the chapter argues for, as numbers, and the printed table under it is the caption.

**Knobs.** `PF6_EV` (the events, their labels and markers — any column of `F` that is a cosmic time, plus
`t_dark`), `PF6_BOXES`, `PF6_UNIQ`, `PF6_NMIN`, `PF6_XLIM`.

In [ ]:
# ── Part 15 — PAPER FIGURE 6: THE TIMELINE — the order of the events in each pathway, on one clock ─────────
# The closing figure: for each group and each resolution, WHEN things happen relative to the truncation —
# the jet activation, the first X-ray firing, the end of the transition (t_QT), the moment the dust falls
# permanently below the threshold (t_dark, Part 5) and where the galaxy is observed (the anchor).  Each event
# is its median with the 16-84 spread over the galaxies (Kaplan-Meier quartiles for t_dark, which is
# censored); the number behind every bar is printed.  It is the sequence the chapter argues for, as numbers.
PF6_STEM  = "ba_paper_timeline"
PF6_BOXES = [b for b in BOXES if b in _BX1]
PF6_EV    = [("t_jet", "the jet activation", "v"), ("t_x1", "the first X-ray firing", "|"),
             ("t_qt", r"the end of the transition ($t_{q,\,\rm end}$)", "D"),
             ("t_dark", r"the dust falls below $10^{%g}$ for good" % DUSTY_CUT, "o"),
             ("t_anchor", r"observed ($t_{\rm obs}$)", "*")]
PF6_UNIQ  = True
PF6_NMIN  = 5
PF6_XLIM  = None            # None = the data, padded
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}


def _pf6_km(t, ev):
    ts, S, _, n = _km5(t, ev)
    q = []
    for lev in (0.75, 0.5, 0.25):
        i = np.flatnonzero(S <= lev)
        q.append(float(ts[i[0]]) if len(i) else np.nan)
    return q[1], q[0], q[2], n


ROW15 = []
for _b in PF6_BOXES:
    _F = B1[_b]["F"]
    _keep = _F["uniq"].to_numpy(bool) if PF6_UNIQ else np.ones(len(_F), bool)
    _t0 = _F["t_sft"].to_numpy(float)
    for _g in GK:
        _m = _keep & (_F["grp"].to_numpy() == _g) & np.isfinite(_t0)
        for _ev, _lab, _mk in PF6_EV:
            if _ev == "t_dark":
                if "L5" not in globals() or not len(L5):
                    ROW15.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), event=_ev, med=np.nan, lo=np.nan,
                                      hi=np.nan, n=0, censored=np.nan)); continue
                _s = L5[(L5["box"] == _b) & (L5["grp"] == _g)]
                _s = _s[np.isfinite(_s["t_dark"].to_numpy(float))]
                if len(_s) < PF6_NMIN:
                    ROW15.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), event=_ev, med=np.nan, lo=np.nan,
                                      hi=np.nan, n=len(_s), censored=np.nan)); continue
                _md, _lo, _hi, _n = _pf6_km(_s["t_dark"].to_numpy(float), _s["ev"].to_numpy(bool))
                ROW15.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), event=_ev, med=_md, lo=_lo, hi=_hi, n=_n,
                                  censored=float((~_s["ev"]).mean())))
                continue
            _d = _F[_ev].to_numpy(float)[_m] - _t0[_m]
            _d = _d[np.isfinite(_d)]
            if len(_d) < PF6_NMIN:
                ROW15.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), event=_ev, med=np.nan, lo=np.nan,
                                  hi=np.nan, n=len(_d), censored=np.nan)); continue
            ROW15.append(dict(box=_b, grp=_g, group=GSH.get(_g, _g), event=_ev, med=float(np.median(_d)),
                              lo=float(np.percentile(_d, 16)), hi=float(np.percentile(_d, 84)), n=len(_d),
                              censored=np.nan))
P15 = pd.DataFrame(ROW15)

fig, AX = plt.subplots(len(PF6_BOXES), 1, figsize=(10.4, 2.35 * len(PF6_BOXES) + 0.9), squeeze=False, sharex=True)
_xall = P15[["lo", "hi", "med"]].to_numpy(float); _xall = _xall[np.isfinite(_xall)]
_xlo, _xhi = (PF6_XLIM if PF6_XLIM else (min(-0.3, np.nanmin(_xall) - 0.3), np.nanmax(_xall) + 0.6))
for _r, _b in enumerate(PF6_BOXES):
    ax = AX[_r, 0]
    for _gi, _g in enumerate(GK):
        _y = 1.0 - _gi                              # the dusty lane on top
        ax.axhspan(_y - 0.42, _y + 0.42, color=PFC[_g], alpha=0.06, lw=0)
        for _k, (_ev, _lab, _mk) in enumerate(PF6_EV):
            _row = P15[(P15["box"] == _b) & (P15["grp"] == _g) & (P15["event"] == _ev)]
            if not len(_row) or not np.isfinite(_row["med"].iloc[0]):
                continue
            _row = _row.iloc[0]
            _dy = 0.22 - 0.11 * _k                  # each event on its own sub-lane, so the bars do not pile up
            _x0 = _row["lo"] if np.isfinite(_row["lo"]) else _row["med"]
            if np.isfinite(_row["hi"]):
                ax.plot([_x0, _row["hi"]], [_y + _dy, _y + _dy], color=PFC[_g], lw=2.6, alpha=0.45,
                        solid_capstyle="round", zorder=2)
            else:                                   # censored: the late quartile never came
                ax.annotate("", xy=(_xhi - 0.1, _y + _dy), xytext=(_x0, _y + _dy),
                            arrowprops=dict(arrowstyle="->", color=PFC[_g], lw=1.8, alpha=0.45))
            ax.plot([_row["med"]], [_y + _dy], marker=_mk, ms=(11 if _mk == "*" else 7.5 if _mk == "|" else 6.5),
                    mfc=PFC[_g] if _mk != "|" else PFC[_g], mec=PFC[_g] if _mk == "|" else "k", mew=1.4 if _mk == "|" else 0.7,
                    ls="none", zorder=5)
            ax.text(_row["med"], _y + _dy + 0.055, f"{_row['med']:+.2f}", ha="center", va="bottom", fontsize=8.45,
                    color=PFC[_g], zorder=6)
        _n = P15[(P15["box"] == _b) & (P15["grp"] == _g) & (P15["event"] == "t_jet")]["n"]
        ax.text(_xlo + 0.05, _y + 0.33, f"{GSH.get(_g, _g)}  (N {int(_n.iloc[0]) if len(_n) else 0})",
                fontsize=11.1, color=PFC[_g], va="top", fontweight="bold")
    ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.7)
    ax.text(0.0, 1.62, r" $t_{q,\,\rm start}$", fontsize=10.4, color="0.25", va="top")
    ax.set_yticks([]); ax.set_ylim(-0.6, 1.6)
    ax.set_ylabel(BOX_LABEL.get(_b, _b).split(" ")[0], fontsize=13.7, fontweight="bold")
    ax.set_xlim(_xlo, _xhi)
    for _s in ("top", "right", "left"):
        ax.spines[_s].set_visible(False)
    ax.set_axisbelow(True)
AX[-1, 0].set_xlabel(r"$t - t_{\rm SFT}$  [Gyr]")
fig.suptitle("The two pathways as a sequence: the median of each event on the truncation clock, with its 16-84 spread "
             "(Kaplan-Meier quartiles for the dust)", y=0.995, fontsize=14.3)
fig.tight_layout(rect=[0, 0.07, 1, 0.965], h_pad=1.2)
fig.legend(handles=[Line2D([], [], color="0.3", marker=mk, ls="none", ms=(10 if mk == "*" else 7), mfc="0.3",
                           mec="0.3" if mk == "|" else "k", label=lab) for _, lab, mk in PF6_EV]
           + [Line2D([], [], color=PFC[g], lw=6, alpha=0.3, label=GLAB.get(g, g)) for g in GK],
           loc="lower center", ncol=4, fontsize=11.1, frameon=False)
paper_save(fig, PF6_STEM); plt.show()

P15.to_csv(os.path.join(OUT, f"{PF6_STEM}.csv"), index=False)
print("THE TIMELINE  —  median [16-84] of each event relative to the truncation, in Gyr (t_dark: KM median [quartiles]):")
_evw = max(len(l) for _, l, _ in PF6_EV)
for _b in PF6_BOXES:
    print(f"  {_b}")
    for _ev, _lab, _ in PF6_EV:
        _line = f"    {_lab.replace('$', '').replace(chr(92), '')[:40]:42s}"
        for _g in GK:
            _row = P15[(P15["box"] == _b) & (P15["grp"] == _g) & (P15["event"] == _ev)]
            if not len(_row) or not np.isfinite(_row["med"].iloc[0]):
                _line += f"  {GSH.get(_g, _g)}: n/a"; continue
            _row = _row.iloc[0]
            _line += (f"  {GSH.get(_g, _g)} {_row['med']:+5.2f} [{_row['lo']:+.2f}, "
                      + (f"{_row['hi']:+.2f}]" if np.isfinite(_row["hi"]) else ">win]")
                      + f" (N {int(_row['n'])}" + (f", {_row['censored'] * 100:.0f} % censored" if np.isfinite(_row["censored"]) else "") + ")")
        print(_line)
print("table ->", os.path.join(OUT, f"{PF6_STEM}.csv"))

## Part 16 — paper figure 7: the observer's guidance

Part 9 asked the observer's question once — one dust threshold, one morphology cut, against redshift. This is
the version an observer can **use**: one panel per anchor of the ladder, the probability that a quenched
galaxy selected as **disky** is **dusty**, as a continuous function of how dusty "dusty" has to be (the
abscissa, $\log M_{\rm dust}/M_\star \geq x$) and for several degrees of "disky" (the curves), with the share
of the quenched population each cut keeps written in the panel — so the conditional probability is the
curve and the joint one is the curve times that share (`PF7_MODE = "joint"` draws the joint directly).

**"Disky" is stellar.** It is Part 9's two-component reading of $\kappa_{\rm rot}(\star)$ as a bulge-to-total
ratio, $B/T = (\kappa_{\rm disc} - \kappa_\star)/(\kappa_{\rm disc} - 1/3)$ with $\kappa_{\rm disc}$ =
`PF7_KDISC`: a relabelling of $\kappa_\star$ in units an observer recognises — monotone in $\kappa_\star$, so it
cannot rank galaxies differently, and a *measured* $B/T$ needs a bulge–disc decomposition of the images. The
legend gives the $\kappa_\star$ each cut corresponds to.

**The sample is the whole quenched population** of the box at each anchor, under Part 9's rule (`PF7_RULE`),
not the peel's galaxies: this is a forecast, not the selection. Bands are the 68 % Beta-posterior credible intervals of each fraction (Part 0's `frac_band`, `BETA_PRIOR` flat or Jeffreys; the sampling band of one box, not its cosmic variance); a panel
needs `PF7_NMIN` quenched galaxies and a curve `PF7_CMIN` disky ones.

**Knobs.** `PF7_BOX`, `PF7_ZS` (`None` = every anchor with enough galaxies), `PF7_BT` (the degrees of disky),
`PF7_XG` (the range of dust thresholds), `PF7_MODE`, `PF7_ALL` (the no-morphology curve), `PF7_SHARE`,
`PF7_XMARK`, `PF7_NCOL`, `PF7_COL`.

In [ ]:
# ── Part 16 — PAPER FIGURE 7: THE OBSERVER'S GUIDANCE — a disky quenched galaxy: how likely is it dusty, at
#    each redshift, for every degree of "disky" and every degree of "dusty" ──────────────────────────────────
# Part 9 asked it once, at one threshold and one morphology cut, against redshift.  This is the version an
# observer can USE: one panel per anchor of the ladder, the probability that a quenched galaxy selected as
# disky is dusty, as a CONTINUOUS function of how dusty "dusty" has to be (the abscissa) and for several
# degrees of "disky" (the curves), with the share of the quenched population each cut keeps written in the
# panel — so the conditional and the joint probability are both on the page.
# "Disky" is STELLAR: Part 9's two-component reading of kappa_rot(*) as a bulge-to-total ratio,
#     B/T = (kappa_disc - kappa_*) / (kappa_disc - 1/3),   kappa_disc = P16_KDISC,
# a relabelling of kappa_* in units an observer recognises (monotone in kappa_*, so it cannot rank galaxies
# differently; a measured B/T needs a bulge-disc decomposition of the images).
# The sample is the WHOLE quenched population of the box at each anchor — the catalogue under Part 9's rule —
# not the peel's galaxies: this is a forecast, not the selection.
PF7_STEM  = "ba_paper_guidance"
PF7_BOX   = "cis100"          # the box with the statistics (m50n512 has 1/8 of the volume)
PF7_ZS    = [0.3, 0.7, 1.0, 1.3, 1.8, 2.0]   # the anchors nearest these (None = every anchor with PF7_NMIN QGs)
PF7_BT    = [0.95, 0.85, 0.75]  # "disky" = B/T at or below this — the diskiest cut first.  WHERE THE POPULATION
                              # IS: under kappa_disc = 0.85 the median quenched galaxy of m100n1024 sits at
                              # kappa_* = 0.30, i.e. B/T = 1 — the relabelling saturates — and B/T <= 0.8 / 0.9 /
                              # 0.95 (kappa_* >= 0.44 / 0.39 / 0.36) are its 5 / 13 / 20 % most rotation-
                              # supported members.  B/T <= 0.7 keeps 1.5 %, B/T <= 0.5 a few dozen galaxies.
PF7_KDISC = P9_KDISC if "P9_KDISC" in dir() else 0.85     # the kappa_rot a pure disc is given (Part 9)
PF7_RULE  = "ssfr"            # the quenched rule, as in Part 9: "ssfr" | "notebook"
PF7_MASS  = MASS_FLOOR
PF7_XG    = np.linspace(-5.5, -2.5, 61)      # the dust threshold: "dusty" = log M_dust/M* at or above x
PF7_NMIN  = 80                # quenched galaxies an anchor needs to get a panel
PF7_CMIN  = 10                # disky galaxies a curve needs
PF7_MODE  = "conditional"     # "conditional" = P(dusty | disky QG);  "joint" = P(disky AND dusty | QG)
PF7_NCOL  = 3
PF7_YMAX  = 80                # the y axis stops here [%]: no curve goes above it and the rest is room for the labels
PF7_COL   = ["#5c2a0a", "#b5541b", "#e0a06a", "#f0d0a8"]   # the cuts, diskiest first, in dust: dark to pale
                              # brown; every QG is grey
PF7_ALL   = True              # the no-morphology curve: every quenched galaxy
PF7_SHARE = True              # write the share of the quenched population each cut keeps
PF7_XMARK = [DUSTY_CUT]       # thresholds marked as vertical lines
from matplotlib.patches import Patch
if "frac_band" not in globals():
    raise RuntimeError("Part 16 needs Part 0 (frac_band, CAT)")

_C16 = CAT[CAT["box"] == PF7_BOX].copy()
if not len(_C16):
    raise RuntimeError(f"Part 16: {PF7_BOX} is not in the catalogue")
with np.errstate(all="ignore"):
    _ms = _C16["mstar"].to_numpy(float)
    _C16["lm"] = np.log10(np.where(_ms > 0, _ms, np.nan))
    _C16["ssfr"] = _C16["sfr"].to_numpy(float) / np.where(_ms > 0, _ms, np.nan)
    _fd = _C16["mdust"].to_numpy(float) / np.where(_ms > 0, _ms, np.nan)
    _C16["lfd"] = np.where(_fd > 0, np.log10(np.where(_fd > 0, _fd, 1.0)), FLOOR)
    _q = (_C16["lm"].to_numpy(float) > PF7_MASS) & (_C16["ssfr"].to_numpy(float)
                                                    < PASSIVE_FACTOR / (_C16["t_H_gyr"].to_numpy(float) * 1e9))
    if PF7_RULE == "notebook":
        _q &= ((_C16["ngas"].to_numpy(float) >= NGAS_MIN) & (_C16["nstar"].to_numpy(float) >= NSTAR_MIN)
               & (_C16["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * _C16["mh2"].to_numpy(float)))
    _C16["bt"] = np.clip((PF7_KDISC - _C16["kappa_star"].to_numpy(float)) / (PF7_KDISC - KAPPA_ISO), 0.0, 1.0)
Q16 = _C16[_q & np.isfinite(_C16["bt"].to_numpy(float))].copy()
_kap_of = lambda b: PF7_KDISC - b * (PF7_KDISC - KAPPA_ISO)          # the kappa_* a B/T cut corresponds to
_zall = sorted(Q16["z"].round(3).unique())
if PF7_ZS is None:
    _Z16 = [z for z in _zall if int((Q16["z"].round(3) == z).sum()) >= PF7_NMIN]
else:
    _Z16 = sorted({min(_zall, key=lambda a: abs(a - float(z))) for z in PF7_ZS})
if not _Z16:
    raise RuntimeError(f"Part 16: no anchor of {PF7_BOX} holds {PF7_NMIN} quenched galaxies")
_CUTS = [(f"B/T ≤ {b:g}", b) for b in PF7_BT] + ([("every quenched galaxy", None)] if PF7_ALL else [])
_CCOL = {lab: (PF7_COL[i] if b is not None else "0.45") for i, (lab, b) in enumerate(_CUTS)}


def _pf7_curve(lfd_sel, n_all):
    """P(dusty at x | selected) — or, in the joint mode, P(selected and dusty at x) — with its 68 % Beta-posterior
    band (Part 0's frac_band, BETA_PRIOR), for every threshold of PF7_XG at once (a sorted sample and one searchsorted)."""
    v = np.sort(np.asarray(lfd_sel, float)); v = v[np.isfinite(v)]; n = len(v)
    k = n - np.searchsorted(v, PF7_XG, side="left")          # how many are at or above each threshold
    P, L, H = frac_band(k, n if PF7_MODE == "conditional" else n_all)
    return P, L, H, n


import time as _time
_t16 = _time.perf_counter()
ROW16 = []
_nr16 = int(np.ceil(len(_Z16) / PF7_NCOL))
with plt.rc_context(PAPER_RC):
    fig, AX = plt.subplots(_nr16, PF7_NCOL, figsize=(4.7 * PF7_NCOL + 0.4, 3.9 * _nr16 + 1.2), squeeze=False,
                           sharex=True, sharey=True)
    for _i, _z in enumerate(_Z16):
        ax = AX[_i // PF7_NCOL, _i % PF7_NCOL]
        _S = Q16[Q16["z"].round(3) == _z]
        _nq = len(_S)
        _shares = []
        for _lab, _b in _CUTS:
            _sel = (_S["bt"].to_numpy(float) <= _b) if _b is not None else np.ones(_nq, bool)
            _n = int(_sel.sum())
            _shares.append((_lab, _b, _n / _nq if _nq else np.nan, _n))
            if _n < PF7_CMIN:
                continue
            _P, _L, _H, _ = _pf7_curve(_S["lfd"].to_numpy(float)[_sel], _nq)
            ax.fill_between(PF7_XG, 100 * _L, 100 * _H, color=_CCOL[_lab], alpha=0.16, lw=0)
            ax.plot(PF7_XG, 100 * _P, color=_CCOL[_lab], lw=2.6 if _b is not None else 2.0,
                    ls="-" if _b is not None else "--", zorder=4)
            for _x, _p, _l, _h in zip(PF7_XG, _P, _L, _H):
                ROW16.append(dict(z=_z, cut=_lab, bt_max=_b, kappa_min=(_kap_of(_b) if _b is not None else np.nan),
                                  n_qg=_nq, n_sel=_n, share_sel=_n / _nq, x=float(_x), p=_p, lo=_l, hi=_h,
                                  mode=PF7_MODE))
        for _xm in PF7_XMARK:
            ax.axvline(_xm, color="0.35", ls=":", lw=1.3, zorder=1)
        ax.text(0.96, 0.95, rf"$z = {_z:.2f}$", transform=ax.transAxes, ha="right", va="top", fontsize=22.1,
                fontweight="bold", color="0.15", bbox=dict(boxstyle="round,pad=0.35", fc="w", ec="0.45", lw=1.1))
        ax.text(0.96, 0.78, rf"$N_{{\rm QG}} = {_nq}$", transform=ax.transAxes, ha="right", va="top",
                fontsize=20.2, color="0.3")
        if PF7_SHARE:
            for _j, (_lab, _b, _sh, _n) in enumerate([s for s in _shares if s[1] is not None]):
                ax.text(0.96, 0.65 - 0.115 * _j, f"{_lab}:  {_sh * 100:.0f} %", transform=ax.transAxes,
                        ha="right", va="top", fontsize=19.5, color=_CCOL[_lab])
        ax.set_ylim(0, PF7_YMAX)
        ax.set_xlim(PF7_XG[0], PF7_XG[-1])
        ax.set_xticks([t for t in np.arange(-5.0, -2.9, 0.5)])          # no tick on the shared edges
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f} %"))
        ax.set_axisbelow(True)
        for _s in ("top", "right"):
            ax.spines[_s].set_visible(False)
        if _i % PF7_NCOL == 0:
            ax.set_ylabel(r"$P(\rm dusty \mid disky\ QG)$" if PF7_MODE == "conditional"
                          else r"$P(\rm disky\ and\ dusty \mid QG)$")
        if _i // PF7_NCOL == _nr16 - 1 or _i + PF7_NCOL >= len(_Z16):
            ax.set_xlabel(r"dusty if  $\log M_{\rm dust}/M_\star \geq$")
    for _i in range(len(_Z16), _nr16 * PF7_NCOL):
        AX[_i // PF7_NCOL, _i % PF7_NCOL].axis("off")
    _h16 = [Line2D([], [], color=_CCOL[lab], lw=2.6, label=f"disky: {lab}") for lab, b in _CUTS if b is not None]
    # (the kappa_* each B/T cut corresponds to is in the printout; the figure speaks B/T only)
    if PF7_ALL:
        _h16 += [Line2D([], [], color="0.45", lw=2.0, ls="--", label="every quenched galaxy, no morphology cut")]
    _h16 += [Patch(fc="0.5", alpha=0.25, ec="none", label=f"68 % credible interval ({BETA_PRIOR} prior)"),
             Line2D([], [], color="0.35", ls=":", lw=1.3, label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$")]
    fig.tight_layout(rect=[0, 0.15 if _nr16 <= 2 else 0.11, 1, 1], h_pad=1.6, w_pad=1.2)
    fig.legend(handles=_h16, loc="lower center", ncol=3, frameon=False, fontsize=18.9)
    _t16b = _time.perf_counter()
    paper_save(fig, PF7_STEM); plt.show()
    print(f"  [Part 16: {_t16b - _t16:.1f} s to build the figure, {_time.perf_counter() - _t16b:.1f} s to save it at {PAPER_DPI} dpi "
          f"— if the second number is the slow one, lower PAPER_DPI in Part 0]")

P16 = pd.DataFrame(ROW16)
P16.to_csv(os.path.join(OUT, f"{PF7_STEM}.csv"), index=False)
print(f"THE OBSERVER'S GUIDANCE ({BOX_LABEL.get(PF7_BOX, PF7_BOX)}; quenched = {PF7_RULE!r}, log M* > {PF7_MASS:g}; "
      f"disky = B/T at or below the cut with kappa_disc = {PF7_KDISC:g}, i.e. kappa_* at or above "
      + ", ".join(f"{_kap_of(b):.2f} (B/T {b:g})" for b in PF7_BT) + f"; mode = {PF7_MODE})")
_plain = lambda lab: lab.replace("≤", "<=")
print(f"    {'z':>5s} {'N_QG':>6s}" + "".join(f"{_plain(lab):>34s}" for lab, _ in _CUTS))
print(f"    {'':>5s} {'':>6s}" + "".join(f"{'share   P(-4.0)  P(-3.5)  P(-3.0)':>34s}" for _ in _CUTS))
for _z in _Z16:
    _line = f"    {_z:5.2f} {int((Q16['z'].round(3) == _z).sum()):6d}"
    for _lab, _b in _CUTS:
        _r = P16[(P16["z"] == _z) & (P16["cut"] == _lab)]
        if not len(_r):
            _line += f"{'too few':>34s}"; continue
        _pv = [float(np.interp(x, _r["x"], _r["p"])) for x in (-4.0, -3.5, -3.0)]
        _line += f"   {_r['share_sel'].iloc[0] * 100:4.0f} %  " + "  ".join(f"{p * 100:5.0f} %" for p in _pv)
    print(_line)
print("    share = the fraction of the quenched population the cut keeps (P(disky | QG)); P(x) = the probability that\n"
      "    one of those has log M_dust/M* >= x.  The two multiplied is the joint probability (PF7_MODE = 'joint' draws it).")
_kq = np.nanpercentile(Q16["kappa_star"].to_numpy(float), [50, 84, 95, 99])
print(f"    WHERE THE POPULATION IS: kappa_* of the quenched galaxies = {_kq[0]:.2f} (median), {_kq[1]:.2f} (84th), "
      f"{_kq[2]:.2f} (95th), {_kq[3]:.2f} (99th) — under kappa_disc = {PF7_KDISC:g} the median quenched galaxy has B/T = "
      f"{np.clip((PF7_KDISC - _kq[0]) / (PF7_KDISC - KAPPA_ISO), 0, 1):.2f}: the relabelling saturates, which is why the\n"
      "    cuts sit at B/T 0.8-0.95 and are labelled by kappa_* first.  PF7_KDISC calibrated on SIMBA's own star-forming discs\n"
      "    (their median kappa_* is ~0.43) would map the same cuts to lower B/T without changing a single curve.")
print("table ->", os.path.join(OUT, f"{PF7_STEM}.csv"))

## Part 16b — paper figure 7b: the dusty fraction along the morphology sequence

Part 16 answers "how likely is a disky quenched galaxy dusty" for nested cuts against the population average. This
is the version that makes the trend itself the figure: **every** quenched galaxy of the anchor is put in exactly one
B/T class (`PF7B_EDGES`: B/T ≤ 0.8, 0.8–0.9, 0.9–1, and the pure spheroids, B/T = 1, i.e. $\kappa_\star \leq 1/3$
where the relabelling saturates — `PF7B_SAT` keeps them as their own class), and each class gets its own
$P(\text{dusty at } x)$ normalised by its own size, so the class populations cannot enter the curves and equal
numbers of each are compared. The lower strip is every class divided by the pure spheroids — how many dusty galaxies
an observer finds per galaxy of that class for every one found per pure spheroid — with a band drawn numerically
from the Beta posteriors (`PF7B_NDRAW` draws, drawn only where both classes hold `PF7B_KMIN` dusty galaxies). `PF7B_DGR` writes on each panel the typical dust-to-molecular-gas ratio of every class, $\delta_{\rm DGR} = \log
M_{\rm dust}/M_{\rm H_2}$ (helium-corrected as Part 1's `ldh2`, `PF7B_DGR_HE`), the median over the class's galaxies with
H$_2$ and dust, in the class colours; `PF7B_YMARK` marks the 30 % level with a line and a
light band below it, and the probability axis is set per row (`PF7B_YMAX` None) so the low-redshift row is not squashed by
the high-redshift one. Nothing else is written on the figure: $N_{\rm QG}$, the dusty-disc / dusty-spheroid counts, the AUC
of $\kappa_\star$ between dusty and other quenched galaxies, $P(\text{ordered})$ and the population-wide Spearman $\rho$
(≈ 0: the link is in the dusty tail, not the bulk) are in the printout and the CSV (`PF7B_STATS_FULL`). The ratio strip is drawn as its own
panel — tinted, framed, set off from the probabilities — because it is a separate quantity, not a footer. The
printout gives the same thing as counts per 100 galaxies of each class at $f_{\rm dust} \geq 10^{-4}$. Same sample
and rule as Part 16 (`Q16`), which must have run.


In [ ]:
# ── Part 16b — PAPER FIGURE 7b: THE DUSTY FRACTION ALONG THE MORPHOLOGY SEQUENCE — the more spheroidal, the less
#    dusty?  (2026-09-18; v2 the same day: exclusive B/T classes instead of one cut) ────────────────────────────
# Part 16 draws P(dusty | disky QG) for NESTED cuts against the population average; read quickly, the shares of the
# population each cut keeps look like the probabilities.  Here every quenched galaxy is in exactly ONE class of B/T
# and each class gets its own P(dusty at x), normalised by ITS OWN size — equal numbers of each class are compared —
# and the ratio of every class to the pure spheroids, with a band from the Beta posteriors, is the trend in numbers.
PF7B_STEM  = "ba_paper_guidance_contrast"
PF7B_EDGES = [0.8, 0.9]      # THE CLASSES, by B/T: <= edges[0], then (edges[i-1], edges[i]], then (edges[-1], 1), then
                             #   B/T = 1 (PF7B_SAT).  Under kappa_disc = 0.85: B/T 0.8 / 0.9 = kappa_* 0.44 / 0.39.
PF7B_SAT   = True            # the galaxies the relabelling SATURATES (kappa_* <= 1/3, B/T clipped to 1 — the pure
                             #   spheroids, ~60 % of the quenched population) as their own class and the REFERENCE
                             #   of the ratio strip; False folds them into the last bin
PF7B_ZS    = None            # None = Part 16's anchors (PF7_ZS) ; or a list of redshifts
PF7B_XG    = PF7_XG          # the dust thresholds
PF7B_KMIN  = 3               # the ratio is drawn only where BOTH classes still hold this many dusty galaxies
PF7B_NDRAW = 4000            # the ratio band: draws from the two Beta posteriors — numerical, no approximation
PF7B_RLIM  = (0.5, 20.0)     # the ratio axis (log)
PF7B_RBG   = "#f1f1ee"       # the ratio strip's own background: it is its own piece of information, not a footer
PF7B_RGAP  = 0.12            # the gap between a probability panel and its ratio strip (fraction of the strip's height)
PF7B_NCOL  = 3
PF7B_COL   = ["#5c2a0a", "#b5541b", "#e0a06a", "#4a6fa5", "#9aa7b8"]   # the classes, diskiest first; the reference
                             #   (the pure spheroids) always takes the LAST colour
PF7B_SEED  = SEED
PF7B_DGR   = True            # THE BLOCK ON EACH PANEL (2026-09-18, v4): the typical dust-to-molecular-gas ratio of every
                             #   class, delta_DGR = log M_dust / M_H2 (PF7B_DGR_HE: M_H2 x HE, Part 1's ldh2 convention — SIMBA's
                             #   M_H2 is hydrogen only), the PF7B_DGR_STAT ("median" | "mean") over the class's galaxies that
                             #   hold H2 and dust, in the class colours.  No counts, no test statistics on the figure: those
                             #   go to the printout and the CSV (PF7B_STATS_FULL).
PF7B_DGR_STAT = "median"
PF7B_DGR_HE = True
PF7B_STATS_FS = 15.0         # the block's font size
PF7B_ZFS   = 17.0            # the redshift label's font size (smaller than the block, which sits under it)
PF7B_DISC_BT = 0.85          # the printout's dusty DISCS: B/T at or below this and f_dust >= DUSTY_CUT ...
PF7B_SPH_BT  = 0.9           # ... and its dusty SPHEROIDS: B/T above this
PF7B_STATS_FULL = True       # the printout: N_QG, dusty discs vs dusty spheroids, AUC(kappa_*), P(ordered), rho_S per anchor
PF7B_YMARK = 30.0            # THE LEVEL TO EMPHASISE [%]: a marked line with a light band below it on every probability panel
PF7B_YMAX  = None            # the probability axis: None = per ROW of panels, the smallest multiple of 10 % above every band
                             #   in the row (the low-z row then stops at ~50 % instead of 80 %); a number fixes it everywhere
from matplotlib.gridspec import GridSpec
if "Q16" not in globals():
    raise RuntimeError("Part 16b reads Part 16's quenched sample Q16 — run Part 16 first")
_Z16B = _Z16 if PF7B_ZS is None else sorted({min(_zall, key=lambda a: abs(a - float(z))) for z in PF7B_ZS})
_A16B = {"flat": 1.0, "jeffreys": 0.5}[BETA_PRIOR]
# THE CLASSES: (label, mask function of (bt, kappa_*)), diskiest first, the reference last
_E = sorted(float(e) for e in PF7B_EDGES)
_CLS16B = [(rf"B/T $\leq$ {_E[0]:g}", lambda bt, k, e=_E[0]: bt <= e)]
for _lo, _hi in zip(_E[:-1], _E[1:]):
    _CLS16B.append((rf"B/T {_lo:g}–{_hi:g}", lambda bt, k, lo=_lo, hi=_hi: (bt > lo) & (bt <= hi)))
if PF7B_SAT:
    _CLS16B.append((rf"B/T {_E[-1]:g}–1", lambda bt, k, lo=_E[-1]: (bt > lo) & (k > KAPPA_ISO)))
    _CLS16B.append((r"B/T $= 1$  (pure spheroids, $\kappa_\star \leq 1/3$)", lambda bt, k: k <= KAPPA_ISO))
else:
    _CLS16B.append((rf"B/T $>$ {_E[-1]:g}", lambda bt, k, lo=_E[-1]: bt > lo))
_NC = len(_CLS16B)
_COL16B = {lab: (PF7B_COL[-1] if i == _NC - 1 else PF7B_COL[i]) for i, (lab, _) in enumerate(_CLS16B)}
_REF = _CLS16B[-1][0]


def _counts16b(lfd):
    """How many of `lfd` are at or above every threshold of PF7B_XG (a sorted sample, one searchsorted)."""
    v = np.sort(np.asarray(lfd, float)); v = v[np.isfinite(v)]
    return len(v) - np.searchsorted(v, PF7B_XG, side="left"), len(v)


def _ratio16b(k1, n1, k2, n2):
    """P(dusty | class) / P(dusty | reference) at every threshold: the observed ratio, and the 16-84 band of the
    ratio of PF7B_NDRAW draws from the two Beta posteriors.  NaN where either holds < PF7B_KMIN dusty."""
    rng = np.random.default_rng(PF7B_SEED)
    p1 = rng.beta(k1[None, :] + _A16B, n1 - k1[None, :] + _A16B, size=(PF7B_NDRAW, len(k1)))
    p2 = rng.beta(k2[None, :] + _A16B, n2 - k2[None, :] + _A16B, size=(PF7B_NDRAW, len(k2)))
    with np.errstate(all="ignore"):
        lo, hi = np.nanpercentile(p1 / p2, [16, 84], axis=0)
        pt = (k1 / n1) / np.where(k2 > 0, k2 / n2, np.nan)
    ok = (k1 >= PF7B_KMIN) & (k2 >= PF7B_KMIN)
    return np.where(ok, pt, np.nan), np.where(ok, lo, np.nan), np.where(ok, hi, np.nan)


ROW16B, _EQ16B = [], []
_nblk = int(np.ceil(len(_Z16B) / PF7B_NCOL))
with plt.rc_context(PAPER_RC):
    fig = plt.figure(figsize=(4.7 * PF7B_NCOL + 0.4, 5.6 * _nblk + 1.6))
    from matplotlib.gridspec import GridSpecFromSubplotSpec
    _gs = GridSpec(_nblk, PF7B_NCOL, figure=fig, hspace=0.16, wspace=0.14)
    _AXP, _AXR = [], []
    for _i, _z in enumerate(_Z16B):
        _b, _c = divmod(_i, PF7B_NCOL)
        _gsi = GridSpecFromSubplotSpec(2, 1, subplot_spec=_gs[_b, _c], height_ratios=[1.0, 0.5], hspace=PF7B_RGAP)
        axp = fig.add_subplot(_gsi[0], sharey=_AXP[_b * PF7B_NCOL] if _c else None)      # y shared along a ROW
        axr = fig.add_subplot(_gsi[1], sharex=axp, sharey=_AXR[0] if _AXR else None)
        _AXP.append(axp); _AXR.append(axr)
        _S = Q16[Q16["z"].round(3) == _z]
        _bt = _S["bt"].to_numpy(float); _ks = _S["kappa_star"].to_numpy(float); _lfd = _S["lfd"].to_numpy(float)
        _K, _N, _M16B, _ytop = {}, {}, {}, 0.0
        for _lab, _fn in _CLS16B:
            _m = _fn(_bt, _ks); _M16B[_lab] = _m
            _K[_lab], _N[_lab] = _counts16b(_lfd[_m])
            if _N[_lab] < PF7_CMIN:
                continue
            _P, _L, _H = frac_band(_K[_lab], _N[_lab])
            _ytop = max(_ytop, float(np.nanmax(100 * _H)))
            axp.fill_between(PF7B_XG, 100 * _L, 100 * _H, color=_COL16B[_lab], alpha=0.16, lw=0)
            axp.plot(PF7B_XG, 100 * _P, color=_COL16B[_lab], lw=2.6 if _lab != _REF else 2.2,
                     ls="-" if _lab != _REF else (0, (5, 2)), zorder=4)
        _j = int(np.argmin(np.abs(PF7B_XG - DUSTY_CUT)))
        if PF7B_DGR:
            with np.errstate(all="ignore"):
                _mg = _S["mh2"].to_numpy(float) * (HE if PF7B_DGR_HE else 1.0); _md = _S["mdust"].to_numpy(float)
                _ldg = np.where((_mg > 0) & (_md > 0), np.log10(np.where(_mg > 0, _md, 1.0) / np.where(_mg > 0, _mg, 1.0)), np.nan)
            _SHORT = {lab: (r"B/T $= 1$" if lab == _REF else lab) for lab, _ in _CLS16B}
            _dgr = {lab: (float(np.nanmedian(_ldg[_M16B[lab]])) if PF7B_DGR_STAT == "median" else float(np.nanmean(_ldg[_M16B[lab]])))
                    if np.isfinite(_ldg[_M16B[lab]]).sum() >= 3 else np.nan for lab, _ in _CLS16B}
            _y0, _dy = 0.78, 0.088
            _bb = dict(boxstyle="square,pad=0.12", fc="w", ec="none", alpha=0.8)
            axp.text(0.96, _y0, rf"{PF7B_DGR_STAT} $\delta_{{\rm DGR}}$", transform=axp.transAxes, ha="right", va="top",
                     fontsize=PF7B_STATS_FS, color="0.3", zorder=9, bbox=_bb)
            for _k, (_lab, _) in enumerate(_CLS16B):
                axp.text(0.96, _y0 - _dy * (_k + 1), rf"{_SHORT[_lab]}:  ${_dgr[_lab]:.2f}$" if np.isfinite(_dgr[_lab]) else f"{_SHORT[_lab]}:  –",
                         transform=axp.transAxes, ha="right", va="top", fontsize=PF7B_STATS_FS, color=_COL16B[_lab], zorder=9, bbox=_bb)
        if PF7B_STATS_FULL:
            _rngS = np.random.default_rng(PF7B_SEED + 1)
            _drw = [_rngS.beta(_K[l][_j] + _A16B, _N[l] - _K[l][_j] + _A16B, PF7B_NDRAW) for l, _ in _CLS16B if _N[l] >= PF7_CMIN]
            _pord = float(np.mean(np.all(np.diff(np.vstack(_drw), axis=0) < 0, axis=0))) if len(_drw) >= 2 else np.nan
            _ptop = float(np.mean(_drw[0] > _drw[-1])) if len(_drw) >= 2 else np.nan
            _okS = np.isfinite(_ks) & np.isfinite(_lfd)
            _rho, _pv = spearmanr(_ks[_okS], _lfd[_okS]) if _okS.sum() >= 10 else (np.nan, np.nan)
            _dm = _okS & (_lfd >= DUSTY_CUT)                       # the dusty QGs of the anchor, against all the others
            _auc, _pauc = np.nan, np.nan
            if _dm.sum() >= PF7_CMIN and (_okS & ~_dm).sum() >= PF7_CMIN:
                _auc, _pauc, _, _ = mw_auc(_ks[_okS], _dm[_okS])      # Part 0: (AUC, Mann-Whitney p, n1, n0)
            _ptxt = ((f"$p < 10^{{-3}}$" if _pauc < 1e-3 else f"$p = {_pauc:.3f}$" if _pauc < 0.01 else f"$p = {_pauc:.2f}$")
                     if np.isfinite(_pauc) else "")
            _dusty = _okS & (_lfd >= DUSTY_CUT)
            _ndd = int((_dusty & (_bt <= PF7B_DISC_BT)).sum()); _nds = int((_dusty & (_bt > PF7B_SPH_BT)).sum())
            ROW16B.append(dict(z=_z, cls="(stats)", x=np.nan, n_cls=len(_S), n_ref=_N[_REF], k_cls=-1, k_ref=-1,
                               p_cls=np.nan, p_ref=np.nan, ratio=np.nan, lo=np.nan, hi=np.nan,
                               n_dusty_discs=_ndd, n_dusty_sph=_nds,
                               **({f"ldg_{i}": _dgr[lab] for i, (lab, _) in enumerate(_CLS16B)} if PF7B_DGR else {}),
                               auc=_auc, p_auc=_pauc, rho_s=_rho, p_rho=_pv, p_ordered=_pord, p_top_gt_ref=_ptop))
        for _lab, _ in _CLS16B[:-1]:
            if _N[_lab] < PF7_CMIN or _N[_REF] < PF7_CMIN:
                continue
            _R, _RL, _RH = _ratio16b(_K[_lab], _N[_lab], _K[_REF], _N[_REF])
            axr.fill_between(PF7B_XG, _RL, _RH, color=_COL16B[_lab], alpha=0.14, lw=0)
            axr.plot(PF7B_XG, _R, color=_COL16B[_lab], lw=2.2, zorder=4)
            for _x, _r, _l, _h, _k1, _k2 in zip(PF7B_XG, _R, _RL, _RH, _K[_lab], _K[_REF]):
                ROW16B.append(dict(z=_z, cls=_lab, x=float(_x), n_cls=_N[_lab], n_ref=_N[_REF], k_cls=int(_k1),
                                   k_ref=int(_k2), p_cls=_k1 / _N[_lab], p_ref=_k2 / _N[_REF], ratio=_r, lo=_l, hi=_h))
            _EQ16B.append((_z, _lab, _N[_lab], 100 * _K[_lab][_j] / _N[_lab], _R[_j], _RL[_j], _RH[_j]))
        if _N[_REF] >= PF7_CMIN:
            _EQ16B.append((_z, _REF, _N[_REF], 100 * _K[_REF][_j] / _N[_REF], 1.0, np.nan, np.nan))
        for ax in (axp, axr):
            for _xm in PF7_XMARK:
                ax.axvline(_xm, color="0.35", ls=":", lw=1.3, zorder=1)
            ax.set_axisbelow(True)
        for _s in ("top", "right"):
            axp.spines[_s].set_visible(False)
        # THE RATIO STRIP IS ITS OWN PANEL: tinted, fully framed, a gap above it, the quantity on its axis
        axr.set_facecolor(PF7B_RBG)
        for _s in axr.spines.values():
            _s.set_visible(True); _s.set_linewidth(1.2); _s.set_color("0.35")
        axr.axhline(1.0, color=_COL16B[_REF], ls=(0, (5, 2)), lw=1.8, zorder=1)
        axr.set_yscale("log"); axr.set_ylim(*PF7B_RLIM)
        axr.set_yticks([1, 3, 10]); axr.set_yticklabels(["1", "3", "10"]); axr.minorticks_off()
        if PF7B_YMARK:                                       # the level the paper's numbers live under
            axp.axhspan(0, PF7B_YMARK, color="0.5", alpha=0.07, lw=0, zorder=0)
            axp.axhline(PF7B_YMARK, color="0.25", ls=(0, (1, 1.5)), lw=1.4, zorder=1)
        _YT16B = globals().setdefault("_YT16B", {}); _YT16B[(_b, _c)] = _ytop
        axp.set_xlim(PF7B_XG[0], PF7B_XG[-1])
        axp.set_xticks([t for t in np.arange(-5.0, -2.9, 0.5)])
        axp.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f} %"))
        plt.setp(axp.get_xticklabels(), visible=False)
        axp.text(0.96, 0.955, rf"$z = {_z:.2f}$", transform=axp.transAxes, ha="right", va="top", fontsize=PF7B_ZFS,
                 fontweight="bold", color="0.15", bbox=dict(boxstyle="round,pad=0.3", fc="w", ec="0.45", lw=1.1))
        if _c == 0:
            axp.set_ylabel(r"$P(\rm dusty \mid QG)$")
            axr.set_ylabel("ratio to\nB/T $= 1$", fontsize=16)
        else:
            plt.setp(axp.get_yticklabels(), visible=False); plt.setp(axr.get_yticklabels(), visible=False)
        if _b == _nblk - 1 or _i + PF7B_NCOL >= len(_Z16B):
            axr.set_xlabel(r"dusty if  $\log M_{\rm dust}/M_\star \geq$")
        else:
            plt.setp(axr.get_xticklabels(), visible=False)
        if _i == 0:
            axr.text(0.02, 0.93, r"$P(\rm dusty \mid class)\,/\,P(\rm dusty \mid B/T = 1)$", transform=axr.transAxes,
                     ha="left", va="top", fontsize=13, color="0.3")
    for _b in range(_nblk):                                  # the y range of each ROW: every band of the row fits
        _ym = PF7B_YMAX if PF7B_YMAX else min(100.0, 10.0 * np.ceil(max(v for (bb, _), v in _YT16B.items() if bb == _b) / 10.0 + 0.5))
        _AXP[_b * PF7B_NCOL].set_ylim(0, _ym)
        _AXP[_b * PF7B_NCOL].set_yticks(np.arange(0, _ym + 1e-6, 10.0 if _ym <= 60 else 20.0))
    _h16b = [Line2D([], [], color=_COL16B[lab], lw=2.6 if lab != _REF else 2.2, ls="-" if lab != _REF else (0, (5, 2)),
                    label=lab) for lab, _ in _CLS16B]
    _h16b += [Patch(fc="0.5", alpha=0.25, ec="none", label=f"68 % credible interval ({BETA_PRIOR} prior)"),
              Line2D([], [], color="0.35", ls=":", lw=1.3, label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$")]
    if PF7B_YMARK:
        _h16b += [Line2D([], [], color="0.25", ls=(0, (1, 1.5)), lw=1.4, label=rf"$P = {PF7B_YMARK:g}\,\%$")]
    fig.subplots_adjust(left=0.085, right=0.985, top=0.985, bottom=0.135 + 0.12 / _nblk)
    fig.legend(handles=_h16b, loc="lower center", ncol=3, frameon=False, fontsize=18.2, bbox_to_anchor=(0.5, 0.005))
    paper_save(fig, PF7B_STEM); plt.show()

P16B = pd.DataFrame(ROW16B)
P16B.to_csv(os.path.join(OUT, f"{PF7B_STEM}.csv"), index=False)
_plain16b = lambda s: s.replace("$\\leq$", "<=").replace("$>$", ">").replace("$= 1$", "= 1").replace(r"$\kappa_\star \leq 1/3$", "kappa_* <= 1/3").replace("  (", " (")
print(f"THE DUSTY FRACTION ALONG THE MORPHOLOGY SEQUENCE ({BOX_LABEL.get(PF7_BOX, PF7_BOX)}; quenched = {PF7_RULE!r}; "
      f"classes by B/T with kappa_disc = {PF7_KDISC:g}; the pure spheroids = kappa_* <= 1/3, where B/T saturates at 1)")
print(f"  PER 100 quenched galaxies OF EACH CLASS, how many are dusty at log M_dust/M* >= {DUSTY_CUT:g}, and the ratio to the pure spheroids:")
print(f"    {'z':>5s}  " + "".join(f"{_plain16b(lab)[:22]:>30s}" for lab, _ in _CLS16B))
for _z in _Z16B:
    _line = f"    {_z:5.2f}  "
    for _lab, _ in _CLS16B:
        _r = [e for e in _EQ16B if e[0] == _z and e[1] == _lab]
        if not _r:
            _line += f"{'too few':>30s}"; continue
        _, _, _n, _p, _rt, _l, _h = _r[0]
        _line += f"  {_n:5d}: {_p:5.1f}" + (f"  x{_rt:4.1f} [{_l:.1f},{_h:.1f}]" if np.isfinite(_l) else "   (reference)   ")
    print(_line)
print("    N: dusty per 100 [x the ratio to the pure spheroids, its 16-84 band].  Every class is normalised by its own size, so the\n"
      "    answer does not depend on how many galaxies of each class there are.")
if PF7B_STATS_FULL and (P16B["cls"] == "(stats)").any():
    _ST = P16B[P16B["cls"] == "(stats)"]
    if PF7B_DGR:
        print(f"  THE BLOCK ON EACH PANEL: the {PF7B_DGR_STAT} delta_DGR = log M_dust/M_H2{' (M_H2 x HE)' if PF7B_DGR_HE else ''} of every class "
              f"(its galaxies with H2 and dust), diskiest first:")
        for _, r in _ST.iterrows():
            print(f"    z {r.z:.2f}  " + "   ".join(f"{r[f'ldg_{i}']:6.2f}" if np.isfinite(r[f'ldg_{i}']) else "     –" for i in range(_NC)))
    print(f"  N_QG, dusty discs (B/T <= {PF7B_DISC_BT:g}, f_dust >= 10^{DUSTY_CUT:g}) and dusty spheroids (B/T > {PF7B_SPH_BT:g}):")
    for _, r in _ST.iterrows():
        print(f"    z {r.z:.2f}  N_QG {int(r.n_cls):5d}   dusty discs {int(r.n_dusty_discs):4d}   dusty spheroids {int(r.n_dusty_sph):4d}")
if PF7B_STATS_FULL and (P16B["cls"] == "(stats)").any():
    print("  THE TEST STATISTICS (printout only): AUC = P(a random DUSTY QG of the anchor rotates more than a random dust-poor one), kappa_*\n"
          "    with no class and no cut (0.5 = no link; Mann-Whitney p); P(ordered) = the posterior probability, from the Beta draws at\n"
          "    f_dust = DUSTY_CUT, that EVERY class is dustier than the next more spheroidal one; P(top > ref) = the diskiest class\n"
          "    alone against the pure spheroids.  rho_S = Spearman of kappa_* vs log f_dust over EVERYONE — ~0 at every z: the bulk\n"
          "    of the quenched population sits near the isotropic kappa_* with little dust, so the disky-dusty link is a property\n"
          "    of the dusty TAIL, not a rank correlation across the population (that is why rho_S is not on the figure).")
    for _, r in _ST.iterrows():
        print(f"    z {r.z:.2f}  N_QG {int(r.n_cls):5d}  AUC {r.auc:.3f} (p {r.p_auc:.1e})  P(ordered) {100 * r.p_ordered:5.1f} %  "
              f"P(top > ref) {100 * r.p_top_gt_ref:5.1f} %   rho_S {r.rho_s:+.3f} (p {r.p_rho:.1e})")
print("table ->", os.path.join(OUT, f"{PF7B_STEM}.csv"))


## Part 16c — where does "disky" begin? The $\kappa_\star$ (B/T) distributions of star-forming and quenched galaxies

A diagnostic for the cut Parts 16 and 16b use. At each anchor, the distribution of $\kappa_{\rm rot}(\star)$ for the
star-forming galaxies and for the quenched ones (same mass floor and rule as Part 16), with the B/T relabelling
($\kappa_{\rm disc}$ = `PF7_KDISC`) on the top axis and the cuts in play marked. B/T saturates at 1 for
$\kappa_\star \leq 1/3$, so the histogram is drawn in $\kappa_\star$, where nothing piles up. The printout gives
the percentiles of both populations, the $\kappa_\star$ where the two densities cross (the point past which a galaxy
is more likely star-forming than quenched — the natural "disky" threshold if one wants one), and the share of each
population on either side of it. Knobs: `PF7C_ZS`, `PF7C_BINS`, `PF7C_POOL` (one pooled panel instead of one per anchor).


In [ ]:
# ── Part 16c — WHERE DOES "DISKY" BEGIN?  kappa_*(and B/T) of the star-forming and the quenched galaxies, per anchor ──
# The observer's cuts of Parts 16 / 16b are B/T thresholds chosen by hand.  This shows the two populations they are
# cut from: does the quenched distribution have a disky tail that stands apart, and where does it cross the star-
# forming one?  Same catalogue, mass floor and quenched rule as Part 16 (CAT, PF7_MASS, PASSIVE_FACTOR).
PF7C_STEM = "ba_bt_populations"
PF7C_ZS   = PF7_ZS            # the anchors (None = every one with PF7_NMIN galaxies of each kind)
PF7C_BINS = np.linspace(0.0, 1.0, 41)
PF7C_POOL = False             # True = one panel, every anchor pooled
PF7C_CUTS = sorted(set(list(PF7_BT) + [PF7B_BT]))     # the B/T cuts in play, marked on every panel
PF7C_NCOL = 3
PF7C_COL  = {"SF": "#2b6ca3", "Q": "#c0392b"}
_C16C = CAT[CAT["box"] == PF7_BOX].copy()
with np.errstate(all="ignore"):
    _ms = _C16C["mstar"].to_numpy(float)
    _C16C["lm"] = np.log10(np.where(_ms > 0, _ms, np.nan))
    _C16C["ssfr"] = _C16C["sfr"].to_numpy(float) / np.where(_ms > 0, _ms, np.nan)
    _C16C["bt"] = np.clip((PF7_KDISC - _C16C["kappa_star"].to_numpy(float)) / (PF7_KDISC - KAPPA_ISO), 0.0, 1.0)
_C16C = _C16C[(_C16C["lm"] > PF7_MASS) & np.isfinite(_C16C["kappa_star"].to_numpy(float))]
_qc = _C16C["ssfr"].to_numpy(float) < PASSIVE_FACTOR / (_C16C["t_H_gyr"].to_numpy(float) * 1e9)
_C16C["pop"] = np.where(_qc, "Q", "SF")
_zc = sorted(_C16C["z"].round(3).unique())
_Z16C = ([None] if PF7C_POOL else
         (sorted({min(_zc, key=lambda a: abs(a - float(z))) for z in PF7C_ZS}) if PF7C_ZS is not None else
          [z for z in _zc if all(int(((_C16C["z"].round(3) == z) & (_C16C["pop"] == p)).sum()) >= PF7_NMIN for p in ("SF", "Q"))]))
_kap_of16c = lambda b: PF7_KDISC - b * (PF7_KDISC - KAPPA_ISO)
_bt_of16c = lambda k: (PF7_KDISC - k) / (PF7_KDISC - KAPPA_ISO)
_xc = 0.5 * (PF7C_BINS[1:] + PF7C_BINS[:-1])
ROW16C = []
_nr = int(np.ceil(len(_Z16C) / PF7C_NCOL))
fig, AX = plt.subplots(_nr, min(PF7C_NCOL, len(_Z16C)), figsize=(5.2 * min(PF7C_NCOL, len(_Z16C)), 4.2 * _nr + 0.8), squeeze=False)
for _i, _z in enumerate(_Z16C):
    ax = AX[_i // PF7C_NCOL, _i % PF7C_NCOL]
    _S = _C16C if _z is None else _C16C[_C16C["z"].round(3) == _z]
    _D = {}
    for _p in ("SF", "Q"):
        _k = _S.loc[_S["pop"] == _p, "kappa_star"].to_numpy(float)
        _h, _ = np.histogram(_k, bins=PF7C_BINS, density=True)
        _D[_p] = (_k, _h)
        ax.stairs(_h, PF7C_BINS, fill=True, color=PF7C_COL[_p], alpha=0.25, lw=0)
        ax.stairs(_h, PF7C_BINS, color=PF7C_COL[_p], lw=2.0,
                  label=f"{'star-forming' if _p == 'SF' else 'quenched'}  ({len(_k)})")
    # THE CROSSING: the kappa_* above which a galaxy is more likely star-forming than quenched, in DENSITY (each
    # population normalised to one — the shapes, not the numbers) — the natural threshold if one wants one
    _hs, _hq = _D["SF"][1], _D["Q"][1]
    _d = _hs - _hq
    _cross = np.nan
    _jj = np.flatnonzero((_d[:-1] <= 0) & (_d[1:] > 0) & (_xc[:-1] > KAPPA_ISO))
    if len(_jj):
        _j = int(_jj[-1]) if _xc[_jj[-1]] < 0.7 else int(_jj[0])
        _cross = float(np.interp(0.0, [_d[_j], _d[_j + 1]], [_xc[_j], _xc[_j + 1]]))
        ax.axvline(_cross, color="0.2", ls="-", lw=1.4)
        ax.text(_cross, 0.98, f"  cross {_cross:.2f}\n  (B/T {_bt_of16c(_cross):.2f})", transform=ax.get_xaxis_transform(),
                ha="left", va="top", fontsize=9, color="0.2")
    for _b in PF7C_CUTS:
        ax.axvline(_kap_of16c(_b), color="0.45", ls=":", lw=1.1)
        ax.text(_kap_of16c(_b), 0.02, f" B/T {_b:g}", transform=ax.get_xaxis_transform(), rotation=90,
                ha="left", va="bottom", fontsize=7.5, color="0.45")
    ax.axvline(KAPPA_ISO, color="0.6", ls="--", lw=1.0)
    ax.set_xlim(0, 1); ax.set_xlabel(r"$\kappa_{\rm rot}(\star)$")
    if _i % PF7C_NCOL == 0:
        ax.set_ylabel("density")
    ax.set_title("every anchor pooled" if _z is None else f"$z = {_z:.2f}$", fontsize=11)
    # THE B/T AXIS STOPS AT 1: the relabelling is a straight line through (kappa_disc, 0) and (1/3, 1), and every galaxy
    # below the isotropic 1/3 is CLIPPED to B/T = 1 by the cuts (np.clip in Parts 16 / 16b) — a pure spheroid; the
    # region is shaded so the axis says what the code does
    ax.axvspan(0, KAPPA_ISO, color="0.5", alpha=0.10, lw=0, zorder=0)
    ax.text(0.5 * KAPPA_ISO, 0.98, "B/T = 1\n(clipped)", transform=ax.get_xaxis_transform(), ha="center", va="top",
            fontsize=8, color="0.4")
    _top = ax.secondary_xaxis("top", functions=(_bt_of16c, _kap_of16c))
    _top.set_xlabel("B/T", fontsize=9); _top.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); _top.tick_params(labelsize=8)
    _top.spines["top"].set_bounds(KAPPA_ISO, PF7_KDISC)          # the axis line only where B/T is defined
    if _i == 0:
        ax.legend(loc="center right", fontsize=8.5, frameon=False)
    _pk = {p: np.nanpercentile(_D[p][0], [16, 50, 84]) for p in ("SF", "Q")}
    _sh = {p: (float(np.mean(_D[p][0] >= _cross)) if np.isfinite(_cross) else np.nan) for p in ("SF", "Q")}
    ROW16C.append(dict(z=(np.nan if _z is None else _z), n_sf=len(_D["SF"][0]), n_q=len(_D["Q"][0]),
                       sf_p16=_pk["SF"][0], sf_p50=_pk["SF"][1], sf_p84=_pk["SF"][2],
                       q_p16=_pk["Q"][0], q_p50=_pk["Q"][1], q_p84=_pk["Q"][2],
                       cross_kappa=_cross, cross_bt=(_bt_of16c(_cross) if np.isfinite(_cross) else np.nan),
                       sf_above_cross=_sh["SF"], q_above_cross=_sh["Q"],
                       **{f"q_share_bt{b:g}": float(np.mean(_D["Q"][0] >= _kap_of16c(b))) for b in PF7C_CUTS},
                       **{f"sf_share_bt{b:g}": float(np.mean(_D["SF"][0] >= _kap_of16c(b))) for b in PF7C_CUTS}))
for _i in range(len(_Z16C), _nr * min(PF7C_NCOL, len(_Z16C))):
    AX[_i // PF7C_NCOL, _i % PF7C_NCOL].axis("off")
fig.suptitle(f"kappa_*(B/T) of the star-forming and the quenched galaxies, {BOX_LABEL.get(PF7_BOX, PF7_BOX)}, "
             f"log M* > {PF7_MASS:g} — the cuts of Parts 16 / 16b dotted, the density crossing solid", y=0.995, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])
paper_save(fig, PF7C_STEM); plt.show()
P16C = pd.DataFrame(ROW16C)
P16C.to_csv(os.path.join(OUT, f"{PF7C_STEM}.csv"), index=False)
print(f"WHERE DOES 'DISKY' BEGIN?  kappa_* (16 / 50 / 84 %) of each population, the density crossing, and what each cut keeps:")
print(f"    {'z':>6s} {'N_SF':>6s} {'N_Q':>6s}   {'SF 16/50/84':>18s}   {'Q 16/50/84':>18s}   {'cross':>5s} {'B/T':>5s}  {'SF>cross':>8s} {'Q>cross':>8s}"
      + "".join(f"  Q>B/T{b:g}" for b in PF7C_CUTS))
for _, r in P16C.iterrows():
    print(f"    {('pooled' if not np.isfinite(r['z']) else f'{r.z:.2f}'):>6s} {int(r.n_sf):6d} {int(r.n_q):6d}   "
          f"{r.sf_p16:.2f} / {r.sf_p50:.2f} / {r.sf_p84:.2f}   {r.q_p16:.2f} / {r.q_p50:.2f} / {r.q_p84:.2f}   "
          + (f"{r.cross_kappa:5.2f} {r.cross_bt:5.2f}  {100 * r.sf_above_cross:6.0f} %  {100 * r.q_above_cross:6.0f} %" if np.isfinite(r.cross_kappa) else f"{'none':>5s} {'':>5s}  {'':>8s} {'':>8s}")
          + "".join(f"   {100 * r[f'q_share_bt{b:g}']:5.1f} %" for b in PF7C_CUTS))
print("    cross = the kappa_* where the two DENSITIES cross (each population normalised to one): above it a galaxy's shape\n"
      "    is more like a star-forming one's than a quenched one's; the B/T beside it is the same point relabelled.")
print("table ->", os.path.join(OUT, f"{PF7C_STEM}.csv"))


## Part 17 — paper figure 8: the radial profiles

The paper version of Part 7 (B). It reads the m25 imaging products' truth tables — the projected annuli of
every galaxy at its anchor, four sightlines each — and draws, annulus by annulus out to 100 kpc, the median
over the galaxies of each group (one value per galaxy = its sightline median) with the bootstrap 16–84 of
that median; the star-forming galaxies of the same catalogue are the grey dashed line, what a profile looks
like before quenching. The outermost annulus (32–100 kpc) is the projected CGM — where displaced dust would
show.

| panel | from |
|---|---|
| $\Sigma_{\rm dust}$, $\Sigma_{\rm H_2}$, $M_{\rm dust}/M_{\rm gas}$ of the annulus | `annulus_ism_truth.fits` (m25 Part 8a) |
| $M_{\rm dust}/M_\star$ in the annulus, the stellar age | with `aperture_truth.fits` (m25 Part 7e; an annulus needs `PF8_NSTAR` stars) |
| $A_V$ of the annulus, all sightlines | `annulus_av_allincl.fits` (m25 Part 8b) |

The tables cover the powderday sample, so only the peel's galaxies that are in it are drawn — the N in the
legend says how many. **Knobs.** `PF8_ANN` (the annuli), `PF8_Q` (the panels), `PF8_SF`, `PF8_NSTAR`,
`PF8_NMIN`, `PF8_NCOL`; `P7_M25_OVERRIDE` points the cell at another `output/<box>` tree.

In [ ]:
# ── Part 17 — PAPER FIGURE 8: THE RADIAL PROFILES — where the dust, the gas and the stars are, annulus by annulus,
#    out to 100 kpc, for the two groups against the star-forming galaxies of the same box ──────────────────
# The paper version of Part 7 (B).  It reads the m25 imaging products' truth tables — the projected annuli of
# every galaxy at its anchor, four sightlines each — and draws, per annulus, the median over the galaxies of
# each group (one value per galaxy = its sightline median) with the bootstrap 16-84 of that median.  The
# outermost annulus (32-100 kpc) is the projected CGM: it is where the displaced dust would show.
#   Sigma_dust, Sigma_H2     the surface densities                          (annulus_ism_truth, m25 Part 8a)
#   M_dust / M_*             the dust per unit stellar mass IN THE ANNULUS   (with aperture_truth, m25 Part 7e)
#   D/G                      the dust-to-gas ratio of the annulus            (annulus_ism_truth)
#   A_V                      the attenuation of the annulus, all sightlines  (annulus_av_allincl, m25 Part 8b)
#   stellar age              mass-weighted, of the stars in the annulus      (aperture_truth)
# The star-forming galaxies of the same catalogue (powderday_quenched_selection, pop = SF) are the grey line:
# what a profile looks like before quenching.
import re
PF8_STEM  = "ba_paper_profiles"
PF8_BOX   = "cis25"
PF8_M25   = globals().get("P7_M25_OVERRIDE", os.path.join(os.getcwd(), "output", PF8_BOX))
PF8_ANN   = [("ap1kpc", 0.0, 1.0), ("ann3kpc", 1.0, 3.2), ("ann10kpc", 3.2, 10.0), ("ann32kpc", 10.0, 32.0),
             ("ann100kpc", 32.0, 100.0)]          # (the table's name, r_in, r_out) [kpc]
PF8_Q     = [("Sigma_dust", r"$\Sigma_{\rm dust}$  [M$_\odot$ kpc$^{-2}$]", "log"),
             ("Sigma_H2", r"$\Sigma_{\rm H_2}$  [M$_\odot$ kpc$^{-2}$]", "log"),
             ("fdust_ap", r"$M_{\rm dust}/M_\star$", "log"),
             ("A_V", r"$A_V$  [mag]", "log"),
             ("Sigma_star", r"$\Sigma_\star$  [M$_\odot$ kpc$^{-2}$]", "log"),
             ("age_ap", "stellar age  [Gyr]", "lin")]
PF8_TOT   = "ap10kpc"        # THE TOTAL: the same quantities inside this aperture (the user's choice, r < 10 kpc;
                             # "ap32kpc" / "ap100kpc" are the other rungs), drawn as one point per group at the
                             # right of every panel — the galaxy as a whole, not radially
_RTOT8 = int(re.sub(r"\D", "", PF8_TOT)) if re.sub(r"\D", "", PF8_TOT) else 0     # the radius the label quotes
PF8_SF    = True              # the star-forming galaxies of the box as the grey reference
PF8_NSTAR = 20                # stars an annulus needs for M_* and the age to be trusted (Part 7's rule)
PF8_NMIN  = 3
PF8_NBOOT = NBOOT
PF8_NCOL  = 3
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
_ANN8 = os.path.join(PF8_M25, "tables", "annulus_ism_truth.fits")
_APT8 = os.path.join(PF8_M25, "tables", "aperture_truth.fits")
_AV8  = os.path.join(PF8_M25, "tables", "annulus_av_allincl.fits")
_SEL8 = os.path.join(PF8_M25, "tables", "powderday_quenched_selection.fits")


def _fr8(path, cols):
    """A FITS table -> a DataFrame of the named columns, masked values as NaN and byte strings decoded."""
    t = Table.read(path); d = {}
    for c in cols:
        if c not in t.colnames:
            continue
        v = t[c]; v = v.filled(np.nan) if hasattr(v, "filled") else v
        v = np.asarray(v)
        d[c] = (np.array([x.decode() if isinstance(x, bytes) else str(x) for x in v]) if v.dtype.kind in "SU"
                else v.astype(float))
    return pd.DataFrame(d)


if PF8_BOX not in _BX1:
    print(f"Part 17: {PF8_BOX} is not in this run's boxes ({_BX1}) — skipped.")
elif not (os.path.exists(_ANN8) and os.path.exists(_APT8)):
    print(f"Part 17: {_ANN8} or {_APT8} is missing — the m25 notebook's Parts 8a / 7e build them (cluster).  Skipped.")
else:
    _F8 = B1[PF8_BOX]["F"]
    KEY8 = {(int(s_), int(i_)): g_ for s_, i_, g_, u_ in zip(_F8["snap"], _F8["gal_id"], _F8["grp"], _F8["uniq"])
            if u_ and g_ in GK}
    _K4 = ["snap", "gal_id", "incl", "aperture"]
    R8 = _fr8(_ANN8, _K4 + ["r_in_kpc", "r_out_kpc", "area_kpc2", "M_dust", "M_gas", "M_H2", "Sigma_dust", "Sigma_H2", "Sigma_gas", "DGR"])
    R8 = R8.merge(_fr8(_APT8, _K4 + ["nstar_ap", "mstar", "age_m_star_myr"]), on=_K4, how="left")
    if os.path.exists(_AV8):
        R8 = R8.merge(_fr8(_AV8, _K4 + ["A_V"]), on=_K4, how="left")
    R8 = R8[R8["aperture"].isin([a for a, _, _ in PF8_ANN] + [PF8_TOT])].reset_index(drop=True)
    SF8 = set()
    if PF8_SF and os.path.exists(_SEL8):
        _s8 = _fr8(_SEL8, ["snap", "gal_id", "pop"])
        SF8 = {(int(s_), int(i_)) for s_, i_, p_ in zip(_s8["snap"], _s8["gal_id"], _s8["pop"]) if p_ == "SF"}
    R8["grp"] = [KEY8.get((int(s_), int(i_)), "sf" if (int(s_), int(i_)) in SF8 else None)
                 for s_, i_ in zip(R8["snap"], R8["gal_id"])]
    R8 = R8[R8["grp"].notna()].reset_index(drop=True)
    with np.errstate(all="ignore"):
        _ms8 = np.where(R8["nstar_ap"].to_numpy(float) >= PF8_NSTAR, R8["mstar"].to_numpy(float), np.nan)
        _md8 = R8["M_dust"].to_numpy(float)
        R8["fdust_ap"] = np.where(np.isfinite(_md8) & (_md8 > 0) & (_ms8 > 0), _md8 / _ms8, np.nan)
        R8["age_ap"] = np.where(np.isfinite(_ms8), R8["age_m_star_myr"].to_numpy(float) / 1e3, np.nan)
        _ar8 = R8["area_kpc2"].to_numpy(float)
        R8["Sigma_star"] = np.where((_ms8 > 0) & (_ar8 > 0), _ms8 / np.where(_ar8 > 0, _ar8, np.nan), np.nan)
        for c in ("Sigma_dust", "Sigma_H2", "Sigma_gas", "DGR", "A_V"):
            if c in R8.columns:
                v = R8[c].to_numpy(float); R8[c] = np.where(np.isfinite(v) & (v > 0), v, np.nan)
    _G8 = (["sf"] if len(SF8) else []) + list(GK)
    _C8 = dict(PFC, sf="0.55"); _L8 = dict(GLAB, sf=f"star-forming galaxies of {BOX_LABEL.get(PF8_BOX, PF8_BOX)}")
    _LW8 = {"sf": 1.6, "burst": 2.8, "aging": 2.8}
    _rc8 = np.array([np.sqrt(max(a, 0.25) * b) for _, a, b in PF8_ANN])     # the geometric centre of each annulus
    _xtot8 = 300.0                                                          # where the TOTAL point sits on the log axis
    _APS8 = list(PF8_ANN) + [(PF8_TOT, np.nan, np.nan)]                     # the annuli, then the total
    _Q8 = [(c, l, k) for c, l, k in PF8_Q if c in R8.columns]
    ROW17, N8 = [], {g: 0 for g in _G8}
    _rng8 = np.random.default_rng(5)
    _nr8 = int(np.ceil(len(_Q8) / PF8_NCOL))
    with plt.rc_context(PAPER_RC):
        fig, AX = plt.subplots(_nr8, PF8_NCOL, figsize=(6 * PF8_NCOL + 1, 3.9 * _nr8 + 1.0), squeeze=True)
        for _qi, (_c, _lab, _kind) in enumerate(_Q8):
            ax = AX[_qi // PF8_NCOL, _qi % PF8_NCOL]
            for _g in _G8:
                _m, _lo, _hi, _n = [], [], [], []
                for _a, _r0, _r1 in _APS8:
                    _sub = R8[(R8["grp"] == _g) & (R8["aperture"] == _a)]
                    _v = _sub[_c].to_numpy(float); _ok = np.isfinite(_v)
                    _per = {}
                    for s_, i_, vv in zip(_sub["snap"].to_numpy(float)[_ok], _sub["gal_id"].to_numpy(float)[_ok], _v[_ok]):
                        _per.setdefault((int(s_), int(i_)), []).append(vv)          # the sightlines -> one value per galaxy
                    _pg = np.array([np.median(vv) for vv in _per.values()])
                    if len(_pg) >= PF8_NMIN:
                        _bs = np.median(_pg[_rng8.integers(0, len(_pg), (PF8_NBOOT, len(_pg)))], axis=1)
                        _m.append(float(np.median(_pg))); _lo.append(float(np.percentile(_bs, 16))); _hi.append(float(np.percentile(_bs, 84)))
                    else:
                        _m.append(np.nan); _lo.append(np.nan); _hi.append(np.nan)
                    _n.append(len(_pg))
                    ROW17.append(dict(quantity=_c, grp=_g, group=_L8.get(_g, _g), annulus=_a, r_in=_r0, r_out=_r1,
                                      n_gal=len(_pg), med=_m[-1], lo=_lo[-1], hi=_hi[-1]))
                N8[_g] = max(N8[_g], max(_n))
                _m, _lo, _hi = map(np.asarray, (_m, _lo, _hi))
                _mp, _lp, _hp = _m[:-1], _lo[:-1], _hi[:-1]; _ok = np.isfinite(_mp)       # the profile ...
                if _ok.any():
                    ax.fill_between(_rc8[_ok], _lp[_ok], _hp[_ok], color=_C8[_g], alpha=0.16, lw=0)
                    ax.plot(_rc8[_ok], _mp[_ok], "o-", color=_C8[_g], lw=_LW8[_g], ms=6.5, mec="w", mew=0.8,
                            ls="--" if _g == "sf" else "-", zorder=4 if _g != "sf" else 3)
                if np.isfinite(_m[-1]):                                                    # ... and the total
                    _dx = {"sf": 0.86, "burst": 1.0, "aging": 1.16}[_g]
                    ax.errorbar([_xtot8 * _dx], [_m[-1]], yerr=[[max(_m[-1] - _lo[-1], 0)], [max(_hi[-1] - _m[-1], 0)]],
                                fmt="D", color=_C8[_g], ms=8, mec="k", mew=0.7, capsize=3, lw=1.6, zorder=6)
            if _kind == "log":
                ax.set_yscale("log")
            if _c == "fdust_ap":
                ax.axhline(10 ** DUSTY_CUT, color=PFC["burst"], ls="--", lw=1.5)
            ax.axvline(160, color="0.75", lw=1.0, ls=":", zorder=0)
            ax.set_xscale("log"); ax.set_xlim(0.35, 520)
            ax.set_xticks([0.5, 1.8, 5.7, 18, 57, _xtot8])
            ax.set_xticklabels(["0-1", "1-3", "3-10", "10-30", "30-100", f"total\n(< {_RTOT8:g})"])
            ax.set_ylabel(_lab)
            if _qi // PF8_NCOL == _nr8 - 1:
                ax.set_xlabel("annulus  [kpc]")
            ax.set_axisbelow(True)
            for _s in ("top", "right"):
                ax.spines[_s].set_visible(False)
        for _qi in range(len(_Q8), _nr8 * PF8_NCOL):
            AX[_qi // PF8_NCOL, _qi % PF8_NCOL].axis("off")
        fig.tight_layout(rect=[0, 0.07, 1, 1], h_pad=1.5, w_pad=1.6)
        fig.legend(handles=[Line2D([], [], color=_C8[g], marker="o", ms=6.5, mec="w", lw=_LW8[g],
                                   ls="--" if g == "sf" else "-", label=f"{_L8.get(g, g)}  (N ≤ {N8[g]})") for g in _G8]
                   + [Line2D([], [], color="0.3", marker="D", ms=8, mec="k", ls="none", label=f"the whole galaxy (r < {_RTOT8:g} kpc)"),
                      Line2D([], [], color=PFC["burst"], ls="--", lw=1.5, label=rf"$f_{{\rm dust}} = 10^{{{DUSTY_CUT:g}}}$")],
                   loc="lower center", ncol=3, frameon=False)
        paper_save(fig, PF8_STEM); plt.show()
    P17 = pd.DataFrame(ROW17)
    P17.to_csv(os.path.join(OUT, f"{PF8_STEM}.csv"), index=False)
    print(f"THE RADIAL PROFILES ({BOX_LABEL.get(PF8_BOX, PF8_BOX)}, at the anchor; one value per galaxy = its sightline median; "
          f"N: " + ", ".join(f"{_L8.get(g, g).split(' ')[0]} {N8[g]}" for g in _G8) + ")")
    for _c, _lab, _kind in _Q8:
        _line = f"    {_c:11s}"
        for _a, _r0, _r1 in _APS8:
            _line += (f"  {_r0:g}-{_r1:g}: " if np.isfinite(_r0) else "  total: ")
            for _g in _G8:
                _r = P17[(P17["quantity"] == _c) & (P17["grp"] == _g) & (P17["annulus"] == _a)].iloc[0]
                _line += (f"{_r['med']:.2g}/" if np.isfinite(_r["med"]) else "–/")
            _line = _line.rstrip("/")
        print(_line)
    print("    (the order inside each annulus is " + " / ".join(_L8.get(g, g).split(' ')[0] for g in _G8) + ")")
    print("table ->", os.path.join(OUT, f"{PF8_STEM}.csv"))

## Part 18 — paper figure 9: where the dust goes, and whether it comes back

A pure re-read of the m25 particle census (`dusty_qgs_kappa_selection` Parts 13 + 17 — Part 6b's `Z6B` when it
has run, rebuilt from the same file otherwise), re-split on this notebook's groups and re-clocked on its
$t_{\rm SFT}$. Every galaxy's dust is followed **by place** — the ISM (caesar's members) and the CGM in four
shells out to 100 kpc — and **by flux** through the shell at $r = 10$ kpc, in both directions.

| | |
|---|---|
| **(a, b) the budget by place** | the dust of each group in the ISM and the four CGM shells against $t - t_{\rm SFT}$, every galaxy normalised to its own total (ISM + CGM) at the truncation: what leaves the ISM, where it goes, and what is destroyed or beyond 100 kpc (the white above the stack) in one picture |
| **(c) the fluxes** | the dust crossing $r = 10$ kpc outward and inward, and the cold part of the inward flux |
| **(d) the return** | the cumulative inflow since the truncation over the cumulative outflow — of what left, how much has come back (all phases, and cold only) |
| **(e) does it matter** | the cold dust returning per unit time against the dust the ISM *grows* per unit time (Part 17's grain clocks, $\dot m = m_d f_Z/\tau_{\rm grow}$ summed over the SF and non-SF ISM): the return counts only where this ratio is not small |
| **(f) can it return at all** | the bound share of the CGM dust (Part 13's escape velocities) |

The nodes are 0.12 Gyr apart in the median, so the cumulative books are trapezoid integrals over the nodes
from the truncation on. **Knobs.** `PF9_TB` (the clock bins), `PF9_REFW` (how close to the truncation the
reference node must be), `PF9_PLACES` (the places and their colours), `PF9_GROWZ` (which growth rates the
return is measured against), `PF9_NMIN`, `PF9_NBOOT`.

In [ ]:
# ── Part 18 — PAPER FIGURE 9: WHERE THE DUST GOES, AND WHETHER IT COMES BACK — the dust budget by place on the
#    clock, the fluxes through 10 kpc, and the return measured against what the ISM makes itself ────────────
# A pure re-read of the m25 particle census (dusty_qgs_kappa_selection Parts 13 + 17; Part 6b's Z6B when it has
# run), re-split on this notebook's groups and re-clocked on its t_SFT.  Every galaxy's dust is followed by
# PLACE — the ISM (caesar's members) and the CGM in four shells out to 100 kpc — and by FLUX through the shell
# at r = 10 kpc, in both directions.  Nothing is opened but the census.
#   (a, b)  INSIDE / OUTSIDE      where the dust that is still within 100 kpc sits — the ISM and the four CGM
#           shells as SHARES of the current total, stacked on the clock, one panel per group.
#   (c)     THE RETURN            the cumulative inflow through r = 10 kpc since the truncation over the cumulative
#           outflow — of what left, how much has come back — and, on the second axis, IN WHAT the returning dust
#           comes back: the share of the inward dust flux carried by COLD gas (T < 1e5 K).  The census splits the
#           fluxes at 1e5 K only (cold / hot), so what is not cold is hot; there is no warm flux to draw.
#   (d)     DOES IT DOMINATE      all the dust returning per unit time against the dust the ISM GROWS per unit
#           time (Part 17's grain clocks, dm/dt = m_d f_Z / tau_grow, summed over the SF and non-SF ISM): above
#           the line the return outweighs what the ISM makes itself.
#   The total within 100 kpc relative to the truncation, the fluxes, the cold-only return and the bound share
#   of the CGM dust stay in the CSV and the printout.
PF9_STEM  = "ba_paper_displacement"
PF9_BOX   = "cis25"
PF9_NODES = os.path.join(KSEL, "kappa_cis25_dust_regrowth_nodes.csv")     # Part 17's census (a superset of Part 13's)
PF9_TB    = np.arange(-1.0, 4.001, 0.5)         # the clock bins [Gyr]; a per-galaxy median inside a bin first
                                                # (from one Gyr before the truncation: earlier bins hold too few
                                                # dusty galaxies with a census node and would leave holes)
PF9_REFW  = 0.3                                 # Gyr: the reference node for the normalisation is the nearest to
                                                # the truncation within this
PF9_PLACES = [("mdust_ism", "ISM", "#67000d"), ("mdust_s10", "CGM 0–10 kpc", "#cb181d"),
              ("mdust_s30", "10–30 kpc", "#fb6a4a"), ("mdust_s60", "30–60 kpc", "#fcae91"),
              ("mdust_s100", "60–100 kpc", "#fee5d9")]
PF9_NMIN  = 3
PF9_NBOOT = None
PF9_UNIQ  = True
PF9_GROWZ = ["grow_ism_nsf", "grow_ism_sf"]      # the in-situ growth rates summed [Msun / Gyr] (Part 17's census)
PF9_LS    = {"burst": "-", "aging": "--"}         # the line style SAYS THE GROUP in the two line panels: dusty solid,
                                                  # dust-poor dashed — so colour is free to say the QUANTITY in (c)
PF9_CRET  = "#333333"                             # (c) left axis: returned / ejected
PF9_CCOLD = "#1b9e77"                             # (c) right axis: the cold fraction of the return (axis coloured too)
if "PFC" not in globals():
    PFC = {"burst": P1D_COL[P1D_NAMES[0]], "aging": P1D_COL[P1D_NAMES[1]]}
from matplotlib.patches import Patch

if not os.path.exists(PF9_NODES):
    print(f"Part 18: {PF9_NODES} is missing — dusty_qgs_kappa_selection Part 17 builds it (cluster only).  Skipped.")
elif PF9_BOX not in _BX1:
    print(f"Part 18: {PF9_BOX} is not in this run's boxes ({_BX1}) — skipped.")
else:
    # ── the census, on this notebook's groups and clock (Part 6b's Z6B when it exists, rebuilt otherwise) ──
    if "Z6B" in globals() and isinstance(globals().get("Z6B"), pd.DataFrame) and "trel" in Z6B.columns:
        Z9 = Z6B.copy()
    else:
        Z9 = pd.read_csv(PF9_NODES)
        _F9 = B1[PF9_BOX]["F"]
        _k9 = (_F9["uniq"].to_numpy(bool) if PF9_UNIQ else np.ones(len(_F9), bool)) & (_F9["grp"].to_numpy() != "")
        _K9 = _F9.loc[_k9, ["snap", "gal_id", "grp", "t_sft"]].rename(columns={"snap": "anchor_snap"})
        Z9 = Z9.drop(columns=[c for c in ("grp", "t_sft", "cls", "dusty") if c in Z9.columns]).merge(_K9, on=["anchor_snap", "gal_id"], how="inner")
        Z9["trel"] = Z9["t"].to_numpy(float) - Z9["t_sft"].to_numpy(float)
        Z9["gid"] = Z9["anchor_snap"].astype(str) + ":" + Z9["gal_id"].astype(str)
    Z9 = Z9.sort_values(["gid", "t"]).reset_index(drop=True)
    _TC9 = 0.5 * (PF9_TB[:-1] + PF9_TB[1:])
    _PL = [p for p, _, _ in PF9_PLACES if p in Z9.columns]
    with np.errstate(all="ignore"):
        Z9["md_tot"] = np.nansum(np.column_stack([Z9[p].to_numpy(float) for p in _PL]), axis=1)
        # the per-galaxy reference: the total dust at the node nearest the truncation
        _ref = {}
        for _g, _d in Z9.groupby("gid"):
            _i = int(np.argmin(np.abs(_d["trel"].to_numpy(float))))
            _ref[_g] = float(_d["md_tot"].iloc[_i]) if abs(float(_d["trel"].iloc[_i])) <= PF9_REFW and float(_d["md_tot"].iloc[_i]) > 0 else np.nan
        Z9["md_ref"] = Z9["gid"].map(_ref)
        for p in _PL:
            Z9[f"rel_{p}"] = Z9[p].to_numpy(float) / Z9["md_ref"].to_numpy(float)
            Z9[f"sh_{p}"] = np.where(Z9["md_tot"] > 0, Z9[p].to_numpy(float) / np.where(Z9["md_tot"] > 0, Z9["md_tot"], 1.0), np.nan)
        Z9["rel_tot"] = Z9["md_tot"] / Z9["md_ref"]
        # the fluxes and the cumulative books, per galaxy, from the truncation on  (mdot in Msun/yr, t in Gyr)
        Z9["cum_in"] = np.nan; Z9["cum_out"] = np.nan; Z9["cum_in_c"] = np.nan
        for _g, _d in Z9.groupby("gid"):
            _m = _d["trel"].to_numpy(float) >= 0
            if _m.sum() < 2:
                continue
            _t = _d["t"].to_numpy(float)[_m] * 1e9
            _ci = np.concatenate([[0.0], np.cumsum(0.5 * (np.nan_to_num(_d["mdot_in_d"].to_numpy(float)[_m])[1:] + np.nan_to_num(_d["mdot_in_d"].to_numpy(float)[_m])[:-1]) * np.diff(_t))])
            _co = np.concatenate([[0.0], np.cumsum(0.5 * (np.nan_to_num(_d["mdot_out_d"].to_numpy(float)[_m])[1:] + np.nan_to_num(_d["mdot_out_d"].to_numpy(float)[_m])[:-1]) * np.diff(_t))])
            _cc = np.concatenate([[0.0], np.cumsum(0.5 * (np.nan_to_num(_d["mdot_in_dc"].to_numpy(float)[_m])[1:] + np.nan_to_num(_d["mdot_in_dc"].to_numpy(float)[_m])[:-1]) * np.diff(_t))])
            Z9.loc[_d.index[_m], "cum_in"] = _ci; Z9.loc[_d.index[_m], "cum_out"] = _co; Z9.loc[_d.index[_m], "cum_in_c"] = _cc
        Z9["f_ret"] = np.where(Z9["cum_out"] > 0, Z9["cum_in"] / np.where(Z9["cum_out"] > 0, Z9["cum_out"], 1.0), np.nan)
        Z9["f_ret_c"] = np.where(Z9["cum_out"] > 0, Z9["cum_in_c"] / np.where(Z9["cum_out"] > 0, Z9["cum_out"], 1.0), np.nan)
        _grow = np.nansum(np.column_stack([Z9[c].to_numpy(float) for c in PF9_GROWZ if c in Z9.columns]), axis=1) if any(c in Z9.columns for c in PF9_GROWZ) else np.full(len(Z9), np.nan)
        Z9["r_ret_grow"] = np.where(_grow > 0, Z9["mdot_in_dc"].to_numpy(float) * 1e9 / np.where(_grow > 0, _grow, 1.0), np.nan)   # both in Msun/Gyr
        Z9["lmdot_out"] = np.log10(np.where(Z9["mdot_out_d"] > 0, Z9["mdot_out_d"], np.nan))
        Z9["lmdot_in"] = np.log10(np.where(Z9["mdot_in_d"] > 0, Z9["mdot_in_d"], np.nan))
        Z9["lmdot_in_c"] = np.log10(np.where(Z9["mdot_in_dc"] > 0, Z9["mdot_in_dc"], np.nan))
        Z9["lr_ret_grow"] = np.log10(np.where(Z9["r_ret_grow"] > 0, Z9["r_ret_grow"], np.nan))
        Z9["recyc_share"] = np.where(Z9["mdust_ism"] > 0, Z9["cum_in_c"] / np.where(Z9["mdust_ism"] > 0, Z9["mdust_ism"], 1.0), np.nan)
        _din = Z9["mdot_in_d"].to_numpy(float)
        Z9["sh_in_cold"] = np.where(_din > 0, Z9["mdot_in_dc"].to_numpy(float) / np.where(_din > 0, _din, 1.0), np.nan)
        Z9["sh_in_hot"] = np.where(_din > 0, Z9["mdot_in_dh"].to_numpy(float) / np.where(_din > 0, _din, 1.0), np.nan)
        Z9["sh_in_warm"] = np.clip(1.0 - Z9["sh_in_cold"].to_numpy(float) - Z9["sh_in_hot"].to_numpy(float), 0.0, 1.0)
        Z9["r_ret_grow_all"] = np.where(_grow > 0, _din * 1e9 / np.where(_grow > 0, _grow, 1.0), np.nan)
        Z9["lr_ret_grow_all"] = np.log10(np.where(Z9["r_ret_grow_all"] > 0, Z9["r_ret_grow_all"], np.nan))
    _ng9 = {g: Z9.loc[Z9["grp"] == g, "gid"].nunique() for g in GK}
    print(f"Part 18 — WHERE THE DUST GOES ({BOX_LABEL.get(PF9_BOX, PF9_BOX)}): {len(Z9)} nodes, "
          + ", ".join(f"{GSH.get(g, g)} {_ng9[g]}" for g in GK) + f" galaxies; a reference total at the truncation for "
          f"{int(np.isfinite(Z9.groupby('gid')['md_ref'].first()).sum())} of them")

    def _piv9(D, col):
        """(galaxy x clock-bin) medians of `col`, Part 6's rule."""
        d = D[["gid", "trel", col]].copy()
        d["ib"] = np.digitize(d["trel"].to_numpy(float), PF9_TB) - 1
        d = d[(d["ib"] >= 0) & (d["ib"] < len(_TC9)) & np.isfinite(d[col].to_numpy(float))]
        if not len(d):
            return None
        return d.groupby(["gid", "ib"])[col].median().unstack("ib").reindex(columns=np.arange(len(_TC9))).to_numpy(float)

    def _band9(D, col):
        A = _piv9(D, col)
        if A is None:
            return (np.full(len(_TC9), np.nan),) * 3 + (np.zeros(len(_TC9), int),)
        return med_band(A, nmin=PF9_NMIN, nboot=PF9_NBOOT, grid=_TC9)

    ROW18 = []
    with plt.rc_context(PAPER_RC):
        fig, AX = plt.subplots(2, 2, figsize=(15.6, 11.2))
        # ── (a, b) inside / outside: the shares of the current total ───────────────────────────────────
        for _k, _g in enumerate(GK):
            ax = AX[0, _k]
            _D = Z9[Z9["grp"] == _g]
            _sh = np.array([_band9(_D, f"sh_{p}")[0] for p in _PL])
            _n = _band9(_D, f"sh_{_PL[0]}")[3]
            _sh = np.where(np.isfinite(_sh), _sh, np.nan)
            _norm = np.nansum(_sh, axis=0); _sh = np.where(_norm > 0, _sh / np.where(_norm > 0, _norm, 1.0), np.nan)
            _base = np.zeros(len(_TC9))
            for _p, (_pc, _pl, _col) in zip(_PL, [x for x in PF9_PLACES if x[0] in _PL]):
                _h = _sh[_PL.index(_p)]; _ok = np.isfinite(_h)
                ax.fill_between(_TC9, _base, _base + np.where(_ok, _h, 0.0), where=_ok, color=_col, alpha=0.95, lw=0.5, ec="w")
                for _t, _s, _nn in zip(_TC9, _h, _n):
                    if np.isfinite(_s):
                        ROW18.append(dict(block="share", grp=_g, group=GSH.get(_g, _g), place=_pl, t=_t, value=_s, n=int(_nn)))
                _base = _base + np.where(_ok, _h, 0.0)
            ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.6)
            ax.text(0.03, 0.96, GLAB.get(_g, _g) + f"  ({_ng9[_g]})", transform=ax.transAxes, va="top", ha="left",
                    fontsize=17.6, fontweight="bold", color="w", bbox=dict(fc=PFC[_g], ec="none", alpha=0.85, pad=2.5))
            ax.set_ylim(0, 1.0); ax.set_xlim(PF9_TB[0], PF9_TB[-1])
            ax.set_ylabel("dust share within 100 kpc" if _k == 0 else "")
            if _k == 1:
                ax.set_yticklabels([])
            ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]")
        # ── (c) the return, and in what it comes back ───────────────────────────────────────────────────
        ax = AX[1, 0]
        for _g in GK:
            _c, _lo, _hi, _n = _band9(Z9[Z9["grp"] == _g], "f_ret")
            ax.fill_between(_TC9, _lo, _hi, color=PF9_CRET, alpha=0.12, lw=0)
            ax.plot(_TC9, _c, color=PF9_CRET, lw=3.0, ls=PF9_LS[_g], marker="o", ms=5)
            for _t, _v, _nn in zip(_TC9, _c, _n):
                if np.isfinite(_v):
                    ROW18.append(dict(block="return", grp=_g, group=GSH.get(_g, _g), place="f_ret", t=_t, value=_v, n=int(_nn)))
        ax.axhline(1.0, color="0.3", ls=":", lw=1.2)
        ax.set_xlim(0, PF9_TB[-1]); ax.set_ylim(0, None)
        ax.set_ylabel("returned / ejected", color=PF9_CRET)
        ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]"); ax.set_axisbelow(True)
        ax2 = ax.twinx()
        for _g in GK:
            _c, _lo, _hi, _n = _band9(Z9[Z9["grp"] == _g], "sh_in_cold")
            ax2.fill_between(_TC9, 100 * _lo, 100 * _hi, color=PF9_CCOLD, alpha=0.10, lw=0)
            ax2.plot(_TC9, 100 * _c, color=PF9_CCOLD, lw=2.4, ls=PF9_LS[_g], alpha=0.95)
            for _col, _pl in (("sh_in_cold", "cold"), ("sh_in_hot", "hot")):
                _c2, _, _, _n2 = _band9(Z9[Z9["grp"] == _g], _col)
                for _t, _v, _nn in zip(_TC9, _c2, _n2):
                    if np.isfinite(_v):
                        ROW18.append(dict(block="return phase", grp=_g, group=GSH.get(_g, _g), place=_pl, t=_t, value=_v, n=int(_nn)))
        ax2.set_ylim(0, 40)
        ax2.set_ylabel("cold fraction  [%]", color=PF9_CCOLD)
        ax2.tick_params(axis="y", colors=PF9_CCOLD); ax2.spines["right"].set_color(PF9_CCOLD)
        ax2.spines["top"].set_visible(False)
        # ── (d) does it dominate: all returning dust against the in-situ growth ─────────────────────────
        ax = AX[1, 1]
        for _g in GK:
            _c, _lo, _hi, _n = _band9(Z9[Z9["grp"] == _g], "lr_ret_grow_all")
            ax.fill_between(_TC9, _lo, _hi, color=PFC[_g], alpha=0.18, lw=0)
            ax.plot(_TC9, _c, color=PFC[_g], lw=3.0, ls=PF9_LS[_g], marker="o", ms=5)
            for _t, _v, _nn in zip(_TC9, _c, _n):
                if np.isfinite(_v):
                    ROW18.append(dict(block="dominate", grp=_g, group=GSH.get(_g, _g), place="all return / in-situ growth", t=_t, value=_v, n=int(_nn)))
        ax.axhline(0.0, color="0.3", ls=":", lw=1.2); ax.axvline(0.0, color="k", ls="--", lw=1.0, alpha=0.6)
        ax.set_xlim(PF9_TB[0], PF9_TB[-1]); ax.set_ylim(None, 1.2)        # head-room for the legend
        ax.set_ylabel(r"$\log$ (returning / grown in the ISM)")
        ax.set_xlabel(r"$t - t_{q,\,\rm start}$  [Gyr]"); ax.set_axisbelow(True)
        for ax in AX.ravel():
            for _s in ("top", "right"):
                ax.spines[_s].set_visible(False)
        AX[1, 0].spines["right"].set_visible(True); AX[1, 0].spines["right"].set_color(PF9_CCOLD)
        # computed for the CSV, not drawn
        for _g in GK:
            for _col, _pl in (("rel_tot", "dust within 100 kpc / the dust at the truncation"), ("fbnd_d_cgm", "bound share of the CGM dust"),
                              ("lmdot_out", "log Mdot_out"), ("lmdot_in", "log Mdot_in"), ("lmdot_in_c", "log Mdot_in cold"),
                              ("f_ret_c", "returned cold / ejected"), ("lr_ret_grow", "log cold return / in-situ growth")):
                _D = Z9[Z9["grp"] == _g] if _col != "rel_tot" else Z9[(Z9["grp"] == _g) & np.isfinite(Z9["md_ref"])]
                _c, _lo, _hi, _n = _band9(_D, _col)
                for _t, _v, _nn in zip(_TC9, _c, _n):
                    if np.isfinite(_v):
                        ROW18.append(dict(block="not drawn", grp=_g, group=GSH.get(_g, _g), place=_pl, t=_t, value=_v, n=int(_nn)))
        fig.tight_layout(rect=[0, 0, 1, 1], h_pad=2.0, w_pad=2.0)
        fig.subplots_adjust(wspace=0.50)                      # room between the two stacks for their legend
        _hp = [Patch(fc=c, ec="w", label=l) for p, l, c in PF9_PLACES if p in _PL]
        _hl = [Line2D([], [], color=PFC[g], lw=3.0, ls=PF9_LS[g], label=GLAB.get(g, g)) for g in GK] \
            + [Line2D([], [], color=PF9_CRET, lw=3.0, marker="o", label="returned / ejected (left axis)"),
               Line2D([], [], color=PF9_CCOLD, lw=2.4, label=r"cold fraction of the return (right axis)")]
        _p0, _p1 = AX[0, 0].get_position(), AX[0, 1].get_position()
        fig.legend(handles=_hp, loc="center", bbox_to_anchor=(0.5 * (_p0.x1 + _p1.x0) - 0.015, 0.5 * (_p0.y0 + _p0.y1)),
                   frameon=False, fontsize=20, handlelength=1.8, handleheight=1.3, labelspacing=1.1)
        AX[1, 1].legend(handles=_hl, loc="upper left", frameon=True, framealpha=0.95, edgecolor="none", fontsize=15)
        paper_save(fig, PF9_STEM); plt.show()

    P18 = pd.DataFrame(ROW18)
    P18.to_csv(os.path.join(OUT, f"{PF9_STEM}.csv"), index=False)
    _W = Z9[(Z9["trel"] >= 0.5) & (Z9["trel"] <= 2.5)]
    print(f"\nIN +0.5 .. +2.5 Gyr AFTER THE TRUNCATION  (one value per galaxy = its median there):")
    for _col, _lab in (("rel_tot", "the total dust, relative to the truncation"), ("sh_mdust_ism", "share of it in the ISM"),
                       ("sh_mdust_s10", "share in the CGM inside 10 kpc"), ("sh_mdust_s100", "share at 60-100 kpc"),
                       ("f_ret", "returned / ejected since the truncation (all phases)"), ("f_ret_c", "returned COLD / ejected"),
                       ("recyc_share", "cumulative cold return / the ISM dust now"), ("r_ret_grow", "cold return rate / in-situ growth rate"),
                       ("r_ret_grow_all", "ALL return rate / in-situ growth rate"), ("sh_in_cold", "share of the returning dust in cold gas"),
                       ("sh_in_warm", "share of the returning dust in warm gas"), ("sh_in_hot", "share of the returning dust in hot gas"),
                       ("fbnd_d_cgm", "bound share of the CGM dust")):
        if _col not in _W.columns:
            continue
        _g1 = _W.groupby(["gid", "grp"])[_col].median().reset_index()
        _v = {g: _g1.loc[_g1["grp"] == g, _col].to_numpy(float) for g in GK}
        _a, _p, _n1, _n0 = mw_auc(_g1[_col].to_numpy(float), (_g1["grp"] == "burst").to_numpy())
        print(f"    {_lab:52s}" + "".join(f"  {GSH.get(g, g)} {np.nanmedian(_v[g]) if len(_v[g]) else np.nan:7.3g} (N {len(_v[g])})" for g in GK)
              + (f"   AUC {_a:.2f}" if np.isfinite(_a) else ""))
    print("    READ IT AS: the stack says where the dust still within 100 kpc sits; the return fraction says how much of what\n"
          "    crossed 10 kpc outward has crossed back, and the phase shares in what it comes back; the last ratio is the test\n"
          "    of 'dominating': all the dust coming back per Gyr against the dust the ISM grows per Gyr.")
    print("table ->", os.path.join(OUT, f"{PF9_STEM}.csv"))

## Part 19 — paper figure 10: the Kennicutt–Schmidt plane

The paper version of Part 7 (C). It reads the K–S track products of `ks_tracks_quenched_m25` (run tag `pt`):
$\Sigma_{\rm H_2}$ and $\Sigma_{\rm SFR}$ inside the face-on H₂ half-mass radius at every critical point of
every m25 galaxy, and draws the two groups' median tracks $t_{q,\rm start} \to t_{q,\rm end} \to$ end with the
individual tracks faint, over the star-forming galaxies of the box.

**The reference is VERTICO** (Jiménez-Donaire et al. 2023; 37 Virgo spirals at 720 pc): the ensemble
depletion time $t_{\rm dep} = 1.86$ Gyr, slope $N = 0.97 \pm 0.07$, 0.42 dex scatter, written as
$\log \Sigma_{\rm SFR} = N(\log \Sigma_{\rm mol} - \log S_{\rm piv}) + \log(S_{\rm piv}/t_{\rm dep})$ with the
depletion time holding at the pivot `piv` (10 M⊙ pc⁻²; a slope of 1 is a constant depletion time, 0.97 bends
it by 0.03 dex per dex). Bigiel+08 and Kennicutt 98 stay in `KS_RELATIONS` and are one entry of `PF10_RELS`
away.

**The conventions are made the same on both sides.** The simulated H₂ is hydrogen-only and the tracks carry
it × `HE_FACTOR` = 1.36, because VERTICO's $\alpha_{\rm CO} = 4.35$ includes helium; and the reference
$\Sigma_{\rm SFR}$ is moved to SIMBA's Chabrier IMF by `KS_IMF_TO_CHABRIER[KS_REF_IMF]` (Madau & Dickinson 2014:
Kroupa → Chabrier ×0.94, Salpeter → Chabrier ×0.63). Which IMF the z0MGS FUV + WISE4 calibration VERTICO uses
is on decides a 0.03 or 0.20 dex shift, so it is a knob and the shift applied is printed with the figure.

**Knobs.** `PF10_STAGES` (the critical points drawn), `PF10_AP` (the aperture), `PF10_RELS`, `KS_REF_IMF`,
`KS_RELATIONS` (the constants), `PF10_SF`, `PF10_NMIN`, `PF10_XLIM` / `PF10_YLIM`.

In [ ]:
# ── Part 19 — PAPER FIGURE 10: THE KENNICUTT-SCHMIDT PLANE — the two groups walked from the truncation to the
#    end of their tracks, against the VERTICO resolved relation, on one IMF ──────────────────────────────────
# The paper version of Part 7 (C).  It reads the K-S track products of ks_tracks_quenched_m25 (run tag 'pt'):
# Sigma_H2 and Sigma_SFR inside the face-on H2 half-mass radius at every critical point of every m25 galaxy,
# and draws the two groups' median tracks SFT -> QT -> end with the individual tracks faint, over the
# star-forming galaxies of the box.  The reference is the VERTICO resolved molecular K-S relation (Jimenez-
# Donaire et al. 2023, 720 pc, Virgo): the ensemble depletion time t_dep = 1.86 Gyr, slope N = 0.97 +- 0.07,
# 0.42 dex scatter — written here as log Sigma_SFR = N (log Sigma_mol - log S_piv) + log(S_piv / t_dep), the
# depletion time holding at the pivot S_piv (a slope of 1 is a constant depletion time; 0.97 bends it by
# 0.03 dex per dex).  THE CONVENTIONS ARE MADE THE SAME ON BOTH SIDES: the simulated H2 is hydrogen-only and
# is multiplied by HE_FACTOR (VERTICO's alpha_CO = 4.35 includes helium), and the observed Sigma_SFR is moved
# to SIMBA's Chabrier IMF by KS_IMF_TO_CHABRIER[KS_REF_IMF] — the correction the user asked for; which IMF the
# z0MGS FUV + WISE4 calibration VERTICO uses is on is a knob, because the answer decides a 0.03 (Kroupa) or
# 0.20 dex (Salpeter) shift and it is printed with the figure.
PF10_STEM = "ba_paper_ks"
PF10_BOX  = "cis25"
PF10_KS   = os.path.join(os.getcwd(), "output", PF10_BOX, "ks_tracks_pt", "ks_tracks.fits")
PF10_AP   = "R50_H2"                       # the aperture: the face-on H2 half-mass radius (the observed convention)
PF10_STAGES = [("sft", "$t_{q,\\,\\rm start}$", "^"), ("qt", "$t_{q,\\,\\rm end}$", "s"), ("end", "Observed", "o")]
PF10_TERSE = True                          # the legend names things and nothing else (no N, no criteria, no run, no fit numbers)
PF10_NMIN = 3
PF10_RELS = ["VERTICO"]                    # which reference relations to draw (any key of KS_RELATIONS)
PF10_SF   = True                           # the star-forming galaxies of the box: their contours (+ a faint cloud)
PF10_TRACKS = False                        # the individual galaxy tracks, faint (off: the medians alone)
PF10_CONT = True                           # KDE contours of the OBSERVED points — the SF galaxies, and each group at
                                           # PF10_CSTAGE — enclosing PF10_CLEV of the density
PF10_CSTAGE = "end"
PF10_CLEV = (0.5, 0.9)
PF10_CMIN = 5                              # points a contour needs
PF10_ZMAX = 0.5                            # only galaxies anchored below this redshift (None = every anchor) — the
                                           # epoch a z = 0 Virgo reference can be held against; the SF cloud too
PF10_SPLIT = "fdust"                       # what "dusty" / "dust-poor" mean HERE: "fdust" = the observable threshold,
                                           # log M_dust/M* >= DUSTY_CUT at the anchor, applied to EVERY quenched galaxy
                                           # of the box in the tracks (the catalogue's pop label — many more points);
                                           # "peel" = Part 1b's two groups (the sample of the rest of the notebook)
PF10_XLIM, PF10_YLIM = (-0.5, 4.4), (-4.6, 2.2)
HE_FACTOR_KS = 1.36                        # ks_tracks_lib.HE_FACTOR, verbatim (the tracks store the He-corrected column)
KS_REF_IMF = "kroupa"                      # the IMF the reference relation's SFR is on: "kroupa" | "chabrier" | "salpeter"
KS_IMF_TO_CHABRIER = {"chabrier": 1.0, "kroupa": 0.94, "salpeter": 0.63}   # multiply an SFR by this to put it on Chabrier
                                           # (Madau & Dickinson 2014: Salpeter -> Kroupa 0.67, -> Chabrier 0.63)
KS_RELATIONS = {
    # log Sigma_SFR [Msun/yr/kpc^2] = N (log Sigma_mol [Msun/pc^2] - log piv) + A0,  A0 = log(piv * 1e6 / (tdep * 1e9))
    "VERTICO": dict(N=0.97, tdep_gyr=1.86, piv=10.0, sig=0.42, color="#7b3294", ls="-",
                    label="Jiménez-Donaire+23"),
    "B08":     dict(N=0.96, A=-2.06 - 0.96, sig=0.20, color="#1b9e77", ls="--", label=r"Bigiel+08, 750 pc"),
    "K98":     dict(N=1.4, A=float(np.log10(2.5e-4 * 0.63)), sig=0.30, color="#d95f02", ls=":", label="Kennicutt 98 (Chabrier)"),
}
_IMF_SHIFT = float(np.log10(KS_IMF_TO_CHABRIER[KS_REF_IMF]))    # dex, applied to the reference relations' Sigma_SFR


def _ks_rel(key, x):
    r = KS_RELATIONS[key]
    a0 = r["A"] if "A" in r else float(np.log10(r["piv"] * 1e6 / (r["tdep_gyr"] * 1e9)))
    x0 = 0.0 if "A" in r else float(np.log10(r["piv"]))
    return a0 + r["N"] * (np.asarray(x, float) - x0) + (_IMF_SHIFT if key != "K98" else 0.0)   # K98 is already on Chabrier


def _kde_levels(x, y, xlim, ylim, fracs, n=160):
    """A 2-D Gaussian KDE on a grid and the density thresholds enclosing the given mass fractions."""
    k = gaussian_kde(np.vstack([x, y]))
    gx = np.linspace(xlim[0], xlim[1], n); gy = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(gx, gy)
    Z = k(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    f = np.sort(Z.ravel())[::-1]; cs = np.cumsum(f) / f.sum()
    lev = sorted({float(f[min(int(np.searchsorted(cs, q)), len(f) - 1)]) for q in fracs})
    return X, Y, Z, lev


if PF10_BOX not in _BX1:
    print(f"Part 19: {PF10_BOX} is not in this run's boxes ({_BX1}) — skipped.")
elif not os.path.exists(PF10_KS):
    print(f"Part 19: {PF10_KS} is missing — ks_tracks_quenched_m25 Part 4 (run tag 'pt', cluster).  Skipped.")
else:
    _F10 = B1[PF10_BOX]["F"]
    if PF10_SPLIT == "peel":
        KEY10 = {(int(s_), int(i_)): g_ for s_, i_, g_, u_ in zip(_F10["snap"], _F10["gal_id"], _F10["grp"], _F10["uniq"])
                 if u_ and g_ in GK}
        LAB10 = {g: GLAB.get(g, g) for g in GK}
    else:                                   # the observable label, from the catalogue, for every galaxy of the box
        _C10 = CAT[CAT["box"] == PF10_BOX]
        with np.errstate(all="ignore"):
            _ms10 = _C10["mstar"].to_numpy(float)
            _lfd10 = np.log10(np.where(_ms10 > 0, _C10["mdust"].to_numpy(float) / np.where(_ms10 > 0, _ms10, 1.0), np.nan))
        KEY10 = {(int(s_), int(i_)): ("burst" if f_ >= DUSTY_CUT else "aging")
                 for s_, i_, f_ in zip(_C10["snap"], _C10["gal_id"], _lfd10) if np.isfinite(f_)}
        LAB10 = {"burst": rf"dusty QGs  ($\log M_{{\rm dust}}/M_\star \geq {DUSTY_CUT:g}$ at $t_{{\rm obs}}$)",
                 "aging": rf"dust-poor QGs  ($< {DUSTY_CUT:g}$)"}
    T10 = Table.read(PF10_KS)
    _s10 = lambda a: np.array([x.decode() if isinstance(x, bytes) else str(x) for x in np.asarray(a)])
    ap10, pop10, stg10, gk10 = _s10(T10["ap_label"]), _s10(T10["pop"]), _s10(T10["stage"]), _s10(T10["gkey"])
    x10, y10, ul10 = np.asarray(T10["logSigmaH2"], float), np.asarray(T10["logSigmaSFR"], float), np.asarray(T10["is_ul"], bool)
    grp10 = np.array([KEY10.get(tuple(int(v) for v in g_.split("_"))) for g_ in gk10], dtype=object)
    base10 = ap10 == PF10_AP
    if PF10_ZMAX is not None:                # the anchor redshift of every track, from the catalogue
        _CZ = CAT[CAT["box"] == PF10_BOX]
        _zk = {(int(s_), int(i_)): float(z_) for s_, i_, z_ in zip(_CZ["snap"], _CZ["gal_id"], _CZ["z"])}
        _zrow = np.array([_zk.get(tuple(int(v) for v in g_.split("_")), np.nan) for g_ in gk10])
        base10 &= np.isfinite(_zrow) & (_zrow < PF10_ZMAX)
    _STG = [s for s, _, _ in PF10_STAGES]
    ROW19 = []
    with plt.rc_context(PAPER_RC):
        fig, ax = plt.subplots(figsize=(9.0, 8.4))
        if PF10_SF:
            _sf = base10 & (pop10 == "SF") & np.isfinite(x10) & np.isfinite(y10)
            ax.scatter(x10[_sf], y10[_sf], s=7, c="0.75", alpha=0.18, lw=0, zorder=1, rasterized=True)
            if PF10_CONT and _sf.sum() >= PF10_CMIN:
                _X, _Y, _Z, _lev = _kde_levels(x10[_sf], y10[_sf], PF10_XLIM, PF10_YLIM, PF10_CLEV)
                ax.contourf(_X, _Y, _Z, levels=_lev + [_Z.max() * 1.01], colors=["0.85", "0.7"][:len(_lev)], alpha=0.5, zorder=1)
                ax.contour(_X, _Y, _Z, levels=_lev, colors="0.45", linewidths=[1.2, 1.8][:len(_lev)], linestyles=["--", "-"][:len(_lev)], zorder=2)
        _xs = np.linspace(PF10_XLIM[0], PF10_XLIM[1], 80)
        for _rk in PF10_RELS:
            _r = KS_RELATIONS[_rk]
            ax.fill_between(_xs, _ks_rel(_rk, _xs) - _r["sig"], _ks_rel(_rk, _xs) + _r["sig"], color=_r["color"], alpha=0.10, lw=0, zorder=0)
            ax.plot(_xs, _ks_rel(_rk, _xs), color=_r["color"], lw=2.0, ls=_r["ls"], zorder=2)
        for _g in GK:
            _mk = base10 & (pop10 == "Q") & (grp10 == _g) & np.isin(stg10, _STG)
            if PF10_TRACKS:
                for _gk in np.unique(gk10[_mk]):
                    _r = np.flatnonzero(_mk & (gk10 == _gk))
                    _r = _r[np.argsort([_STG.index(s_) for s_ in stg10[_r]])]
                    _ok = np.isfinite(x10[_r]) & np.isfinite(y10[_r])
                    if _ok.sum() >= 2:
                        ax.plot(x10[_r][_ok], y10[_r][_ok], "-", color=PFC[_g], lw=0.9, alpha=0.35, zorder=3)
            if PF10_CONT:                                   # the group's observed points, as contours
                _ob = _mk & (stg10 == PF10_CSTAGE) & ~ul10 & np.isfinite(x10) & np.isfinite(y10)
                if _ob.sum() >= PF10_CMIN:
                    _X, _Y, _Z, _lev = _kde_levels(x10[_ob], y10[_ob], PF10_XLIM, PF10_YLIM, PF10_CLEV)
                    ax.contour(_X, _Y, _Z, levels=_lev, colors=PFC[_g], linewidths=[1.3, 2.2][:len(_lev)],
                               linestyles=["--", "-"][:len(_lev)], alpha=0.85, zorder=3)
                    ax.scatter(x10[_ob], y10[_ob], s=18, c=PFC[_g], alpha=0.5, lw=0, zorder=3)
            _mx, _my = [], []
            for _st, _sl, _mrk in PF10_STAGES:
                _ms = _mk & (stg10 == _st) & ~ul10 & np.isfinite(x10) & np.isfinite(y10)
                if _ms.sum() >= PF10_NMIN:
                    _mx.append(float(np.median(x10[_ms]))); _my.append(float(np.median(y10[_ms])))
                    ax.scatter(_mx[-1], _my[-1], s=150, marker=_mrk, c=PFC[_g], edgecolors="k", linewidths=0.9, zorder=6)
                    _off = _my[-1] - float(_ks_rel(PF10_RELS[0], _mx[-1]))
                    ROW19.append(dict(grp=_g, group=GSH.get(_g, _g), stage=_st, n=int(_ms.sum()), logSigmaH2=_mx[-1],
                                      logSigmaSFR=_my[-1], offset_from_ref=_off, ref=PF10_RELS[0],
                                      n_censored=int((_mk & (stg10 == _st) & ul10).sum())))
            if len(_mx) >= 2:
                ax.plot(_mx, _my, "-", color=PFC[_g], lw=3.2, zorder=5)
        ax.set_xlabel(r"$\log\,\Sigma_{\rm H_2}$  [M$_\odot$ pc$^{-2}$]   inside $R_{50}({\rm H_2})$, He included")
        ax.set_ylabel(r"$\log\,\Sigma_{\rm SFR}$  [M$_\odot$ yr$^{-1}$ kpc$^{-2}$]")
        ax.set_xlim(*PF10_XLIM); ax.set_ylim(*PF10_YLIM)
        ax.set_axisbelow(True)
        for _s in ("top", "right"):
            ax.spines[_s].set_visible(False)
        _ng = {g: len(np.unique(gk10[base10 & (pop10 == "Q") & (grp10 == g) & np.isin(stg10, _STG)])) for g in GK}
        _h19 = [Line2D([], [], color=PFC[g], lw=3.2, label=(GSH.get(g, g) if PF10_TERSE else f"{LAB10[g]}  (N = {_ng[g]})")) for g in GK]
        if PF10_ZMAX is not None:
            ax.text(0.03, 0.97, rf"$z < {PF10_ZMAX:g}$", transform=ax.transAxes, va="top", ha="left", fontsize=20.8,
                    fontweight="bold", color="0.2")
        _h19 += [Line2D([], [], ls="", marker=m, ms=11, mfc="0.6", mec="k", label=l) for _, l, m in PF10_STAGES]
        if PF10_SF:
            _h19 += [Line2D([], [], color="0.45", lw=1.8, label="star-forming galaxies" if PF10_TERSE else f"star-forming galaxies of {BOX_LABEL.get(PF10_BOX, PF10_BOX)}")]
        if PF10_CONT:
            _h19 += [Line2D([], [], color="0.3", lw=2.2, ls="-", label=f"{int(PF10_CLEV[0] * 100)} % contour" if PF10_TERSE else f"{int(PF10_CLEV[0] * 100)} % contour of the observed points ({PF10_CSTAGE})"),
                     Line2D([], [], color="0.3", lw=1.3, ls="--", label=f"{int(PF10_CLEV[1] * 100)} % contour")]
        for _rk in PF10_RELS:
            _r = KS_RELATIONS[_rk]
            _lab = _r["label"].split(",")[0] if PF10_TERSE else (_r["label"] + ("\n" + rf"$t_{{\rm dep}}$ = {_r['tdep_gyr']:g} Gyr, ±{_r['sig']:g} dex" if "tdep_gyr" in _r else rf"  (±{_r['sig']:g} dex)")
                                                                     + (", Chabrier IMF" if _rk != "K98" and _IMF_SHIFT != 0 else ""))
            _h19 += [Line2D([], [], color=_r["color"], lw=2.0, ls=_r["ls"], label=_lab)]
        ax.legend(handles=_h19, loc="lower right", frameon=True, framealpha=0.92, edgecolor="none", fontsize=15.6,
                  handlelength=2.0, labelspacing=0.55)
        fig.tight_layout()
        paper_save(fig, PF10_STEM); plt.show()
    P19 = pd.DataFrame(ROW19)
    P19.to_csv(os.path.join(OUT, f"{PF10_STEM}.csv"), index=False)
    print(f"THE K-S PLANE ({BOX_LABEL.get(PF10_BOX, PF10_BOX)}, inside R50(H2), the He-corrected Sigma_H2"
          + (f", anchors at z < {PF10_ZMAX:g}" if PF10_ZMAX is not None else "") + "; the split is "
          + ("the OBSERVABLE one, f_dust at the anchor against 1e%g, over every quenched galaxy in the tracks" % DUSTY_CUT
             if PF10_SPLIT == "fdust" else "Part 1b's peel")
          + f"): the group medians at each critical point and their offset from {PF10_RELS[0]} (dex above the relation)")
    for _g in GK:
        for _st, _sl, _ in PF10_STAGES:
            _r = P19[(P19["grp"] == _g) & (P19["stage"] == _st)]
            if len(_r):
                _r = _r.iloc[0]
                print(f"    {GSH.get(_g, _g):14s} {_st:4s}  log Sigma_H2 {_r['logSigmaH2']:5.2f}  log Sigma_SFR {_r['logSigmaSFR']:6.2f}  "
                      f"offset {_r['offset_from_ref']:+5.2f} dex  (N {int(_r['n'])}, {int(_r['n_censored'])} censored)")
    print(f"    the reference: VERTICO's ensemble depletion time {KS_RELATIONS['VERTICO']['tdep_gyr']:g} Gyr at Sigma_mol = "
          f"{KS_RELATIONS['VERTICO']['piv']:g} Msun/pc^2 with slope {KS_RELATIONS['VERTICO']['N']:g} and {KS_RELATIONS['VERTICO']['sig']:g} dex scatter; "
          f"its Sigma_SFR moved from {KS_REF_IMF} to Chabrier by {_IMF_SHIFT:+.3f} dex (KS_REF_IMF / KS_IMF_TO_CHABRIER); the simulated H2 x {HE_FACTOR_KS:g} for helium, "
          f"as VERTICO's alpha_CO = 4.35 includes it.")
    print("table ->", os.path.join(OUT, f"{PF10_STEM}.csv"))

## Conventions, what is read, and what each knob does

### What is read (nothing here is rebuilt)

| file | produced by | used by |
|---|---|---|
| `output/box_resolution/ism_prediction/ism_prediction_catalogue.fits` | `paper_ism_prediction_boxes` Part 1 | Part 0 — the sample |
| `…/ism_prediction_agn_classes.csv` | its Part 6a | Part 0 — $t_{\rm SFT}$, $t_{\rm QT}$, $\tau_q$ |
| `…/ism_prediction_histories_<box>.hdf5` | its Part 6a | Part 1 — the main-branch chains, with the ladder's own `z` (Part 10's redshift row) and `zgas` (the dust-to-metal track) |
| `output/box_resolution/agn_states/agn_states_kin_histories_<box>.hdf5` | `dusty_qgs_kappa_selection` Part 3 | Part 1 — $\kappa_{\rm gas}$, $R_{50}$(gas), $R_{50}(\star)$, $N_{\rm gas}$ along the branch |
| `output/box_resolution/kappa_selection/kappa_m25_cgm_nodes.csv` | its Part 13 | Part 6 — the m25 CGM census |
| `…/kappa_cis25_dust_regrowth_nodes.csv` | its Part 17 | Parts 6b, 14, 18 — the grain clocks, the dust by place, the fluxes through 10 kpc |
| `output/cis25/tables/{annulus_ism_truth,aperture_truth,annulus_av_allincl,powderday_quenched_selection}.fits` | the m25 imaging notebook (Parts 8a / 7e / 8b / 3) | Parts 7 (B) and 17 — the projected annuli at the anchor |
| `output/cis25/ks_tracks_pt/ks_tracks.fits` | `ks_tracks_quenched_m25` Part 4a | Parts 7 (C) and 19 — the K–S tracks |

### What is written — all into `output/box_resolution/burst_aging/`

`ba_galaxies.csv` (Part 1, one row per branch) · `ba_peel_<q>_<clock>.{png,pdf,csv}` + `ba_peel_<q>_where.{png,pdf}` +
`ba_peel_members.csv` (Part 1b) · `ba_peel_ltauq.{png,pdf,csv}` (Part 1c) · `ba_split_rule.csv` + `ba_division.csv` +
`ba_split.{png,pdf}` (Part 2) ·
`ba_clocks.csv` + `ba_clocks.{png,pdf}` (Part 3) · `ba_observe.csv` + `ba_observe.{png,pdf}` (Part 4) ·
`ba_lifetime{,_galaxies}.csv` + `ba_lifetime.{png,pdf}` (Part 5) · `ba_cgm{,_anchor}.csv` +
`ba_cgm.{png,pdf}` (Part 6) · `ba_grain_clocks.csv` + `ba_grain_clocks.{png,pdf}` (Part 6b) ·
`ba_plane_<y>_<x>.{png,pdf,csv}`, one per plane of `P3B_PLANES` (Part 3b) ·
**`ba_paper_dust_modes.{png,pdf,csv}` (Part 10)** · **`ba_paper_selection.{png,pdf,csv}` (Part 10b)** · **`ba_paper_peel.{png,pdf,csv}` (Part 10c)** · **`ba_paper_tracks.{png,pdf,csv}` (Part 11)** ·
**`ba_paper_onset.{png,pdf,csv}` + `ba_paper_onset_tracks.csv` (Part 12)** · **`ba_paper_onset_m25{.png,.pdf,.csv}` + `ba_paper_onset_m25_tracks.csv` (Part 12b)** · **`ba_paper_origin{,_galaxies}.csv` + `ba_paper_origin.{png,pdf}`
(Part 13)** · **`ba_paper_timescales.{png,pdf,csv}` (Part 14)** · **`ba_paper_timeline.{png,pdf,csv}` (Part 15)** ·
**`ba_paper_guidance.{png,pdf,csv}` (Part 16)** · **`ba_paper_profiles.{png,pdf,csv}` (Part 17)** ·
**`ba_paper_displacement.{png,pdf,csv}` (Part 18)** · **`ba_paper_ks.{png,pdf,csv}` (Part 19)** — the ten paper figures. **Nothing is written into `kappa_selection/`.**

### The knobs most likely to be wanted

- **Part 0** — `DUSTY_CUT` (−4.0, i.e. $f_{\rm dust}>10^{-4}$; it is the threshold of Parts 4 and 5 too),
  `MASS_MIN`, `AGE_MIN`, `Z_MAX`, `ROT_SIG` (what "rotating" means in Part 4), `BA_NBOOT` (env).
- **Part 1** — **`P1_WIN` / `P1_WOFF`, the window** every `_pk` / `_mn` / `_w0` column, $C$ and $t_{\rm feed}$
  are taken over: two event keys (`jet` `x1` `gate` `big` `ign` `acc` `sft` `qt` `anc` `first`) and an offset
  in Gyr for either end. `P1_TRK` (what is tracked), `P1_PKQ` / `P1_MNQ` (whose peak and minimum times are
  recorded), `P1_TOL` (how near a snapshot must be to an event to be read at it).
- **Part 1b, THE SELECTION** — the map: `P1B_CLOCK` (`sft` / `jet` / `x1` / `qt`), `P1B_UNIQ`,
  `P1B_NMIN` / `P1B_NPER` (galaxies a column needs to be used, and to have its valley trusted), **`P1B_BW`**
  (the per-column KDE bandwidth in units of Scott's rule — the knob that decides how much structure is
  real), `P1B_PROM` / `P1B_DEEP` (how high a peak and how deep a valley must be), `P1B_FLOORS` (the
  censoring floors that are never modes), `P1B_YLIM`.  The peel: **`P1D_CAL`** (the boxes the regions are calibrated on, pooled; `None` = each box
  its own), `P1D_ON`, `P1D_T0` (only look after this epoch), `P1D_BRIGHT` (how bright a cell must be,
  relative to the peak of its own side of the valley), `P1D_NHIT` (epochs a track must spend inside a region
  to join it), `P1D_BRIDGE` (the gap in epochs a region may jump), `P1D_MINCELL` / `P1D_MINGAL` (what is too
  small to be a region), `P1D_REG` (two rectangles by hand instead), `P1D_NAMES`, `P1D_KEEP` (write
  `FR["grp_peel"]`, which Part 2 divides on by default), `P1D_Q` (what the second figure carries the
  groups onto), `P1D_NAMES`.
- **Part 1c, the quenching time** — `P1C_X` (the axis tested), `P1C_FAST`, `P1C_MINN`, `P1C_TEST` (env
  `BA_TEST=1` adds Silverman's p on the axis itself).
- **Part 2, the division** — `P2_ADOPT = "peel"` adopts the selection and nothing else in the cell runs;
  a spec instead puts the browser back:
  - **`P2_SHOW`**, the candidate axes drawn (one row of FIG 1 each), and **`P2_ADOPT`**, the one that divides.
    An axis is `"<q> @ <epoch>"`, `"<q> @ <epochA> > <epochB>"` (their difference), the same with
    `": max|min|mean|rng|slope|d"`, or any column of `FR` (`"ltauq"`, `"comp_c"`, `"comp_dw"`, `"t_feed"`)
    taken as it stands. `<q>` is anything in `P1_TRK`; an epoch is any event of `P1_EV` with an offset in Gyr
    (`sft`, `qt+0.5`, `jet-0.3`). An illegal name raises with the legal ones listed.
  - **`P2_CUT`**, where the cut goes: `"dip"` (the KDE minimum, the median when there is none), `"median"`,
    `"tercile"`, `"q70"`, a number you type, a `(low, high)` **region** that leaves the middle in neither
    group, or a dict of any of those keyed by box. `P2_MINFRAC` is the smallest share a dip may leave on a
    side before the median takes over; `P2_PERBOX = False` pools the boxes into one cut.
  - **`P2_LABELS`** renames the two groups in every figure downstream — do it whenever the division stops
    being about H₂ density, since `burst` / `aging` are only names for the HIGH and LOW side of the axis.
  - `P2_TOL` (how near a snapshot must be to an epoch to be read at it), **`P2_CLAMP`** (read an epoch that
    falls off the end of a branch at the end instead of returning NaN), **`P2_ZCTRL`** (cut the axis *ranked
    inside epoch × mass cells*, which removes the confound from the division rather than only from the
    scores), `P2_UNIQ`, `P2_BW`, `P2_MINN`.
- **Part 3** — `P3_CLOCK` (`"sft"` / `"jet"` / `"x1"`), `P3_TRK`.
- **Part 3b** — **`P3B_PLANES`**, the (abscissa, ordinate) pairs drawn one figure each (the age plane and the
  $\kappa_{\rm gas}$ plane by default), `P3B_BOXES` (the X-ray-on resolutions; the control is not a panel),
  `P3B_CLOCK`, `P3B_TLIM`, `P3B_EV`. A band is drawn only where the track is single-valued in the abscissa;
  where it doubles back the 16–84 are two dotted lines.
- **Part 4** — `P4_FDUST`, `P4_KZ`, `P4_LADDER`, `P4_POST`, `BA_NDRAW` (env).
- **Part 5** — `P5_THR`, `P5_SLOPE` (the window the initial rate is fitted over).
- **Part 6** — `P6_POST` (the scoring window), `P6_COLD` (the cold-rich cut; `None` = the m25 median),
  `P6_NMIN` (galaxies a clock bin needs before a point is drawn — with ~30 galaxies in the m25 census this is
  what puts the gaps in the curves).
- **Part 10, paper figure 1** — `PF1_BOXES` (rows 1-3, the X-ray-on resolutions), **`PF1_NOX`** (row 1 only:
  the X-ray-off box as a further map panel, the same cells laid over it — the count it prints for the
  dust-poor region is the control), **`PF1_ZSPLIT`** (row 3: the already-quenched subset of the reference box as
  a thin second curve in the same panel — the control on the whole row), **`PF1_KDE`** / **`PF1_ANNOT`** (the
  per-box KDE curves and the in-panel statistics: diagnostics, off in the paper version, which is the histogram
  of the reference box and the division line alone), `PF1_COUNTS`, `PF1_ROWLAB` (the boxed label at the left
  of each row),
  `PF1_EPOCHS` / `PF1_ZS` (the five events and the five redshifts, and their labels), **`PF1_ZQ`** (row 3
  over the branches already quenched at that redshift instead of every branch alive there), `PF1_Q`,
  `PF1_BW`, `PF1_FTOL` (how close to `FLOOR` still counts as the floor), `PF1_NMIN`, `PF1_MED`, `PF1_CUT`,
  `PF1_BOXCOL` / `PF1_BOXLS`, `BA_PSILV=1` (env — Silverman's p per panel, off by default). The paper
  colours `PFC` are defined here, the peel's own red and blue, and Part 11 reuses them.
- **Part 10b, paper figure 1b** — the selection alone (Part 10's first row, standalone): `PFS_BOXES`, `PFS_NOX`
  (the X-ray-off box as a further panel), `PFS_MED` (the median tracks back on the map), `PFS_COUNTS`, `PFS_CBAR`,
  `PFS_ALPHA` (the fill of the regions; 0 = outline only), `PFS_TLIM` / `PFS_YLIM`, `PFS_CUT` (the observable
  threshold as a line; it is not the selection). The CSV carries the counts and the extent of each region.
- **Part 10c, paper figure 1c** — the peel stage by stage (Part 1b's FIG 1 row, paper version, one box): `PFP_BOX`
  (m100n1024), `PFP_STAGES`, `PFP_DIM` (columns dimmed by the share still in play, Part 1b's convention), `PFP_MED`
  (the median of what is still in play), `PFP_TEXT` (the two-line note per panel), `PFP_CBAR`, `PFP_ALPHA`,
  `PFP_TLIM` / `PFP_YLIM`. The CSV carries the galaxies in play and the region taken at each stage.
- **Part 11, paper figure 2** — `PF2_TRK` (the six quantities), `PF2_CLOCK`, **`PF2_BOX`** (the reference
  box: bands, counts and event marks) and `PF2_THIN` (the other resolutions, thin medians), `PF2_HIST` /
  `PF2_HFRAC` (the galaxy counts in the back and how much of the panel the largest takes), `PF2_MARKS`,
  `PF2_HLINE`, `PF2_NCOL`, `PF2_NMIN`, `PF2_XLIM`, `PF2_TSHOW`.
- **Part 12, paper figure 3** — **`PF3_STYLE`** (`"tracks"`: the median track + bootstrap band of each group
  on `PF3_CLOCK` with the critical points on it, each at the group's median offset and the median value read AT
  the event with its 16-84 bar; `"violins"`: the earlier version, the distributions at the events alone),
  `PF3_Q` (the quantities), `PF3_CLOCK`, `PF3_EV` (the events and their labels), `PF3_MARK` (one marker per
  event), `PF3_TLIM` (the stretch of the clock drawn — a marker past it is skipped and printed), `PF3_XERR` (a
  horizontal bar = the 16-84 spread of the event's own offset), `PF3_NBOOT`, `PF3_LAB` (the plain labels),
  `PF3_AUC` (the AUCs written in the panels; off — they are always printed), `PF3_LEAD` (the lead-time panel;
  `None`, it is a rung of Part 14), `PF3_LEADLOG`, `PF3_BOXES`, `PF3_YLIM` (violins), `PF3_HLINE`, `PF3_NMIN`.
  The tracks go to `ba_paper_onset_tracks.csv`, the event reads and the AUCs to `ba_paper_onset.csv` (with the
  events' clock offsets, `dt_med` / `dt_p16` / `dt_p84`). The track panel is a helper, `_pf3_track_panel`, and
  **Part 12b follows the same style** (`PF3B_STYLE = None` = `PF3_STYLE`; `"tracks"` / `"violins"` on its own)
  but has ITS OWN clock, events and range: `PF3B_CLOCK = "sft"` (the tracks aligned at the truncation, the dashed
  line), `PF3B_EV = ["jet", "qt", "anc"]` (the jet and the end of the transition as markers, the observation in
  the strip — `PF3B_STRIP`), `PF3B_TLIM = (-6, 4)` (the tracks are re-gridded from the raw series, so the range is
  not bound to `TG`: the dusty group's jet, 2.9 Gyr before the truncation, is on the figure), `PF3B_TSTEP`,
  `PF3B_HLINE = {}` (no reference lines); six panels of m25 with the sSFR floor censored (`PF3B_FLOORS`),
  `PF3B_NBOOT`; its tracks go to `ba_paper_onset_m25_tracks.csv`. Part 12's `PF3_EVLAB` carries the label of
  every event either cell may draw, `PF3_TSTEP` the grid step. Part 12 itself now defaults to the truncation clock
  (`PF3_CLOCK = "sft"`), the jet / quenching-end / observation events, `PF3_TLIM = (-4, 4)` and no run names
  (`PF3_NAMES`); `PF3_CLOCK = "jet"` with `("sft", ...)` in `PF3_EV` gives the jet-aligned version back.
- **Part 13, paper figure 4** — **`PF4_FROM`** (`"qt"` / `"sft"`, where the ex-situ leg starts), **`PF4_JUMP`**
  (dex of $\log M_\star$ gained that counts as a merger; 0.04 = 1:10), `PF4_REB` (dex of rebound that counts as
  regrowth), `PF4_BOXES`, `PF4_NMIN`, `PF4_CLS`.
- **Part 14, paper figure 5** — `PF5_BOX` (m25, the one population of the ladder), **`PF5_WHEN`** (`"all"` =
  the grain clocks and the turnover over every node of a galaxy's history; `"post"` = inside `PF5_POST` after
  the truncation — the printout gives both), `PF5_SEC`, `PF5_NMIN`, `PF5_HUB`.
- **Part 16, paper figure 7** — `PF7_BOX`, `PF7_ZS` (`None` = every anchor with `PF7_NMIN` quenched galaxies),
  **`PF7_BT`** (the degrees of disky, as B/T under `PF7_KDISC`; labelled by the $\kappa_\star$ they correspond
  to, because the relabelling saturates at B/T = 1 for the median quenched galaxy), `PF7_XG` (the dust
  thresholds), **`PF7_MODE`** (`"conditional"` P(dusty | disky QG) or `"joint"` P(disky and dusty | QG)),
  `PF7_RULE`, `PF7_ALL`, `PF7_SHARE`, `PF7_XMARK`, `PF7_NCOL`.
- **Part 17, paper figure 8** — `PF8_ANN` (the annuli, out to 32–100 kpc), `PF8_Q` (the panels), `PF8_SF` (the
  star-forming reference), `PF8_NSTAR`, `PF8_NMIN`, `PF8_NCOL`; `P7_M25_OVERRIDE` for another `output/<box>` tree.
- **Part 18, paper figure 9** — `PF9_TB` (the clock bins), `PF9_REFW` (how close to the truncation the reference
  node must be), `PF9_PLACES` (the ISM and the four shells, with their colours), **`PF9_GROWZ`** (the in-situ
  growth rates the return is measured against), **`PF9_LS`** (the line style says the group in the two line
  panels: dusty solid, dust-poor dashed), `PF9_CRET` / `PF9_CCOLD` (the two quantities of panel c and its
  coloured right axis), `PF9_NMIN`, `PF9_NBOOT`. The fluxes, the bound share and "how much is left" are computed
  into the CSV and not drawn.
- **Part 19, paper figure 10** — `PF10_RELS` (the reference relations drawn; VERTICO alone by default),
  **`KS_REF_IMF`** (the IMF the reference SFR is on, converted to Chabrier by `KS_IMF_TO_CHABRIER`),
  `KS_RELATIONS` (the constants), `PF10_STAGES`, `PF10_AP`, `PF10_SF`, **`PF10_CONT`** / `PF10_CSTAGE` / `PF10_CLEV`
  (KDE contours of the observed points — the star-forming cloud and each group at `PF10_CSTAGE` — enclosing
  50 / 90 % of the density), `PF10_TRACKS` (the individual tracks, off), `PF10_NMIN`.
- **Part 16** prints how long it took to build and to save the figure; the save at `PAPER_DPI` = 300 is the
  slow half — lower `PAPER_DPI` in Part 0 if it matters. `PF7_YMAX` (80 %) is where its y axis stops.
- **No grids on the paper figures** (`PAPER_RC["axes.grid"] = False`, and no `grid()` call in Parts 3b, 10–19).
  Part 18's legends sit inside the figure: the places between the two stacks, the lines in panel (1, 1).
- **Part 16** — the degrees of "disky" are drawn as B/T only (`PF7_BT`); the $\kappa_\star$ each corresponds to is
  in the printout. `PF7_ZS` picks the anchors nearest 0.3, 0.7, 1.0, 1.3, 1.8, 2.0.
- **Part 17** — `PF8_TOT` is the aperture the "total" points are read in (r < 32 kpc), drawn as diamonds at the
  right of every panel.
- **Part 15, paper figure 6** — `PF6_EV` (the events, labels and markers), `PF6_BOXES`, `PF6_UNIQ`, `PF6_NMIN`,
  `PF6_XLIM`.
- **Part 6b, the grain clocks** — `P6B_ZONES` (which zones are drawn), `P6B_POST`, `P6B_NMIN`, `P6B_SKIP`
  (the two panels that are physically empty: no $\tau_{\rm sp}$ in the star-forming ISM, no
  $\tau_{\rm grow}$ above $10^5$ K). The timescales themselves are Part 17's and are not recomputed.

### Caveats, in the order they bite

1. **A valley is not two populations, and neither is a floor.** Part 1b grades a column only when it has
   enough galaxies in it, refuses a shoulder, refuses a pile-up on a known censoring floor, and follows the
   galaxies across the valley to see whether they stay on their side of it *above the chance rate*. Quote the
   run length and the persistence, never the mode count alone.
2. **$\Sigma_{\rm H_2}$ is a density, not a resolved size.** caesar carries no H₂ radius, so the H₂ mass is
   spread over the *cold-gas* half-mass radius. It is the same quantity in every box, but it confounds the H₂
   mass with the gas size, and it is not the resolved $R_{50,\rm H_2}$ the m25 particle products can measure.
   **And a maximum is a biased estimator**: a branch sampled at more snapshots inside the window has more
   chances to reach a high value. The window width and its snapshot count are printed, and they differ between
   boxes (m25 ~5 snapshots, `cis50nox` ~18) — so compare *within* a box before comparing across them.
3. **The division is a cut, not a boundary,** unless Part 2c says otherwise — and with `P2_TEST` off it has
   not even been asked. Read the Silverman p before writing "bimodal" anywhere; a KDE mode count is not a
   test and a dip is not a boundary.
4. **A duration is not scale-invariant, and below the snapshot gap it is not even a duration.** Part 2b tests
   $t_{\rm feed}$ in Gyr *and* in $\log_{10}$ and prints both, because the two disagree wherever the short tail
   is dense. Below one snapshot gap (~0.09 Gyr here) $t_{\rm feed}$ is a rounded count of snapshots, so a
   spike there is discreteness; the fraction that sits there is printed and the region is shaded.
5. **$C$ has a hard floor and a length-dependent bias, and neither is a physical result.** $C \ge 0$ by
   construction, so a pile-up sits at exactly 0 (the branches whose gas only ever grows after the jet) and a
   spike beside a continuum will make any multimodality test reject — which is why the test is repeated on
   $C > 0$ alone. And the minimum of $n$ points falls further as $n$ grows, so $C$ means nothing except
   against its own pre-jet null $C_0$, measured over an equally long stretch of the same branch. Quote
   $C - C_0$, or the fraction of branches with $C > C_0$, never the raw $C$ across boxes.
6. **A change measured over a window that varies in length is partly that length.** `"comp @ sft > qt+0.5"`
   is measured over $\tau_q + 0.5$ Gyr, and $\tau_q$ is what the quenching-mode axis is made of: Part 2c
   prints the rank correlation of the axis with its own window width and the width of the two groups. If that
   is what is happening, `": slope"` asks the question per Gyr, and two ends at fixed offsets from one event
   (`"comp @ sft > sft+1.0"`) make every window equal.
7. **The groups differ in cosmic time and in stellar mass.** Part 2c prints every score twice, raw and at fixed
   epoch + mass; the second column is the one to quote. `P2_ZCTRL = True` moves that control into the division.
8. **The de-duplication is a fingerprint**, not the exact progenitor index — $\log M_\star$ at three fixed
   cosmic times. A branch too short to fingerprint keeps its own identity rather than being merged.
9. **Part 4's two probabilities answer different questions.** The one-draw number is the question as posed
   (one galaxy, one observation); the survey number is what a snapshot survey returns, where a long-lived
   object is genuinely caught more often. Quote the one you mean.
10. **Part 5 censors, it does not drop.** A branch still above the threshold at its anchor is a lower limit.
   In `cis50nox` essentially *every* branch is censored — the dust never dies there, which is the whole point
   of the control column, and the medians are correctly `NaN` rather than a number.
11. **Part 10's redshift row is not a survey.** A chain ends at its own anchor, so the sample behind the
   panels *changes with redshift*: the high-$z$ ones hold the progenitors of the whole sample, almost none of
   them quenched yet, and the low-$z$ ones only the galaxies still in the quenched sample there. The panel
   prints both — the count and the quenched share — and `PF1_ZQ = True` restricts it to the quenched ones,
   at the cost of emptying the high-redshift end. What the row can say is *when a second mode exists at all*,
   not what a flux-limited survey would count.
12. **Part 13's merger axis is a mass difference, not a tree.** Stellar mass gained after $t_{q,\rm end}$ is
   ex-situ because a passive galaxy forms almost nothing in situ — but a branch whose $\log M_\star$ *falls*
   by a large amount over that leg is a progenitor swap, not physics, and the panels stop at the 2nd percentile
   (never below −0.15 dex) so one such branch cannot set the axis. The class is measured against the
   *observable* threshold at the anchor, so a "dusty" group that is mostly *faded* by its anchor is Part 5's
   lifetime read from the other end, not a contradiction of the peel.
13. **Part 14's rungs are contrasts, not values.** Both grain laws are linear in the assumed grain size and
   $\tau_{\rm sp}$ is drawn as the mass clock ($\tau_{\rm sp}/3$); the CGM turnover divides a reservoir
   by an instantaneous inflow. Read the order of the rungs and the gap between the groups; the third digit
   means nothing.
14. **Part 16's "B/T" is $\kappa_\star$ in other units.** Under $\kappa_{\rm disc} = 0.85$ the median quenched
   galaxy of m100n1024 ($\kappa_\star = 0.30$) has $B/T = 1$: the relabelling saturates, so the degrees of
   "disky" are the 5 / 13 / 20 % most rotation-supported quenched galaxies ($\kappa_\star \geq 0.44 / 0.39 /
   0.36$) and are labelled by $\kappa_\star$ first. A different $\kappa_{\rm disc}$ moves the B/T numbers and
   not one curve; a measured B/T needs the mock images.
15. **Part 18's budget is a budget within 100 kpc.** The shares are of the dust in the ISM plus the CGM
   inside Part 13's aperture; what has left that volume or been destroyed is not in the stack (the total
   relative to the truncation is in the CSV). The return fraction counts every crossing of the 10 kpc shell,
   so dust that cycles through it several times counts several times — it is a flux ratio, not a tracer of
   individual grains. The census splits the fluxes at $10^5$ K only, cold / hot: "in cold gas" is the share
   below that, the rest is hot, and there is no warm flux to draw.
16. **Part 19's reference is a depletion time with a slope.** VERTICO's ensemble numbers ($t_{\rm dep}$,
   $N$, the scatter) come from its abstract; the line is anchored so that $t_{\rm dep}$ holds at the pivot
   surface density, and the IMF the z0MGS SFR calibration assumes is a knob because it is not stated on the
   pages reachable from here — the shift applied (−0.03 dex for Kroupa, −0.20 for Salpeter) is printed. The
   simulated $\Sigma_{\rm H_2}$ carries helium so that it matches an $\alpha_{\rm CO}$ that includes it.
17. **Part 6's census was measured under Part 13's knobs** (phase edges, shell radius, the 100 kpc aperture).
   It is re-clocked and re-split here, but not re-measured. The ISM/CGM boundary is caesar's member list, not
   a radius.